# noisy_cost_comparison — 物理誤り率を与えたときの、Kobayashi 方式（D4 code switching）と Litinski 方式（15-to-1 蒸留）の資源比較

**問い**：物理誤り率 $p$ と要求される出力誤り $\varepsilon_{\rm target}$ が与えられたとき、論理 $T$ 魔法状態 1 個を作るのに、どちらの方式が少ない時空間体積（物理量子ビット × 時間ステップ）で済むか。

**このノートがすること**（重い計算はしない。数分で終わる）
1. **Kobayashi 方式**：`D4_noisy_simulation.ipynb` の回路レベル・モンテカルロの結果 $\varepsilon_K(d,p)$（非 Clifford 回路を Takada らの方法で厳密にシミュレーション）を読み込み、$d$ と $p$ への依存を当てはめる。資源は同じ回路から数える。
2. **Litinski 方式**：Litinski（2019）の公開コードの誤り予算（本研究で再実装し、公開値 7 件を再現：`validation.ipynb` N6）を、*A Game of Surface Codes*（GoSC）Sec. 3 の 11-tile 15-to-1 と 176-tile 225-to-1、および Litinski（2019）の最適化された蒸留器に適用する。パッチの論理誤り率 $p_L(d,p)$ は**同じ雑音モデル**の表面符号シミュレーションで較正する。
3. 同じ単位（物理量子ビット × 時間ステップ、全ゲート 1 時間ステップ）で比較する。

**雑音のシナリオ**：A＝アイドル雑音あり（$p_{\rm idle}=p$）、B＝アイドル雑音なし（Takada らと同じ）。ゲート・初期化・測定の誤りはどちらも $p$。

**結果の位置づけ**：Kobayashi 方式の数値は**本研究で構成した復号器**（[KZH] が引用する JIT decoder [Dav] は参照できなかった）と**最適化していない測定スケジュール**に依存し、最適な復号器・スケジュールで達成できる値の上界（保守的）である。Litinski 方式は解析的な誤り予算モデル（文献の手法）である。手法・仮定の詳細は `docs/methods.md`。

In [ ]:
# ==== 環境設定（Google Colab / ローカル共通）====
# このノートブックは単独で実行できる（共通コードは下に埋め込まれており、他のファイルを読み込まない）。
# 出力（results/, figures/）はこのランタイムの作業ディレクトリに保存される。Colab ではランタイム終了で
# 消えるので、残したい場合は Google Drive を使う：
#   from google.colab import drive; drive.mount('/content/drive')
#   os.makedirs('/content/drive/MyDrive/D4_vs_MSD', exist_ok=True); os.chdir('/content/drive/MyDrive/D4_vs_MSD')
import os, sys, subprocess, json
IN_COLAB = 'google.colab' in sys.modules
RESULTS, FIGURES = 'results', 'figures'
os.makedirs(RESULTS, exist_ok=True); os.makedirs(FIGURES, exist_ok=True)
SEED = 20261024
print('cwd =', os.getcwd(), '| Colab =', IN_COLAB)

In [ ]:
# ==== 外部ライブラリ（Colab に無いものだけをインストール）====
import importlib.util
for _mod, _pkg in [('stim', 'stim'), ('pymatching', 'pymatching'), ('mpmath', 'mpmath'), ('scipy', 'scipy'),
                   ('sympy', 'sympy'), ('pandas', 'pandas'), ('matplotlib', 'matplotlib')]:
    if importlib.util.find_spec(_mod) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg], check=True)
import numpy as np, pandas as pd, scipy, stim, pymatching, mpmath
print('numpy', np.__version__, '| scipy', scipy.__version__, '| stim', stim.__version__,
      '| pymatching', pymatching.__version__, '| mpmath', mpmath.__version__)

In [ ]:
# ==== 図のスタイル（明るい背景・細い線・識別は色＋線種＋マーカー）====
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb', 'savefig.facecolor': '#fcfcfb',
                     'axes.edgecolor': '#52514e', 'axes.labelcolor': '#0b0b0b', 'text.color': '#0b0b0b',
                     'xtick.color': '#52514e', 'ytick.color': '#52514e', 'axes.grid': True,
                     'grid.color': '#e4e3df', 'grid.linewidth': 0.8, 'lines.linewidth': 2,
                     'lines.markersize': 7, 'axes.spines.top': False, 'axes.spines.right': False,
                     'font.size': 10, 'legend.frameon': False})
C_KOB, C_LIT, C_AUX = '#2a78d6', '#eb6834', '#1baf7a'     # categorical slots 1,2,3 (validated palette)
C_R, C_G, C_B = '#e34948', '#008300', '#2a78d6'            # colours of the r, g, b qubit species

## 共通ライブラリ（埋め込み・関数定義のみ）
Google Colab ではノートブックごとに別のランタイムで実行されるため、他のファイルを読み込まずに済むよう、共通コードをこのノートブックに埋め込んでいる（各ノートの共通コードは同一のソースから自動生成されており、内容は一致する）。
**このセクションは上から順に実行するだけでよく、編集する必要はない**（Colab では見出しの左の三角で折りたためる）。

### 共通ライブラリ（F2 線形代数・ゲート／ガジェット・スケジューラ・Clifford+T 分解・計数ルール R1–R3）

In [ ]:
"""common_lib: F2 linear algebra, gate/gadget model, scheduler, decompositions, cost model rules."""
import itertools, math
import numpy as np
from collections import Counter, defaultdict

In [ ]:
# ---------------------------------------------------------------- F2 algebra
def f2_rref(M):
    """Row-reduce a uint8 matrix over F2. Returns (R, pivot_cols)."""
    M = (np.array(M, dtype=np.uint8) & 1).copy()
    rows, cols = M.shape
    piv = []
    r = 0
    for c in range(cols):
        if r >= rows:
            break
        nz = np.nonzero(M[r:, c])[0]
        if len(nz) == 0:
            continue
        p = r + nz[0]
        if p != r:
            M[[r, p]] = M[[p, r]]
        others = np.nonzero(M[:, c])[0]
        others = others[others != r]
        if len(others):
            M[others] ^= M[r]
        piv.append(c)
        r += 1
    return M[:r], piv


def f2_rank(M):
    if len(M) == 0:
        return 0
    return len(f2_rref(M)[1])


def f2_nullspace(M, n=None):
    """Basis (rows) of {x : M x = 0} over F2."""
    M = np.array(M, dtype=np.uint8)
    if M.size == 0:
        return np.eye(n, dtype=np.uint8)
    n = M.shape[1]
    Rm, piv = f2_rref(M)
    free = [c for c in range(n) if c not in set(piv)]
    basis = []
    for f in free:
        v = np.zeros(n, dtype=np.uint8)
        v[f] = 1
        for i, p in enumerate(piv):
            if Rm[i, f]:
                v[p] = 1
        basis.append(v)
    return np.array(basis, dtype=np.uint8).reshape(len(basis), n)


def f2_in_rowspace(Rref, piv, v):
    """Is v in the row space of the RREF matrix Rref (pivots piv)?"""
    v = np.array(v, dtype=np.uint8) & 1
    w = v.copy()
    for i, p in enumerate(piv):
        if w[p]:
            w ^= Rref[i]
    return not w.any()


def f2_solve(A, b):
    """Solve x A = b (x row vector, A rows = generators). Returns x or None."""
    A = np.array(A, dtype=np.uint8)
    m, n = A.shape
    aug = np.concatenate([A, np.eye(m, dtype=np.uint8)], axis=1)
    Rm, piv = f2_rref(aug)
    w = np.concatenate([np.array(b, dtype=np.uint8) & 1, np.zeros(m, dtype=np.uint8)])
    for i, p in enumerate(piv):
        if p >= n:
            break
        if w[p]:
            w ^= Rm[i]
    if w[:n].any():
        return None
    return w[n:]




import itertools
from collections import Counter, defaultdict

ARITY = {'INIT_X': 1, 'INIT_Z': 1, 'MEAS_X': 1, 'MEAS_Z': 1, 'H': 1, 'X': 1, 'Z': 1, 'S': 1, 'SDG': 1,
         'T': 1, 'TDG': 1, 'CNOT': 2, 'CZ': 2, 'CS': 2, 'CSDG': 2, 'CCZ': 3}
NONCLIFFORD = {'T', 'TDG', 'CS', 'CSDG', 'CCZ'}

In [ ]:
# ------------------------------------------------------------------ minimum-weight logical (MILP)
def min_weight_odd(Hc, lvec, cols):
    """min |x| over x in F2^cols with Hc x = 0 and lvec . x = 1 (exact MILP, HiGHS)."""
    from scipy.optimize import milp, LinearConstraint, Bounds
    H = np.array(Hc, dtype=float)[:, cols]
    H = H[np.any(H, axis=1)]
    l = np.array(lvec, dtype=float)[cols]
    m, n = H.shape
    # variables: x (n, binary), y (m, integer), z (1, integer)
    nv = n + m + 1
    c = np.concatenate([np.ones(n), np.zeros(m + 1)])
    A1 = np.concatenate([H, -2 * np.eye(m), np.zeros((m, 1))], axis=1)
    A2 = np.concatenate([l[None, :], np.zeros((1, m)), -2 * np.ones((1, 1))], axis=1)
    cons = [LinearConstraint(A1, 0, 0), LinearConstraint(A2, 1, 1)]
    integrality = np.ones(nv)
    ub = np.concatenate([np.ones(n), np.full(m + 1, np.inf)])
    res = milp(c, constraints=cons, integrality=integrality, bounds=Bounds(np.zeros(nv), ub))
    assert res.status == 0, res.message
    x = np.round(res.x[:n]).astype(np.uint8)
    out = np.zeros(len(lvec), dtype=np.uint8)
    out[cols] = x
    return int(x.sum()), out

In [ ]:
# ------------------------------------------------------------------ gadgets
def pauli_check_gadget(anc, xq=(), zq=()):
    """Measure X(xq) Z(zq) with ancilla anc (standard CNOT circuit)."""
    g = [('INIT_X', anc)]
    g += [('CNOT', anc, q) for q in sorted(xq)]           # ancilla controls -> X on data
    g += [('CZ', anc, q) for q in sorted(zq)]             # controlled Z on data
    g += [('MEAS_X', anc)]
    return g


def z_check_gadget(anc, zq):
    g = [('INIT_Z', anc)] + [('CNOT', q, anc) for q in sorted(zq)] + [('MEAS_Z', anc)]
    return g


def clifford_x_gadget(anc, parts):
    """Measure a product of X-type Clifford operators S = S_1 S_2 ... with
    S_k = X(s_k) prod CZ(pairs_k): ancilla |+>, controlled-S (= CNOTs from anc and
    CCZ(anc,i,j) for each CZ pair; rightmost factor first), measure X.
    parts: list of (xqubits, czpairs) in operator order S_1 S_2 ..."""
    g = [('INIT_X', anc)]
    for xq, pairs in reversed(parts):
        g += [('CCZ', anc, i, j) for (i, j) in pairs]      # CZ part acts first within S_k
        g += [('CNOT', anc, q) for q in sorted(xq)]
    g += [('MEAS_X', anc)]
    return g

In [ ]:
# ------------------------------------------------------------------ scheduling
def asap_layers(gates):
    """Greedy ASAP layering of a gate list respecting qubit order (no reordering of gates that
    share a qubit).  Returns list of layers (lists of gates)."""
    ready = defaultdict(int)
    layers = []
    for gt in gates:
        qs = gt[1:]
        t = max(ready[q] for q in qs)
        while len(layers) <= t:
            layers.append([])
        layers[t].append(gt)
        for q in qs:
            ready[q] = t + 1
    return layers


def gadget_support(gad):
    s = set()
    for gt in gad:
        s |= set(gt[1:])
    return s


def disjoint_groups(gadgets):
    """Greedy colouring: partition gadgets so that gadgets in a group touch disjoint qubits.
    Measurements within a group are then exactly simultaneous and the sequential-group
    schedule is valid for commuting (on the relevant subspace) generators."""
    groups = []          # list of (used_qubits, [gadgets])
    order = sorted(range(len(gadgets)), key=lambda i: -len(gadget_support(gadgets[i])))
    for i in order:
        sup = gadget_support(gadgets[i])
        for used, members in groups:
            if not (used & sup):
                used |= sup
                members.append(gadgets[i])
                break
        else:
            groups.append((set(sup), [gadgets[i]]))
    return [m for _, m in groups]


def layer_duration(layer, dur):
    return max(dur[g[0]] for g in layer)


def circuit_time(layers, dur):
    return sum(layer_duration(l, dur) for l in layers)


def round_schedule(gadgets, dur):
    """Upper-bound schedule of one round: disjoint groups executed sequentially, each group
    ASAP-layered.  Returns dict with depth (layers), time (weighted by dur), groups."""
    groups = disjoint_groups(gadgets)
    depth = 0
    time = 0
    for grp in groups:
        gl = asap_layers([gt for gad in grp for gt in gad])
        depth += len(gl)
        time += circuit_time(gl, dur)
    return {'groups': len(groups), 'depth_ub': depth, 'time_ub': time}


def round_lower_bound(gadgets, dur):
    """Lower bound on the duration of one round: every qubit executes its gates sequentially,
    so time >= max_q sum of durations of gates acting on q."""
    busy = defaultdict(float)
    cnt = defaultdict(int)
    for gad in gadgets:
        for gt in gad:
            for q in gt[1:]:
                busy[q] += dur[gt[0]]
                cnt[q] += 1
    return {'depth_lb': max(cnt.values()), 'time_lb': max(busy.values())}


def gate_counts(gates):
    c = Counter(g[0] for g in gates)
    return dict(c)

In [ ]:
# ------------------------------------------------------------------ decompositions
# Explicit Clifford+T decompositions (verified numerically in the validation notebook).
def ccz_to_clifford_t(a, b, c):
    """CCZ with 7 T/T^dag and 6 CNOT (standard Toffoli decomposition, Nielsen & Chuang Fig. 4.9,
    with the target Hadamards removed)."""
    return [('CNOT', b, c), ('TDG', c), ('CNOT', a, c), ('T', c), ('CNOT', b, c), ('TDG', c),
            ('CNOT', a, c), ('T', b), ('T', c), ('CNOT', a, b), ('T', a), ('TDG', b), ('CNOT', a, b)]


def cs_to_clifford_t(a, b):
    """CS = diag(1,1,1,i) = T_a T_b CNOT(a,b) T^dag_b CNOT(a,b)."""
    return [('T', a), ('T', b), ('CNOT', a, b), ('TDG', b), ('CNOT', a, b)]


def csdg_to_clifford_t(a, b):
    return [('TDG', a), ('TDG', b), ('CNOT', a, b), ('T', b), ('CNOT', a, b)]


def decompose(gates):
    out = []
    for g in gates:
        if g[0] == 'CCZ':
            out += ccz_to_clifford_t(*g[1:])
        elif g[0] == 'CS':
            out += cs_to_clifford_t(*g[1:])
        elif g[0] == 'CSDG':
            out += csdg_to_clifford_t(*g[1:])
        else:
            out.append(g)
    return out


DUR_UNIT = {k: 1 for k in ARITY}            # every primitive = 1 time unit (ASSUMPTION, parametrised)

In [ ]:
# ------------------------------------------------- commuting-class scheduler
DIAG_GATES = {'CZ', 'CCZ', 'CS', 'CSDG', 'T', 'TDG', 'S', 'SDG', 'Z'}


def gate_roles(gt):
    """Per-qubit role of an entangling/1q unitary: 'Z' (diagonal in Z basis), 'X' (X-type)."""
    name = gt[0]
    if name in DIAG_GATES:
        return {q: 'Z' for q in gt[1:]}
    if name == 'CNOT':
        return {gt[1]: 'Z', gt[2]: 'X'}
    if name == 'X':
        return {gt[1]: 'X'}
    return {q: '?' for q in gt[1:]}      # H, INIT, MEAS: never shared between gadgets


def gadget_roles(gad):
    roles = {}
    for gt in gad:
        for q, r in gate_roles(gt).items():
            if roles.get(q, r) != r:
                roles[q] = '?'
            else:
                roles[q] = r
    return roles


def commuting_classes(gadgets, n_orders=6, seed=0):
    """Best (fewest classes) of several greedy colourings (different gadget orders)."""
    import random
    best = None
    orders = [list(range(len(gadgets)))]
    orders.append(sorted(orders[0], key=lambda i: tuple(sorted(set(gadget_roles(gadgets[i]).values())))))
    rnd = random.Random(seed)
    for _ in range(n_orders):
        o = orders[0][:]
        rnd.shuffle(o)
        orders.append(o)
    for o in orders:
        cl = _commuting_classes_greedy([gadgets[i] for i in o])
        if best is None or len(cl) < len(best):
            best = cl
    return best


def _commuting_classes_greedy(gadgets):
    """Greedy colouring of the gadget conflict graph.  Two gadgets are compatible if on every
    shared qubit both act with the same role ('Z' or 'X'); then all their gates commute and the
    joint circuit is equivalent to measuring them one after another (sufficient condition)."""
    roles = [gadget_roles(g) for g in gadgets]
    classes = []     # list of (role_map, [idx])
    for i, rm in enumerate(roles):
        placed = False
        for cmap, members in classes:
            ok = all(cmap.get(q, r) == r and r != '?' for q, r in rm.items() if q in cmap)
            if ok:
                for q, r in rm.items():
                    cmap[q] = r
                members.append(i)
                placed = True
                break
        if not placed:
            classes.append((dict(rm), [i]))
    return [[gadgets[i] for i in m] for _, m in classes]


def class_layers(gadgets):
    """Layering of one commuting class: init layer, greedy layering of the (mutually commuting)
    entangling gates, measurement layer."""
    inits = [g for gad in gadgets for g in gad if g[0].startswith('INIT')]
    meas = [g for gad in gadgets for g in gad if g[0].startswith('MEAS')]
    body = [g for gad in gadgets for g in gad if not (g[0].startswith('INIT') or g[0].startswith('MEAS'))]
    # greedy: most-constrained gates first
    body = sorted(body, key=lambda g: -len(g))
    layers = []
    busy = []
    for g in body:
        qs = set(g[1:])
        for li, b in enumerate(busy):
            if not (b & qs) and layers[li][0][0] == g[0]:     # keep one gate type per layer
                b |= qs
                layers[li].append(g)
                break
        else:
            busy.append(set(qs))
            layers.append([g])
    return ([inits] if inits else []) + layers + ([meas] if meas else [])


def round_schedule_classes(gadgets, dur):
    classes = commuting_classes(gadgets)
    depth = 0
    time = 0
    for c in classes:
        ls = class_layers(c)
        depth += len(ls)
        time += circuit_time(ls, dur)
    return {'classes': len(classes), 'depth_ub': depth, 'time_ub': time}


import math
from collections import Counter

SOURCE_LIT = 'literature'
SOURCE_COUNT = 'counted (explicit circuit)'
SOURCE_ASSUME = 'assumption (parameter)'
SOURCE_UNRES = 'unresolved (excluded / illustrative)'

DEFAULT_PARAMS = dict(
    kappa0=1.0,      # FSC |+_L> preparation rounds = ceil(kappa0*d)   [ASSUMPTION; Kobayashi: not specified]
    kappa_in=1.0,    # switch-in rounds            = ceil(kappa_in*d)  [ASSUMPTION; Kobayashi: O(d)]
    kappa_out=1.0,   # FSC EC rounds after return  = ceil(kappa_out*d) [ASSUMPTION; Kobayashi: O(d)]
    tau_rounds=1.0,  # Litinski: 1 time step = tau_rounds*d code cycles [LITERATURE: d cycles]
    t_inj_FT=6,      # Litinski App. A (Li's injection): 3 + 3 code cycles, d-independent [LITERATURE]
    t_inj_NL=1,      # noiseless-minimal injection: 1 round [ASSUMPTION]
)


def rounds_for(model, kappa, d):
    if model == 'NL_min':
        return 1
    return int(math.ceil(kappa * d))

In [ ]:
# ----------------------------------------------------------------- summaries
def summarize(rows):
    tot = Counter()
    for r in rows:
        for k, v in r['gates'].items():
            tot[k] += v
    peak = max(r['qubits'] for r in rows)
    rounds = sum(r['rounds'] for r in rows)
    tl = sum(r['time_lb'] for r in rows if r['time_lb'] is not None)
    tu = sum(r['time_ub'] for r in rows if r['time_ub'] is not None)
    v_round = sum(r['qubits'] * r['rounds'] for r in rows)
    v_tl = sum(r['qubits'] * r['time_lb'] for r in rows if r['time_lb'] is not None)
    v_tu = sum(r['qubits'] * r['time_ub'] for r in rows if r['time_ub'] is not None)
    nonclif = {k: v for k, v in tot.items() if k in NONCLIFFORD}
    tcount = 7 * tot.get('CCZ', 0) + 3 * (tot.get('CS', 0) + tot.get('CSDG', 0)) + tot.get('T', 0) + tot.get('TDG', 0)
    return dict(peak_qubits=peak, rounds=rounds, time_lb=tl, time_ub=tu, V_qubit_round=v_round,
                V_qubit_time_lb=v_tl, V_qubit_time_ub=v_tu, V_peak_x_rounds_UB=peak * rounds,
                gates=dict(tot), nonclifford=nonclif, T_count_if_decomposed=tcount)


def uncounted_costs(protocol, comparison):
    """Costs that are known to exist but are not included in the totals (listed, never summed as 0)."""
    out = [('classical decoding / control latency', 'both comparisons, both protocols')]
    if protocol == 'Kobayashi_D4':
        out.append(('JIT-decoder classical processing', 'not modelled'))
        if comparison == 'B':
            out.append(('FSC -> standard rotated surface-code patch conversion', 'UNRESOLVED: no protocol constructed'))
    else:
        out.append(('twist-based Z_m (x) Y_0 seam checks between |m> and |0>', 'O(d) measurement qubits, not constructed'))
    return out

In [ ]:
# ------------------------------------------------------------------ table helpers
def jsonify_columns(df, cols=('gates', 'nonclifford')):
    """Store dict-valued columns as JSON strings (same format as the saved CSV files)."""
    import json
    for col in cols:
        if col in df.columns:
            df[col] = df[col].apply(lambda g: g if isinstance(g, str) else json.dumps({k: float(v) for k, v in g.items()}))
    return df


def identify_polynomial(values, npts=4):
    """Interpolate a polynomial in d through the first npts points (degree <= npts-1) and report
    whether it reproduces ALL given points exactly.  values: dict d -> integer.  Returns (expr, ok)."""
    import sympy
    x = sympy.Symbol('d')
    ds = sorted(values)
    if len(ds) <= npts:
        return None, False
    p = sympy.interpolate([(dd, int(round(values[dd]))) for dd in ds[:npts]], x)
    ok = all(sympy.nsimplify(p.subs(x, dd) - int(round(values[dd]))) == 0 for dd in ds)
    return sympy.expand(p), ok

### D4 ライブラリ（境界付き D4 符号・transversal $U$・code switching・疎状態シミュレータ・資源モデル）

In [ ]:
"""d4_lib: D4 Clifford stabilizer code (Kobayashi-Zhu-Hsin) on the bounded square lattice, transversal U, code switching, resource model."""
import itertools, cmath
import numpy as np
from collections import Counter

R, G, B = 0, 1, 2
CNAME = "rgb"

In [ ]:
# ------------------------------------------------- phase-shift operators
class PSOp:
    """Operator  O |x> = (-1)^{f(x)} |x + s>  on n qubits (computational basis x in F2^n).

    f(x) = const + sum_{i in lin} x_i + sum_{(i,j) in quad} x_i x_j   (mod 2), i<j.
    X-type Clifford stabilizers of the D4 code, Pauli X/Z strings and products
    thereof all have this form.  Composition rule (A o B)|x> = A B |x>:
        s = s_A + s_B,   f(x) = f_B(x) + f_A(x + s_B).
    """
    __slots__ = ("s", "lin", "quad", "const")

    def __init__(self, s=frozenset(), lin=frozenset(), quad=frozenset(), const=0):
        self.s = frozenset(s)
        self.lin = frozenset(lin)
        self.quad = frozenset(tuple(sorted(p)) for p in quad)
        self.const = const & 1

    def shifted_phase(self, t):
        """Return (lin, quad, const) of x -> f(x + t)."""
        lin = set(self.lin)
        const = self.const
        for i in self.lin:
            if i in t:
                const ^= 1
        for (i, j) in self.quad:
            ti, tj = i in t, j in t
            if ti and tj:
                const ^= 1
            if tj:
                lin ^= {i}
            if ti:
                lin ^= {j}
        return frozenset(lin), self.quad, const

    def __matmul__(self, other):
        lA, qA, cA = self.shifted_phase(other.s)
        return PSOp(self.s ^ other.s, lA ^ other.lin, qA ^ other.quad, cA ^ other.const)

    def phase_at(self, x):
        """f(x) for x given as a set/frozenset of 1-positions."""
        v = self.const
        for i in self.lin:
            if i in x:
                v ^= 1
        for (i, j) in self.quad:
            if i in x and j in x:
                v ^= 1
        return v

    def apply(self, x):
        """O|x> = sign |x'> ; returns (x', sign_bit)."""
        return frozenset(x) ^ self.s, self.phase_at(x)

    def support(self):
        sup = set(self.s) | set(self.lin)
        for p in self.quad:
            sup |= set(p)
        return sup


def commutator_affine(A, B):
    """A B = (-1)^{g(x)} B A on |x>.  Returns g as (lin, const) -- quadratic terms cancel."""
    AB = A @ B
    BA = B @ A
    assert AB.s == BA.s
    q = AB.quad ^ BA.quad
    assert not q, "commutator has quadratic part (should not happen)"
    return AB.lin ^ BA.lin, AB.const ^ BA.const

In [ ]:
# ------------------------------------------------------------- the lattice
class D4Lattice:
    def __init__(self, L):
        assert L >= 1
        self.L = L
        self.edges = []          # (kind, x, y), kind 'h' or 'v'
        self.eidx = {}
        for y in range(L + 1):
            for x in range(L):
                self._add(('h', x, y))
        for x in range(L + 1):
            for y in range(L):
                self._add(('v', x, y))
        self.E = len(self.edges)
        self.n = 3 * self.E
        self.vertices = [(x, y) for y in range(L + 1) for x in range(L + 1)]
        self.plaquettes = [(x, y) for y in range(L) for x in range(L)]

    def _add(self, e):
        self.eidx[e] = len(self.edges)
        self.edges.append(e)

    def q(self, e, c):
        if not isinstance(e, int):
            e = self.eidx[e]
        return 3 * e + c

    # --- geometry --------------------------------------------------------
    def endpoints(self, e):
        k, x, y = self.edges[e] if isinstance(e, int) else e
        return ((x, y), (x + 1, y)) if k == 'h' else ((x, y), (x, y + 1))

    def incident(self, v):
        x, y = v
        out = []
        for e in [('h', x, y), ('h', x - 1, y), ('v', x, y), ('v', x, y - 1)]:
            if e in self.eidx:
                out.append(self.eidx[e])
        return out

    def plaq_edges(self, p):
        """(e01, e13, e02, e23) of plaquette p=(x,y)."""
        x, y = p
        ei = self.eidx
        return ei[('h', x, y)], ei[('v', x + 1, y)], ei[('v', x, y)], ei[('h', x, y + 1)]

    def plaq_boundary(self, p):
        return list(self.plaq_edges(p))

    def has_plaq(self, x, y):
        return 0 <= x < self.L and 0 <= y < self.L

    def boundary_of_edge(self, e):
        k, x, y = self.edges[e]
        L = self.L
        if k == 'v' and x == 0:
            return 'Lr'
        if k == 'h' and y == L:
            return 'Lb'
        if k == 'h' and y == 0:
            return 'Lrb'
        if k == 'v' and x == L:
            return 'Lrb'
        return None

    def boundary_edges(self, name):
        return [e for e in range(self.E) if self.boundary_of_edge(e) == name]

    # --- Z-type checks (rows of H_Z) --------------------------------------
    def z_checks(self, colours=(R, G, B)):
        """List of (label, set_of_qubits) Z-type stabilizer generators.
        Plaquettes for every colour + single-edge boundary conditions (Fig. 4 / eq. (7))."""
        rows = []
        for p in self.plaquettes:
            for c in colours:
                rows.append((('plaq', p, CNAME[c]), {self.q(e, c) for e in self.plaq_boundary(p)}))
        for e in range(self.E):
            bd = self.boundary_of_edge(e)
            if bd == 'Lr' and R in colours:
                rows.append((('bdZ', e, 'r'), {self.q(e, R)}))
            elif bd == 'Lb' and B in colours:
                rows.append((('bdZ', e, 'b'), {self.q(e, B)}))
            elif bd == 'Lrb':
                if R in colours and B in colours:
                    rows.append((('bdZ', e, 'rb'), {self.q(e, R), self.q(e, B)}))
                if G in colours:
                    rows.append((('bdZ', e, 'g'), {self.q(e, G)}))
        return rows

    # --- X-type Clifford stabilizers ----------------------------------------
    def cz_pairs(self, v, c):
        """CZ pairs (qubit, qubit) of the dressing of S^c_{X;v} (SM eq. for S_X, cup products
        evaluated with the hypercube formula; plaquettes outside the lattice are dropped)."""
        x, y = v
        pairs = []
        if c == R:      # int v u e' u e'' : v = vertex 0 of p(x,y);   CZ^{g,b}
            if self.has_plaq(x, y):
                e01, e13, e02, e23 = self.plaq_edges((x, y))
                pairs += [(self.q(e01, G), self.q(e13, B)), (self.q(e02, G), self.q(e23, B))]
        elif c == G:    # int e'' u v u e' : v = vertex 1 of p(x-1,y), vertex 2 of p(x,y-1); CZ^{r,b}
            if self.has_plaq(x - 1, y):
                e01, e13, e02, e23 = self.plaq_edges((x - 1, y))
                pairs.append((self.q(e01, R), self.q(e13, B)))
            if self.has_plaq(x, y - 1):
                e01, e13, e02, e23 = self.plaq_edges((x, y - 1))
                pairs.append((self.q(e02, R), self.q(e23, B)))
        else:           # int e' u e'' u v : v = vertex 3 of p(x-1,y-1); CZ^{r,g}
            if self.has_plaq(x - 1, y - 1):
                e01, e13, e02, e23 = self.plaq_edges((x - 1, y - 1))
                pairs += [(self.q(e01, R), self.q(e13, G)), (self.q(e02, R), self.q(e23, G))]
        return pairs

    def SX(self, v, c, dressed=True):
        s = {self.q(e, c) for e in self.incident(v)}
        quad = self.cz_pairs(v, c) if dressed else []
        # CZ applied first, then X (S = (prod X)(prod CZ)); X and CZ act on disjoint colours.
        return PSOp(s=s, quad=quad)

    def vertex_x_types(self, v, colours=(R, G, B)):
        """Allowed colour combinations at vertex v: the F2 span of colour sets whose
        X-part commutes with the boundary Z conditions.  Returns a list of colour tuples
        (a basis), derived automatically from the boundary Z checks."""
        bdrows = [rows for (lab, rows) in self.z_checks(colours) if lab[0] == 'bdZ']
        inc = self.incident(v)
        cand = []
        for k in range(1, len(colours) + 1):
            for combo in itertools.combinations(colours, k):
                s = {self.q(e, c) for e in inc for c in combo}
                if all(len(s & rw) % 2 == 0 for rw in bdrows):
                    cand.append(combo)
        # reduce to an F2 basis over colour space, preferring single colours
        basis, vecs = [], []
        for combo in sorted(cand, key=len):
            vec = np.zeros(3, dtype=np.uint8)
            vec[list(combo)] = 1
            if f2_rank(np.array(vecs + [vec])) > len(vecs):
                vecs.append(vec)
                basis.append(combo)
        return basis

    def x_generators(self, colours=(R, G, B), dressed=True):
        """List of (label, PSOp).  At a vertex where only a product of colours is allowed
        (L_rb), the generator is the ordered product S^r S^b (Fig. 4 shows S^r S^b there)."""
        gens = []
        for v in self.vertices:
            for combo in self.vertex_x_types(v, colours):
                op = PSOp()
                for c in combo:
                    op = op @ self.SX(v, c, dressed)
                gens.append((('X', v, ''.join(CNAME[c] for c in combo)), op))
        return gens

    def vec(self, qubits):
        v = np.zeros(self.n, dtype=np.uint8)
        v[list(qubits)] = 1
        return v

In [ ]:
# ---------------------------------------------------------- code data
class CodeData:
    """Linear-algebra data of a (D4 or folded-surface) code on a lattice."""

    def __init__(self, lat, colours=(R, G, B), dressed=True):
        self.lat = lat
        self.colours = tuple(colours)
        self.zlabels, zr = zip(*lat.z_checks(colours))
        self.HZ = np.array([lat.vec(r) for r in zr], dtype=np.uint8)
        self.xgens = lat.x_generators(colours, dressed)
        self.S = np.array([lat.vec(op.s) for _, op in self.xgens], dtype=np.uint8)
        self.cols = sorted({lat.q(e, c) for e in range(lat.E) for c in colours})
        self.HZr, self.HZpiv = f2_rref(self.HZ)
        self.Sr, self.Spiv = f2_rref(self.S)
        # logical representatives
        CZ = f2_nullspace(self.HZ[:, self.cols])
        self.t_log = None
        for row in CZ:
            t = np.zeros(lat.n, dtype=np.uint8)
            t[self.cols] = row
            if not f2_in_rowspace(self.Sr, self.Spiv, t):
                self.t_log = t
                break
        KS = f2_nullspace(self.S[:, self.cols])
        self.z_log = None
        for row in KS:
            u = np.zeros(lat.n, dtype=np.uint8)
            u[self.cols] = row
            if not f2_in_rowspace(self.HZr, self.HZpiv, u):
                self.z_log = u
                break
        self.k = len(self.cols) - f2_rank(self.HZ[:, self.cols]) - f2_rank(self.S[:, self.cols])


def distances(cd):
    """(d_Z, d_X_shift) : min weight of a logical Z-type Pauli, and min weight of an
    X-shift lying in C_Z but outside the stabilizer shift span."""
    dz, uz = min_weight_odd(cd.S, cd.t_log, cd.cols)
    dx, tx = min_weight_odd(cd.HZ, cd.z_log, cd.cols)
    return dz, dx, uz, tx

In [ ]:
# ------------------------------------------- transversal automorphism U = W V
class DiagPhase:
    """Diagonal unitary |x> -> omega^{w(x)} |x>, omega = exp(i pi/4), w integer mod 8.
    w(x) = sum_k c_k * prod_{i in mono_k} x_i  (monomials of degree <= 2)."""

    def __init__(self, terms=()):
        self.terms = list(terms)   # (coef mod 8, tuple_of_qubits)

    def value(self, x):
        v = 0
        for c, mono in self.terms:
            if all(i in x for i in mono):
                v += c
        return v % 8

    def support(self):
        s = set()
        for _, m in self.terms:
            s |= set(m)
        return s

    def terms_touching(self, qs):
        return DiagPhase([(c, m) for c, m in self.terms if set(m) & qs])


def W_operator(lat, cs_signs=(+1, -1), boundary_T=-1):
    """W of eq. (8):  prod_p CS^{r,b}_{e01,e13} (CS^{r,b}_{e02,e23})^dag  x  prod_{e in bdry_rb} (T^r_e)^dag.
    CS = diag(1,1,1,i) -> omega^{2 x y};  T^dag -> omega^{-x}."""
    terms = []
    for p in lat.plaquettes:
        e01, e13, e02, e23 = lat.plaq_edges(p)
        terms.append(((2 * cs_signs[0]) % 8, (lat.q(e01, R), lat.q(e13, B))))
        terms.append(((2 * cs_signs[1]) % 8, (lat.q(e02, R), lat.q(e23, B))))
    if boundary_T:
        for e in lat.boundary_edges('Lrb'):
            terms.append((boundary_T % 8, (lat.q(e, R),)))
    return DiagPhase(terms)


def V_map(lat, x):
    """V = prod_e CNOT^{(r,g)}_e CNOT^{(b,g)}_e :  x_g <- x_g + x_r + x_b  (set representation)."""
    x = set(x)
    out = set(x)
    for e in range(lat.E):
        qr, qg, qb = lat.q(e, R), lat.q(e, G), lat.q(e, B)
        if (qr in x) ^ (qb in x):
            out ^= {qg}
    return frozenset(out)


def V_support_map(lat, qs):
    """Qubits whose values determine (V x) restricted to qs."""
    out = set()
    for q in qs:
        e, c = divmod(q, 3)
        out.add(q)
        if c == G:
            out |= {3 * e + R, 3 * e + B}
    return out


def CZ_basis(cd):
    if not hasattr(cd, '_CZbasis'):
        nb = f2_nullspace(cd.HZ[:, cd.cols])
        full = np.zeros((nb.shape[0], cd.lat.n), dtype=np.uint8)
        full[:, cd.cols] = nb
        cd._CZbasis = full
    return cd._CZbasis


def project_CZ(cd, Q, functionals=()):
    """Exact image of C_Z under x -> (x|_Q, phi_1(x), ..., phi_k(x)) for F2-linear functionals
    phi_j (callables on a set of qubits, evaluated on basis vectors).  Returns a list of
    configurations (frozenset of Q-qubits) or, if functionals are given, (config, bits)."""
    Q = sorted(Q)
    Bs = CZ_basis(cd)
    P = Bs[:, Q]
    if functionals:
        extra = np.array([[phi(frozenset(np.nonzero(b)[0])) for phi in functionals] for b in Bs],
                         dtype=np.uint8).reshape(len(Bs), len(functionals))
        P = np.concatenate([P, extra], axis=1)
    Pr, piv = f2_rref(P)
    k = Pr.shape[0]
    assert k <= 24, f"local projection too large ({k})"
    out = []
    nq = len(Q)
    for bits in itertools.product((0, 1), repeat=k):
        v = np.zeros(P.shape[1], dtype=np.uint8)
        for b, row in zip(bits, Pr):
            if b:
                v ^= row
        cfg = frozenset(Q[i] for i in np.nonzero(v[:nq])[0])
        out.append((cfg, tuple(int(z) for z in v[nq:])) if functionals else cfg)
    return out


def check_conjugation(cd, Wop, gen_op, target_op):
    """Exhaustively check  U S U^dag == S'  on C_Z, where U = W V.
    Returns (ok, n_local_configs)."""
    lat = cd.lat
    As = V_map(lat, gen_op.s)
    if As != target_op.s:
        return False, 0
    # qubits on which the phase difference can depend
    Wt = Wop.terms_touching(set(As))
    Q = set(Wt.support()) | V_support_map(lat, gen_op.support()) | target_op.support() | set(As)
    configs = project_CZ(cd, Q)
    for x in configs:
        # omega^{ w(x+As) - w(x) + 4 f(A x) - 4 f'(x) }   (only W-terms touching As contribute)
        d = Wt.value(x ^ As) - Wt.value(x) + 4 * gen_op.phase_at(V_map(lat, x)) - 4 * target_op.phase_at(x)
        if d % 8:
            return False, len(configs)
    return True, len(configs)


def stabilizer_path_phase(cd, x, target):
    """Given x and target = x + (sum of generator shifts), apply generators to |x> and
    return the accumulated sign bit, i.e. psi(target) = (-1)^bit psi(x) for code states."""
    shifts = cd.S
    delta = cd.lat.vec(set(x) ^ set(target))
    coeff = f2_solve(shifts, delta)
    if coeff is None:
        return None
    bit = 0
    cur = frozenset(x)
    for i in np.nonzero(coeff)[0]:
        op = cd.xgens[i][1]
        cur, b = op.apply(cur)
        bit ^= b
    assert cur == frozenset(target)
    return bit


def logical_phases(cd, Wop):
    """Eigenphases (in units of pi/4) of U on |0_L>, |1_L>, assuming U preserves the code space
    (checked separately).  Uses x0 = 0 and x1 = t_log."""
    lat = cd.lat
    res = []
    for x in [frozenset(), frozenset(np.nonzero(cd.t_log)[0])]:
        Ax = V_map(lat, x)
        bit = stabilizer_path_phase(cd, x, Ax)
        assert bit is not None, "V does not map the coset to itself"
        res.append((Wop.value(Ax) + 4 * bit) % 8)
    return res

In [ ]:
# ------------------------------------------- per-configuration protocol phase
def lrb_vertices(lat):
    vs = set()
    for e in lat.boundary_edges('Lrb'):
        vs |= set(lat.endpoints(e))
    return vs


def cprime(lat, a):
    """0-cochain c' with  delta c' = a_r + a_b  and c' = 0 on L_rb vertices (BFS).
    a: set of qubits (configuration of r,b qubits).  Returns dict vertex -> bit.
    Raises if a_r + a_b is not exact relative to L_rb."""
    from collections import deque
    val = {v: 0 for v in lrb_vertices(lat)}
    dq = deque(val)
    while dq:
        u = dq.popleft()
        for e in lat.incident(u):
            p, q = lat.endpoints(e)
            w = q if p == u else p
            bit = val[u] ^ (lat.q(e, R) in a) ^ (lat.q(e, B) in a)
            if w in val:
                if val[w] != bit:
                    raise ValueError("a_r + a_b not exact")
            else:
                val[w] = bit
                dq.append(w)
    return val


def g_phase_poly(lat, v):
    """CZ^{r,b} pairs of S^g_{X;v}  (f^g_v)."""
    return lat.cz_pairs(v, G)


def protocol_phase(lat, Wop, a, gverts):
    """Phase exponent (mod 8, units of pi/4) acquired by FSC configuration a through
    switch-in (S^g measurement) -> U -> Z^g measurement -> S^g correction -> Pauli frame,
    i.e.  F(a) = w(a) + 4 * sum_v c'_v(a) f^g_v(a)."""
    cp = cprime(lat, a)
    phi = 0
    for v in gverts:
        if cp[v]:
            for (i, j) in g_phase_poly(lat, v):
                if i in a and j in a:
                    phi ^= 1
    return (Wop.value(a) + 4 * phi) % 8


def check_protocol_invariance(lat, fsc, Wop, gverts, gen_op):
    """Exhaustive local proof that F(a + t) = F(a) (mod 8) for all a in C_FSC, t = shift of gen_op."""
    t = frozenset(gen_op.s)
    # c'(t) (linear in a)
    ct = cprime(lat, t)
    N2 = {v for v in gverts if ct[v]}
    N1 = {v for v in gverts if set(q for pr in g_phase_poly(lat, v) for q in pr) & t}
    Wt = Wop.terms_touching(set(t))
    Q = set(Wt.support()) | set(t)
    for v in N1 | N2:
        Q |= {q for pr in g_phase_poly(lat, v) for q in pr}
    # paths from a reference vertex w0 to each w in N1 (BFS on the lattice graph)
    w0 = min(N1 | N2 | {lat.endpoints(next(iter(t)) // 3)[0]})
    paths = {}
    from collections import deque
    prev = {w0: None}
    dq = deque([w0])
    while dq:
        u = dq.popleft()
        for e in lat.incident(u):
            p, q = lat.endpoints(e)
            w = q if p == u else p
            if w not in prev:
                prev[w] = (u, e)
                dq.append(w)
    for w in N1:
        es = []
        cur = w
        while prev[cur] is not None:
            u, e = prev[cur]
            es.append(e)
            cur = u
        paths[w] = es
        Q |= {lat.q(e, c) for e in es for c in (R, B)}
    beta_fn = lambda b: cprime(lat, b)[w0]          # c'_{w0}(a) is F2-linear in a
    configs = project_CZ(fsc, Q, functionals=[beta_fn])
    for a, (beta,) in configs:
        if True:
            cpa = {}
            for w in N1:
                bit = beta
                for e in paths[w]:
                    bit ^= (lat.q(e, R) in a) ^ (lat.q(e, B) in a)
                cpa[w] = bit
            at = a ^ t
            d = Wt.value(at) - Wt.value(a)
            phi = 0
            for w in N1:
                if cpa[w]:
                    for (i, j) in g_phase_poly(lat, w):
                        phi ^= ((i in at) and (j in at)) ^ ((i in a) and (j in a))
            for w in N2:
                for (i, j) in g_phase_poly(lat, w):
                    phi ^= ((i in at) and (j in at))
            if (d + 4 * phi) % 8:
                return False, len(configs)
    return True, len(configs)


OMEGA = cmath.exp(1j * cmath.pi / 4)


class SparseState:
    def __init__(self, amps):
        self.a = {k: v for k, v in amps.items() if abs(v) > 1e-14}

    def norm2(self):
        return sum(abs(v) ** 2 for v in self.a.values())

    def normalize(self):
        n = self.norm2() ** 0.5
        self.a = {k: v / n for k, v in self.a.items()}
        return self

    def apply_psop(self, op):
        out = {}
        for x, v in self.a.items():
            y, b = op.apply(x)
            out[y] = out.get(y, 0) + (-v if b else v)
        return SparseState(out)

    def apply_diag(self, D):
        return SparseState({x: v * OMEGA ** D.value(x) for x, v in self.a.items()})

    def apply_perm(self, f):
        out = {}
        for x, v in self.a.items():
            y = f(x)
            out[y] = out.get(y, 0) + v
        return SparseState(out)

    def project(self, op, sign):
        """(1 + sign*O)/2 |psi>  (unnormalised)."""
        Opsi = self.apply_psop(op).a
        out = {}
        for k in set(self.a) | set(Opsi):
            out[k] = 0.5 * (self.a.get(k, 0) + sign * Opsi.get(k, 0))
        return SparseState(out)

    def expect(self, op):
        Opsi = self.apply_psop(op).a
        return sum(np.conj(self.a.get(k, 0)) * v for k, v in Opsi.items())

    def measure_op(self, op, rng, forced=None):
        """Projective measurement of a Hermitian involution O; returns (outcome bit, state)."""
        p0 = self.project(op, +1)
        n0 = p0.norm2()
        if forced is None:
            bit = 0 if rng.random() < n0 else 1
        else:
            bit = forced
        st = p0 if bit == 0 else self.project(op, -1)
        prob = n0 if bit == 0 else 1 - n0
        return bit, st.normalize() if prob > 1e-12 else st, prob

    def measure_z(self, q, rng, forced=None):
        p1 = sum(abs(v) ** 2 for x, v in self.a.items() if q in x)
        bit = (1 if rng.random() < p1 else 0) if forced is None else forced
        st = SparseState({x: v for x, v in self.a.items() if (q in x) == bool(bit)})
        prob = p1 if bit else 1 - p1
        return bit, st.normalize() if prob > 1e-12 else st, prob

    def overlap(self, other):
        return sum(np.conj(v) * other.a.get(k, 0) for k, v in self.a.items())


def coset_state(cd, rep, phases=True):
    """Code state supported on rep + span(shifts):  psi(rep)=1, psi(x+s_v)=(-1)^{f_v(x)} psi(x)."""
    amps = {frozenset(rep): 1.0}
    # breadth-first application of generators
    frontier = [frozenset(rep)]
    while frontier:
        new = []
        for x in frontier:
            for _, op in cd.xgens:
                y, b = op.apply(x)
                if y not in amps:
                    amps[y] = amps[x] * (-1 if b else 1)
                    new.append(y)
                else:
                    if phases:
                        assert abs(amps[y] - amps[x] * (-1 if b else 1)) < 1e-9, "inconsistent code state"
        frontier = new
    return SparseState(amps).normalize()




def frozen_qubits(lat):
    """Qubits fixed to |0> by single-qubit boundary Z stabilizers (L_r: r, L_b: b, L_rb: g)."""
    fz = set()
    for lab, rows in lat.z_checks():
        if lab[0] == 'bdZ' and len(rows) == 1:
            fz |= rows
    return fz


class D4Protocol:
    def __init__(self, L, prune_frozen=True):
        self.L = L
        self.lat = lat = D4Lattice(L)
        self.frozen = frozen_qubits(lat) if prune_frozen else set()
        self.prune = prune_frozen
        self.d4 = CodeData(lat)
        self.fsc = CodeData(lat, (R, B), dressed=False)
        # ancilla allocation: one ancilla per stabilizer generator label (D4 superset)
        self.anc = {}
        nxt = lat.n
        labels = [lab for lab, _ in self.d4.xgens] + list(self.d4.zlabels)
        labels += [lab for lab, _ in self.fsc.xgens] + list(self.fsc.zlabels)
        for lab in labels:
            key = self._anc_key(lab)
            if key not in self.anc and not self._is_frozen_check(lab):
                self.anc[key] = nxt
                nxt += 1
        self.gverts = [lab[1] for lab, op in self.d4.xgens if lab[2] == 'g']

    @staticmethod
    def _anc_key(lab):
        return lab

    def _is_frozen_check(self, lab):
        return lab[0] == 'bdZ' and lab[2] in ('r', 'b', 'g') and self.prune

    def data_qubits(self, colours):
        return {self.lat.q(e, c) for e in range(self.lat.E) for c in colours} - self.frozen

    # ---------------------------------------------------------- gadgets
    def _keep_pair(self, i, j):
        return i not in self.frozen and j not in self.frozen

    def x_gadget(self, lab, dressed):
        _, v, cols = lab
        parts = []
        for ch in cols:
            c = CNAME.index(ch)
            xq = {self.lat.q(e, c) for e in self.lat.incident(v)} - self.frozen
            pairs = [p for p in self.lat.cz_pairs(v, c) if self._keep_pair(*p)] if dressed else []
            parts.append((xq, pairs))
        return clifford_x_gadget(self.anc[lab], parts)

    def z_gadget(self, lab, rows):
        return z_check_gadget(self.anc[lab], set(rows) - self.frozen)

    def round_gadgets(self, code, which='all'):
        """Gadgets of one stabilizer round.  code: 'D4' or 'FSC'.
        which: 'all' | 'X' | 'Z' | 'g' (D4 S^g only)."""
        cd = self.d4 if code == 'D4' else self.fsc
        dressed = code == 'D4'
        gads = []
        if which in ('all', 'X', 'g'):
            for lab, op in cd.xgens:
                if which == 'g' and lab[2] != 'g':
                    continue
                gads.append(self.x_gadget(lab, dressed))
        if which in ('all', 'Z'):
            for lab, (l2, rows) in zip(cd.zlabels, self.lat.z_checks(cd.colours)):
                if self._is_frozen_check(lab):
                    continue
                gads.append(self.z_gadget(lab, rows))
        return gads

    # ---------------------------------------------------------- transversal U
    def U_gates(self):
        lat, fz = self.lat, self.frozen
        gates = []
        for e in range(lat.E):
            qr, qg, qb = lat.q(e, R), lat.q(e, G), lat.q(e, B)
            for ctrl in (qr, qb):
                if ctrl in fz:
                    continue
                if qg in fz:
                    continue   # L_rb: CNOT(r,g) CNOT(b,g) = identity on Z^r Z^b = +1 (g removed)
                gates.append(('CNOT', ctrl, qg))
        for p in lat.plaquettes:
            e01, e13, e02, e23 = lat.plaq_edges(p)
            for (a, b, name) in [(lat.q(e01, R), lat.q(e13, B), 'CS'), (lat.q(e02, R), lat.q(e23, B), 'CSDG')]:
                if a in fz or b in fz:
                    continue
                gates.append((name, a, b))
        for e in lat.boundary_edges('Lrb'):
            gates.append(('TDG', lat.q(e, R)))
        return gates

    # ---------------------------------------------------------- switch-out
    def switch_out_gates(self, worst_case=True):
        lat = self.lat
        gq = sorted(self.data_qubits((G,)))
        gates = [('MEAS_Z', q) for q in gq]
        cz = []
        for v in self.gverts:
            for p in lat.cz_pairs(v, G):
                if self._keep_pair(*p):
                    cz.append(('CZ',) + tuple(p))
        return gates, cz      # cz: worst-case correction (all vertices in kappa)

    def qubit_sets(self):
        fsc_data = self.data_qubits((R, B))
        d4_data = self.data_qubits((R, G, B))
        fsc_anc = {self.anc[l] for l, _ in self.fsc.xgens} | {self.anc[l] for l in self.fsc.zlabels
                                                              if not self._is_frozen_check(l)}
        d4_anc = {self.anc[l] for l, _ in self.d4.xgens} | {self.anc[l] for l in self.d4.zlabels
                                                            if not self._is_frozen_check(l)}
        return dict(fsc_data=fsc_data, d4_data=d4_data, fsc_anc=fsc_anc, d4_anc=d4_anc)

In [ ]:
# ----------------------------------------------------------------- Kobayashi (D4)
class D4CostData:
    """All d-dependent counts of the D4 protocol, computed once per L from explicit circuits."""

    def __init__(self, L, dur=None, decomposed=False):
        dur = dur or DUR_UNIT
        P = D4Protocol(L)
        self.L = L
        qs = P.qubit_sets()
        self.n = {k: len(v) for k, v in qs.items()}
        self.n_total = len(qs['d4_data'] | qs['d4_anc'])
        dec = decompose if decomposed else (lambda g: g)
        gad = {
            'D4': [dec(g) for g in P.round_gadgets('D4')],
            'D4g': [dec(g) for g in P.round_gadgets('D4', 'g')],
            'FSC': P.round_gadgets('FSC'),
        }
        self.round = {}
        for k, g in gad.items():
            ub = round_schedule_classes(g, dur)
            lb = round_lower_bound(g, dur)
            self.round[k] = dict(time_lb=lb['time_lb'], time_ub=ub['time_ub'], depth_lb=lb['depth_lb'],
                                 depth_ub=ub['depth_ub'], classes=ub['classes'],
                                 gates=Counter(gate_counts([x for gg in g for x in gg])),
                                 n_anc=len(g))
        self.n_anc_g = len(gad['D4g'])
        U = dec(P.U_gates())
        Ul = class_layers([U])              # all U gates mutually commute (validated)
        self.U = dict(gates=Counter(gate_counts(U)), time=circuit_time(Ul, dur), depth=len(Ul))
        meas, cz = P.switch_out_gates()
        czl = class_layers([cz]) if cz else []
        ml = [meas]
        self.out = dict(gates_worst=Counter(gate_counts(meas + cz)),
                        gates_expected=Counter({**gate_counts(meas), 'CZ': len(cz) / 2}),
                        time=circuit_time(ml + czl, dur), depth=len(ml) + len(czl))
        self.x_corr = dict(time=dur['X'], gates_worst=Counter({'X': self.n['fsc_data']}))
        self.init = Counter({'INIT_X': self.n['fsc_data'], 'INIT_Z': len(qs['d4_data']) - self.n['fsc_data']})


def d4_stages(cd, model, params=DEFAULT_PARAMS, comparison='A', data_only=False):
    d = cd.L
    n = cd.n
    fsc_q = n['fsc_data'] + (0 if data_only else n['fsc_anc'])
    d4_q = n['d4_data'] + (0 if data_only else n['d4_anc'])
    rows = []

    def add(stage, qubits, rounds, rkey, extra_time, gates, source, note=''):
        rt = cd.round[rkey] if rkey else dict(time_lb=0, time_ub=0)
        rows.append(dict(protocol='Kobayashi_D4', comparison=comparison, model=model, d=d, stage=stage,
                         qubits=qubits, rounds=rounds,
                         time_lb=rounds * rt['time_lb'] + extra_time,
                         time_ub=rounds * rt['time_ub'] + extra_time,
                         gates=dict(gates), source=source, note=note))

    r0 = rounds_for(model, params['kappa0'], d)
    g0 = Counter(cd.init)
    for k, v in cd.round['FSC']['gates'].items():
        g0[k] += v * r0
    g0['X'] += cd.x_corr['gates_worst']['X']
    add('S0 FSC |+_L> preparation', fsc_q, r0, 'FSC', cd.x_corr['time'], g0,
        SOURCE_ASSUME if model == 'FT_ref' else SOURCE_COUNT,
        'kappa0*d rounds (not specified in Kobayashi); X-correction worst case counted')
    if model == 'NL_min':
        r1, key = 1, 'D4g'
        q1 = n['d4_data'] + (0 if data_only else cd.n_anc_g)
        note1 = 'noiseless minimum: one round of S^g_X measurements only'
    else:
        r1, key = rounds_for(model, params['kappa_in'], d), 'D4'
        q1 = d4_q
        note1 = 'O(d) rounds of all D4 stabilizers (Kobayashi step 2); kappa_in*d assumed'
    g1 = Counter()
    for k, v in cd.round[key]['gates'].items():
        g1[k] += v * r1
    add('S1 switch-in (gauging) to D4', q1, r1, key, 0, g1,
        SOURCE_ASSUME if model == 'FT_ref' else SOURCE_COUNT, note1)
    add('S2 transversal U = W V (logical T^dag)', n['d4_data'], 0, None, cd.U['time'], cd.U['gates'],
        SOURCE_COUNT, 'constant depth; gates from eq. (8)')
    add('S3 switch-out: Z^g meas + S^g correction', n['d4_data'], 0, None, cd.out['time'],
        cd.out['gates_worst'], SOURCE_COUNT, 'CZ correction counted worst case (expected = half)')
    r4 = 0 if model == 'NL_min' else rounds_for(model, params['kappa_out'], d)
    g4 = Counter()
    for k, v in cd.round['FSC']['gates'].items():
        g4[k] += v * r4
    add('S4 FSC error-correction rounds', fsc_q, r4, 'FSC', 0, g4,
        SOURCE_ASSUME if model == 'FT_ref' else SOURCE_COUNT,
        'O(d) rounds (Kobayashi step 4); kappa_out*d assumed; 0 in the noiseless minimum')
    add('S5 logical X frame: T^dag|+> -> T|+>', 0, 0, None, 0, Counter(), SOURCE_COUNT,
        'Pauli frame update, no physical cost')
    # NOTE (comparison B): the FSC -> standard rotated-patch conversion is UNRESOLVED.  It is deliberately
    # NOT added as a zero-cost stage; it is reported separately by uncounted_costs().
    return rows



def d4_conversion_illustrative(cd, model, params=DEFAULT_PARAMS, data_only=False):
    """ILLUSTRATIVE ONLY: conversion modelled as one lattice-surgery time step (tau) during which the
    FSC plus 2 tiles (target patch + ancilla strip) are active.  Not taken from literature."""
    d = cd.L
    r = rounds_for(model, params['tau_rounds'], d)
    q = cd.n['fsc_data'] + (0 if data_only else cd.n['fsc_anc']) + 2 * (d * d if data_only else 2 * d * d)
    return dict(protocol='Kobayashi_D4', comparison='B_illustrative', model=model, d=d,
                stage='S6 conversion (ILLUSTRATIVE: 1 tau lattice surgery)', qubits=q, rounds=r,
                time_lb=None, time_ub=None, gates={}, source=SOURCE_UNRES,
                note='illustrative assumption, not a validated protocol')

In [ ]:
# ---------------------------------------------------------- consistency checks of a code
def code_consistency(cd):
    """Exact F2 checks of a (possibly non-commuting) Clifford stabilizer code on its Z-subspace C_Z:
      shift_in_CZ : every X-type generator preserves C_Z (its X part commutes with all Z checks)
      noncommuting_pairs : generator pairs that do NOT commute on C_Z
      bad_squares : generators with S^2 != 1 on C_Z
      k : number of logical qubits = dim C_Z - rank(shifts)   (generators have independent shifts)"""
    lat = cd.lat
    HZ, S = cd.HZ, cd.S
    res = {}
    res['shift_in_CZ'] = not ((HZ.astype(int) @ S.T.astype(int)) % 2).any()
    supp = [op.support() for _, op in cd.xgens]
    bad = []
    for i in range(len(cd.xgens)):
        for j in range(i + 1, len(cd.xgens)):
            if not (supp[i] & supp[j]):
                continue
            lin, const = commutator_affine(cd.xgens[i][1], cd.xgens[j][1])
            if const or not f2_in_rowspace(cd.HZr, cd.HZpiv, lat.vec(lin)):
                bad.append((cd.xgens[i][0], cd.xgens[j][0]))
    res['noncommuting_pairs'] = bad
    sq = []
    for lab, op in cd.xgens:
        s2 = op @ op
        if s2.s or s2.quad or s2.const or not f2_in_rowspace(cd.HZr, cd.HZpiv, lat.vec(s2.lin)):
            sq.append(lab)
    res['bad_squares'] = sq
    res['n_qubits'] = len(cd.cols)
    res['dim_CZ'] = len(cd.cols) - f2_rank(HZ[:, cd.cols])
    res['rank_shifts'] = f2_rank(S[:, cd.cols])
    res['n_xgens'] = len(cd.xgens)
    res['k'] = res['dim_CZ'] - res['rank_shifts']
    res['shifts_independent'] = res['rank_shifts'] == res['n_xgens']
    return res


def U_target(lat, lab):
    """Image of generator `lab` under U predicted by SM eq. (permutingstabilizers):
    S^r -> S^r S^g, S^g -> S^g, S^b -> S^g S^b, and (boundary L_rb) S^r S^b -> S^r S^b."""
    _, v, cols = lab
    if cols == 'rb':
        return lat.SX(v, R) @ lat.SX(v, B)
    op = PSOp()
    for ch in cols:
        c = CNAME.index(ch)
        for cc in ([c] if c == G else [c, G]):
            op = op @ lat.SX(v, cc)
    return op

In [ ]:
# ---------------------------------------------------------- Knill-Laflamme distance lower bound
def _milp_binary(c_obj, A_eq_mod2, rhs_mod2, extra_ub=None, fixed_zero=()):
    """min c.x over binary x subject to A x = rhs (mod 2) (rows of A_eq_mod2), and optional
    linear constraints extra_ub = [(row, ub), ...] (row.x <= ub).
    Returns (value, x) if an optimum was found, (None, None) only if the problem is proven
    infeasible (scipy status 2).  Any other status (time/iteration limit, unbounded, numerical
    failure) raises RuntimeError, so that a solver failure is never mistaken for "no solution"."""
    from scipy.optimize import milp, LinearConstraint, Bounds
    A = np.array(A_eq_mod2, dtype=float)
    m, n = A.shape
    nv = n + m
    c = np.concatenate([np.asarray(c_obj, float), np.zeros(m)])
    cons = [LinearConstraint(np.concatenate([A, -2 * np.eye(m)], axis=1), rhs_mod2, rhs_mod2)]
    for row, ub in (extra_ub or []):
        cons.append(LinearConstraint(np.concatenate([np.asarray(row, float), np.zeros(m)])[None, :], -np.inf, ub))
    ub_ = np.concatenate([np.ones(n), np.full(m, np.inf)])
    for i in fixed_zero:
        ub_[i] = 0
    res = milp(c, constraints=cons, integrality=np.ones(nv), bounds=Bounds(np.zeros(nv), ub_))
    if res.status == 2:
        return None, None
    if res.status != 0:
        raise RuntimeError(f'MILP solver did not finish (status {res.status}: {res.message})')
    return round(res.fun), np.round(res.x[:n]).astype(np.uint8)


def low_weight_stabilizer_shifts(cd, wmax, frozen=()):
    """Exact enumeration (MILP with no-good cuts) of all t in span(S) with 1 <= |t| <= wmax,
    where span(S) = C_Z  intersect  z_log^perp  (k = 1).  Frozen qubits are excluded
    (an X on a frozen qubit violates its single-qubit Z check, so t would leave C_Z).
    Returns (found, stop) where stop records why the enumeration ended:
    'infeasible' (no further t with 1 <= |t| <= wmax) or 'min weight > wmax'."""
    cols = [c for c in cd.cols if c not in set(frozen)]
    H = np.vstack([cd.HZ[:, cols], cd.z_log[cols][None, :]])
    H = H[np.any(H, axis=1)]
    rhs = np.zeros(len(H))
    n = len(cols)
    cuts = [(np.ones(n), wmax), (-np.ones(n), -1)]
    found = []
    while True:
        val, x = _milp_binary(np.ones(n), H, rhs, extra_ub=cuts)
        if val is None:
            stop = 'infeasible'
            break
        if val > wmax:
            stop = 'min weight > wmax'
            break
        sup = np.nonzero(x)[0]
        t = np.zeros(cd.lat.n, dtype=np.uint8)
        t[[cols[i] for i in sup]] = 1
        found.append(t)
        row = -np.ones(n)
        row[sup] = 1
        cuts.append((row, len(sup) - 1))                  # exclude exactly this vector
    return found, stop


def kl_distance_lower_bound(cd, wmax, frozen=()):
    r"""Rigorous lower-bound argument for the Knill-Laflamme distance of the (non-Pauli) code cd.

    For a Pauli E = X^t Z^u (up to phase):
      * t not in C_Z           -> E maps the code space to an orthogonal space: detectable;
      * t in the logical coset -> |t| >= X-shift distance (checked separately);
      * t in span(S)           -> X^t = S_c F_c with S_c a stabilizer and F_c = (-1)^{f_c} diagonal,
        so P E P = P F_c Z^u P.  Expanding (-1)^{f_c} in Z-type Paulis Z^v with v supported on
        Q_f = supp(f_c), a violation of KL requires some v + u to be a logical Z operator l.
        Hence |E| >= |t| + |u \ t| >= |t| + min_l |l \ (Q_f u t)|  =: LB_t.
    Returns (min over enumerated t of LB_t, ts, details, stop) where stop is the termination
    reason of the enumeration (see low_weight_stabilizer_shifts).  t = 0 is covered by d_Z."""
    lat = cd.lat
    ts, stop = low_weight_stabilizer_shifts(cd, wmax, frozen)
    cols = [c for c in cd.cols if c not in set(frozen)]
    Srows = cd.S[:, cols]
    Srows = Srows[np.any(Srows, axis=1)]
    best, details = None, []
    for t in ts:
        coef = f2_solve(cd.S, t)
        op = PSOp()
        for i in np.nonzero(coef)[0]:
            op = op @ cd.xgens[i][1]
        assert op.s == frozenset(np.nonzero(t)[0])
        Qf = {q for pr in op.quad for q in pr} | set(op.lin) | set(np.nonzero(t)[0])
        cost = np.array([0.0 if c in Qf else 1.0 for c in cols])
        A = np.vstack([Srows, cd.t_log[cols][None, :]])
        rhs = np.concatenate([np.zeros(len(Srows)), [1.0]])
        val, _ = _milp_binary(cost, A, rhs)
        assert val is not None                            # a logical Z always exists
        lb = int(t.sum()) + val
        details.append(dict(weight_t=int(t.sum()), Qf_size=len(Qf), LB=lb))
        best = lb if best is None else min(best, lb)
    return best, ts, details, stop

In [ ]:
# ---------------------------------------------------------- full resource tables (used by main + comparison)
def d4_resource_tables(D_LIST, dur=None, params=None, verbose=True):
    """Per-stage and summary tables of the Kobayashi protocol for every d in D_LIST, gate set
    ('primitive' | 'clifford_t'), counting variant (data_only), time model and comparison A/B.
    Returns (stage_df, sum_df, blocks)."""
    import pandas as pd, time
    dur = dur or DUR_UNIT
    params = params or DEFAULT_PARAMS
    records, summaries, blocks = [], [], {}
    for d in D_LIST:
        t0 = time.time()
        for gs in ['primitive', 'clifford_t']:
            cd = D4CostData(d, dur, decomposed=(gs == 'clifford_t'))
            for data_only in [False, True]:
                for model in ['FT_ref', 'NL_min']:
                    for comp in ['A', 'B']:
                        st = d4_stages(cd, model, params, comp, data_only=data_only)
                        for r in st:
                            r.update(gateset=gs, data_only=data_only)
                        records += st
                        s = summarize(st)
                        s.update(protocol='Kobayashi_D4', d=d, model=model, comparison=comp, gateset=gs,
                                 data_only=data_only)
                        summaries.append(s)
            blocks[f'{d}|{gs}'] = dict(d=d, gateset=gs, n=cd.n, n_total=cd.n_total, n_anc_g=cd.n_anc_g,
                                       round={k: {kk: (dict(vv) if isinstance(vv, Counter) else vv) for kk, vv in v.items()}
                                              for k, v in cd.round.items()},
                                       U_time=cd.U['time'], U_gates=dict(cd.U['gates']), out_time=cd.out['time'],
                                       out_gates=dict(cd.out['gates_worst']), xcorr_time=cd.x_corr['time'])
        if verbose:
            print(f'Kobayashi d={d}: {time.time() - t0:.1f}s')
    return jsonify_columns(pd.DataFrame(records)), jsonify_columns(pd.DataFrame(summaries)), blocks

### 雑音付き D4 code switching ライブラリ（回路・雑音モデル・Takada らの Pauli proxy 法による非 Clifford シミュレーション・復号器）

In [ ]:
"""noisy_d4_lib: circuit-level noisy simulation of the Kobayashi-Zhu-Hsin D4 code switching protocol.

Non-Clifford simulation method: Clifford-stabilizer simulation with Pauli proxies
(Y. Takada, S. D. Bartlett, D. J. Williamson, arXiv:2609.16929, Algorithm 2 (Sec. VII D) and Sec. VIII).
For every Monte Carlo shot the noisy non-Clifford circuit is mapped to a Clifford circuit (the
"Pauli-proxy circuit") with the same measurement statistics:
  * every diagonal third-level gate D (CCZ of the D4 gadgets, CS / CS^dag / T^dag of U) is removed;
  * the X component e of the sampled Pauli errors and of applied X corrections, propagated forward
    through the Clifford gates of the true circuit, is tracked ("continuing errors"); where e meets
    D the diagonal Clifford gate C_e = D^dag X^e D X^e (a product of S, Z, CZ) is inserted
    (= the controlled Pauli dressing Lambda(D_U(E)) of Takada Fig. 14 / Eq. (165));
  * Z components commute with every diagonal gate and need no propagation.
All Clifford gates, sampled Pauli errors and inserted dressings are executed by stim's TableauSimulator.

Requires (as globals, embedded before this code): numpy as np, stim, common_lib, d4_lib.
"""
import itertools
import numpy as np

In [ ]:
# ------------------------------------------------------------------ gate algebra
DIAG_THETA = {                     # phase polynomials (units of pi/4) of the diagonal 3rd-level gates
    'CCZ': lambda x: 4 * x[0] * x[1] * x[2],
    'NCCZ': lambda x: 4 * x[0] * x[1] * x[2],      # virtual noiseless CCZ (deferred switch-out correction)
    'CS': lambda x: 2 * x[0] * x[1],
    'CSDG': lambda x: 6 * x[0] * x[1],
    'T': lambda x: x[0],
    'TDG': lambda x: 7 * x[0],
}
NONCLIFF = set(DIAG_THETA)
_DRESS = {}


def diag_clifford_gates(delta, k):
    """delta: dict bits -> phase exponent mod 8 (even).  Returns gates ('S'|'Z'|'SDG', i), ('CZ', i, j)
    (local indices) equal to diag(omega^delta) up to a global phase; asserts the result is exact."""
    c0 = delta[(0,) * k]
    a, gates = [], []
    for i in range(k):
        e = tuple(int(j == i) for j in range(k))
        ai = (delta[e] - c0) % 8
        assert ai % 2 == 0, 'dressing is not Clifford'
        a.append(ai // 2)
        if ai // 2:
            gates.append(({1: 'S', 2: 'Z', 3: 'SDG'}[ai // 2], i))
    b = {}
    for i, j in itertools.combinations(range(k), 2):
        e = tuple(int(m in (i, j)) for m in range(k))
        v = (delta[e] - c0 - 2 * a[i] - 2 * a[j]) % 8
        assert v in (0, 4), 'dressing has a non-Clifford quadratic term'
        b[(i, j)] = v // 4
        if v:
            gates.append(('CZ', i, j))
    for x in itertools.product((0, 1), repeat=k):
        pred = c0 + sum(2 * a[i] * x[i] for i in range(k)) + sum(4 * b[ij] * x[ij[0]] * x[ij[1]] for ij in b)
        assert (pred - delta[x]) % 8 == 0, 'dressing has a cubic term'
    return gates


def dressing(name, e):
    """C_e = D^dag X^e D X^e for diagonal gate `name` and X pattern e (tuple of bits), local indices."""
    key = (name, e)
    if key not in _DRESS:
        th = DIAG_THETA[name]
        k = len(e)
        delta = {x: (th(tuple(xi ^ ei for xi, ei in zip(x, e))) - th(x)) % 8
                 for x in itertools.product((0, 1), repeat=k)}
        _DRESS[key] = diag_clifford_gates(delta, k)
    return _DRESS[key]

In [ ]:
# ------------------------------------------------------------------ noise model
class NoiseModel:
    """Standard circuit-level Pauli noise (Takada et al. App. A 2) plus optional idling noise:
    after every k-qubit unitary a k-qubit depolarizing channel (probability p; p_ccz for CCZ, p_nc for
    the gates of U), preparation followed by a flip (p_reset), measurement preceded by a flip (p_meas),
    every idle qubit of a time step depolarized with p_idle."""

    def __init__(self, p, p_meas=None, p_reset=None, p_idle=None, p_ccz=None, p_nc=None):
        self.p = p
        self.p_meas = p if p_meas is None else p_meas
        self.p_reset = p if p_reset is None else p_reset
        self.p_idle = p if p_idle is None else p_idle
        self.p_ccz = p if p_ccz is None else p_ccz
        self.p_nc = p if p_nc is None else p_nc

    def locations(self, ops, idle):
        """Noise locations of one time step: list of (when, kind, qubits, prob), when in {'pre','post'}."""
        loc = []
        for op in ops:
            nm, qs = op[0], op[1:]
            if nm in ('NCCZ', 'MZ0'):
                continue
            if nm == 'FLIPX':
                loc.append(('pre', 'X', qs, self.p_meas))
            elif nm == 'MZ':
                loc.append(('pre', 'X', qs, self.p_meas))
            elif nm == 'MX':
                loc.append(('pre', 'Z', qs, self.p_meas))
            elif nm == 'RZ':
                loc.append(('post', 'X', qs, self.p_reset))
            elif nm == 'RX':
                loc.append(('post', 'Z', qs, self.p_reset))
            elif nm == 'CCZ':
                loc.append(('post', 'D3', qs, self.p_ccz))
            elif nm in ('CS', 'CSDG', 'T', 'TDG'):
                loc.append(('post', 'D%d' % len(qs), qs, self.p_nc))
            else:
                loc.append(('post', 'D%d' % len(qs), qs, self.p))
        for q in idle:
            loc.append(('post', 'D1', (q,), self.p_idle))
        return [l for l in loc if l[3] > 0]


PAULI_CHOICES = {k: [''.join(p) for p in itertools.product('IXYZ', repeat=k) if set(p) != {'I'}]
                 for k in (1, 2, 3)}

In [ ]:
# ------------------------------------------------------------------ the noisy protocol circuit
class D4SwitchingCircuit:
    """Time-step level description of the noisy KZH protocol on the bounded square lattice.

    Blocks (in order):
      ('S0', r)  r = 1..R0   FSC |+_L> preparation: data |+>, rounds of all FSC checks
      ('S1', r)  r = 1..Rin  switch-in: g |0>, rounds of all D4 stabilizers ([KZH] step 2)
      ('U',)                 transversal U = W V ([KZH] eq. (8), step 3)
      ('SO',)                switch-out: Z^g readout ([KZH] step 4; deferred-correction form)
      ('S4', r)  r = 1..Rout FSC rounds after switching back ([KZH] step 4)
      ('FIN',)               one noiseless FSC round (fictitious error correction, Takada Sec. VI D)
    Qubits are re-indexed compactly: data (non-frozen r,g,b) first, then ancillas.
    sequential=True measures the gadgets of a commuting class one after another (validation only)."""

    def __init__(self, L, R0, Rin, Rout, sequential=False, n_pre_u=1):
        self.n_pre_u = n_pre_u
        P = D4Protocol(L)
        lat = P.lat
        self.P, self.lat, self.L = P, lat, L
        self.R0, self.Rin, self.Rout = R0, Rin, Rout
        data = sorted(P.data_qubits((R, G, B)))
        gadgets = {'D4': P.round_gadgets('D4'), 'FSC': P.round_gadgets('FSC')}
        ancs = sorted({g[0][1] for k in gadgets for g in gadgets[k]})
        c = {q: i for i, q in enumerate(data + ancs)}
        self.c = c
        self.nq, self.nd = len(data) + len(ancs), len(data)
        self.data_rb = sorted(c[q] for q in P.data_qubits((R, B)))
        self.data_g = sorted(c[q] for q in P.data_qubits((G,)))
        self.data_all = sorted(c[q] for q in data)
        conv = {'INIT_X': 'RX', 'INIT_Z': 'RZ', 'MEAS_X': 'MX', 'MEAS_Z': 'MZ', 'CNOT': 'CX'}
        cv = lambda g: (conv.get(g[0], g[0]),) + tuple(c[q] for q in g[1:])
        # label of every ancilla, per code
        self.lab_of = {}
        for code, cd in (('D4', P.d4), ('FSC', P.fsc)):
            labs = [l for l, _ in cd.xgens] + [l for l in cd.zlabels if not P._is_frozen_check(l)]
            for l in labs:
                self.lab_of[(code, c[P.anc[l]])] = l

        def round_moments(code, which='all'):
            gl = gadgets[code]
            if which != 'all':
                gl = [g for g in gl if g[0][0] == ('INIT_Z' if which == 'Z' else 'INIT_X')]
            classes = commuting_classes(gl)
            if sequential:
                classes = [[g] for cls in classes for g in cls]
            moms = []
            for cls in classes:
                for layer in class_layers(cls):
                    moms.append([cv(g) for g in layer])
            return moms
        # D4 rounds: Pauli Z checks first, then (after the real-time flux correction) the X-type
        # Clifford stabilizers, so that fluxes detected by the Z checks are removed before they meet CCZs
        self.round_moms = {'FSC': round_moments('FSC'), 'D4z': round_moments('D4', 'Z'),
                           'D4x': round_moments('D4', 'X')}
        self.anc = {code: sorted({c[g[0][1]] for g in gadgets[code]}) for code in gadgets}
        self.U_moms = [[cv(g) for g in layer] for layer in class_layers([P.U_gates()])]
        # ---- switch-out data: S^g generators, Gamma (g configuration -> kappa), CZ^{r,b} pairs
        self.gens_g = [(lab, op) for lab, op in P.d4.xgens if lab[2] == 'g']
        gpos = {q: i for i, q in enumerate(self.data_g)}
        Sg = np.zeros((len(self.gens_g), len(self.data_g)), dtype=np.uint8)
        for i, (lab, op) in enumerate(self.gens_g):
            for q in op.s:
                if q in c and c[q] in gpos:
                    Sg[i, gpos[c[q]]] = 1
        self.Sg = Sg
        self.Gamma = left_inverse_T(Sg)
        self.gpairs = [[(c[i], c[j]) for (i, j) in op.quad if i in c and j in c] for lab, op in self.gens_g]
        self.network = [('NCCZ', self.data_g[e], i, j)
                        for v in range(len(self.gens_g)) for e in np.nonzero(self.Gamma[v])[0]
                        for (i, j) in self.gpairs[v]]
        # Pauli-frame strings Z^r Z^b for the S^g eigenvalues (protocol step 5 of the noiseless check V7)
        self.frame_strings = []
        for lab, op in self.gens_g:
            x0, y0 = lab[1]
            qs = [lat.q(lat.eidx[('v', x0, y)], cc) for y in range(y0) for cc in (R, B)]
            self.frame_strings.append(sorted(c[q] for q in qs if q in c))
        # ---- Z checks of the FSC (rb data, frozen checks removed) -> used for the S1 X frame
        self.zchecks = {}
        for code, cd in (('D4', P.d4), ('FSC', P.fsc)):
            rows = []
            for lab, (l2, qs) in zip(cd.zlabels, lat.z_checks(cd.colours)):
                if P._is_frozen_check(lab):
                    continue
                rows.append((lab, sorted(c[q] for q in qs if q in c)))
            self.zchecks[code] = rows
        self.xgens = {}
        for code, cd in (('D4', P.d4), ('FSC', P.fsc)):
            self.xgens[code] = [(lab, sorted(c[q] for q in op.s if q in c),
                                 [(c[i], c[j]) for (i, j) in op.quad if i in c and j in c]) for lab, op in cd.xgens]
        self._logical()
        self.blocks = self._blocks()

    # ------------------------------------------------------------ logical Clifford stabilizer
    def _logical(self):
        """Ideal output = D_F |+_L>_FSC with F(a) = w(a) + 4 sum_v c'_v(a) f^g_v(a) (protocol phase,
        validated in V7/V7b/V8).  Its logical Clifford stabilizer is H = D_F X^t D_F^dag = X^t D_Delta,
        Delta(x) = F(x+t) - F(x) = c0 + sum 2 a_i x_i + sum 4 b_ij x_i x_j (mod 8)."""
        P, lat, c = self.P, self.lat, self.c
        F = {}

        def add(mono, coef):
            m = frozenset(mono)
            if any(q not in c for q in m):
                return                                   # frozen qubits are |0>
            m = frozenset(c[q] for q in m)
            F[m] = (F.get(m, 0) + coef) % 8
        for coef, mono in W_operator(lat).terms:
            add(mono, coef)
        # c'_v(a): path sums along a BFS tree from the L_rb vertices (linear in a)
        from collections import deque
        root = lrb_vertices(lat)
        path = {v: [] for v in root}
        dq = deque(root)
        while dq:
            u = dq.popleft()
            for e in lat.incident(u):
                p, q = lat.endpoints(e)
                w = q if p == u else p
                if w not in path:
                    path[w] = path[u] + [e]
                    dq.append(w)
        for v in P.gverts:
            lin = [lat.q(e, cc) for e in path[v] for cc in (R, B)]
            for (i, j) in lat.cz_pairs(v, G):
                for k in lin:
                    add({k, i, j}, 4)
        self.Fpoly = {m: v for m, v in F.items() if v}
        t = sorted(c[int(q)] for q in np.nonzero(P.fsc.t_log)[0])
        self.t_log = t
        tset = set(t)
        D = {}
        for m, coef in self.Fpoly.items():           # F(x+t): substitute x_i -> 1 - x_i for i in t
            fixed = [q for q in m if q not in tset]
            flip = [q for q in m if q in tset]
            for sub in itertools.product((0, 1), repeat=len(flip)):
                mono = frozenset(fixed + [q for q, s in zip(flip, sub) if s])
                sign = (-1) ** sum(sub)
                D[mono] = (D.get(mono, 0) + coef * sign) % 8
        for m, coef in self.Fpoly.items():
            D[m] = (D.get(m, 0) - coef) % 8
        D = {m: v for m, v in D.items() if v}
        self.Delta_c0 = D.pop(frozenset(), 0)
        self.Delta_a, self.Delta_b = {}, {}
        for m, v in D.items():
            if len(m) == 1:
                assert v % 2 == 0, 'logical stabilizer not Clifford (odd linear term)'
                self.Delta_a[next(iter(m))] = v // 2 % 4
            elif len(m) == 2:
                assert v % 4 == 0, 'logical stabilizer not Clifford (quadratic term)'
                self.Delta_b[tuple(sorted(m))] = v // 4 % 2
            else:
                raise AssertionError('logical stabilizer not Clifford (cubic term)')
        self.Delta_a = {q: a for q, a in self.Delta_a.items() if a}
        self.Delta_b = {ij: b for ij, b in self.Delta_b.items() if b}

    def logical_proxy(self, fx, sign=1):
        """stim.PauliString of the Pauli proxy of the dressed logical stabilizer X^e H X^e
        (e = X frame fx on the data): proxy(X^t D_Delta R_e) = X^t * prod_{a_i=2} Z_i * R_e, where
        R_e = D_Delta^dag X^e D_Delta X^e = phase * Z-string (Takada Secs. VI D, VII B)."""
        n = self.nq
        ps = stim.PauliString(n)
        for q in self.t_log:
            ps[q] = 'X'
        zs = stim.PauliString(n)
        for q, a in self.Delta_a.items():
            if a == 2:
                zs[q] = 'Z'
        R = stim.PauliString(n)
        ph = 1
        for q, a in self.Delta_a.items():
            if fx[q] and a % 2 == 1:
                ph *= 1j ** a
                R *= _single(n, q, 'Z')
            elif fx[q] and a == 2:
                ph *= -1
        for (i, j), b in self.Delta_b.items():
            if fx[i] and fx[j]:
                ph *= -1
                R *= _single(n, i, 'Z') * _single(n, j, 'Z')
            elif fx[i]:
                R *= _single(n, j, 'Z')
            elif fx[j]:
                R *= _single(n, i, 'Z')
        out = ps * zs * R
        return out * complex(ph * sign)

    # ------------------------------------------------------------ blocks
    def _blocks(self):
        B_ = []
        rb, g, alld = self.data_rb, self.data_g, self.data_all
        for r in range(1, self.R0 + 1):
            pre = [[('RX', q) for q in rb]] if r == 1 else []
            B_.append(dict(name=('S0', r), code='FSC', moms=pre + self.round_moms['FSC'], data=rb))
        for r in range(1, self.Rin + 1 + self.n_pre_u):
            pre = [[('RZ', q) for q in g]] if r == 1 else []
            B_.append(dict(name=('S1', r, 'z'), code='D4', moms=pre + self.round_moms['D4z'], data=alld))
            if r <= self.Rin:            # rounds Rin+1.. have only the Z checks (flux checks before U)
                B_.append(dict(name=('S1', r, 'x'), code='D4', moms=self.round_moms['D4x'], data=alld))
        B_.append(dict(name=('U',), code=None, moms=self.U_moms, data=alld))
        B_.append(dict(name=('SO',), code=None,
                       moms=[[('FLIPX', q) for q in g], list(self.network), [('MZ0', q) for q in g]], data=alld))
        for r in range(1, self.Rout + 1):
            B_.append(dict(name=('S4', r), code='FSC', moms=self.round_moms['FSC'], data=rb))
        B_.append(dict(name=('FIN',), code='FSC', moms=self.round_moms['FSC'], data=rb, noiseless=True))
        return B_


def _single(n, q, p):
    s = stim.PauliString(n)
    s[q] = p
    return s


def left_inverse_T(S):
    """Gamma (m x n) over F2 with Gamma @ S^T = I_m, for S (m x n) of full row rank."""
    m, n = S.shape
    A = (S.T % 2).astype(np.uint8)
    M = np.concatenate([A, np.eye(n, dtype=np.uint8)], axis=1)
    r = 0
    for col in range(m):
        nz = np.nonzero(M[r:, col])[0]
        assert len(nz), 'S^T is not of full column rank'
        p = r + nz[0]
        M[[r, p]] = M[[p, r]]
        for rr in range(n):
            if rr != r and M[rr, col]:
                M[rr] ^= M[r]
        r += 1
    Gm = M[:m, m:]
    assert not ((Gm.astype(int) @ A.astype(int)) % 2 - np.eye(m, dtype=int)).any()
    return Gm

In [ ]:
# ------------------------------------------------------------------ exact reference (validation only)
class TrueSim:
    """Exact sparse state-vector simulation of the TRUE (non-Clifford) circuit, used to validate the
    Pauli-proxy simulation.  Basis states are uint64 bit masks: data qubits 0..nd-1, ancillas get a
    slot nd+k while they are alive (reset ... measurement)."""
    OMEGA = np.exp(1j * np.pi / 4)

    def __init__(self, nd, keys, amps):
        self.nd = nd
        self.keys = np.asarray(keys, dtype=np.uint64)
        self.amps = np.asarray(amps, dtype=complex)
        self.slot = {}
        self.free = []
        self.nslots = 0

    def bit(self, q):
        if q < self.nd:
            return q
        if q not in self.slot:
            raise KeyError(q)
        return self.slot[q]

    def _alloc(self, q):
        if q < self.nd:
            return q
        if q in self.slot:
            return self.slot[q]
        b = self.free.pop() if self.free else self.nd + self.nslots
        if b == self.nd + self.nslots:
            self.nslots += 1
        self.slot[q] = b
        assert b < 64
        return b

    def _bits(self, b):
        return ((self.keys >> np.uint64(b)) & np.uint64(1)).astype(np.int64)

    def _merge(self):
        u, inv = np.unique(self.keys, return_inverse=True)
        re = np.bincount(inv, weights=self.amps.real, minlength=len(u))
        im = np.bincount(inv, weights=self.amps.imag, minlength=len(u))
        a = re + 1j * im
        keep = np.abs(a) > 1e-12
        self.keys, self.amps = u[keep], a[keep]

    def X(self, q):
        self.keys = self.keys ^ np.uint64(1 << self.bit(q))

    def Z(self, q):
        self.amps = self.amps * (1 - 2 * self._bits(self.bit(q)))

    def Y(self, q):
        self.Z(q)
        self.X(q)
        self.amps = self.amps * 1j

    def H(self, q):
        b = self.bit(q)
        m = np.uint64(1 << b)
        v = self._bits(b)
        k0 = self.keys & ~m
        a = self.amps / np.sqrt(2)
        self.keys = np.concatenate([k0, k0 | m])
        self.amps = np.concatenate([a, a * (1 - 2 * v)])
        self._merge()

    def diag(self, qs, theta):
        bits = [self._bits(self.bit(q)) for q in qs]
        ph = np.array([theta(t) for t in zip(*bits)], dtype=float) if len(self.keys) else np.zeros(0)
        self.amps = self.amps * self.OMEGA ** ph

    def CX(self, c, t):
        vc = self._bits(self.bit(c)).astype(np.uint64)
        self.keys = self.keys ^ (vc << np.uint64(self.bit(t)))

    def measure_z(self, q, forced):
        b = self.bit(q)
        v = self._bits(b)
        p1 = float(np.sum(np.abs(self.amps[v == 1]) ** 2))
        prob = p1 if forced else 1 - p1
        keep = v == forced
        self.keys, self.amps = self.keys[keep], self.amps[keep]
        if prob > 1e-12:
            self.amps = self.amps / np.sqrt(prob)
        return prob

    def release(self, q, outcome):
        if q >= self.nd:
            if outcome:
                self.X(q)
            self.free.append(self.slot.pop(q))

    def reset(self, q, basis, rng):
        b = self._alloc(q)
        if q < self.nd:                      # data reset: measure and flip to |0>
            v = self._bits(b)
            p1 = float(np.sum(np.abs(self.amps[v == 1]) ** 2))
            o = int(rng.random() < p1)
            self.measure_z(q, o)
            if o:
                self.X(q)
        if basis == 'X':
            self.H(q)

    def run_moment(self, mom, errs, outcomes, rng):
        """Execute a time step with the given sampled errors; measurements forced to `outcomes`
        (list in the order of the moment's measurements).  Returns list of probabilities."""
        for when, ps, qs in errs:
            if when == 'pre':
                for pc, q in zip(ps, qs):
                    if pc != 'I':
                        getattr(self, pc)(q)
        probs = []
        k = 0
        for op in mom:
            nm, qs = op[0], op[1:]
            if nm in DIAG_THETA:
                self.diag(qs, DIAG_THETA[nm])
            elif nm == 'CX':
                self.CX(*qs)
            elif nm == 'CZ':
                self.diag(qs, lambda x: 4 * x[0] * x[1])
            elif nm in ('S', 'SDG'):
                self.diag(qs, (lambda x: 2 * x[0]) if nm == 'S' else (lambda x: 6 * x[0]))
            elif nm in ('X', 'Z', 'Y'):
                getattr(self, nm)(qs[0])
            elif nm in ('RX', 'RZ'):
                self.reset(qs[0], nm[1], rng)
            elif nm in ('MX', 'MZ', 'MZ0'):
                q = qs[0]
                if nm == 'MX':
                    self.H(q)
                o = outcomes[k]
                k += 1
                probs.append(self.measure_z(q, o))
                if q >= self.nd:
                    self.release(q, o)
            elif nm == 'FLIPX':
                pass
            else:
                raise ValueError(nm)
        for when, ps, qs in errs:
            if when == 'post':
                for pc, q in zip(ps, qs):
                    if pc != 'I':
                        getattr(self, pc)(q)
        return probs

    def expect_logical(self, circ):
        """<Psi| X^t D_Delta |Psi> on the data qubits (all ancillas released)."""
        assert not self.slot
        x = self.keys
        ph = np.full(len(x), circ.Delta_c0, dtype=float)
        bits = {}
        get = lambda q: bits.setdefault(q, ((x >> np.uint64(q)) & np.uint64(1)).astype(np.int64))
        for q, a in circ.Delta_a.items():
            ph += 2 * a * get(q)
        for (i, j), b in circ.Delta_b.items():
            ph += 4 * b * get(i) * get(j)
        Da = self.amps * self.OMEGA ** ph
        tm = np.uint64(sum(1 << q for q in circ.t_log))
        k2 = x ^ tm
        order = np.argsort(x)
        xs = x[order]
        pos = np.searchsorted(xs, k2)
        pos[pos >= len(xs)] = 0
        hit = xs[pos] == k2
        amp_t = np.zeros(len(x), dtype=complex)
        amp_t[hit] = self.amps[order][pos[hit]]
        return complex(np.sum(np.conj(amp_t) * Da))

In [ ]:
# ------------------------------------------------------------------ compiled protocol (fast Monte Carlo)
_KIND = {'X': 0, 'Z': 1, 'D1': 2, 'D2': 3, 'D3': 4}


class CompiledMoment:
    __slots__ = ('ops', 'base', 'meas', 'cx_c', 'cx_t', 'resets', 'clears', 'nonc', 'loc_slice')


class CompiledProtocol:
    """Pre-compiled time steps of a D4SwitchingCircuit + flattened noise-location table."""

    def __init__(self, circ, noise):
        self.circ, self.noise = circ, noise
        locs = []
        self.blocks = []
        live = set()
        for blk in circ.blocks:
            moms = []
            for mom in blk['moms']:
                cm = CompiledMoment()
                cm.ops = mom
                lines, meas, cxc, cxt, rs, cl, nc = [], [], [], [], [], [], []
                for op in mom:
                    nm, qs = op[0], op[1:]
                    if nm in NONCLIFF:
                        nc.append((nm, qs))
                    elif nm == 'FLIPX':
                        pass
                    elif nm == 'CX':
                        lines.append('CX %d %d' % qs)
                        cxc.append(qs[0]); cxt.append(qs[1])
                    elif nm == 'CZ':
                        lines.append('CZ %d %d' % qs)
                    elif nm in ('RX', 'RZ'):
                        lines.append(('RX %d' if nm == 'RX' else 'R %d') % qs[0])
                        rs.append(qs[0])
                    elif nm in ('MX', 'MZ', 'MZ0'):
                        lines.append(('MX %d' if nm == 'MX' else 'M %d') % qs[0])
                        meas.append((qs[0], 'X' if nm == 'MX' else 'Z'))
                        if nm == 'MZ0':
                            cl.append(qs[0])
                    else:
                        lines.append('%s %d' % (_STIM1[nm], qs[0]))
                cm.base = stim.Circuit('\n'.join(lines)) if lines else None
                cm.meas = meas
                cm.cx_c, cm.cx_t = np.array(cxc, dtype=np.int64), np.array(cxt, dtype=np.int64)
                cm.resets, cm.clears = np.array(rs, dtype=np.int64), np.array(cl, dtype=np.int64)
                cm.nonc = nc
                touched = {q for op in mom for q in op[1:]}
                idle = [q for q in blk['data'] if q not in touched] + [q for q in live if q not in touched]
                ml = [] if blk.get('noiseless') else noise.locations(mom, idle)
                s0 = len(locs)
                locs += ml
                cm.loc_slice = (s0, len(locs))
                for op in mom:
                    if op[0] in ('RX', 'RZ') and op[1] >= circ.nd:
                        live.add(op[1])
                    if op[0] in ('MX', 'MZ'):
                        live.discard(op[1])
                moms.append(cm)
            self.blocks.append(dict(name=blk['name'], code=blk['code'], moms=moms))
        self.loc_when = np.array([0 if l[0] == 'pre' else 1 for l in locs], dtype=np.int8)
        self.loc_kind = np.array([_KIND[l[1]] for l in locs], dtype=np.int8)
        self.loc_p = np.array([l[3] for l in locs])
        self.loc_q = [l[2] for l in locs]
        self.n_loc = len(locs)


class FastShot:
    """One Monte Carlo shot of the Pauli-proxy circuit (same semantics as ProxyRun, compiled)."""
    _G1 = {'S': 's', 'SDG': 's_dag', 'Z': 'z'}

    def __init__(self, comp, rng, forced=None):
        self.comp, self.rng, self.circ = comp, rng, comp.circ
        self.sim = stim.TableauSimulator(seed=int(rng.integers(2 ** 62)))
        self.sim.set_num_qubits(comp.circ.nq)
        self.fx = np.zeros(comp.circ.nq, dtype=np.uint8)
        if forced is not None:                  # {location index: pauli string}
            self.errs = {i: (comp.loc_when[i], ps, comp.loc_q[i]) for i, ps in forced.items()}
            self.n_err = len(forced)
            self.n_dress = 0
            return
        hit = np.nonzero(rng.random(comp.n_loc) < comp.loc_p)[0]
        self.errs = {}
        for i in hit:
            k = comp.loc_kind[i]
            qs = comp.loc_q[i]
            if k == 0:
                ps = 'X'
            elif k == 1:
                ps = 'Z'
            else:
                ch = PAULI_CHOICES[len(qs)]
                ps = ch[rng.integers(len(ch))]
            self.errs[i] = (comp.loc_when[i], ps, qs)
        self.n_err = len(hit)
        self.n_dress = 0

    def _pauli(self, ps, qs):
        sim, fx = self.sim, self.fx
        for pc, q in zip(ps, qs):
            if pc == 'X':
                sim.x(q); fx[q] ^= 1
            elif pc == 'Y':
                sim.y(q); fx[q] ^= 1
            elif pc == 'Z':
                sim.z(q)

    def run_block(self, bi):
        """Run block bi; returns list of (qubit, basis, outcome) of its measurements."""
        blk = self.comp.blocks[bi]
        sim, fx = self.sim, self.fx
        nmeas = 0
        meas = []
        for cm in blk['moms']:
            a, b = cm.loc_slice
            errs = [self.errs[i] for i in range(a, b) if i in self.errs] if b > a and self.errs else []
            for w, ps, qs in errs:
                if w == 0:
                    self._pauli(ps, qs)
            for nm, qs in cm.nonc:
                e = tuple(int(fx[q]) for q in qs)
                if any(e):
                    for gt in dressing(nm, e):
                        if gt[0] == 'CZ':
                            sim.cz(qs[gt[1]], qs[gt[2]])
                        else:
                            getattr(sim, self._G1[gt[0]])(qs[gt[1]])
                        self.n_dress += 1
            if cm.base is not None:
                sim.do_circuit(cm.base)
            if len(cm.cx_c):
                fx[cm.cx_t] ^= fx[cm.cx_c]
            if len(cm.resets):
                fx[cm.resets] = 0
            if len(cm.clears):
                fx[cm.clears] = 0
            for w, ps, qs in errs:
                if w == 1:
                    self._pauli(ps, qs)
            nmeas += len(cm.meas)
            meas += cm.meas
        if not nmeas:
            return []
        rec = sim.current_measurement_record()[-nmeas:]
        return [(q, bb, int(o)) for (q, bb), o in zip(meas, rec)]

    def apply_x(self, qubits):
        for q in qubits:
            self.sim.x(q)
            self.fx[q] ^= 1

    def init_frame(self, zrows, qubits):
        """X frame at the start of the first non-Clifford block (S1).  S0 is a Clifford circuit, so stim's
        state is the true state E|ideal>, |ideal> = FSC|+_L> (all checks +1); the X part of E is any
        solution x of H_Z x = s (s = current Z-check signs): solutions differ by X stabilizers / logical X
        of |ideal>, which act trivially on it, and Algorithm 2 is exact for any such representation."""
        s = []
        for lab, qs in zrows:
            ps = stim.PauliString(self.circ.nq)
            for q in qs:
                ps[q] = 'Z'
            v = self.sim.peek_observable_expectation(ps)
            assert v in (1, -1), 'Z check not deterministic at the end of S0'
            s.append(0 if v == 1 else 1)
        H = np.zeros((len(zrows), len(qubits)), dtype=np.uint8)
        pos = {q: i for i, q in enumerate(qubits)}
        for r, (lab, qs) in enumerate(zrows):
            for q in qs:
                H[r, pos[q]] = 1
        x = f2_solve(H.T, np.array(s, dtype=np.uint8))      # x H^T = s  <=>  H x = s
        assert x is not None
        self.fx[:] = 0
        for i in np.nonzero(x)[0]:
            self.fx[qubits[i]] = 1
        return np.array(s)

    _G = {'CX': 'cx', 'CZ': 'cz', 'S': 's', 'SDG': 's_dag', 'Z': 'z', 'X': 'x', 'Y': 'y'}

    def run_block_lockstep(self, bi, ts, rng):
        """Validation: same as run_block, but op by op, with the exact simulation `ts` (TrueSim) of the
        TRUE non-Clifford circuit run in lockstep with the same errors.  Before every measurement stim's
        peek value (+1/-1 deterministic, 0 random) is compared with the exact probability of the outcome.
        Returns list of (qubit, basis, outcome, peek, exact probability)."""
        blk = self.comp.blocks[bi]
        sim, fx = self.sim, self.fx
        out = []
        for cm in blk['moms']:
            a, b = cm.loc_slice
            errs = [self.errs[i] for i in range(a, b) if i in self.errs]
            terrs = [('pre' if w == 0 else 'post', ps, qs) for w, ps, qs in errs]
            for w, ps, qs in errs:
                if w == 0:
                    self._pauli(ps, qs)
            for nm, qs in cm.nonc:
                e = tuple(int(fx[q]) for q in qs)
                if any(e):
                    for gt in dressing(nm, e):
                        getattr(sim, self._G[gt[0]])(*[qs[i] for i in gt[1:]])
                        self.n_dress += 1
            outs, peeks = [], []
            for op in cm.ops:
                nm, qs = op[0], op[1:]
                if nm in NONCLIFF or nm == 'FLIPX':
                    continue
                if nm in ('RX', 'RZ'):
                    (sim.reset_x if nm == 'RX' else sim.reset_z)(qs[0])
                elif nm in ('MX', 'MZ', 'MZ0'):
                    q = qs[0]
                    if nm == 'MX':
                        peeks.append(sim.peek_x(q)); sim.h(q); o = sim.measure(q); sim.h(q)
                    else:
                        peeks.append(sim.peek_z(q)); o = sim.measure(q)
                    outs.append(int(o))
                else:
                    getattr(sim, self._G[nm])(*qs)
            if len(cm.cx_c):
                fx[cm.cx_t] ^= fx[cm.cx_c]
            if len(cm.resets):
                fx[cm.resets] = 0
            if len(cm.clears):
                fx[cm.clears] = 0
            for w, ps, qs in errs:
                if w == 1:
                    self._pauli(ps, qs)
            probs = ts.run_moment(cm.ops, terrs, outs, rng)
            out += [(q, bb, o, pk, pr) for (q, bb), o, pk, pr in zip(cm.meas, outs, peeks, probs)]
        return out

In [ ]:
# ------------------------------------------------------------------ decoders
def _wt(p):
    p = min(max(p, 1e-12), 0.5 - 1e-9)
    return float(np.log((1 - p) / p))


_XFRAC = {0: 1.0, 1: 0.0, 2: 2 / 3, 3: 8 / 15, 4: 32 / 63}       # prob. that a location puts X/Y on a qubit


class FluxDecoder:
    """Real-time ("just-in-time") decoding of the Pauli Z checks (non-Abelian fluxes = X errors).

    [KZH] step 2 pairs the fluxes "in real time ... whenever the observation time of the syndromes is
    comparable to the separation of the syndromes" with the decoder of [Dav] (not accessible here).
    Implementation used in this study (own construction, documented as such): after every round t the
    space-time matching graph of all rounds <= t (with an open time boundary after round t) is decoded
    by minimum-weight perfect matching (PyMatching); the space-like edges whose round is <= t - delay
    are committed and the corresponding X corrections are applied physically at once (feed-forward).
    Before the first non-Clifford block (end of S0) and before U (end of S1) everything is committed.
    The g readout of the switch-out is decoded together with the g fluxes of S1 and gives the
    corrected configuration used for kappa."""

    def __init__(self, comp, delay=0, gauge_w=0.05):
        circ = comp.circ
        self.comp, self.circ, self.delay, self.gauge_w = comp, circ, delay, gauge_w
        zr = {code: circ.zchecks[code] for code in ('D4', 'FSC')}
        g_rows = [(lab, qs) for lab, qs in zr['D4'] if lab[0] == 'plaq' and lab[2] == 'g']
        # flux rounds
        self.fr = []
        for bi, blk in enumerate(comp.blocks):
            nm = blk['name']
            if nm[0] in ('S0', 'S4', 'FIN') or (nm[0] == 'S1' and nm[2] == 'z'):
                self.fr.append(dict(block=bi, name=nm, rows=zr[blk['code']]))
            elif nm[0] == 'SO':
                self.fr.append(dict(block=bi, name=nm, rows=g_rows))
        off = 0
        for k, f in enumerate(self.fr):
            f['off'] = off
            f['idx'] = {lab: off + i for i, (lab, qs) in enumerate(f['rows'])}
            off += len(f['rows'])
        self.n_nodes = off
        # relation for the SO detectors: prod_{q in g-plaq} Z^g Z^r Z^b (pre-V) = sum of D4 rows
        lat, c = circ.lat, circ.c
        inv = {v: k for k, v in c.items()}
        d4rows = zr['D4']
        H = np.zeros((len(d4rows), circ.nq), dtype=np.uint8)
        for i, (lab, qs) in enumerate(d4rows):
            H[i, qs] = 1
        self.so_rel = {}
        for lab, qs in g_rows:
            v = np.zeros(circ.nq, dtype=np.uint8)
            for q in qs:
                e = inv[q] // 3
                for cc in (R, G, B):
                    raw = lat.q(e, cc)
                    if raw in c:
                        v[c[raw]] ^= 1
            sol = f2_solve(H, v)
            assert sol is not None, 'SO relation not in the D4 Z-check span'
            self.so_rel[lab] = [d4rows[i][0] for i in np.nonzero(sol)[0]]
        # rows containing each qubit, per code
        self.rows_of = {code: {} for code in ('D4', 'FSC')}
        for code in ('D4', 'FSC'):
            for lab, qs in zr[code]:
                for q in qs:
                    self.rows_of[code].setdefault(q, []).append(lab)
        self._error_rates()
        self._build()

    def _error_rates(self):
        """Phenomenological edge probabilities estimated from the circuit-level noise locations."""
        comp = self.comp
        self.p_data = []           # per flux round: dict qubit -> prob of X/Y since the previous flux round
        self.p_meas = []           # per flux round: dict label -> prob that the check outcome is flipped
        prev_block = -1
        for f in self.fr:
            blk = comp.blocks[f['block']]
            pd, pa = {}, {}
            for bj in range(prev_block + 1, f['block'] + 1):
                for cm in comp.blocks[bj]['moms']:
                    a, b = cm.loc_slice
                    for i in range(a, b):
                        fr_ = _XFRAC[comp.loc_kind[i]] * comp.loc_p[i]
                        for q in comp.loc_q[i]:
                            if q < self.circ.nd:
                                pd[q] = pd.get(q, 0) + fr_
                            elif bj == f['block']:
                                pa[q] = pa.get(q, 0) + fr_
            prev_block = f['block']
            pm = {}
            if blk['code']:
                for lab, qs in f['rows']:
                    a = self.circ.P.anc[lab]
                    pm[lab] = pa.get(self.circ.c[a], 0)
            self.p_data.append(pd)
            self.p_meas.append(pm)

    def _build(self):
        """Matching graphs M_t for t = 0..len(fr)-1 (rounds <= t, open boundary after t)."""
        self.mt = []
        nfr = len(self.fr)
        for t in range(nfr):
            m = pymatching.Matching()
            look = {}

            def add(u, v, w, info):
                key = (u, v) if v == -1 else (min(u, v), max(u, v))
                if key in look and look[key][0] <= w:
                    return
                look[key] = (w, info)
            for k in range(t + 1):
                f = self.fr[k]
                code = self.comp.blocks[f['block']]['code'] or 'D4'
                rows = f['rows']
                rowset = {lab for lab, _ in rows}
                # space edges: X on a data qubit before round k
                qrows = {}
                for lab, qs in rows:
                    for q in qs:
                        qrows.setdefault(q, []).append(f['idx'][lab])
                for q, nodes in qrows.items():
                    if k == 0:
                        w = self.gauge_w           # random first-round outcomes of S0 (gauge choice)
                    else:
                        w = _wt(self.p_data[k].get(q, 1e-6))
                    if len(nodes) == 2:
                        add(nodes[0], nodes[1], w, ('s', q, k))
                    elif len(nodes) == 1:
                        add(nodes[0], -1, w, ('s', q, k))
                # time edges to the previous occurrence of the same label
                if k > 0:
                    for lab, qs in rows:
                        prev = self._prev(k, lab)
                        if prev is not None:
                            kp, node_p = prev
                            w = _wt(self.p_meas[kp].get(lab, 1e-6))
                            add(node_p, f['idx'][lab], w, ('t', lab, kp))
            # switch-out round: a measurement error of a check of the last D4 Z round enters the reference
            # of the g-readout detectors (hyper-edge with its next detector); represent it by an edge between
            # the two nodes known at this time and fix the stored reference when it is committed ('m')
            if self.fr[t]['name'][0] == 'SO':
                f = self.fr[t]
                kp = t - 1
                for lab, qs in f['rows']:
                    for l2 in self.so_rel[lab]:
                        if l2 != lab and l2 in self.fr[kp]['idx']:
                            add(self.fr[kp]['idx'][l2], f['idx'][lab], _wt(self.p_meas[kp].get(l2, 1e-6)),
                                ('m', l2, kp))
            # open time boundary: nodes whose label is measured again after t
            for k in range(t + 1):
                f = self.fr[k]
                for lab, qs in f['rows']:
                    if self._next(k, lab) is not None and self._next(k, lab) > t and self._latest(k, lab, t):
                        add(f['idx'][lab], -1, _wt(self.p_meas[k].get(lab, 1e-6) + 1e-9), ('f', lab, k))
            for (u, v), (w, info) in look.items():
                if v == -1:
                    m.add_boundary_edge(u, weight=w)
                else:
                    m.add_edge(u, v, weight=w)
            nn = self.fr[t]['off'] + len(self.fr[t]['rows'])
            self.mt.append((m, {k_: v[1] for k_, v in look.items()}, nn))

    def _prev(self, k, lab):
        for kp in range(k - 1, -1, -1):
            if lab in self.fr[kp]['idx']:
                return kp, self.fr[kp]['idx'][lab]
        return None

    def _next(self, k, lab):
        for kn in range(k + 1, len(self.fr)):
            if lab in self.fr[kn]['idx']:
                return kn
        return None

    def _latest(self, k, lab, t):
        """True if round k holds the latest occurrence of lab among rounds <= t."""
        for kk in range(k + 1, t + 1):
            if lab in self.fr[kk]['idx']:
                return False
        return True

    # ---------------------------------------------------------------- per shot
    def start(self):
        self.last = {}
        self.active = set()
        self.k = -1

    def push_round(self, k, outcomes_by_label=None, readout=None):
        """Add the detection events of flux round k.  outcomes_by_label: dict label -> bit (Z checks);
        readout: dict g-qubit -> bit (switch-out)."""
        f = self.fr[k]
        self.k = k
        if f['name'][0] == 'SO':
            par = {}
            for lab, qs in f['rows']:
                v = 0
                for q in qs:
                    v ^= readout[q]
                ref = 0
                for l2 in self.so_rel[lab]:
                    ref ^= self.last.get(l2, 0)
                par[lab] = v
                if v ^ ref:
                    self.active ^= {f['idx'][lab]}
            self.readout = dict(readout)
            return
        for lab, qs in f['rows']:
            m = outcomes_by_label[lab]
            if m ^ self.last.get(lab, 0):
                self.active ^= {f['idx'][lab]}
            self.last[lab] = m

    def decode(self, flush=False, min_age=None):
        """Decode with all rounds <= current; returns list of committed space corrections (qubit, k).
        Commits space-like edges with round k <= t - delay (flush: k <= t - min_age, default all)."""
        t = self.k
        m, look, nn = self.mt[t]
        if not self.active:
            return []
        syn = np.zeros(m.num_detectors, dtype=np.uint8)
        for u in self.active:
            syn[u] = 1
        edges = m.decode_to_edges_array(syn)
        out = []
        for u, v in edges:
            u, v = int(u), int(v)
            key = (u, -1) if v == -1 else ((v, -1) if u == -1 else (min(u, v), max(u, v)))
            info = look[key]
            if info[0] == 'm' and flush:
                self.active ^= {key[0], key[1]}
                self.last[info[1]] = self.last.get(info[1], 0) ^ 1     # it was a measurement error
                continue
            if info[0] != 's':
                continue
            q, k = info[1], info[2]
            lim = (t - (min_age or 0)) if flush else (t - self.delay)
            if k > lim:
                continue
            out.append((q, k))
            if key[1] == -1:
                self.active ^= {key[0]}
            else:
                self.active ^= {key[0], key[1]}
        return out

    def register_correction(self, q, code):
        """A physical X correction on data qubit q: flip the stored reference of every Z check of the
        current code containing q (as if it had been applied before that check was measured)."""
        for lab in self.rows_of[code].get(q, []):
            if lab in self.last:
                self.last[lab] ^= 1


class ChargeDecoder:
    """Decoding of the X-type checks (Abelian charges = Z errors) after the protocol: minimum-weight
    matching on the detector error model of the Pauli-proxy circuit (stim + PyMatching).  The DEM
    describes the Pauli errors of the circuit; the extra randomness produced by non-Abelian dressings
    is not in the model (it is present in the simulated data).  Output: predicted flip of the logical
    observable (proxy of the logical Clifford stabilizer, including the Pauli-frame strings)."""

    def __init__(self, comp, dem_noise=None, aligned=True):
        circ = comp.circ
        self.comp, self.circ = comp, circ
        nz = dem_noise or comp.noise
        # measurement bookkeeping
        meas_lab = []          # per absolute measurement: (block_name, kind, label or qubit)
        blk_meas = []
        for bi, blk in enumerate(comp.blocks):
            ids = []
            for cm in blk['moms']:
                for q, b in cm.meas:
                    if blk['code']:
                        lab = circ.lab_of[(blk['code'], q)]
                        meas_lab.append((blk['name'], b, lab))
                    else:
                        meas_lab.append((blk['name'], 'R', q))
                    ids.append(len(meas_lab) - 1)
            blk_meas.append(ids)
        self.n_meas = len(meas_lab)
        self.blk_meas = blk_meas
        idx = {}
        for i, (bn, b, lab) in enumerate(meas_lab):
            idx[(bn, lab)] = i
        self.meas_idx = idx
        sup = {code: {lab: set(xs) for lab, xs, quad in circ.xgens[code]} for code in ('D4', 'FSC')}
        names = [blk['name'] for blk in comp.blocks]
        S0 = [n for n in names if n[0] == 'S0']
        S1 = [n for n in names if n[0] == 'S1' and n[2] == 'x']
        S4 = [n for n in names if n[0] in ('S4', 'FIN')]
        dets = []
        fscl = [lab for lab, _, _ in circ.xgens['FSC']]
        d4l = [lab for lab, _, _ in circ.xgens['D4']]
        for j, n in enumerate(S0):
            for lab in fscl:
                dets.append([idx[(n, lab)]] + ([idx[(S0[j - 1], lab)]] if j else []))

        def span_solve(target, basis_labels, code):
            M = np.zeros((len(basis_labels), circ.nq), dtype=np.uint8)
            for i, l in enumerate(basis_labels):
                M[i, list(sup[code][l])] = 1
            v = np.zeros(circ.nq, dtype=np.uint8)
            v[list(target)] = 1
            sol = f2_solve(M, v)
            return None if sol is None else [basis_labels[i] for i in np.nonzero(sol)[0]]
        for j, n in enumerate(S1):
            for lab in d4l:
                if j == 0:
                    if not S0:
                        continue
                    combo = span_solve(sup['D4'][lab], fscl, 'FSC')
                    if combo is None:
                        continue                       # S^g: random first outcome (gauging)
                    dets.append([idx[(n, lab)]] + [idx[(S0[-1], l)] for l in combo])
                else:
                    dets.append([idx[(n, lab)], idx[(S1[j - 1], lab)]])
        lat, c = circ.lat, circ.c
        inv = {v: k for k, v in c.items()}
        for j, n in enumerate(S4):
            first = {}
            for lab in fscl:
                if j == 0:
                    tgt = set(sup['FSC'][lab])
                    for q in list(sup['FSC'][lab]):
                        gq = lat.q(inv[q] // 3, G)
                        if gq in c:
                            tgt ^= {c[gq]}
                    combo = span_solve(tgt, d4l, 'D4')
                    assert combo is not None, 'S4 X check not determined by the last D4 round'
                    first[lab] = [idx[(n, lab)]] + [idx[(S1[-1], l)] for l in combo]
                else:
                    dets.append([idx[(n, lab)], idx[(S4[j - 1], lab)]])
            if j == 0:
                # detector basis across the switch-out: aligned=True uses, at a vertex with separate r and
                # b checks, D_r and D_r + D_b (S^g of the last D4 round cancels in D_r + D_b), so that a g
                # charge created during the last D4 round is a graph-like error, but an r charge then has
                # 4 symptoms; aligned=False (D_r, D_b) has the opposite property.  D4Experiment decodes
                # with both and keeps the lower-weight solution (no single basis makes all graph-like)
                for lab, d in first.items():
                    v, col = lab[1], lab[2]
                    if aligned and col == 'b' and ('X', v, 'r') in first:
                        dets.append(sorted(set(d) ^ set(first[('X', v, 'r')])))
                    else:
                        dets.append(d)
        self.dets = dets
        # logical observable: proxy logical (measured by MPP at the end of the DEM circuit) + frame terms
        tset = set(circ.t_log)
        self.cross = [len(set(fs) & tset) % 2 for fs in circ.frame_strings]
        self.obs_meas = [idx[(S1[-1], lab)] for (lab, op), cr in zip(circ.gens_g, self.cross) if cr]
        self._dem_circuit(nz)
        # sparse detector evaluation
        self.det_ptr = np.cumsum([0] + [len(d) for d in dets])
        self.det_idx = np.concatenate([np.array(d, dtype=np.int64) for d in dets])

    def _dem_circuit(self, nz):
        circ, comp = self.circ, self.comp
        lines = []
        for bi, blk in enumerate(comp.blocks):
            noiseless = blk['name'][0] == 'FIN'
            for cm in blk['moms']:
                pre, post = [], []
                touched = {q for op in cm.ops for q in op[1:]}
                a, b = cm.loc_slice
                for i in range(a, b):
                    k, p, qs = comp.loc_kind[i], comp.loc_p[i], comp.loc_q[i]
                    if k == 0:
                        ln = 'X_ERROR(%.12g) %d' % (p, qs[0])
                    elif k == 1:
                        ln = 'Z_ERROR(%.12g) %d' % (p, qs[0])
                    elif k == 2:
                        ln = 'DEPOLARIZE1(%.12g) %d' % (p, qs[0])
                    elif k == 3:
                        ln = 'DEPOLARIZE2(%.12g) %d %d' % (p, qs[0], qs[1])
                    else:
                        ln = 'DEPOLARIZE1(%.12g) %d %d %d' % (p * 48 / 63, *qs)
                    (pre if comp.loc_when[i] == 0 else post).append(ln)
                lines += pre
                if cm.base is not None:
                    lines.append(str(cm.base))
                lines += post
            lines.append('TICK')
        n = circ.nq
        lp = ['X%d' % q for q in circ.t_log if circ.Delta_a.get(q) != 2]
        lp += ['Y%d' % q for q in circ.t_log if circ.Delta_a.get(q) == 2]
        lp += ['Z%d' % q for q, a in circ.Delta_a.items() if a == 2 and q not in set(circ.t_log)]
        lines.append('MPP ' + '*'.join(lp))
        tot = self.n_meas + 1
        for d in self.dets:
            lines.append('DETECTOR ' + ' '.join('rec[%d]' % (i - tot) for i in d))
        lines.append('OBSERVABLE_INCLUDE(0) rec[-1] ' + ' '.join('rec[%d]' % (i - tot) for i in self.obs_meas))
        self.stim_circuit = stim.Circuit('\n'.join(lines))
        dem = self.stim_circuit.detector_error_model(decompose_errors=True, ignore_decomposition_failures=True,
                                                     approximate_disjoint_errors=True)
        self.matching = pymatching.Matching.from_detector_error_model(dem)
        # graph-like components of the DEM (for the flux-informed re-weighting)
        comp = {}
        for ins in dem.flattened():
            if ins.type != 'error':
                continue
            p = ins.args_copy()[0]
            cur_d, cur_o = [], 0
            parts = []
            for t in ins.targets_copy():
                if t.is_separator():
                    parts.append((tuple(sorted(cur_d)), cur_o)); cur_d, cur_o = [], 0
                elif t.is_relative_detector_id():
                    cur_d.append(t.val)
                elif t.is_logical_observable_id():
                    cur_o ^= 1
            parts.append((tuple(sorted(cur_d)), cur_o))
            for key in parts:
                if not key[0] or len(key[0]) > 2:
                    continue
                q0 = comp.get(key, 0.0)
                comp[key] = q0 * (1 - p) + p * (1 - q0)
        keys = list(comp)
        self.comp_keys = keys
        self.comp_index = {k[0]: i for i, k in enumerate(keys)}
        from scipy.sparse import csc_matrix
        rows, cols = [], []
        for j, (dd, o) in enumerate(keys):
            for dtt in dd:
                rows.append(dtt); cols.append(j)
        nd = self.stim_circuit.num_detectors
        self.H = csc_matrix((np.ones(len(rows), dtype=np.uint8), (rows, cols)), shape=(nd, len(keys)))
        self.O = csc_matrix((np.ones(sum(o for _, o in keys), dtype=np.uint8),
                             ([0] * sum(o for _, o in keys), [j for j, (dd, o) in enumerate(keys) if o])),
                            shape=(1, len(keys)))
        self.w0 = np.array([_wt(comp[k]) for k in keys])
        # detectors containing each measurement
        self.dets_of_meas = {}
        for j, d_ in enumerate(self.dets):
            for i in d_:
                self.dets_of_meas.setdefault(i, []).append(j)

    def detectors(self, outcomes):
        o = np.asarray(outcomes, dtype=np.uint8)[self.det_idx]
        return np.bitwise_xor.reduceat(o, self.det_ptr[:-1]) if len(o) else np.zeros(0, np.uint8)

    def flux_erasures(self, flux_events, w_erase=0.1):
        """Flux-informed re-weighting (our construction, following [KZH]'s description of cleaning up the
        Abelian charges after the fluxes have been paired): a flux on qubit q present during the X-type
        round ('S1', r, 'x') randomizes the outcomes of the D4 stabilizers whose CZ dressing contains q and
        leaves Z errors on the CZ partners with probability 1/2.  The corresponding time-like components
        and the space-like components of the partner Z errors get the weight w_erase."""
        circ = self.circ
        if not hasattr(self, '_cz_of'):
            self._cz_of = {}
            for lab, xs, quad in circ.xgens['D4']:
                for (i, j) in quad:
                    self._cz_of.setdefault(i, []).append((lab, j))
                    self._cz_of.setdefault(j, []).append((lab, i))
            self._stars = {}
            for lab, xs, quad in circ.xgens['D4']:
                for q in xs:
                    self._stars.setdefault(q, []).append(lab)
        er = set()
        for q, r in flux_events:
            blk = ('S1', r, 'x')
            nxt = ('S1', r + 1, 'x')
            for lab, partner in self._cz_of.get(q, []):
                m = self.meas_idx.get((blk, lab))
                if m is None:
                    continue
                dd = tuple(sorted(self.dets_of_meas.get(m, [])))
                if 1 <= len(dd) <= 2 and dd in self.comp_index:
                    er.add(self.comp_index[dd])
                # space-like component: Z on the partner flips the two stars containing it
                st = self._stars.get(partner, [])
                for rr in (blk, nxt):
                    ds = []
                    for l2 in st:
                        m2 = self.meas_idx.get((rr, l2))
                        if m2 is not None:
                            ds += [j for j in self.dets_of_meas.get(m2, []) if self.dets[j][0] == m2]
                    dd = tuple(sorted(ds))
                    if 1 <= len(dd) <= 2 and dd in self.comp_index:
                        er.add(self.comp_index[dd])
        return er

    def predict(self, outcomes, erasures=None, w_erase=0.1):
        """Returns (predicted flip of the logical observable, weight of the matching)."""
        det = self.detectors(outcomes)
        if not erasures:
            pred, w = self.matching.decode(det, return_weight=True)
            return int(pred[0]), float(w)
        w = self.w0.copy()
        w[list(erasures)] = w_erase
        m = pymatching.Matching.from_check_matrix(self.H, weights=w, faults_matrix=self.O)
        pred, wt = m.decode(det, return_weight=True)
        return int(pred[0]), float(wt)


class D4Experiment:
    """Noisy KZH protocol + decoders.  run(shots) returns per-shot logical infidelity samples
    (0, 1/2 or 1; Takada Sec. VI D with the infidelity (1 - <H>)/2 of the output) and diagnostics."""

    def __init__(self, L, R0, Rin, Rout, noise, delay=0, pre_u_age=0, n_pre_u=1, flux_informed=True,
                 quiet_window=False):
        self.pre_u_age = pre_u_age
        self.flux_informed = flux_informed
        self.circ = D4SwitchingCircuit(L, R0, Rin, Rout, n_pre_u=n_pre_u)
        self.comp = CompiledProtocol(self.circ, noise)
        self.noise = noise
        self.flux = FluxDecoder(self.comp, delay=delay)
        self.charges = [ChargeDecoder(self.comp, aligned=a) for a in (True, False)]
        # diagnostic (not a physical model): no noise in the switching window = the last D4 round, the Z
        # rounds before U, U, the switch-out and the first FSC round, where the single faults that our
        # decoders fail to correct are located (validation N5).  Gives an optimistic bound.  The decoders keep
        # the priors of the full noise model (they are built before the window is silenced).
        self.quiet_window = quiet_window
        if quiet_window:
            win = {('S1', Rin, 'z'), ('S1', Rin, 'x'), ('U',), ('SO',), ('S4', 1)}
            win |= {('S1', r, 'z') for r in range(Rin + 1, Rin + 1 + n_pre_u)}
            for blk in self.comp.blocks:
                if blk['name'] in win:
                    a, b = blk['moms'][0].loc_slice[0], blk['moms'][-1].loc_slice[1]
                    self.comp.loc_p[a:b] = 0.0
        self.gset = set(self.circ.data_g)

    def shot(self, rng, decode=True, forced=None):
        circ, comp, fl = self.circ, self.comp, self.flux
        sh = FastShot(comp, rng, forced)
        fl.start()
        outs = []
        kfr = 0
        m_g_last = None
        flux_events = []
        stats = dict(n_err=sh.n_err, n_corr=0)
        names = [b['name'] for b in comp.blocks]
        last_S0 = max([i for i, n in enumerate(names) if n[0] == 'S0'], default=-1)
        last_S1 = max(i for i, n in enumerate(names) if n[0] == 'S1')          # Z-only round before U
        last_S1x = max(i for i, n in enumerate(names) if n[0] == 'S1' and n[2] == 'x')
        for bi, blk in enumerate(comp.blocks):
            nm = blk['name']
            if nm == ('S1', 1, 'z'):
                sh.init_frame(circ.zchecks['FSC'], circ.data_rb)
            res = sh.run_block(bi)
            outs += [o for (_, _, o) in res]
            if bi == last_S1x:
                gl = {circ.lab_of[('D4', q)]: o for (q, b, o) in res if b == 'X'}
                m_g_last = [gl[lab] for lab, _ in circ.gens_g]
            if nm[0] in ('S0', 'S4', 'FIN') or (nm[0] == 'S1' and nm[2] == 'z'):
                code = blk['code']
                zl = {circ.lab_of[(code, q)]: o for (q, b, o) in res if b == 'Z'}
                fl.push_round(kfr, zl)
                kfr += 1
                if decode:
                    flush = bi in (last_S0, last_S1) or nm[0] == 'FIN'
                    # before U: corrections supported only by the newest round stay undecided (decoded
                    # after the switch-out with the S4 / g-readout data)
                    cc = fl.decode(flush=flush, min_age=self.pre_u_age if bi == last_S1 else 0)
                    corr = [q for q, k in cc]
                    if nm[0] == 'S1':
                        # the flux was present during the X-type round preceding this Z round
                        flux_events += [(q, nm[1] - 1) for q in corr if nm[1] >= 2]
                    sh.apply_x(corr)
                    for q in corr:
                        fl.register_correction(q, code)
                    stats['n_corr'] += len(corr)
            elif nm[0] == 'SO':
                readout = {q: o for (q, b, o) in res}
                fl.push_round(kfr, readout=readout)
                kfr += 1
                a_dec = dict(readout)
                if decode:
                    phys = []
                    for q, k in fl.decode(flush=True):
                        if q in self.gset:
                            a_dec[q] ^= 1
                        else:
                            phys.append(q)
                    sh.apply_x(phys)
                    for q in phys:
                        fl.register_correction(q, 'FSC')
                    stats['n_corr'] += len(phys)
                self._switch_out_fix(sh, readout, a_dec, rng, noisy=forced is None and not self.quiet_window)
        # Pauli frame of the S^g eigenvalues (last S1 round), then the logical Clifford stabilizer
        for v, m in enumerate(m_g_last):
            if m:
                for q in circ.frame_strings[v]:
                    sh.sim.z(q)
        val = sh.sim.peek_observable_expectation(circ.logical_proxy(sh.fx))
        if decode:
            sols = [ch.predict(outs, ch.flux_erasures(flux_events) if self.flux_informed else None)
                    for ch in self.charges]
            flip = min(sols, key=lambda s: s[1])[0]
        else:
            flip = 0
        val = val * (-1) ** flip
        stats['n_dress'] = sh.n_dress
        return (1 - val) / 2, stats

    def _switch_out_fix(self, sh, readout, a_dec, rng, noisy=True):
        """kappa = Gamma(a): the virtual network applied CZ(Gamma(a_raw)); the protocol applies
        CZ(Gamma(a_dec)) physically -> add the difference (Clifford) and the noise of the physical CZs."""
        circ = self.circ
        ar = np.array([readout[q] for q in circ.data_g], dtype=np.uint8)
        ad = np.array([a_dec[q] for q in circ.data_g], dtype=np.uint8)
        kr = (circ.Gamma.astype(int) @ ar) % 2
        kd = (circ.Gamma.astype(int) @ ad) % 2
        p = self.noise.p
        for v in range(len(circ.gens_g)):
            if kr[v] ^ kd[v]:
                for (i, j) in circ.gpairs[v]:
                    sh.sim.cz(i, j)
            if kd[v] and p > 0 and noisy:
                for (i, j) in circ.gpairs[v]:
                    if rng.random() < p:
                        ps = PAULI_CHOICES[2][rng.integers(15)]
                        sh._pauli(ps, (i, j))

    def run(self, shots, rng, decode=True):
        inf = np.zeros(shots)
        st = []
        for s in range(shots):
            inf[s], d = self.shot(rng, decode)
            st.append(d)
        return inf, st

In [ ]:
# ------------------------------------------------------------------ applicability of Takada et al.
def takada_conditions(L):
    """Algebraic checks of the sufficient conditions of Takada et al. (Definition 2 and Sec. VIII) for
    the D4 code switching protocol on the L x L lattice.  Returns dict name -> (passed, detail)."""
    P = D4Protocol(L)
    lat, d4, fsc = P.lat, P.d4, P.fsc
    res = {}
    # (a) admissible factorisation: every X-type Clifford stabilizer = X factors x CZ factors with disjoint
    #     supports (transversal, so every effectively measured operator stays Hermitian, Eq. (131))
    bad = 0
    for lab, op in d4.xgens:
        qs = list(op.s) + [q for pr in op.quad for q in pr]
        bad += len(qs) != len(set(qs))
    res['factors transversal (X and CZ factors on disjoint qubits)'] = (bad == 0, '%d generators' % len(d4.xgens))
    # (b) Pauli dressings: X/Y errors on the Clifford factors (CZ; S, CZ in the logical stabilizer; S,
    #     CZ dressings) give weight-1 Z dressings; all dressings commute (Eq. (91))
    I2 = np.eye(2); X = np.array([[0, 1], [1, 0]]); Y = np.array([[0, -1j], [1j, 0]]); Zm = np.diag([1, -1])
    CZm = np.diag([1, 1, 1, -1]); Sm = np.diag([1, 1j])
    ok = True
    for E in (X, Y):
        D = np.kron(E, I2) @ CZm @ np.kron(E, I2).conj().T @ CZm.conj().T       # [E_1, CZ]
        ok &= np.allclose(np.abs(D), np.abs(np.kron(I2, Zm)))
        D1 = E @ Sm @ E.conj().T @ Sm.conj().T
        ok &= np.allclose(np.abs(D1), np.abs(Zm))
    res['dressings of CZ / S by X, Y are single-qubit Z (dressing axis Z on every qubit)'] = (bool(ok), '')
    # (c) effective commutation: group commutators of measured generators are Z-type Paulis in the
    #     persistent Z-stabilizer span (D4 Z checks), Eq. (58) / Lemma 1
    HZr, HZp = f2_rref(d4.HZ)
    nbad, npair = 0, 0
    for (l1, a), (l2, b) in itertools.combinations(d4.xgens, 2):
        lin, const = commutator_affine(a, b)
        npair += 1
        if lin and not f2_in_rowspace(HZr, HZp, lat.vec(lin)):
            nbad += 1
    res['D4 generators: commutators in the Z-stabilizer span'] = (nbad == 0, '%d pairs' % npair)
    # switch-in: S^g measured on FSC|+_L> (x) |0>_g: commutators with the FSC generators and the logical X
    # must lie in the span of the FSC Z checks and the single-qubit Z^g (current Z stabilizers)
    # The state before switch-in, FSC|+_L> (x) |0>_g, is stabilized by the FSC Z checks, the single-qubit
    # Z^g, and by the D4-dressed X-type operators S^r_v, S^b_v (their CZ^{g,.} dressing acts trivially
    # on |0>_g) and a dressed logical X = X^t D (D diagonal Clifford, also trivial on |0>_g).  Takada
    # Sec. VIII B: the generators may be chosen such that the new measurements (S^g) effectively commute
    # or anticommute with them.  Check this choice:
    zin = [rows for lab, rows in lat.z_checks((R, B))] + [{lat.q(e, G)} for e in range(lat.E)]
    Hin = np.array([lat.vec(r) for r in zin], dtype=np.uint8)
    Hir, Hip = f2_rref(Hin)
    gens_in = [op for lab, op in d4.xgens if lab[2] != 'g']
    nbad = 0
    for lab, op in d4.xgens:
        for g in gens_in:
            lin, const = commutator_affine(op, g)
            if lin and not f2_in_rowspace(Hir, Hip, lat.vec(lin)):
                nbad += 1
    # dressed logical X: find D = prod CZ^{c_k} over the CZ pairs of the D4 generators with
    # [X^t D, S] in the D4 Z-stabilizer span for every X-type generator S (linear system over F2)
    t = set(int(q) for q in np.nonzero(fsc.t_log)[0])
    pairs = sorted({pr for _, op in d4.xgens for pr in op.quad})
    rows_eq = []                          # for each generator: commutator linear part as function of c
    HZr2, HZp2 = f2_rref(d4.HZ)
    # basis of the complement of the Z span to express "lin in span" as linear equations
    Npre = f2_nullspace(d4.HZ)            # vectors orthogonal to the span: lin in span <=> N.lin = 0
    A_cols, b_vec = [], None
    base_list = []
    for lab, op in d4.xgens:
        lin0, _ = commutator_affine(PSOp(s=t), op)
        base_list.append(lat.vec(lin0) if lin0 else np.zeros(lat.n, dtype=np.uint8))
    for pr in pairs:
        col = []
        for lab, op in d4.xgens:
            lin, _ = commutator_affine(PSOp(quad=[pr]), op)
            col.append(lat.vec(lin) if lin else np.zeros(lat.n, dtype=np.uint8))
        A_cols.append(col)
    # equations: N @ (base_v + sum_k c_k col_k,v) = 0 for all v
    eqA = np.array([[int((Npre.astype(int) @ A_cols[k][v].astype(int) % 2)[i]) for k in range(len(pairs))]
                    for v in range(len(d4.xgens)) for i in range(Npre.shape[0])], dtype=np.uint8)
    eqb = np.array([int((Npre.astype(int) @ base_list[v].astype(int) % 2)[i])
                    for v in range(len(d4.xgens)) for i in range(Npre.shape[0])], dtype=np.uint8)
    sol = f2_solve(eqA.T, eqb) if eqA.size else np.zeros(0, dtype=np.uint8)
    logical_ok = sol is not None
    if logical_ok:
        D = [pr for pr, cbit in zip(pairs, sol) if cbit]
        Xbar = PSOp(s=t, quad=D)
        # dressing trivial on |0>_g: every CZ of D touches a g qubit
        triv = all(any(q % 3 == G for q in pr) for pr in D)
        for lab, op in d4.xgens:
            lin, const = commutator_affine(Xbar, op)
            if lin and not f2_in_rowspace(Hir, Hip, lat.vec(lin)):
                nbad += 1
        logical_ok = triv
    res['switch-in: S^g vs dressed generators of FSC|+_L>|0>_g (current Z span), dressed logical X exists'] = (
        nbad == 0 and logical_ok, 'logical dressing with %s CZ pairs' % (len(D) if logical_ok else '-'))
    # (d) primary-support exactness (Definition 1): X supports of the X-type generators (and the logical
    #     X) are linearly independent, so no product of them is a non-Pauli operator without X part
    Ssup = np.vstack([d4.S, d4.t_log[None, :]])
    rk = f2_rank(Ssup[:, d4.cols])
    res['primary-support exactness (independent X supports)'] = (rk == Ssup.shape[0], 'rank %d / %d' % (rk, Ssup.shape[0]))
    # (e) the transversal U and the deferred switch-out network consist of CNOTs and diagonal third-level
    #     gates only (Sec. VIII A, Eq. (178)): Pauli / diagonal-Clifford errors stay of that form
    names = {g[0] for g in P.U_gates()}
    res['U = CNOT + diagonal 3rd-level gates (Sec. VIII A)'] = (names <= {'CNOT', 'CS', 'CSDG', 'TDG'}, str(sorted(names)))
    # (f) logical Clifford stabilizer of the output: Clifford (no cubic / odd terms) and effectively
    #     commuting with every FSC X check (dressing = Z string in the FSC Z span with sign +-1)
    circ = D4SwitchingCircuit(L, 1, 1, 1)
    Hf = np.array([[1 if q in qs else 0 for q in range(circ.nq)] for lab, qs in circ.zchecks['FSC']], dtype=np.uint8)
    Hfr, Hfp = f2_rref(Hf)
    nbad = 0
    for lab, xs, quad in circ.xgens['FSC']:
        fx = np.zeros(circ.nq, dtype=np.uint8)
        fx[xs] = 1
        R_ = circ.logical_proxy(fx) * circ.logical_proxy(np.zeros(circ.nq, dtype=np.uint8))
        zpart = np.array([1 if R_[q] in (2, 3) else 0 for q in range(circ.nq)], dtype=np.uint8)
        xpart = np.array([1 if R_[q] in (1, 2) else 0 for q in range(circ.nq)], dtype=np.uint8)
        if xpart.any() or (zpart.any() and not f2_in_rowspace(Hfr, Hfp, zpart)) or R_.sign.imag != 0:
            nbad += 1
    res['output logical stabilizer: Clifford, effectively commutes with FSC X checks'] = (nbad == 0, '')
    return res


def lockstep_validation(L, Rin, Rout, noise, rng, sequential=False, corr_prob=0.05):
    """Run the Pauli-proxy shot (FastShot) and the exact simulation of the TRUE circuit (TrueSim) in
    lockstep from the start of S1 (initial state E|+_L>_FSC|0>_g with a random Pauli E on the FSC data),
    with the same sampled circuit-level errors and random feed-forward X corrections.  Returns dict with
    the number of measurements, the number of mismatches (deterministic vs random, probability of the
    outcome), and the final <H> from both simulations."""
    circ = D4SwitchingCircuit(L, 0, Rin, Rout, sequential=sequential)
    comp = CompiledProtocol(circ, noise)
    P, c, fsc = circ.P, circ.c, circ.P.fsc
    zero = coset_state(fsc, set())
    one = coset_state(fsc, set(int(q) for q in np.nonzero(fsc.t_log)[0]))
    amps = {}
    for st in (zero, one):
        for k, v in st.a.items():
            amps[k] = amps.get(k, 0) + v / np.sqrt(2)
    keys = [sum(1 << c[q] for q in k) for k in amps]
    ts = TrueSim(circ.nd, keys, list(amps.values()))
    sh = FastShot(comp, rng)
    n = circ.nq
    stabs = []
    for lab, xs, quad in circ.xgens['FSC']:
        ps = stim.PauliString(n)
        for q in xs:
            ps[q] = 'X'
        stabs.append(ps)
    for lab, qs in circ.zchecks['FSC']:
        ps = stim.PauliString(n)
        for q in qs:
            ps[q] = 'Z'
        stabs.append(ps)
    ps = stim.PauliString(n)
    for q in circ.t_log:
        ps[q] = 'X'
    stabs.append(ps)
    for q in range(n):
        if q not in circ.data_rb:
            ps = stim.PauliString(n)
            ps[q] = 'Z'
            stabs.append(ps)
    sh.sim.set_state_from_stabilizers(stabs, allow_redundant=True)
    for q in rng.choice(circ.data_rb, 2, replace=False):
        pc = 'XYZ'[rng.integers(3)]
        getattr(ts, pc)(int(q)); getattr(sh.sim, pc.lower())(int(q))
    sh.init_frame(circ.zchecks['FSC'], circ.data_rb)
    nmeas = nbad = nrand = 0
    m_g = None
    for bi, blk in enumerate(comp.blocks):
        res = sh.run_block_lockstep(bi, ts, rng)
        for q, bb, o, pk, pr in res:
            nmeas += 1
            if pk == 0:
                nrand += 1
                nbad += abs(pr - 0.5) > 1e-9
            else:
                nbad += (o != (0 if pk == 1 else 1)) or abs(pr - 1) > 1e-9
        nm = blk['name']
        if nm[0] == 'S1' and nm[2] == 'x':
            gl = {circ.lab_of[('D4', q)]: o for q, bb, o, pk, pr in res if bb == 'X'}
            m_g = [gl[lab] for lab, _ in circ.gens_g]
        if corr_prob and nm[0] in ('S1', 'S4') and rng.random() < corr_prob * len(blk['moms']):
            q = int(rng.choice(circ.data_rb if nm[0] == 'S4' else circ.data_all))
            sh.apply_x([q]); ts.X(q)
    for v, m in enumerate(m_g):
        if m:
            for q in circ.frame_strings[v]:
                sh.sim.z(q); ts.Z(q)
    true = ts.expect_logical(circ)
    proxy = sh.sim.peek_observable_expectation(circ.logical_proxy(sh.fx))
    return dict(n_meas=nmeas, n_random=nrand, n_mismatch=int(nbad), H_true=true, H_proxy=proxy,
                n_dress=sh.n_dress, agree=bool(nbad == 0 and abs(true - proxy) < 1e-9))

### 雑音付き 15-to-1 ライブラリ（Litinski の誤り予算・密度行列計算・GoSC 配置・表面符号の p_L 較正）

In [ ]:
"""msd_noisy_lib: output error and cost of 15-to-1 magic-state distillation under circuit-level noise.

Error model (D. Litinski, "Magic State Distillation: Not as Costly as You Think", Quantum 3, 205 (2019),
and its published resource-estimation code github.com/litinski/magicstates, Apache-2.0; re-implemented
here, the error budgets are transcribed from that code):
  * the distillation circuit acts on 5 logical qubits (surface-code patches) and consists of Z-type
    pi/8 Pauli-product rotations; a faulty rotation is followed by a P_{pi/2}, P_{-pi/4} or P_{pi/4}
    error with probabilities p1, p2, p3 (faulty magic state: p/3 each; lattice-surgery measurement
    errors and errors in the ancilla region add to p1, p2);
  * patches suffer X / Z storage errors with probability 0.5 * p_L(d) per code cycle each;
  * p_L(p, d) = 0.1 (100 p)^((d+1)/2) per patch and code cycle (GoSC Eq. (10)), or a calibration of the
    same form obtained by stim simulations of the rotated surface code under the SAME circuit-level
    noise model as the D4 simulation (`calibrate_surface_code`);
  * the output error is the infidelity of the post-selected output state (exact density-matrix
    calculation, 128-bit arithmetic), the success probability is the post-selection probability.
Protocols:
  * `litinski2019_one_level(p, dx, dz, dm)` and `litinski2019_two_level(...)`: Litinski's optimised
    factories (transcribed; validated against the outputs stored in his notebook);
  * `gosc_15to1(p, d)` and `gosc_225to1(p, d1, d2)`: the A Game of Surface Codes Sec. 3 layouts
    (Fig. 17c: 11 tiles x 11 tau; Fig. 19: 176 tiles x 15 tau) evaluated with the same error model.
    The per-rotation error budget of these layouts is OUR adaptation of Litinski (2019) to the GoSC
    geometry (single distance d, two ancilla tiles per multi-patch measurement) - see `gosc_steps`.
"""
import numpy as np
import itertools
import mpmath
from mpmath import mp

mp.prec = 128


def plog_gosc(p, d):
    """GoSC Eq. (10) / Litinski (2019): logical error per patch per code cycle."""
    return mpmath.mpf('0.1') * (100 * mpmath.mpf(p)) ** (mpmath.mpf(d + 1) / 2)

In [ ]:
# ------------------------------------------------------------------ 5-qubit density-matrix engine
class DM5:
    """Density matrix of n (=5) qubits.  backend='ld': numpy extended precision (np.clongdouble, ~19 digits,
    fast; used for the searches); backend='mp': numpy object array of mpmath.mpc (128 bits; used to check
    the transcription against Litinski's published outputs down to 1e-25).  Qubit 1 = most significant bit
    (Litinski's kron ordering)."""

    def __init__(self, n=5, singles=None, backend='ld'):
        self.n, self.backend = n, backend
        N = 2 ** n
        idx = np.arange(N)
        self.bits = [(idx >> (n - 1 - q)) & 1 for q in range(n)]
        if singles is None:
            h = 1 / mpmath.sqrt(2) if backend == 'mp' else np.longdouble(1) / np.sqrt(np.longdouble(2))
            singles = [[h, h]] * n
        if backend == 'mp':
            psi = np.array([mpmath.mpc(1)] * N, dtype=object)
            for q, s in enumerate(singles):
                psi = psi * np.array([mpmath.mpc(s[b]) for b in self.bits[q]], dtype=object)
        else:
            psi = np.ones(N, dtype=np.clongdouble)
            for q, s in enumerate(singles):
                psi = psi * np.array([s[b] for b in self.bits[q]], dtype=np.clongdouble)
        self.rho = np.outer(psi, np.conj(psi))

    def _f(self, x):
        return mpmath.mpf(x) if self.backend == 'mp' else np.longdouble(float(x)) if not isinstance(x, np.longdouble) else x

    def _eig(self, axis):
        s = np.ones(2 ** self.n, dtype=int)
        for q, a in enumerate(axis):
            if a:
                s = s * (1 - 2 * self.bits[q])
        return s

    def rot(self, axis, p1=0, p2=0, p3=0):
        """Z-type Pauli-product rotation exp(i pi/8 P) followed by the faulty-rotation channel of
        Litinski's `apply_rot`: with prob p1 / p2 / p3 the angle is 5pi/8 / -pi/8 / 3pi/8."""
        s = self._eig(axis)
        diff = s[:, None] - s[None, :]            # in {-2, 0, 2}
        if self.backend == 'mp':
            p1, p2, p3 = mpmath.mpf(p1), mpmath.mpf(p2), mpmath.mpf(p3)
            w = [(1 - p1 - p2 - p3, mp.pi / 8), (p1, 5 * mp.pi / 8), (p2, -mp.pi / 8), (p3, 3 * mp.pi / 8)]
            m2 = sum(wk * mpmath.expj(2 * ph) for wk, ph in w)
            M = np.where(diff == 2, m2, np.where(diff == -2, mpmath.conj(m2), mpmath.mpf(1)))
        else:
            p1, p2, p3 = (np.longdouble(float(x)) for x in (p1, p2, p3))
            pi = np.longdouble('3.14159265358979323846264338327950288')
            w = [(1 - p1 - p2 - p3, pi / 8), (p1, 5 * pi / 8), (p2, -pi / 8), (p3, 3 * pi / 8)]
            m2 = sum(wk * (np.cos(2 * ph) + 1j * np.sin(2 * ph)) for wk, ph in w)
            M = np.where(diff == 2, m2, np.where(diff == -2, np.conj(m2), 1)).astype(np.clongdouble)
        self.rho = self.rho * M

    def z_err(self, q, p):
        if p == 0:
            return
        p = mpmath.mpf(p) if self.backend == 'mp' else np.longdouble(float(p))
        sg = (1 - 2 * self.bits[q])
        self.rho = self.rho * np.where(sg[:, None] * sg[None, :] == 1, 1, 1 - 2 * p)

    def x_err(self, q, p):
        if p == 0:
            return
        p = mpmath.mpf(p) if self.backend == 'mp' else np.longdouble(float(p))
        f = np.arange(2 ** self.n) ^ (1 << (self.n - 1 - q))
        self.rho = (1 - p) * self.rho + p * self.rho[np.ix_(f, f)]

    def store(self, px, pz):
        for q in range(self.n):
            self.x_err(q, px[q])
            self.z_err(q, pz[q])

    def postselect_x(self, measured):
        """Project the qubits in `measured` onto |+> (X = +1).  Returns (success probability, normalised
        2x2 density matrix of the remaining qubit)."""
        keep = [q for q in range(self.n) if q not in measured]
        assert len(keep) == 1
        k = keep[0]
        out = np.zeros((2, 2), dtype=self.rho.dtype)
        for a in range(2):
            for b in range(2):
                ia = self.bits[k] == a
                ib = self.bits[k] == b
                out[a, b] = self.rho[np.ix_(ia, ib)].sum() / 2 ** len(measured)
        ps = (out[0, 0] + out[1, 1]).real
        return ps, out / ps


def run_steps(steps, n=5, out_qubit=0, init=None, backend='ld'):
    """Evaluate a protocol given as a list of steps
       ('rot', axis (0/1 tuple), p1, p2, p3)   faulty Z-type pi/8 rotation
       ('store', px (list), pz (list))        X / Z Pauli channels
    on |+>^n; the qubits != out_qubit are measured in X and post-selected on +1.  Returns
    (success probability, output infidelity).  The ideal output is taken from the same circuit without
    errors; the infidelity is evaluated as the population <psi_perp|rho|psi_perp> of the orthogonal
    state (no cancellation in 1 - F)."""
    def go(noisy):
        dm = DM5(n, init, backend)
        for st in steps:
            if st[0] == 'rot':
                dm.rot(st[1], *(st[2:5] if noisy else (0, 0, 0)))
            elif st[0] == 'store' and noisy:
                dm.store(st[1], st[2])
        return dm.postselect_x([q for q in range(n) if q != out_qubit])
    ps0, r0 = go(False)
    # ideal output |psi> from the noiseless (pure) reduced state: rho0 = |psi><psi|
    if backend == 'mp':
        a = mpmath.sqrt(r0[0, 0].real)
        psi = [a, r0[1, 0] / a] if a != 0 else [mpmath.mpc(0), mpmath.mpc(1)]
        perp = [-mpmath.conj(psi[1]), mpmath.conj(psi[0])]
    else:
        a = np.sqrt(r0[0, 0].real)
        psi = [a, r0[1, 0] / a]
        perp = [-np.conj(psi[1]), np.conj(psi[0])]
    ps, r = go(True)
    eps = sum(np.conj(perp[i]) * r[i, j] * perp[j] for i in range(2) for j in range(2)).real
    return ps, eps


def infidelity(rho1, ideal1):
    """1 - <ideal|rho|ideal> for a single-qubit rho (2x2) and an ideal ket."""
    f = sum(np.conj(ideal1[a]) * rho1[a, b] * ideal1[b] for a in range(2) for b in range(2))
    return 1 - f.real

In [ ]:
# ------------------------------------------------------------------ Litinski (2019), transcribed
_Z = 1


def _ax(*qs):
    return tuple(int(q in qs) for q in range(5))


def litinski2019_one_level_steps(p, dx, dz, dm, plog=plog_gosc):
    """Error budget of Litinski's (15-to-1)_{dx,dz,dm} (onelevel15to1.py), qubit 1 = output (index 0)."""
    p = mpmath.mpf(p)
    px, pz, pm = plog(p, dx), plog(p, dz), plog(p, dm)
    t = p / 3
    S = []
    single = lambda q: ('rot', _ax(q), t + 0.5 * (dm / dz) * pz * dm, t + 0.5 * dz * pm, t)
    multi = lambda qs, w: ('rot', _ax(*qs), t + 0.5 * pm * dm, t + 0.5 * pm * dm + 0.5 * w * dx / dm * pm, t)
    sx = lambda c0, c: ('store', [c0] + [c] * 4, [0] * 5)
    S += [single(1), single(2), single(3), multi((1, 2, 3), 3 * dz)]
    S += [('store', [0] + [0.5 * (dz / dx) * px * dm] * 3 + [0], [0] + [0.5 * (dx / dz) * pz * dm] * 3 + [0])]
    S += [multi((0, 1, 2), dx + 2 * dz), multi((0, 1, 3), dx + 3 * dz)]
    S += [('store', [0] * 5, [0.5 * ((dx + 2 * dz) + (dx + 3 * dz)) / dx * px * dm, 0, 0, 0, 0])]
    S += [('store', [0.5 * px * dm] + [0.5 * (dz / dx) * px * dm] * 3 + [0],
           [0.5 * px * dm] + [0.5 * (dx / dz) * pz * dm] * 3 + [0])]
    S += [multi((0, 2, 3), dx + 3 * dz), multi((0, 3, 4), dx + 4 * dz), single(4)]
    S += [('store', [0] * 5, [0.5 * ((dx + 3 * dz) + (dx + 4 * dz)) / dx * px * dm, 0, 0, 0, 0])]
    full = lambda a: ('store', [a * px] + [0.5 * (dz / dx) * px * dm] * 4, [a * px] + [0.5 * (dx / dz) * pz * dm] * 4)
    S += [full(0.5 * dm)]
    S += [multi((0, 1, 4), dx + 4 * dz), multi((0, 2, 4), dx + 4 * dz)]
    S += [('store', [0] * 5, [0.5 * ((dx + 4 * dz) + (dx + 4 * dz)) / dx * px * dm, 0, 0, 0, 0])]
    S += [full(0.5 * dm)]
    S += [multi((0, 1, 2, 3, 4), dx + 4 * dz), multi((2, 3, 4), 3 * dz)]
    S += [('store', [0] * 5, [0.5 * (dx + 4 * dz) / dx * px * dm, 0, 0, 0, 0])]
    S += [full(0.5 * (dm + 2 * dx))]
    S += [multi((1, 3, 4), 4 * dz), multi((1, 2, 4), 4 * dz)]
    S += [('store', [0] + [0.5 * (dz / dx) * px * dm] * 4, [0] + [0.5 * (dx / dz) * pz * dm] * 4)]
    return S


def litinski2019_one_level(p, dx, dz, dm, plog=plog_gosc, backend='ld'):
    ps, eps = run_steps(litinski2019_one_level_steps(p, dx, dz, dm, plog), out_qubit=0, backend=backend)
    qubits = 2 * ((dx + 4 * dz) * 3 * dx + 2 * dm)
    cycles = 6 * dm / ps
    return dict(protocol='Litinski2019 (15-to-1)_{%d,%d,%d}' % (dx, dz, dm), eps=float(eps), p_success=float(ps),
                qubits=int(qubits), cycles=float(cycles), volume=float(qubits * cycles), levels=1)


def litinski2019_two_level_steps(p, dx, dz, dm, dx2, dz2, dm2, nl1, plog=plog_gosc, backend='ld'):
    p = mpmath.mpf(p)
    px2, pz2, pm2 = plog(p, dx2), plog(p, dz2), plog(p, dm2)
    ps1, pl1 = run_steps(litinski2019_one_level_steps(p, dx, dz, dm, plog), out_qubit=0, backend=backend)
    ps1, pl1 = mpmath.mpf(float(ps1)) if backend != 'mp' else ps1, mpmath.mpf(float(pl1)) if backend != 'mp' else pl1
    pfail = 1 - ps1
    l1time = max(6 * dm / (mpmath.mpf(nl1) / 2) / (1 - pfail), dm2)
    lmove = 10 * dm2 + mpmath.mpf(nl1) / 4 * (dx + 4 * dz)
    R = lambda qs, w: ('rot', _ax(*qs), pl1 + 0.5 * lmove * pm2,
                       0.5 * lmove * pm2 + 0.5 * w * dx2 / dm2 * pm2, 0)
    st = lambda mask, c0=0: ('store', [c0] + [0.5 * (dz2 / dx2) * px2 * l1time if m else 0 for m in mask],
                             [c0] + [0.5 * (dx2 / dz2) * pz2 * l1time if m else 0 for m in mask])
    zs = lambda v: ('store', [0] * 5, [v, 0, 0, 0, 0])
    a = 0.5 * px2 * l1time
    S = [R((1,), dx2 + dz2 + dm2), R((2,), 3 * dz2 + dm2), st([1, 1, 0, 0]),
         R((3,), dx2 + 3 * dz2 + dm2), R((4,), dz2 + dm2), st([1, 1, 1, 1]),
         R((0, 1, 2), dx2 + 2 * dz2 + dm2), R((1, 2, 3), 4 * dz2 + dm2),
         zs(0.5 * (dx2 + 2 * dz2 + dm2) * dm2 / dx2 * px2), st([1, 1, 1, 1], a),
         R((0, 2, 3), dx2 + 3 * dz2 + dm2), R((0, 1, 3), dx2 + 4 * dz2 + dm2),
         zs(0.5 * ((dx2 + 3 * dz2 + dm2) + (dx2 + 4 * dz2 + dm2)) * dm2 / dx2 * px2), st([1, 1, 1, 1], a),
         R((0, 1, 4), dx2 + 4 * dz2 + dm2), R((0, 3, 4), dx2 + 4 * dz2 + dm2),
         zs(0.5 * 2 * (dx2 + 4 * dz2 + dm2) * dm2 / dx2 * px2), st([1, 1, 1, 1], a),
         R((0, 2, 4), dx2 + 4 * dz2 + dm2), R((0, 1, 2, 3, 4), dx2 + 4 * dz2 + dm2),
         zs(0.5 * 2 * (dx2 + 4 * dz2 + dm2) * dm2 / dx2 * px2), st([1, 1, 1, 1], a),
         R((1, 3, 4), dx2 + 4 * dz2 + dm2), R((2, 3, 4), 3 * dz2 + dm2),
         zs(0.5 * (dx2 + 4 * dz2 + dm2) * dm2 / dx2 * px2), st([1, 1, 1, 1], 0.5 * px2 * (dm2 + 2 * dx2)),
         R((1, 2, 4), 4 * dz2 + dm2), st([1, 1, 0, 1])]
    # note: Litinski's code passes the X/Z storage of qubit 1 only where stated (copied as above)
    return S, ps1, pl1, l1time


def litinski2019_two_level(p, dx, dz, dm, dx2, dz2, dm2, nl1, plog=plog_gosc, backend='ld'):
    S, ps1, pl1, l1time = litinski2019_two_level_steps(p, dx, dz, dm, dx2, dz2, dm2, nl1, plog, backend)
    ps2, eps = run_steps(S, out_qubit=0, backend=backend)
    qubits = 2 * int((dx2 + 4 * dz2) * 3 * dx2 + nl1 * ((dx + 4 * dz) * (3 * dx + dm2 / 2) + 2 * dm)
                     + 20 * dm2 * dm2 + 2 * dx2 * dm2)
    cycles = 7.5 * l1time / ps2
    return dict(protocol='Litinski2019 (15-to-1)x(15-to-1)_{%d,%d,%d;%d,%d,%d;%d}' % (dx, dz, dm, dx2, dz2, dm2, nl1),
                eps=float(eps), p_success=float(ps2), qubits=int(qubits), cycles=float(cycles),
                volume=float(qubits * cycles), levels=2, eps_level1=float(pl1))

In [ ]:
# ------------------------------------------------------------------ surface-code p_L calibration (stim)
def noisy_surface_code(d, rounds, basis, p, p_idle=None):
    """Rotated surface-code memory (stim's generated circuit) with the SAME circuit-level noise model and
    time-step convention as the D4 simulation (noisy_d4_lib.NoiseModel / D4SwitchingCircuit rounds):
    X-type ancillas are prepared / measured with RX / MX (no H layers); stim's combined MR is split into a
    measurement step and a reset step, so that one round = reset, 4 CNOT layers, measurement = 6 time
    steps; 2-qubit gates are followed by DEPOLARIZE2(p), resets by a flip, measurements are preceded by a
    flip, and the idle qubits of every time step are depolarized with p_idle."""
    p_idle = p if p_idle is None else p_idle
    base = stim.Circuit.generated('surface_code:rotated_memory_' + basis, distance=d, rounds=rounds).flattened()
    allq, hq = set(), set()
    for ins in base:
        if ins.name in ('R', 'CX', 'M', 'MR', 'H'):
            allq |= {t.value for t in ins.targets_copy()}
        if ins.name == 'H':
            hq |= {t.value for t in ins.targets_copy()}
    out = stim.Circuit()

    def emit(pre, body, post, touched):
        for e in pre:
            out.append(*e)
        for b in body:
            out.append(*b) if isinstance(b, tuple) else out.append(b)
        for e in post:
            out.append(*e)
        idle = sorted(allq - touched)
        if idle and p_idle > 0:
            out.append('DEPOLARIZE1', idle, p_idle)
        out.append('TICK')

    def flush(layer):
        touched, pre, body, post, resets = set(), [], [], [], []
        for ins in layer:
            nm = ins.name
            ts = [t.value for t in ins.targets_copy() if t.is_qubit_target]
            if nm == 'H':
                continue
            if nm in ('R', 'RX'):
                for q in ts:
                    x = q in hq or nm == 'RX'
                    body.append(('RX' if x else 'R', [q]))
                    post.append(('Z_ERROR' if x else 'X_ERROR', [q], p))
            elif nm in ('MR', 'M', 'MX'):
                # one target per instruction keeps stim's measurement-record order
                for q in ts:
                    x = q in hq or nm == 'MX'
                    pre.append(('Z_ERROR' if x else 'X_ERROR', [q], p))
                    body.append(('MX' if x else 'M', [q]))
                    if nm == 'MR':
                        resets.append(q)
            elif nm == 'CX':
                body.append(('CX', ts))
                post.append(('DEPOLARIZE2', ts, p))
            else:
                body.append(ins)
                continue
            touched |= set(ts)
        if not body:
            return                                   # H-only layer: no time step
        emit(pre, body, post, touched)
        if resets:                                   # the reset half of MR: its own time step
            emit([], [('RX' if q in hq else 'R', [q]) for q in resets],
                 [('Z_ERROR' if q in hq else 'X_ERROR', [q], p) for q in resets], set(resets))
    layer = []
    for ins in base:
        if ins.name == 'TICK':
            flush(layer)
            layer = []
        elif ins.name in ('DETECTOR', 'OBSERVABLE_INCLUDE', 'QUBIT_COORDS', 'SHIFT_COORDS'):
            if layer:
                flush(layer)
                layer = []
            out.append(ins)
        else:
            layer.append(ins)
    if layer:
        flush(layer)
    return out


def surface_code_logical_rate(d, p, shots, rounds=None, seed=0):
    """Logical error per round (X + Z memories summed) of the rotated surface code (MWPM)."""
    rounds = rounds or 3 * d
    tot = 0.0
    out = {}
    for basis in ('x', 'z'):
        c = noisy_surface_code(d, rounds, basis, p)
        m = pymatching.Matching.from_detector_error_model(c.detector_error_model(decompose_errors=True))
        det, obs = c.compile_detector_sampler(seed=seed).sample(shots, separate_observables=True)
        pred = m.decode_batch(det)
        PL = float(np.mean(pred[:, 0] != obs[:, 0]))
        eps = (1 - (1 - 2 * PL) ** (1 / rounds)) / 2 if PL < 0.5 else 0.5
        out[basis] = (PL, eps)
        tot += eps
    return tot, out


def fit_plog(rows):
    """Fit p_L = A (p/p_th)^((d+1)/2) to rows (d, p, p_L) by least squares in log p_L."""
    ds = np.array([r[0] for r in rows], float)
    ps = np.array([r[1] for r in rows], float)
    pl = np.array([r[2] for r in rows], float)
    k = (ds + 1) / 2
    # log pl = log A + k log p - k log pth  -> linear in (log A, log pth)
    A_ = np.stack([np.ones_like(k), -k], axis=1)
    y = np.log(pl) - k * np.log(ps)
    (lA, lpth), *_ = np.linalg.lstsq(A_, y, rcond=None)
    return float(np.exp(lA)), float(np.exp(lpth))


def make_plog(A, pth):
    return lambda p, d: mpmath.mpf(A) * (mpmath.mpf(p) / mpmath.mpf(pth)) ** (mpmath.mpf(d + 1) / 2)

In [ ]:
# ------------------------------------------------------------------ GoSC Sec. 3 layouts with the same model
# the 11 rotations of GoSC Fig. 15 (= msd_lib.fig15_rotations(): columns 5..15 of the 15-to-1 matrix,
# qubit indices 0..4 = qubits 1..5; validated in MSD_15to1_validation M1)
GOSC_ROTATIONS = [(2, 3, 4), (1, 3, 4), (1, 2, 4), (1, 2, 3), (0, 3, 4), (0, 2, 4), (0, 2, 3),
                  (0, 1, 4), (0, 1, 3), (0, 1, 2), (0, 1, 2, 3, 4)]


def gosc_15to1_steps(p, d, plog=plog_gosc, p_in=None, anc_tiles=2, in_channel='pauli'):
    """GoSC Fig. 15 / 17c: qubits 1-4 start in |m> (faulty, error p_in), qubit 5 in |+>; the 11 Z-type
    pi/8 rotations of Fig. 15 (msd_lib.fig15_rotations) are auto-corrected pi/8 rotations of 1 tau = d
    code cycles each; qubits 1-4 are measured in X, qubit 5 is the output.
    Error budget per rotation (our adaptation of Litinski 2019 to the single-distance GoSC layout):
      p1 (P_{pi/2})  = e1 + 0.5 p_L d                        (time-like error of the P x Z_m measurement)
      p2 (P_{-pi/4}) = e2 + 0.5 p_L d + 0.5 anc_tiles p_L d   (space-like errors in the ancilla tiles)
      p3 (P_{pi/4})  = e3
    with (e1, e2, e3) = (p_in/3,)*3 for a raw state with a random Pauli error (in_channel='pauli', GoSC
    Sec. 4 assumption) or (p_in, 0, 0) for a distilled input (in_channel='z', Litinski two-level code);
    every data tile suffers X and Z storage errors 0.5 p_L d per tau."""
    p = mpmath.mpf(p)
    p_in = p if p_in is None else mpmath.mpf(p_in)
    pl = plog(p, d)
    e = (p_in / 3,) * 3 if in_channel == 'pauli' else (p_in, 0, 0)
    S = [('rot', _ax(q), *e) for q in range(4)]               # the four |m> states (0 tau)
    st = ('store', [0.5 * pl * d] * 5, [0.5 * pl * d] * 5)
    for qs in GOSC_ROTATIONS:
        S.append(('rot', _ax(*qs), e[0] + 0.5 * pl * d, e[1] + 0.5 * pl * d + 0.5 * anc_tiles * pl * d, e[2]))
        S.append(st)
    return S


def gosc_15to1(p, d, plog=plog_gosc, backend='ld', **kw):
    """Output error and cost of the GoSC 11-tile 15-to-1 block (Fig. 17c): 11 tiles x 11 tau."""
    ps, eps = run_steps(gosc_15to1_steps(p, d, plog, **kw), out_qubit=4, backend=backend)
    qubits = 11 * 2 * d * d                       # 11 tiles, d^2 data + d^2 measurement qubits each
    cycles = 11 * d / ps
    return dict(protocol='GoSC 15-to-1 (11 tiles x 11 tau), d=%d' % d, eps=float(eps), p_success=float(ps),
                qubits=int(qubits), cycles=float(cycles), volume=float(qubits * cycles), levels=1, d=d)


def gosc_225to1(p, d1, d2, plog=plog_gosc, move_tiles=5, backend='ld', level1=None):
    """GoSC Fig. 19 (176 tiles, 15 tau): 11 level-1 15-to-1 blocks (distance d1) feed a level-2 15-to-1
    whose 15 rotations (incl. the 4 single-qubit ones) consume distilled states.  Level-1 output error
    enters as a Z-type (P_{pi/2}) error (Litinski two-level code); moving a level-1 state through the
    blue ancilla of Fig. 19 adds 0.5 * move_tiles * p_L(d2) * d2 (our assumption, move_tiles = 5)."""
    ps1, e1 = level1 if level1 is not None else run_steps(gosc_15to1_steps(p, d1, plog), out_qubit=4, backend=backend)
    ps1, e1 = mpmath.mpf(float(ps1)), mpmath.mpf(float(e1))
    pl2 = plog(mpmath.mpf(p), d2)
    mv = 0.5 * move_tiles * pl2 * d2
    S = []
    st = ('store', [0.5 * pl2 * d2] * 5, [0.5 * pl2 * d2] * 5)
    for q in range(4):
        S.append(('rot', _ax(q), e1 + mv + 0.5 * pl2 * d2, 0.5 * pl2 * d2 + pl2 * d2, 0))
        S.append(st)
    for qs in GOSC_ROTATIONS:
        S.append(('rot', _ax(*qs), e1 + mv + 0.5 * pl2 * d2, 0.5 * pl2 * d2 + pl2 * d2, 0))
        S.append(st)
    ps2, eps = run_steps(S, out_qubit=4, backend=backend)
    qubits = 2 * (121 * d1 * d1 + 55 * d2 * d2)   # 11 level-1 blocks of 11 tiles + 55 level-2 tiles
    cycles = 15 * d2 / (ps2 * ps1)                # a level-2 step is skipped when no level-1 state is ready
    return dict(protocol='GoSC 225-to-1 (176 tiles x 15 tau), d1=%d, d2=%d' % (d1, d2), eps=float(eps),
                p_success=float(ps2), qubits=int(qubits), cycles=float(cycles), volume=float(qubits * cycles),
                levels=2, d=d2, d1=d1, eps_level1=float(e1))

In [ ]:
print('library loaded')

## §0. パラメータとデータ
* `P_EVAL`：比較する物理誤り率。`TARGETS`：要求出力誤り。
* `STEPS_PER_CYCLE`：表面符号 1 ラウンド（code cycle）の時間ステップ数。較正に使った回路（初期化 → CNOT×4 → 測定）と同じ 6（`validation.ipynb` N7）。
* `PLOG_SOURCE`：`'calibrated'`（同じ雑音モデルでの表面符号シミュレーション、既定）または `'gosc'`（GoSC 式 (10)：$0.1(100p)^{(d+1)/2}$）。
* モンテカルロの結果ファイルが無い場合（Colab で単独実行した場合など）は、このノートに埋め込まれた本番結果を使う。

In [ ]:
P_EVAL = [1e-4, 2e-4, 3e-4, 5e-4, 1e-3]
TARGETS = [1e-4, 1e-6, 1e-8, 1e-10]
SCENARIOS = {'A': 'アイドル雑音あり (p_idle = p)', 'B': 'アイドル雑音なし (p_idle = 0)'}
STEPS_PER_CYCLE = 6
PLOG_SOURCE = 'calibrated'
import io
EMBED_MC = """scenario,window,d,p,p_idle,kappa0,kappa_in,kappa_out,R0,Rin,Rout,shots,sum_infidelity,n_fail,n_half,mean_errors,seconds
A,noisy,9,0.001,0.001,1,1.0,1,9,9,9,40,13.5,14,1,279.475,9.381391048431396
A,noisy,9,0.001,0.001,1,1.0,1,9,9,9,40,4.5,5,1,281.65,8.322658777236938
A,noisy,9,0.001,0.001,1,1.0,1,9,9,9,40,6.0,6,0,277.375,7.431972980499268
A,noisy,9,0.001,0.001,1,1.0,1,9,9,9,40,2.5,3,1,274.3,7.64038610458374
A,noisy,9,0.001,0.001,1,1.0,1,9,9,9,40,16.0,17,2,274.0,6.939210891723633
A,noisy,9,0.001,0.001,1,1.0,1,9,9,9,40,10.5,11,1,280.55,8.157454490661621
A,noisy,9,0.001,0.001,1,1.0,1,9,9,9,40,9.0,10,2,276.475,8.338906526565552
A,noisy,9,0.001,0.001,1,1.0,1,9,9,9,40,8.0,9,2,276.625,7.730153083801269
A,noisy,9,0.001,0.001,1,1.0,1,9,9,9,40,6.0,7,2,275.55,7.070082426071167
A,noisy,9,0.001,0.001,1,1.0,1,9,9,9,40,7.0,7,0,275.35,8.080144882202148
A,noisy,9,0.001,0.001,1,1.0,1,9,9,9,40,9.5,10,1,274.6,8.341097593307495
A,noisy,9,0.001,0.001,1,1.0,1,9,9,9,40,8.5,9,1,282.725,8.11793851852417
A,noisy,9,0.001,0.001,1,1.0,1,9,9,9,40,8.0,8,0,278.4,8.248433113098145
A,noisy,9,0.001,0.001,1,1.0,1,9,9,9,40,12.5,14,3,271.925,7.703518152236938
A,noisy,9,0.001,0.001,1,1.0,1,9,9,9,40,10.0,10,0,281.025,7.859093427658081
A,noisy,9,0.001,0.001,1,1.0,1,9,9,9,40,7.0,7,0,278.7,11.661115407943726
A,noisy,9,0.001,0.001,1,1.0,1,9,9,9,40,8.0,8,0,272.65,11.836674690246582
A,noisy,9,0.001,0.001,1,1.0,1,9,9,9,40,11.0,12,2,276.75,10.211730718612673
A,noisy,9,0.001,0.001,1,1.0,1,9,9,9,40,9.0,9,0,276.6,11.781240940093994
A,noisy,9,0.001,0.001,1,1.0,1,9,9,9,40,11.0,12,2,278.875,11.4064462184906
A,noisy,9,0.001,0.001,1,1.0,1,9,9,9,40,7.0,7,0,277.35,11.720125675201416
A,noisy,9,0.001,0.001,1,1.0,1,9,9,9,40,4.0,5,2,277.975,10.189868211746216
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.675,8.256772994995117
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.5,7.420562505722046
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.625,7.109050989151001
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.775,7.764539003372192
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.975,7.4282917976379395
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,29.775,8.743992567062378
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.275,7.570371866226196
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.5,7.93896222114563
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.425,8.765352964401245
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.85,8.0472412109375
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.15,8.341696739196777
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.875,7.2315673828125
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.0,8.845703363418579
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.125,9.766349077224731
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.075,9.52646017074585
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.15,9.094927787780762
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.9,8.930238485336304
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.025,8.170369386672974
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.35,8.902739524841309
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.7,9.129456043243408
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.75,9.751121520996094
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.625,8.637155532836914
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.425,8.821068525314331
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.15,8.84248685836792
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,29.625,9.303301572799684
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.8,8.081560373306274
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.125,8.932395219802856
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.525,9.468729734420776
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.875,7.634166240692139
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.05,8.937283754348755
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.575,9.854629516601562
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,1.0,1,0,28.225,8.427133798599243
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.3,6.764610767364502
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.225,6.670100688934326
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.375,6.866509437561035
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.475,6.302266359329224
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,29.375,6.62017035484314
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.55,6.668968915939331
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.675,6.319147825241089
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.5,6.429986000061035
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.95,6.285724401473999
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,1.0,1,0,28.775,5.840939998626709
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.075,8.631115674972534
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.3,8.643712043762207
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.2,8.766864538192749
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.575,8.574893236160278
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.025,10.252623319625854
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.875,9.204320669174194
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.325,8.797178268432617
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.1,8.163889408111572
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.975,7.850287437438965
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.075,10.801668882369995
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.525,8.570303916931152
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.225,8.7996244430542
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,29.55,8.233874559402466
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,29.2,8.780856847763062
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.325,9.127042293548584
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.125,8.235900402069092
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.0,8.726582527160645
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.075,8.436462879180908
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.675,8.799728393554688
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.275,8.335170030593872
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.05,9.137412309646606
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.7,8.491922855377197
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.45,9.525848865509031
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.325,8.378228902816772
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.4,8.666909217834473
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.55,8.322157859802246
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.0,9.425703763961792
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.325,8.641777276992798
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.0,8.606811046600342
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.875,8.093513011932373
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.05,9.183897018432615
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.7,8.08739185333252
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,29.325,8.150503635406494
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.625,8.69639778137207
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.55,8.051579236984253
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,25.725,8.580217599868774
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.925,8.778253316879272
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.4,8.172331809997559
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.45,8.25934624671936
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.95,9.247248649597168
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.675,8.164150714874268
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,29.075,8.920418500900269
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.2,8.99917221069336
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.575,9.56814193725586
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,29.7,7.919528484344482
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.35,9.051400423049929
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.15,8.158235311508179
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.5,9.45600724220276
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.85,8.905343294143677
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,29.3,9.20057988166809
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.225,7.497498750686645
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.825,8.360585927963257
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.475,9.458123922348022
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.825,8.305929899215698
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.8,8.584311962127686
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.4,8.508402347564697
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.7,7.709445953369141
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.525,8.548403263092041
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,29.05,7.559617280960083
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.925,8.546588897705078
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.4,8.106040954589844
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.0,7.936240196228027
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.575,8.653456687927246
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.65,8.76790976524353
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.325,8.842401266098022
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.15,7.767880201339722
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.85,7.512773036956787
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.7,8.641100406646729
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.575,9.357255935668944
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,29.55,8.463301181793213
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.975,9.14827585220337
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.725,10.620315313339232
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.125,9.931106328964232
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.625,9.574156284332275
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.25,8.652953147888184
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.375,9.679218053817747
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.225,9.5457923412323
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.275,9.905832290649414
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.8,9.535518407821655
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.8,9.07785415649414
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.6,8.552179336547852
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.725,8.021681785583496
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.925,8.214314222335815
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.2,9.68367338180542
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.975,8.829832315444946
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.55,8.5034818649292
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.475,10.726428270339966
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.975,7.986700057983398
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.275,9.240700960159302
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.35,9.578895330429075
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.825,7.857722997665405
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.7,7.910935640335083
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.975,7.966171503067017
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.95,6.662721872329712
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.65,5.9576404094696045
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.875,5.88067102432251
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,29.275,5.923276424407959
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,28.875,6.331305980682373
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.225,6.648366451263428
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,26.975,7.292458295822143
A,noisy,9,0.0001,0.0001,1,1.0,1,9,9,9,40,0.0,0,0,27.575,7.710115671157837
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.525,6.267235279083252
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.0,6.2591986656188965
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.6,6.202906131744385
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.5,5.896729469299316
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,12.875,5.741172790527344
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.45,5.850831508636475
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.875,5.639860391616821
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.875,6.439458608627319
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.275,5.762155294418335
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.9,6.568835258483887
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,12.925,7.040479898452759
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.925,6.744898796081543
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.95,6.4432456493377686
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.5,7.074397802352905
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.925,6.299597501754761
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.125,6.156167984008789
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.45,6.180095434188843
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.0,6.729671716690064
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,12.725,7.366647958755493
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.425,7.0304388999938965
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.375,6.719177961349487
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.7,6.921894550323486
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.95,6.416116714477539
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.75,7.203822612762451
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.775,7.577880144119263
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,15.15,5.97232985496521
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.225,6.19516396522522
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,1.0,1,0,13.95,6.797954082489014
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.0,7.330453395843506
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.75,6.283450603485107
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,12.8,7.706946849822998
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.475,5.892134428024292
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.45,6.316240310668945
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.575,7.789166212081909
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,11.95,7.124322175979614
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.375,6.758528470993042
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.0,7.678656816482544
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.75,7.918211221694946
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.85,7.760491132736206
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,2.0,2,0,13.45,6.756155014038086
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.575,6.6872313022613525
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.2,6.410901784896851
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.575,6.135254621505737
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.0,6.758872985839844
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,12.75,5.942187070846558
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,12.9,5.75128173828125
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.8,6.155625343322754
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.275,7.982639789581299
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.325,7.8534605503082275
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,1.0,1,0,15.1,6.528109073638916
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,12.7,6.506705045700073
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.35,7.308500289916992
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.375,6.7845299243927
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.575,6.459788084030151
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.0,7.835896968841553
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.85,7.487425088882446
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.825,6.201389312744141
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.9,7.444950103759766
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.575,6.606920957565308
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.25,6.785588502883911
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.9,7.991636514663696
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.65,6.28069543838501
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.25,7.228904247283935
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.35,7.873216152191162
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.475,7.525121688842773
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,12.8,6.586152076721191
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.875,7.058252573013306
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.125,6.397622346878052
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.125,7.021970272064209
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.575,6.251830816268921
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.275,6.225988388061523
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.075,6.815864562988281
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.65,6.707946062088013
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.75,8.821463823318481
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.2,6.596965789794922
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.5,8.201597929000854
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,1.0,1,0,14.425,7.093664407730103
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.625,6.986762046813965
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.2,7.122373580932617
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,12.85,7.070086240768433
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.675,6.939093112945557
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,1.0,1,0,13.2,7.080514430999756
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.475,6.569527864456177
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.425,6.133329153060913
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.85,7.829167366027832
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.575,6.538522720336914
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.7,7.0664167404174805
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.725,8.431325197219849
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.25,8.435667037963867
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,12.7,8.208131313323975
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.15,7.634034872055054
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.2,7.353254795074463
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,1.5,2,1,14.425,7.911154508590698
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.8,7.797419548034668
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.7,7.4569032192230225
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.45,8.690462350845337
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,15.15,7.169319152832031
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,1.0,1,0,12.95,7.82345724105835
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.425,7.517122507095337
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.2,6.3131632804870605
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.4,7.019871950149536
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,1.0,1,0,13.65,7.54050350189209
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.475,6.312023401260376
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.75,6.782472610473633
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,12.925,6.167297124862671
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.125,6.652653217315674
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,12.8,6.819833755493164
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.125,8.0892915725708
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.275,7.411376953125
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.6,7.167707204818726
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.5,1,1,14.95,7.204514026641846
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.15,7.199808359146118
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,1.0,1,0,14.25,5.7911376953125
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.65,6.695411920547485
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.65,6.615119695663452
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.5,7.293256521224976
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.25,7.158975839614868
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.15,7.126487731933594
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.25,7.397129058837891
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.6,6.844560623168945
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.05,6.493939638137817
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.05,6.528740167617798
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.85,6.520333290100098
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.7,6.540598630905151
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.65,6.871095657348633
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.2,7.337568283081055
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.9,6.576014041900635
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.5,1,1,13.875,6.321053743362427
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.275,6.693619966506958
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.575,5.666565656661987
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.7,6.397712707519531
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,1.0,1,0,13.025,6.588059186935425
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.825,7.0624473094940186
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.1,7.032167434692383
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.15,6.666155576705933
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.225,7.266287565231323
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.375,6.4707136154174805
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.75,5.862960338592529
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.025,6.348148584365845
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.975,7.759114980697632
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.15,7.508359670639038
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.325,6.926620721817017
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.7,6.859611034393311
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.3,6.318139791488648
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.65,6.181025505065918
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.325,6.64674186706543
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.425,6.199657201766968
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.175,6.432419061660767
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.175,6.645260334014893
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.0,6.062987327575684
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.775,8.121251106262207
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.2,7.158394813537598
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.275,7.158528804779053
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.075,5.779669761657715
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.925,6.84895396232605
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.375,7.128132581710815
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,12.9,5.6332688331604
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.75,8.055201053619385
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.725,7.838408708572388
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.7,7.753859043121338
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.4,6.595001220703125
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,1.0,1,0,13.025,6.62216067314148
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.725,6.502557039260864
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.85,8.480995178222656
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.1,7.464139938354492
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.5,7.320145606994629
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,14.4,6.618927001953125
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,12.825,7.074201345443726
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,12.975,7.11192512512207
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.85,6.103050470352173
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.725,7.064911842346191
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.55,5.619410753250122
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.675,5.003295183181763
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,13.975,4.685333251953125
B,noisy,9,0.0002,0.0,1,1.0,1,9,9,9,40,0.0,0,0,12.7,4.856600522994995
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,15.8,9.601568937301636
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,17.16,8.595518350601196
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.86,8.423880100250244
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,15.84,7.193397521972656
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.09,7.698802471160889
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.07,7.402424097061157
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.16,8.213037729263306
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.57,8.424339056015015
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.47,9.182578802108765
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.33,8.951931953430176
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.65,9.22136902809143
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.44,7.929232358932495
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.5,1,1,16.22,8.095574378967285
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.6,9.405216932296751
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.73,8.79811978340149
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.31,7.860355615615845
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.7,8.086092472076416
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.33,8.723875999450684
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.27,7.943960189819336
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,16.21,10.40070629119873
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.5,1,1,16.7,8.758880615234375
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.25,7.98965048789978
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.31,8.634502172470093
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.15,7.5577216148376465
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,15.89,7.626079559326172
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.18,7.7581610679626465
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,15.77,10.008943557739258
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.5,1,1,16.2,8.823477983474731
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,16.93,9.405625820159912
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.5,1,1,16.64,8.339160680770874
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.42,8.737503051757812
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,15.14,7.754394054412842
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,17.24,9.973012685775757
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.1,8.476073503494263
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.5,1,1,16.73,7.137808799743652
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.24,7.806087017059326
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.13,8.420619249343872
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,17.03,9.114246606826782
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,15.89,10.045743227005005
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,15.69,8.17084550857544
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,15.59,9.533640146255491
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.5,1,1,16.51,8.876005411148071
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.39,7.326523542404175
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,16.19,8.853045463562012
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.76,8.221312284469604
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.64,8.06198787689209
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.5,1,1,16.31,7.678882598876953
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.54,7.645498037338257
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.24,8.046957015991211
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,15.49,8.373027563095093
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.56,9.095326900482178
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.57,8.343029022216797
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,16.39,8.06710147857666
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.23,8.814745426177979
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,16.51,8.87009572982788
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,2.0,2,0,16.31,7.879565954208374
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.67,9.009212017059326
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,15.92,8.53003454208374
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,16.67,7.970895528793335
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.59,8.301271915435791
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.09,8.241657972335815
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.77,8.687570571899414
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,15.67,7.001798629760742
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.92,8.064099311828613
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.4,7.995224475860596
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,15.8,8.11269760131836
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.11,9.115312814712524
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.37,8.196078300476074
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.44,7.719848394393921
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.79,7.587470293045044
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,15.86,8.446643590927124
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.59,8.648493766784668
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,16.44,8.185625553131104
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.5,2,1,16.12,7.9258811473846436
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,16.63,7.257238388061523
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.25,8.13584852218628
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,16.47,7.797861814498901
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,16.57,8.383086442947388
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.89,8.647479772567749
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,15.52,6.909185409545898
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.9,8.271451950073242
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.61,9.20335078239441
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,16.23,7.706018924713135
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.5,1,1,16.42,7.901308536529541
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,2.0,2,0,17.38,8.263511896133423
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.19,8.088307619094849
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,16.61,8.086162805557251
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,15.77,8.770941257476807
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.73,7.515904903411865
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,16.15,8.466615915298462
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,2,2,16.94,8.37667179107666
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,17.03,8.93982481956482
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.2,7.807558059692383
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,15.87,6.848307847976685
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.38,7.909721612930298
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,15.97,8.685479164123535
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.5,2,1,16.03,9.361400127410889
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,16.4,7.786453247070312
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,15.5,10.034274816513062
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.71,9.37497091293335
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.01,9.84441304206848
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.09,9.639196395874023
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.44,8.452702283859253
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,17.12,8.217760562896729
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.56,8.23105764389038
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.28,8.396699666976929
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.76,8.15377688407898
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,16.09,8.407743692398071
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,15.71,9.598116636276243
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.22,9.8959481716156
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.41,9.320079565048218
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,16.02,9.79362177848816
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,16.58,9.539158582687378
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,16.68,10.11154007911682
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.26,9.308912754058838
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,16.47,8.270998477935791
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.27,8.572943210601807
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.58,8.088515043258667
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.03,9.029513359069824
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.22,8.328868627548218
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,16.98,8.162935972213745
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.27,8.509778022766113
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.5,1,1,16.33,8.172292232513428
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.79,10.221725463867188
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.19,9.21560525894165
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.5,2,1,16.04,8.258901357650757
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,15.97,8.578635215759277
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.48,8.081786155700684
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,16.92,7.524564743041992
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.38,6.942730665206909
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,16.56,8.14206576347351
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,17.24,8.7882719039917
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.5,1,1,16.28,7.932461261749268
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.28,8.551392793655396
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.41,8.026262283325195
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,1.0,1,0,17.36,8.040492534637451
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.24,7.94393515586853
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.5,1,1,16.62,7.475205421447754
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.47,6.684310436248779
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.66,8.883694887161255
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.48,8.719035863876343
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.15,7.5142152309417725
B,noisy,7,0.0005,0.0,1,1.0,1,7,7,7,100,0.0,0,0,16.07,8.914575815200806
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.5,1,1,4.86,3.7396352291107178
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.26,4.176841974258423
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,5.04,4.148861885070801
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.59,3.865978479385376
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.02,3.957592248916626
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.69,3.41941237449646
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.84,3.6653800010681152
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.7,3.325787305831909
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.02,3.9219963550567623
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.85,3.441257238388061
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.0,2.736560344696045
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,2.0,2,0,5.0,2.676417350769043
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.68,2.82723069190979
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.79,2.4668750762939453
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.89,2.8970437049865723
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.86,2.5339272022247314
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,4.87,2.6725876331329346
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.87,2.6901111602783203
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.05,2.766960620880127
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.85,2.452110290527344
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,5.35,2.664815902709961
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.85,2.7819693088531494
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.76,2.851424217224121
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.78,3.285056829452514
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.99,3.61737322807312
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.0,2.6746022701263428
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.7,2.972050428390503
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.93,3.417178153991699
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.95,2.91849684715271
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.78,2.5848445892333984
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.07,2.908071994781494
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.63,2.791154623031616
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.9,3.076894044876098
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.05,3.381927728652954
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.93,3.05159330368042
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.81,2.90828275680542
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.69,2.685903549194336
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.92,3.0292446613311768
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.13,2.972017526626587
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.29,3.88238525390625
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.98,2.780932664871216
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.95,3.3048975467681885
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.85,2.704002618789673
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.7,2.5830814838409424
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.87,3.1861143112182617
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.82,2.885070323944092
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.99,2.870112657546997
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.09,3.131692409515381
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.6,2.7839250564575195
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.21,2.8451387882232666
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.23,2.760270357131958
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.62,2.540367364883423
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.9,3.1014204025268555
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.9,2.7598061561584477
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.0,2.938928604125977
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.82,2.968985319137573
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.86,3.1485135555267334
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.73,3.199215173721313
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.55,2.67795729637146
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.54,2.746693849563598
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.31,3.211268424987793
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.91,3.202312707901001
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.75,2.708528518676758
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.57,2.7521443367004395
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.31,2.8008553981781006
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.05,3.1003835201263428
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.69,2.9725027084350586
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.93,2.5886993408203125
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.12,3.005986452102661
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.74,3.354938268661499
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.96,3.274021625518799
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.85,2.823868989944458
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.87,3.4927546977996826
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.97,2.7843334674835205
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,4.87,2.8623411655426025
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.65,2.546515703201294
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.76,2.633927345275879
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.79,2.669085741043091
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.81,2.859253406524658
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.68,2.6515581607818604
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.89,2.7728874683380127
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.15,2.7992095947265625
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.02,2.6429648399353027
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.81,2.833055257797241
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.85,2.856516122817993
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.15,2.788733720779419
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.58,3.007528066635132
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.83,3.531451940536499
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.93,2.5275418758392334
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.78,2.581599712371826
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.01,2.8341243267059326
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.85,2.7626161575317383
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.89,2.811917781829834
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.04,3.070028781890869
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.79,2.985902547836304
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,5.12,3.752842664718628
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.2,3.044162511825561
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.79,3.027420282363892
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.83,3.568901538848877
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.08,2.905028343200684
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.19,2.7470593452453613
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.03,2.638508081436157
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.21,3.326341152191162
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.99,2.8256819248199463
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.98,2.8503658771514893
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.73,3.04512357711792
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.91,2.81258225440979
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.96,2.6143784523010254
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.23,3.2078969478607178
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.08,2.859737634658813
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.22,2.960576057434082
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.65,2.64068865776062
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.05,3.027482032775879
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.0,2.8680636882781982
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.97,2.8085567951202397
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.8,2.9298481941223145
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.68,3.0953547954559326
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.75,2.866978168487549
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.05,3.032698392868042
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.03,3.094630718231201
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.18,2.832959651947021
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.94,2.784735202789306
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.92,2.9577085971832275
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.88,2.635485887527466
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,5.13,2.8448421955108643
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.33,2.778477907180786
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.7,2.45922589302063
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.99,2.74881362915039
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,4.85,3.0651180744171143
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.97,2.6655526161193848
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.49,2.7354249954223637
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.33,2.845294237136841
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,4.8,3.141577959060669
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.86,2.904993772506714
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,2.0,2,0,4.65,2.495714902877808
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.95,2.934345006942749
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,5.12,2.8226709365844727
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.3,2.908541440963745
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.95,2.833181619644165
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.19,2.8546135425567627
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.85,3.0099856853485107
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.17,3.046183824539185
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.03,3.096410036087036
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.77,3.0480399131774902
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.21,3.124408006668091
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.35,2.897644281387329
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.16,2.8216614723205566
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.85,2.6122796535491943
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.87,2.7541632652282715
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.5,1,1,4.61,2.848363876342773
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.82,2.712621688842773
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.81,2.717343330383301
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.94,2.5251052379608154
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.94,2.7369277477264404
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.07,2.606823444366455
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.17,2.832810401916504
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.78,2.5001535415649414
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.19,2.4545743465423584
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.91,2.7413694858551025
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.77,2.5018539428710938
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.02,2.886948823928833
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.0,2.785043478012085
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.34,2.813192129135132
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,4.63,2.4727001190185547
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.94,2.770147562026977
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.14,2.793076992034912
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.5,1,1,4.98,3.1923842430114746
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.14,3.00326943397522
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.97,3.177171468734741
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.74,2.772526502609253
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.93,3.0270557403564453
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.08,2.8683853149414062
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.49,2.858248472213745
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.5,1,1,4.63,2.47776198387146
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.65,2.9701876640319824
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.98,2.911343097686768
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,5.25,2.736379384994507
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.76,2.786046266555786
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,4.75,2.635748624801636
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.97,2.6126723289489746
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.96,3.354440927505493
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.67,3.321340322494507
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.77,2.958486318588257
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.17,3.455283164978028
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.9,2.5283169746398926
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.5,1,1,4.63,2.6161234378814697
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.76,2.748676061630249
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.94,2.5781874656677246
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.88,2.6382365226745605
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.1,2.9120185375213623
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.63,2.6261656284332275
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.05,2.504623413085937
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.5,1,1,5.33,2.917141914367676
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.85,2.7649378776550293
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.31,3.295085430145264
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.05,3.0019099712371826
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.84,3.4735100269317627
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.14,3.054274082183838
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.24,2.712665319442749
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.84,2.743387460708618
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.87,3.472832679748535
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.23,2.637010335922241
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.89,3.078288078308105
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.09,2.8014347553253174
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.79,3.087839126586914
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.04,3.209302425384521
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.72,3.1281564235687256
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,4.79,2.972557783126831
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.29,2.8071389198303223
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,5.11,2.568603992462158
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.22,2.6964221000671387
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,2.0,2,0,4.69,2.7702767848968506
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.75,2.660561800003052
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.41,2.66328239440918
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.9,2.635404348373413
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.82,2.9513726234436035
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.87,2.732365369796753
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.1,2.885514736175537
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.78,2.4449055194854736
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.08,2.663973569869995
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.5,2,1,4.97,3.172877550125122
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.13,3.661713361740112
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.88,3.044590473175049
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.5,1,1,4.55,3.797964334487915
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.77,3.212357521057129
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.94,2.9705848693847656
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.11,3.133106470108032
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.88,3.287272930145264
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.72,3.9200243949890137
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.11,2.9466452598571777
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.78,3.0004751682281494
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.03,2.968138694763184
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.9,2.979766368865967
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.94,3.29610538482666
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.01,3.5472869873046875
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.96,3.7409303188323975
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.79,3.1959264278411865
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.88,3.119746685028076
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.9,3.0626399517059326
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.79,2.860734224319458
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.23,2.84204649925232
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.99,2.8329296112060547
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.64,2.9668667316436768
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.75,2.999964475631714
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.9,3.0236854553222656
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.08,3.0044150352478027
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.18,3.0206661224365234
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.99,2.884748697280884
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,5.15,2.519239902496338
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.2,3.6659865379333496
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.59,2.768584489822388
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.53,3.4124467372894287
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.2,3.0729029178619385
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.99,2.9929299354553223
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.06,2.731450319290161
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.15,3.5238397121429443
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,4.6,3.0427865982055664
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.71,3.462195634841919
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.91,2.934917688369751
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,5.2,3.058313131332397
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.94,2.8907082080841064
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.99,3.2117950916290283
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.84,3.024458408355713
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.71,3.63360047340393
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.89,3.250800848007202
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.36,3.818449258804321
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.1,3.1044514179229736
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.23,3.1902313232421875
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.75,3.8326611518859854
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.02,3.612300395965576
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.05,2.993303060531616
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.73,3.1377744674682617
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.2,2.765755176544189
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.82,3.231266975402832
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.7,2.757615804672241
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.3,2.7954816818237305
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.5,1,1,5.14,2.9402048587799072
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.93,2.850505828857422
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.78,2.8727927207946777
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.49,2.593719720840454
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.5,1,1,4.91,3.194244623184204
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.96,3.052950143814087
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.84,2.758214473724365
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.27,2.949152708053589
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.97,2.743605852127075
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.05,2.749314069747925
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.91,2.6278011798858643
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,5.0,3.2160017490386963
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.2,2.8082544803619385
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.94,3.343639373779297
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.76,3.158987522125244
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.94,2.750492811203003
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.41,2.659766435623169
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.26,2.920127868652344
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.16,2.762138605117798
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.53,3.0468204021453857
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.32,2.795796155929565
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.15,2.607591390609741
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.05,2.9717931747436523
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.8,2.5908758640289307
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.24,2.969536781311035
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.13,2.988513231277466
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.21,3.011746883392334
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.04,2.8215172290802
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.65,2.5261247158050537
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.75,2.8155670166015625
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.0,2.7920026779174805
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.12,3.049292087554932
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.06,3.12686538696289
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.06,2.7526605129241943
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.81,2.6420693397521973
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.94,3.0341925621032715
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.66,2.5359911918640137
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.31,2.747885227203369
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.0,2.4997096061706543
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.95,3.074244737625122
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.13,2.807324886322021
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.94,2.5534026622772217
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.02,2.845812559127808
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.27,3.0726237297058105
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.64,3.0693366527557373
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.86,3.2593414783477783
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.77,3.1869800090789795
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.81,3.596318006515503
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,4.76,2.9713616371154785
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.85,3.4386935234069824
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.94,3.1754841804504395
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.31,2.966878890991211
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.83,2.8695456981658936
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.44,2.871568202972412
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.88,2.9252872467041016
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.81,2.943911552429199
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.02,3.1861722469329834
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.12,2.881696939468384
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.76,2.7036550045013428
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.32,2.819279909133911
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.06,3.5665652751922607
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.7,2.6563682556152344
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,4.72,3.019860029220581
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,4.84,3.1516404151916504
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.84,3.601933717727661
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.5,3.211035251617432
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.75,3.5635132789611816
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.93,3.1029179096221924
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.02,3.208770275115967
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.84,3.2789864540100098
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.92,3.0274174213409424
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.34,3.178270101547241
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.84,3.0819592475891118
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.22,2.67852783203125
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.66,3.009372472763061
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,5.45,3.177048921585083
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.6,3.1037633419036865
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.92,2.859212875366211
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.11,3.160952091217041
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.11,3.100170135498047
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.95,3.042132377624512
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.01,3.052736520767212
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.05,2.8177850246429443
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.49,2.5373880863189697
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.26,3.1705851554870605
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.98,3.4800970554351807
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,5.4,3.083004951477051
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.07,2.955559015274048
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,4.88,3.0995912551879883
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,5.03,3.981767892837525
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.61,3.2530832290649414
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.56,2.944849729537964
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.98,2.750816345214844
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.97,2.7554585933685303
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,5.15,2.5756995677948
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.77,2.869178056716919
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.91,2.684266328811645
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.16,3.0411901473999023
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.19,3.101746082305908
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.57,2.6193649768829346
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.57,2.578649044036865
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.0,2.38895320892334
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,4.89,2.6539547443389893
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.1,3.2071757316589355
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,1.0,1,0,5.03,3.4214096069335938
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.08,2.8988029956817627
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.12,2.8404579162597656
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.22,2.387502193450928
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.54,2.5429916381835938
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.96,3.4819412231445312
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.01,3.0235495567321777
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.77,3.2523269653320312
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.99,2.917116403579712
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.5,1,1,5.07,2.9320013523101807
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.98,2.8208255767822266
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.87,2.717284679412842
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.53,2.5503385066986084
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.98,2.85432505607605
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.69,3.152280569076538
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.94,2.691932916641236
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.91,3.0229947566986084
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.97,3.364461898803711
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.04,2.8675882816314697
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.8,3.2667105197906494
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,4.86,2.979029893875122
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.12,3.251568078994751
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.02,3.078741312026977
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.5,1,1,5.21,3.0795130729675293
B,noisy,7,0.0002,0.0,1,0.4,1,7,3,7,100,0.0,0,0,5.36,2.616647958755493
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.0,2,0,15.075,8.62977147102356
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,1.0,1,0,15.3575,8.757623672485352
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.0,2,0,15.29,9.036523580551147
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,5.5,6,1,15.2,9.70201301574707
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.5,3,1,14.9725,8.2593355178833
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,0.5,1,1,15.065,9.518683195114136
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,0.5,1,1,14.945,8.903533697128296
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.0,2,0,15.14,8.23839259147644
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,1.0,1,0,15.1475,8.677752494812012
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.0,3,2,15.405,8.734745025634766
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.0,2,0,15.3125,8.507224798202515
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.0,2,0,14.9175,8.417330265045166
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,0.0,0,0,15.1325,10.07457184791565
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,3.5,4,1,15.005,9.445522546768188
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.0,2,0,14.96,9.30550456047058
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,3.0,3,0,15.02,9.22749662399292
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.0,2,0,15.26,8.887807607650757
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,3.0,3,0,14.8575,9.733816146850586
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,3.5,4,1,15.0125,9.5673725605011
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.5,3,1,14.9675,9.719911813735962
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,1.0,1,0,15.05,10.764806270599363
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,5.0,5,0,15.4475,8.904630184173584
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,7.5,8,1,15.0725,10.12111520767212
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,3.0,3,0,15.325,9.561322927474976
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.0,2,0,15.3525,8.470339059829712
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,3.0,3,0,15.0425,9.186007738113403
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.0,2,0,15.2375,8.58267617225647
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,3.0,3,0,14.9875,8.53506064414978
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,0.0,0,0,15.2575,8.99258303642273
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,1.5,2,1,15.445,8.797788381576538
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.5,3,1,15.245,8.461233615875244
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.0,2,0,15.355,8.765074968338013
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,3.0,3,0,15.06,9.685993432998655
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,4.0,4,0,15.035,8.791242599487305
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.0,2,0,15.12,9.079152584075928
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,1.0,1,0,15.1675,8.827478647232056
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,1.5,2,1,15.2225,9.304134845733644
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,0.0,0,0,15.2175,9.397507667541504
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.0,2,0,15.205,8.9013831615448
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,1.5,2,1,15.305,10.473019123077393
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,1.0,1,0,15.2275,9.890167713165283
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,6.0,7,2,15.005,9.577582597732544
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,1.0,2,2,14.8,10.85055446624756
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,1.5,2,1,15.045,10.591293096542358
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,3.0,3,0,15.7175,8.79796838760376
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,4.5,5,1,14.8675,9.384624242782593
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,3.0,3,0,15.18,9.59967875480652
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.5,3,1,14.945,9.97106909751892
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,1.0,1,0,15.3,9.265488862991331
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.5,3,1,15.645,10.324798345565796
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,3.0,4,2,15.07,10.61372447013855
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,1.0,1,0,14.7925,9.61403203010559
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.0,2,0,14.94,8.85726284980774
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,0.5,1,1,15.5,8.9440598487854
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,4.0,5,2,15.285,8.432615995407104
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,3.0,3,0,15.0525,8.839205980300903
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,3.5,4,1,15.51,9.41359543800354
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,3.5,4,1,15.115,8.968698501586914
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,3.0,3,0,15.48,8.543611288070679
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,1.0,1,0,15.415,10.363617420196531
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,1.0,1,0,15.175,9.33355951309204
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,1.5,2,1,14.9875,9.397238969802856
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,1.0,1,0,15.265,9.48697566986084
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,1.5,2,1,15.3125,8.814013004302979
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.5,3,1,15.155,9.27122712135315
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,5.0,5,0,15.305,9.10992431640625
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.5,3,1,15.2675,9.016143560409546
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,1.0,1,0,15.635,9.570401430130005
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,3.0,3,0,14.8125,8.496206521987915
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,1.0,1,0,14.785,8.892499446868896
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,1.0,1,0,15.1425,8.868440628051758
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.0,2,0,15.22,9.980896472930908
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.0,2,0,15.085,9.36958932876587
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,4.0,4,0,15.2225,9.368221759796144
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.0,2,0,14.9875,10.547659635543823
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,0.0,0,0,15.18,8.965896844863892
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,4.5,5,1,15.2275,9.4465069770813
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,1.0,1,0,15.1675,9.090901613235474
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,1.0,1,0,14.965,9.756765842437744
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,1.0,1,0,15.2,8.667838335037231
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,2.0,2,0,14.9925,9.1670503616333
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,0.0,0,0,15.355,8.795706510543823
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,0.0,0,0,15.24,8.289981603622437
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,0.5,1,1,15.085,9.11866307258606
A,noisy,5,0.0003,0.0003,1,1.0,1,5,5,5,400,4.0,5,2,15.375,9.342000246047974
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,5.0,7,4,11.94,7.643635034561157
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,1.5,3,3,12.365,7.648684501647949
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,2.5,3,1,12.11,7.822174549102783
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,2.5,3,1,11.9175,8.339834451675415
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,11.8725,8.235056638717651
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,6.0,6,0,12.49,8.430686473846436
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,5.0,5,0,12.02,7.9163172245025635
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,1.5,2,1,12.0275,7.791282653808594
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,2.0,3,2,12.2925,7.833162069320679
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,1.5,2,1,12.405,8.21427869796753
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,1.5,2,1,12.195,7.911384105682373
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,7.0,7,0,12.3075,7.840509176254272
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,2.0,3,2,12.4775,8.793640613555908
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,12.4275,7.773479700088501
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,11.875,7.951068639755249
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,12.3425,8.210946559906006
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,11.865,7.910457611083984
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,12.0375,7.610671043395996
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,7.0,7,0,12.39,7.849118232727051
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,4.0,5,2,12.1975,7.587919950485229
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,2.0,2,0,12.42,7.627679109573364
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,4.0,4,0,12.2125,7.320048570632935
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,5.0,6,2,11.95,8.61132287979126
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,6.0,7,2,12.115,9.481467723846436
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,1.5,2,1,12.395,8.165978908538818
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,2.5,3,1,12.2475,7.954546213150024
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,12.17,7.501978158950806
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,4.0,4,0,12.1925,7.767120599746704
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,2.5,3,1,12.015,7.929567098617554
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,12.285,8.047842025756836
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,12.5625,7.98487401008606
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,5.0,5,0,12.2525,8.200932264328003
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,2.0,3,2,12.105,7.678199768066406
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,3.5,5,3,12.175,7.8893141746521
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,2.5,3,1,12.285,7.602803707122803
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,2.0,2,0,12.1975,7.257055044174194
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,3.5,5,3,12.22,7.675772666931152
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,4.0,5,2,12.0625,7.737405300140381
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,2.5,3,1,12.0325,8.358926773071289
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,3.0,3,0,12.4325,9.222220420837402
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,3.5,4,1,12.1825,8.30505084991455
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,2.0,3,2,12.435,8.457144737243652
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,2.5,4,3,12.1325,7.931943893432617
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,1.5,2,1,12.3975,7.56687331199646
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,3.5,4,1,12.2475,7.574040174484253
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,3.0,3,0,12.2575,7.8766162395477295
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,4.0,4,0,11.9675,6.98388934135437
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,12.08,7.673023462295532
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,4.0,5,2,12.14,7.381646633148193
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,4.0,4,0,12.175,7.636131286621094
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,2.0,3,2,12.245,8.230940818786621
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,5.0,5,0,12.1125,7.643003225326538
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,2.0,2,0,12.06,7.724832057952881
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,12.235,7.555789947509766
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,2.5,3,1,12.4875,7.0093629360198975
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,12.0675,7.8358399868011475
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,4.0,5,2,12.025,7.72913122177124
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,5.0,6,2,12.1925,7.83511209487915
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,3.5,5,3,12.14,7.583348274230957
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,3.0,4,2,12.285,7.233047485351562
B,noisy,5,0.001,0.0,1,1.0,1,5,5,5,400,4.0,4,0,12.0875,7.895064830780029
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.27,4.354840040206909
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.235,4.368371963500977
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,1.5,2,1,1.1725,4.250482082366943
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.4,5.055371522903442
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,1.5,2,1,1.275,4.54009747505188
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1025,4.575024843215942
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.295,4.5737292766571045
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2325,4.648861646652222
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.295,4.69401216506958
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.245,4.488608360290527
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,1.2175,4.46272349357605
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2225,4.202929258346558
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2025,4.527400970458984
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1725,4.879632949829102
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.255,4.3662965297698975
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.4025,4.611922025680542
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1975,4.678384780883789
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2625,4.479901075363159
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2525,4.541329383850098
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1825,4.290117502212524
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2025,4.272133827209473
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2825,4.495480298995972
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2625,4.660990238189697
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.3,4.714017391204834
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.5,1,1,1.16,4.520708084106445
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.295,4.4388275146484375
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.215,4.323782205581665
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.19,4.222459554672241
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2175,4.081844806671143
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.27,4.2590012550354
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.19,4.547807216644287
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.245,4.684865236282349
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2575,4.645694732666016
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1575,4.872653245925903
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2575,4.790928840637207
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2775,4.157546281814575
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1275,4.338290452957153
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.24,4.845515966415405
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.245,4.48347544670105
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2175,4.530385255813599
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2475,4.207046031951904
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1525,4.242846250534058
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.225,4.871163368225098
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1025,4.630630016326904
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2175,4.661792516708374
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.3025,4.7495033740997314
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2125,4.278346300125122
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2475,4.400028228759766
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,1.2325,4.466531991958618
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.285,4.7617104053497314
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.3,4.2841637134552
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1475,4.335527420043945
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2425,4.364535808563232
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1325,4.24252462387085
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2725,4.554892539978027
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.3225,4.616305112838745
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2375,5.251244306564331
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2225,4.491949081420898
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2275,4.796545743942261
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.5,1,1,1.1925,4.835517406463623
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2125,4.747575998306274
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,1.2525,5.128767728805542
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.28,4.522065877914429
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.255,4.613940000534058
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.175,4.827385902404785
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1925,4.555587768554688
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.3325,4.72819709777832
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.5,1,1,1.22,5.181275606155396
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.3025,5.367567300796509
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.315,4.930322408676148
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.5,1,1,1.185,4.5670084953308105
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.255,4.712371826171875
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2325,4.853632211685181
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.17,4.557880163192749
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.375,5.074949741363525
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.215,4.966902732849121
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1525,4.514731407165527
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.5,1,1,1.215,4.556953430175781
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.26,4.8932178020477295
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2825,4.163203716278076
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2225,4.356502532958984
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,1.32,4.879108428955078
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1875,4.370055913925171
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,1.0575,4.429565191268921
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1775,4.407741785049439
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.5,1,1,1.2175,4.36871600151062
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1875,4.419411897659302
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.27,4.731749773025513
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,1.2975,4.709349155426025
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2275,5.221709728240967
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.205,4.500674486160278
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2025,5.20804762840271
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.21,5.272872447967529
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.5,1,1,1.28,4.913162708282471
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.22,5.250474452972412
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.21,5.37221622467041
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2825,4.714182138442993
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1975,4.437231540679932
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,1.25,4.448566913604736
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,2.0,2,0,1.2325,4.745882987976074
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2225,4.728713750839233
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.295,4.5984601974487305
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.265,4.617619037628174
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.225,3.9723381996154794
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2675,4.398316383361816
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2275,4.601852178573608
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2075,4.170830965042114
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2725,4.266480207443237
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.225,4.699409008026123
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2025,4.504662990570068
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.24,4.6152989864349365
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.11,5.001932621002197
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.5,1,1,1.1925,4.629357576370239
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.265,4.6424400806427
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.5,1,1,1.145,4.226802349090576
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2075,4.334960699081421
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2225,4.300718784332275
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.3,4.71232795715332
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2275,4.533338069915772
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2025,4.608409643173218
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.185,4.477956295013428
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,1.3,4.92748236656189
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,1.205,5.1791627407073975
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2375,4.389551401138306
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1925,4.553401947021484
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2775,4.453895330429077
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2975,4.674983978271484
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.13,4.4986302852630615
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2275,4.560600757598877
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.3525,4.399777173995972
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1725,4.916548252105713
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1625,4.1362059116363525
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.205,4.037216663360596
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2575,4.415266036987305
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.24,4.370346307754517
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,1.2175,4.764031648635864
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.205,4.585614919662476
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1625,4.605668306350708
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2175,4.411402702331543
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.12,4.512991428375244
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2425,4.67345929145813
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1975,4.490109443664551
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,1.12,4.407411098480225
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2425,4.8291778564453125
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.155,4.638944625854492
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2475,4.361438512802124
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.11,4.570725440979004
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.235,4.335106372833252
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.24,4.677479267120361
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,1.2325,4.504243612289429
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1075,4.45623517036438
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.155,4.710028171539307
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,1.23,4.690933704376221
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2025,4.514262914657593
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.28,4.274323225021362
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.3325,4.590766668319702
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.155,4.30188512802124
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2525,4.624057292938232
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.5,1,1,1.13,4.353859663009644
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.215,4.598891019821167
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2025,4.624277591705322
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2525,4.105382204055786
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.23,4.4640092849731445
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.245,4.716772556304932
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,1.235,4.846377372741699
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.16,4.372908592224121
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.225,4.496639490127564
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,1.2925,4.687778234481812
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2125,4.483429431915283
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.205,4.066751480102539
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.195,4.485291481018066
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2125,4.212972402572632
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2625,4.142502546310425
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.23,4.614337921142578
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.23,4.490286350250244
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.19,4.536473035812378
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.235,4.520934104919434
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.195,3.973912477493286
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,2.0,2,0,1.27,4.806153297424316
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2225,4.142019748687744
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1925,4.5553505420684814
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.175,4.41499662399292
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.225,4.541710615158081
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.3325,4.645239353179932
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.21,4.352630376815796
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2975,4.603966474533081
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2925,4.652547836303711
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1325,4.872714996337891
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.255,4.284907579421997
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.235,4.46039605140686
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.255,4.691022872924805
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.205,4.81857705116272
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.205,4.428447961807251
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2575,4.806849241256714
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2575,4.070485353469849
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1475,4.524560689926148
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.0925,4.556807041168213
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.15,4.853933095932007
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2425,4.461458444595337
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2475,4.46913743019104
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.315,4.59567141532898
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.24,4.552423715591431
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2025,4.828872203826904
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,1.235,4.262883186340332
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2175,4.108331203460693
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.165,4.451291799545288
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1125,4.440046072006226
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.235,4.921336650848389
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.27,5.48188066482544
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.21,4.511885643005371
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2275,4.664393186569214
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.16,4.535607099533081
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2075,4.63299036026001
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,1.2275,5.205472469329834
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.21,5.084350347518921
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.195,5.056232452392578
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.21,4.551919460296631
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.255,4.485177993774414
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2425,4.445031404495239
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,1.27,4.691316604614258
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2275,4.49451756477356
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1225,4.41240668296814
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2025,4.676344156265259
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.16,4.269799470901489
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2225,4.3794920444488525
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1675,4.086296319961548
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2525,4.711468935012817
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1125,4.0724709033966064
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.255,4.745828866958618
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1175,4.013134241104126
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1875,4.46994948387146
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2725,4.527509450912476
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.19,4.795246601104736
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2325,4.576289176940918
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.29,4.987680673599243
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.5,1,1,1.2375,4.905855655670166
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2425,4.267421007156372
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.205,3.998628854751587
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2675,4.496715307235718
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1925,4.475730657577515
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2575,4.581823110580444
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.265,4.723889112472534
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,1.2475,4.543118000030518
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1875,4.850334167480469
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.5,1,1,1.2575,4.9719953536987305
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.275,4.70269250869751
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.22,4.469627618789673
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,1.0,1,0,1.1325,4.5524818897247314
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.195,4.428541660308838
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2125,4.390151500701904
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1975,4.457850456237793
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2875,4.812913656234741
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.26,4.576093912124634
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.19,4.69372820854187
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1725,4.464799404144287
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.215,4.885795831680298
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.155,4.728520154953003
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.2375,4.517346858978272
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.18,4.452052116394043
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.205,4.7120041847229
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.295,4.709152698516846
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.245,4.296287059783936
B,noisy,5,0.0001,0.0,1,1.0,1,5,5,5,400,0.0,0,0,1.1475,4.329030752182007
A,noisy,3,0.001,0.001,1,1.0,1,3,3,3,400,69.5,71,3,11.86,2.6302168369293213
A,noisy,3,0.001,0.001,1,1.0,1,3,3,3,400,67.5,70,5,11.92,2.5936367511749268
A,noisy,3,0.001,0.001,1,1.0,1,3,3,3,400,76.0,80,8,12.3325,2.6316781044006348
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,5.5,6,1,3.6675,1.939136505126953
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,2.0,2,0,3.6125,2.036533832550049
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,8.0,8,0,3.5225,1.9299383163452148
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,6.0,6,0,3.5125,2.0085220336914062
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,10.0,10,0,3.6375,1.9571919441223145
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,6.0,6,0,3.825,1.952627658843994
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,8.0,8,0,3.5475,2.0575900077819824
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,8.0,8,0,3.5175,2.1452226638793945
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,6.0,6,0,3.46,2.03238844871521
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,3.5,4,1,3.63,2.06577730178833
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,6.0,6,0,3.555,1.948305368423462
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,6.0,6,0,3.66,2.136629104614258
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,12.0,12,0,3.6775,1.9730637073516848
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,6.0,6,0,3.54,1.979887008666992
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,7.0,7,0,3.5325,2.138366460800171
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,4.0,4,0,3.5025,2.166513442993164
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,12.0,12,0,3.7525,2.0659637451171875
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,5.0,5,0,3.3675,1.9926378726959229
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,5.5,6,1,3.2575,1.9731571674346924
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,10.0,10,0,3.2825,2.276994466781616
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,10.5,11,1,3.6725,2.2568914890289307
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,6.0,6,0,3.5475,2.2452893257141118
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,11.5,12,1,3.515,2.1976022720336914
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,6.0,6,0,3.6475,2.1226611137390137
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,4.0,4,0,3.6575,2.225743055343628
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,7.5,8,1,3.6625,2.0522186756134038
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,3.5,4,1,3.4725,1.886223316192627
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,9.5,10,1,3.625,1.8422870635986328
A,noisy,3,0.0003,0.0003,1,1.0,1,3,3,3,400,8.0,8,0,3.66,1.912135124206543
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,8.5,9,1,2.73,1.4540209770202637
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,4.0,4,0,2.755,1.5624706745147705
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,5.0,5,0,2.89,1.7380874156951904
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,4.5,6,3,2.6925,1.5471858978271484
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,7.0,8,2,2.705,1.5532832145690918
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,3.5,4,1,2.8075,1.7130906581878662
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,8.0,8,0,2.795,1.611842155456543
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,4.0,4,0,2.79,1.6770460605621338
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,4.0,4,0,2.6575,1.7591984272003174
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,1.5,2,1,2.7075,1.9587600231170652
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,8.0,8,0,2.85,1.909573316574097
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,2.0,2,0,2.71,1.7780189514160156
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,3.5,4,1,2.7,1.6618220806121826
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,9.5,10,1,2.665,1.8292255401611328
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,3.0,3,0,2.635,1.7040667533874512
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,4.5,5,1,2.7625,1.6875600814819336
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,2.5,3,1,2.69,1.675205945968628
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,3.0,3,0,2.7025,1.623506784439087
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,1.0,2,2,2.6625,1.516559600830078
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,3.0,3,0,2.5725,1.6137809753417969
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,3.0,3,0,2.765,1.623100996017456
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,5.0,6,2,2.6975,1.9203448295593264
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,3.0,3,0,2.735,1.6631426811218262
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,3.0,3,0,2.7475,1.6138596534729004
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,7.0,8,2,2.775,1.721719741821289
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,4.0,4,0,2.8125,1.8157005310058596
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,5.5,6,1,2.8025,1.793827772140503
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,4.5,5,1,2.7025,1.5547327995300293
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,4.0,5,2,2.6625,1.6346337795257568
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,4.5,5,1,2.6575,1.8540503978729248
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,3.5,4,1,2.7625,1.7688891887664795
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,2.5,3,1,2.6375,1.861889123916626
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,1.0,2,2,2.685,1.7982325553894043
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,5.5,6,1,2.9125,1.7880492210388184
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,6.5,8,3,2.68,1.6109182834625244
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,8.5,10,3,2.8225,1.5848209857940674
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,7.5,8,1,2.6675,1.690073013305664
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,3.0,3,0,2.8825,2.101268529891968
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,5.5,6,1,2.8975,1.6780672073364258
B,noisy,3,0.001,0.0,1,1.0,1,3,3,3,400,4.5,5,1,2.7725,1.745006799697876
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3,1.2394065856933594
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3075,1.248175859451294
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2475,1.401625394821167
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2825,1.6751558780670166
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.27,1.8156416416168213
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2825,1.3997485637664795
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.245,1.2948262691497805
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3175,1.6435716152191162
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.2975,1.402554273605347
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.285,1.316494703292847
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.3468232154846191
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2525,1.4292643070220947
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2425,1.4564332962036133
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.25,1.4699125289916992
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.28,1.358485221862793
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.27,1.2169313430786133
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2725,1.2924039363861084
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.275,1.1071889400482178
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.27,1.6636650562286377
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.24,1.2814695835113523
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.245,1.303654432296753
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2525,1.4034647941589355
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.275,1.2377824783325195
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.25,1.161665439605713
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2425,1.2520949840545654
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.255,1.1602253913879397
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.31,1.408501386642456
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2575,1.2957115173339844
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2975,1.2404773235321045
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.24,1.3006234169006348
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2475,1.1166760921478271
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2825,1.3508644104003906
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2225,1.3944194316864014
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2725,1.2940800189971924
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.2975,1.354750394821167
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.29,1.227198839187622
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.2786521911621094
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2875,1.235386848449707
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.275,1.5749998092651367
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3,1.3695435523986816
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2875,1.2288119792938232
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.23,1.4042932987213137
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.315,1.3414199352264404
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.23,1.1583764553070068
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.2675,1.174877405166626
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2875,1.4863348007202148
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3025,1.2713303565979004
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.255,1.5583865642547607
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2475,1.2496590614318848
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.285,1.2343120574951172
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2775,1.295539379119873
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.235,1.352635622024536
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.285,1.322825908660889
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2825,1.2205662727355957
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.21,1.2202298641204834
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.305,1.1843914985656738
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2825,1.4371650218963623
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.2925,1.341676950454712
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.31,1.3726489543914795
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2475,1.1843678951263428
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2325,1.151078462600708
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.305,1.115692138671875
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2775,1.06048846244812
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.245,1.3085589408874512
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2675,1.1499199867248535
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.32,1.270423173904419
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2625,1.171299934387207
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.245,1.1726603507995603
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.26,1.275167465209961
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.245,1.3648576736450195
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2675,1.3869431018829346
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.6296873092651367
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2825,1.8067083358764648
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.275,1.4161944389343262
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2375,1.424074649810791
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.487846612930298
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2725,1.5697996616363523
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.285,1.34122633934021
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.29,1.4193992614746094
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.24,1.2304408550262451
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.29,1.560494899749756
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.285,1.2361068725585938
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.285,1.475637912750244
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2725,1.357196807861328
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2875,1.5455472469329834
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.285,1.6652626991271973
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2525,1.322303056716919
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2175,1.54961895942688
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2975,1.3097591400146484
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.36,1.4281623363494873
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.255,1.4039759635925293
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.26,1.3677268028259275
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.275,1.553649663925171
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2975,1.5635817050933838
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.2725,1.679973840713501
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2825,1.715568780899048
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.29,1.4372835159301758
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3,1.3892455101013184
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2375,1.3688068389892578
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.255,1.213097095489502
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2325,1.168808937072754
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.2825,1.2832849025726318
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2775,1.3320293426513672
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2425,1.2509851455688477
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2275,1.4167213439941406
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2375,1.3010101318359375
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.23,1.2462024688720703
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.345,1.1890268325805664
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.2931292057037354
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2475,1.339930534362793
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.275,1.2962324619293213
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.26,1.2796003818511963
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.167567014694214
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.267627477645874
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.3175,1.246119737625122
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.2418663501739502
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2775,1.2483689785003662
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.27,1.2938292026519775
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.315,1.223376750946045
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.295,1.304908037185669
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3025,1.332336187362671
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.2575390338897705
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.235,1.2058274745941162
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.295,1.1534388065338137
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.27,1.110457420349121
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2775,1.1950404644012451
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.25,1.2274930477142334
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2725,1.280289888381958
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.29,1.2005751132965088
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.2525,1.2312965393066406
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2675,1.311601638793945
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.295,1.2811591625213623
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2875,1.1750483512878418
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2675,1.2188410758972168
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2475,1.2202088832855225
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.275,1.2826623916625977
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.2825,1.313744068145752
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3,1.2458264827728271
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.225,1.2175474166870115
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.1728942394256592
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.275,1.2724859714508057
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2675,1.2479362487792969
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2675,1.1345810890197754
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.28,1.100250005722046
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.305,1.2633936405181885
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2975,1.2853753566741943
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2575,1.3823764324188232
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2125,1.2117938995361328
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.31,1.2430963516235352
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2775,1.2317214012145996
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.245,1.2489566802978516
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2625,1.153585433959961
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.285,1.2531523704528809
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2875,1.258620262145996
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.27,1.2765774726867676
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.324812889099121
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.31,1.181751012802124
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.2525,1.4390747547149658
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.255,1.2905120849609375
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.275,1.218095302581787
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2475,1.48842191696167
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.25,1.3716826438903809
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2525,1.345862865447998
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.265,1.1358654499053955
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.275,1.2224128246307373
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2525,1.5137670040130615
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.295,1.4261043071746826
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.28,1.225006341934204
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.32,1.2473618984222412
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2475,1.3890478610992432
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.315,1.6928939819335938
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.23,1.4650676250457764
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2775,1.2213246822357178
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2675,1.1712141036987305
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.235,1.244217872619629
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3075,1.2384543418884275
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2975,1.3626155853271484
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.2517549991607666
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.295,1.384528398513794
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2775,1.2284770011901855
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.4370715618133545
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2325,1.2547953128814695
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3,1.5589101314544678
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.23,1.728867769241333
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2725,1.5252461433410645
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.4560203552246094
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.31,1.4186697006225586
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3025,1.2771663665771484
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.22,1.2215044498443604
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2975,1.2806918621063232
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.225,1.2175886631011963
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.25,1.2113068103790283
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.27,1.0732142925262451
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.285,1.2243983745574951
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.255,1.0865471363067627
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3125,1.169816970825195
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.25,1.2587230205535889
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2875,1.3796768188476562
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2625,1.388497829437256
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2475,1.2549207210540771
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3125,1.2909672260284424
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.27,1.289250373840332
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.255,1.2607581615447998
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2725,1.2059829235076904
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.25,1.2442409992218018
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2475,1.291745662689209
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2375,1.1319000720977783
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2425,1.2280352115631104
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.26,1.4281549453735352
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.2809951305389404
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.29,1.415905237197876
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.28,1.2626848220825195
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.245,1.4168894290924072
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.3025,1.1698393821716309
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2725,1.259613037109375
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.351701021194458
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.26,1.3172121047973633
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2525,1.4130797386169434
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.29,1.298444747924805
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2975,1.2056279182434082
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.29,1.2838478088378906
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2625,1.236903429031372
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.25,1.298926591873169
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2925,1.2820096015930176
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.25,1.324037790298462
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.285,1.2431354522705078
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2525,1.1984028816223145
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.275,1.3236443996429443
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.25,1.2588374614715576
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.187638521194458
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.2775,1.209068775177002
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2725,1.3984403610229492
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.31,1.2199468612670898
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.315,1.2793633937835691
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.275,1.2464232444763184
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.315,1.3576748371124268
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.308180332183838
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.3,1.176689386367798
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.27,1.3162434101104736
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.2725,1.2107009887695312
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.32,1.2365984916687012
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3075,1.2104284763336182
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.255,1.265205144882202
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2975,1.4024055004119873
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2725,1.3133981227874756
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2775,1.3348345756530762
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.4095444679260254
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.325,1.4756479263305664
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.295,1.2500910758972168
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2775,1.249006271362305
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.3375,1.4254286289215088
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2775,1.120575189590454
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3125,1.2830564975738523
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.27,1.2378134727478027
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2575,1.295274019241333
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2625,1.2493553161621094
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2775,1.2589993476867676
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.285,1.19545316696167
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2975,1.6364715099334717
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2975,1.181706428527832
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2425,1.3967626094818115
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.24,1.2808127403259275
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.305,1.3796322345733645
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2475,1.2387099266052246
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.295,1.2412316799163818
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.285,1.3254199028015137
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.25,1.2231853008270264
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.2019364833831787
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.2725,1.1472632884979248
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2875,1.127676248550415
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.136256217956543
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.172879934310913
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.325,1.2307071685791016
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.1509034633636477
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.25,1.3015968799591064
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2525,1.2136588096618652
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.255,1.115009307861328
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2875,1.1593763828277588
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.285,1.1983745098114014
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2525,1.4538562297821045
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.275,1.303739309310913
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2825,1.175029754638672
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2475,1.2562077045440674
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.305,1.16103196144104
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2775,1.468315601348877
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2375,1.3580222129821775
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.29,1.2592854499816897
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2875,1.2981693744659424
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2675,1.1991820335388184
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.23,1.2166430950164795
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2975,1.1700913906097412
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.26,1.453929424285889
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.29,1.267137050628662
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.27,1.238847017288208
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.225,1.3458592891693115
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2875,1.3114023208618164
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.26,1.2850911617279053
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2625,1.1805546283721924
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.275,1.0930049419403076
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.153822422027588
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2075,1.2108242511749268
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3,1.283243179321289
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.22,1.1565921306610107
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.113765001296997
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2925,1.1292715072631836
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.26,1.086219310760498
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2475,1.099644899368286
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2375,1.1681852340698242
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2825,1.33290696144104
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.27,1.2079691886901855
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2475,1.194915533065796
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.26,1.1470348834991455
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2875,1.1926367282867432
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.24,1.272369146347046
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.3025,1.2375600337982178
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.295,1.1628658771514893
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.29,1.1714928150177002
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3025,1.3127491474151611
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2875,1.372237205505371
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2525,1.2524096965789795
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.3092937469482422
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.295,1.1645920276641846
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2425,1.293656826019287
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.28,1.3557171821594238
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3075,1.1981215476989746
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.275,1.2276232242584229
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.29,1.2938590049743652
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.225,1.3517932891845703
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2775,1.5466358661651611
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2775,1.3922405242919922
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2175,1.4904465675354004
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.29,1.4872024059295654
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2775,1.1768276691436768
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.1481661796569824
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.255,1.201911449432373
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3075,1.185962200164795
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.32,1.2228517532348633
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.2775,1.3046610355377195
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.295,1.3081915378570557
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.26,1.2525150775909424
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2675,1.195305347442627
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.245,1.194437026977539
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.27,1.1836576461791992
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.305,1.1976475715637207
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2925,1.203507900238037
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.25,1.211536169052124
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2925,1.3676650524139404
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2525,1.3544304370880127
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.25,1.2899086475372314
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.27,1.2882907390594482
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.275,1.282914638519287
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.275,1.1470534801483154
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.26,1.1846213340759275
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2875,1.1966466903686523
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2725,1.2084622383117676
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.3175,1.4552936553955078
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.305,1.2026143074035645
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.31,1.2497129440307615
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2925,1.4333288669586182
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.295,1.3837294578552246
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.24,1.2869625091552734
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2725,1.2364416122436523
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.2475,1.2032783031463623
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.275,1.2508153915405271
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2925,1.3240740299224854
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3,1.1387488842010498
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2775,1.2428746223449707
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.31,1.286409616470337
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2925,1.095569372177124
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.27,1.2140295505523682
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.3075,1.134260654449463
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2675,1.0999510288238523
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2725,1.2964410781860352
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.285,1.2694625854492188
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.27,1.360576629638672
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2375,1.3737611770629885
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.26,1.1289284229278564
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2775,1.1026296615600586
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.25,1.1687414646148682
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2975,1.1556546688079834
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.142930030822754
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.205,1.2347192764282229
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.3175,1.2343263626098633
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.245,1.1544265747070312
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.25,1.166337251663208
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2775,1.340012550354004
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.255,1.3593103885650637
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.2725,1.2362425327301023
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3175,1.2551918029785156
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.325,1.1419110298156738
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.2791979312896729
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.33,1.4249582290649414
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2725,1.2869083881378174
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.24,1.2237606048583984
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.3280360698699951
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.235,1.1503469944000244
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2825,1.151073694229126
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.29,1.179470777511597
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2675,1.4482100009918213
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2525,1.164515256881714
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.295,1.2209022045135498
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.275,1.1450769901275637
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2675,1.0701801776885986
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2825,1.1354601383209229
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.295,1.286674976348877
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2925,1.1853394508361816
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.255,1.3026361465454102
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2825,1.2217986583709717
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2575,1.131871223449707
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2475,1.076141595840454
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.27,1.0885839462280271
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.265,1.135481595993042
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.24,1.110975742340088
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.285,1.1117594242095947
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.0431561470031738
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.285,1.0968115329742432
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2675,1.1788923740386963
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2725,1.109041452407837
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.295,1.20389723777771
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.26,1.241997480392456
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.2842633724212646
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.275,1.3469736576080322
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.295,1.261730670928955
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.255,1.173985481262207
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.315,1.3375327587127686
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.295,1.2333900928497314
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.3125,1.2813591957092283
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.295,1.2563636302947998
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.27,1.2868943214416504
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3025,1.2504515647888184
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.225,1.2588167190551758
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.32,1.3215742111206057
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.31,1.2113804817199707
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2275,1.1699295043945312
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.285,1.2988505363464355
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.27,1.0992305278778076
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.1978166103363037
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2625,1.2497508525848389
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.27,1.0788731575012207
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.28,1.1241276264190674
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.3,1.380746841430664
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.295,1.367776870727539
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2475,1.2413322925567627
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.235,1.259822130203247
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.25,1.1878066062927246
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2725,1.2817225456237793
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2725,1.20558762550354
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2925,1.443578004837036
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.27,1.4415051937103271
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.2625,1.2627248764038086
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.26,1.236156940460205
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.29,1.1690280437469482
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2775,1.237032413482666
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.26,1.2000224590301514
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.27,1.2159225940704346
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2325,1.1750452518463137
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.255,1.1676011085510254
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2425,1.2286407947540283
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.29,1.144451141357422
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2675,1.3128559589385986
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.235,1.2431128025054932
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.26,1.363769769668579
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2825,1.2355432510375977
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.255,1.3670496940612793
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2825,1.154149055480957
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.0964388847351074
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.2175307273864746
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2875,1.1887471675872805
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.275,1.3391945362091064
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.27,1.2682411670684814
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,3.0,3,0,0.26,1.1591465473175049
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2775,1.0816094875335691
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.29,1.2808654308319092
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2475,1.2298507690429688
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.29,1.124100685119629
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.26,1.2473013401031494
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2525,1.1618614196777344
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.255,1.2728052139282229
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3,1.5577716827392578
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.27,1.232689619064331
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2525,1.3010165691375732
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2275,1.1343779563903809
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.295,1.226992130279541
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.34395170211792
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.34,1.1264560222625732
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.0720293521881104
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2925,1.204625368118286
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.24,1.07458758354187
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2825,1.0764360427856443
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2675,1.3018198013305664
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.28,1.2603471279144287
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2675,1.3268020153045654
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2325,1.2307376861572266
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2725,1.181525707244873
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.26,1.1700212955474854
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2725,1.2611916065216064
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2825,1.1461567878723145
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3025,1.0623445510864258
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2625,1.0791282653808594
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.285,1.1888608932495115
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2275,1.130549669265747
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.26,1.1977825164794922
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3175,1.1766343116760254
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.3025,1.1287221908569336
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2875,1.2143676280975342
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.2875,1.3653006553649902
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.255,1.254525661468506
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.235,1.2186355590820312
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3125,1.2358055114746094
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.3125,1.140310525894165
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.31,1.1619744300842283
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.285,1.1710941791534424
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.3175,1.2025175094604492
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.295,1.1416587829589844
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.27,1.3922183513641355
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.22,1.2824842929840088
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.31,1.1609492301940918
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3,1.097018480300903
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3025,1.1422502994537354
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.1497414112091064
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.24,1.2845590114593506
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.265,1.3208239078521729
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2625,1.2806580066680908
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.245,1.1273353099822998
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2975,1.1114447116851809
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.315,1.2584593296051023
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2575,1.2195374965667725
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2925,1.1923649311065674
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3275,1.2285730838775637
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.275,1.2603037357330322
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.275,1.1918044090270996
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3,1.1995301246643066
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3,1.2482185363769531
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2525,1.2778406143188477
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.265,1.2857387065887451
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.295,1.3159527778625488
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.3075,1.2079253196716309
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.2575,1.1685683727264404
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.26,1.381882905960083
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.5152390003204346
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.245,1.5357816219329834
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2675,1.3666040897369385
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.335,1.402677059173584
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2325,1.386117935180664
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2875,1.2151620388031006
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.26,1.5299286842346191
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.285,1.4416401386260986
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2625,1.233811378479004
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.3475284576416016
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2725,1.5147719383239746
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.26,1.261608362197876
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.24,1.2284767627716064
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.3673536777496338
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.215,1.4251267910003662
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2475,1.4752302169799805
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3225,1.2379143238067627
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3075,1.4900248050689695
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2325,1.5409276485443115
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2825,1.271232843399048
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.275,1.1937189102172852
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.275,1.3284106254577637
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3025,1.2238550186157229
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3175,1.327493667602539
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2875,1.0980730056762695
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.275,1.295696258544922
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2875,1.2596213817596436
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2725,1.2047903537750244
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2575,1.2053775787353516
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.32,1.3743104934692385
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2525,1.1656341552734375
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2525,1.147479772567749
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2575,1.1403508186340332
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2675,1.1658332347869873
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.26,1.1227974891662598
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.235,1.114767074584961
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3125,1.0245013236999512
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2825,1.0314340591430664
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2475,1.1948604583740234
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.31,1.15761137008667
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2575,1.3894383907318115
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2675,1.117640256881714
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2675,1.2153289318084717
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.33,1.2130358219146729
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.1589841842651367
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3,1.1277899742126465
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.1209890842437744
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2925,1.2676539421081543
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.26,1.176771879196167
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2875,1.3329615592956543
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.315,1.2352876663208008
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.265,1.2651612758636477
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2475,1.1162872314453125
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.24,1.1048362255096436
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.3302497863769531
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2575,1.1892571449279783
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.29,1.1877427101135254
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2675,1.2550668716430664
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.23,1.1199815273284912
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2825,1.192213773727417
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.2625,1.1513874530792236
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3275,1.2160425186157229
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.255,1.3137123584747314
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2725,1.1086714267730713
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2725,1.338104009628296
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.245,1.1327745914459229
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.3025,1.1834819316864014
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.305,1.1945209503173828
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.29,1.3025362491607666
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.245,1.2804327011108398
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.3175,1.195566177368164
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.1627426147460938
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.225,1.2088727951049805
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.31,1.177586555480957
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3225,1.159301519393921
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2675,1.1722652912139893
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.245,1.152778148651123
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2725,1.1351017951965332
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2,1.251450538635254
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.245,1.220982551574707
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.225,1.4775254726409912
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.22,1.1509475708007812
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.2513015270233154
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2725,1.112919807434082
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2575,1.3111093044281006
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2375,1.216938018798828
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.2329251766204834
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2875,1.2908554077148438
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2525,1.1525835990905762
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.2975,1.2032372951507568
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.316336154937744
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2925,1.3045525550842283
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.275,1.1892011165618896
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.295,1.117617130279541
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2775,1.1446521282196045
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.275,1.1227364540100098
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2975,1.237485647201538
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2625,1.2445697784423828
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2875,1.3531162738800049
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2675,1.348595142364502
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.24796462059021
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.295,1.2931113243103027
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.2704782485961914
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.32,1.313606023788452
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.305,1.1732029914855957
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.27,1.2654905319213867
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3125,1.3528821468353271
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2725,1.2118184566497805
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.255,1.272315263748169
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.32,1.374109983444214
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2325,1.3043460845947266
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.3,1.4473278522491455
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.3025,1.2131798267364502
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.27,1.2318599224090576
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2975,1.245856523513794
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2825,1.2687506675720217
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.265,1.224742412567139
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.285,1.283411979675293
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.245,1.4814512729644775
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.27,1.3723058700561523
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.275,1.2587316036224363
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.315,1.416027307510376
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.265,1.2594258785247805
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.2625,1.254058599472046
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2775,1.358978271484375
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3025,1.1721861362457275
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.25,1.3307619094848633
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.27,1.2208144664764404
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.235,1.1323113441467283
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.23,1.344813346862793
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2475,1.1851739883422852
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.28,1.241360425949097
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.25,1.23006010055542
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.22,1.2235863208770752
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.285,1.1729145050048828
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2825,1.2250704765319824
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2675,1.343523025512695
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2575,1.1233093738555908
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.3,1.2331101894378662
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2925,1.2405149936676023
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2475,1.2506756782531738
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2525,1.2057397365570068
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2525,1.13787841796875
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.305,1.4702661037445068
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.255,1.2798442840576172
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.2525,1.3056011199951172
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.2875,1.534186840057373
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.335,1.190107822418213
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.27,1.1090247631072998
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.245,1.3122944831848145
B,noisy,3,0.0001,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.2425,1.2533223628997805
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,3.0,3,0,137.5,8.561883211135864
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,138.1,8.040160655975342
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,2.0,2,0,138.025,7.800323724746704
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,139.375,6.538822889328003
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,136.85,6.861413955688477
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,139.1,7.296136617660522
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,135.95,7.453582286834717
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,139.2,7.12940788269043
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,137.275,6.543617486953735
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,140.55,7.517033338546753
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,136.475,7.627487897872925
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,141.925,7.615649700164795
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,139.325,7.176959037780762
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,136.575,8.030678272247314
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,140.075,7.5948662757873535
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,136.975,7.936009645462036
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,139.175,9.37891435623169
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,138.5,10.724772930145264
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,139.325,11.211862564086914
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,135.05,9.36261510848999
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.5,1,1,137.725,10.382797479629517
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,136.2,11.145747900009155
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,137.1,10.954287767410278
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,133.85,8.400864124298096
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,142.25,10.435145616531372
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,138.4,8.618158340454102
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,137.825,11.200379371643066
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,3.0,3,0,139.825,9.679877042770386
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,136.325,10.51749324798584
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,139.4,11.358424425125122
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,135.55,9.95195198059082
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,2.0,2,0,137.85,10.932103872299194
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,139.425,9.994757175445557
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,137.05,9.325513124465942
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,138.7,9.765169143676758
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,136.9,10.403064250946043
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,139.375,10.414560079574583
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,2.0,2,0,138.675,10.146268367767334
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,137.075,10.078278064727783
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,136.45,10.557754755020142
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,137.275,10.662759780883787
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,138.675,9.208500385284424
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,141.825,9.807505130767822
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,135.575,9.990141153335571
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.5,2,1,136.25,10.642302989959717
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,140.425,9.65946888923645
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,140.2,10.838728427886965
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,141.475,10.689306735992432
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,2.0,2,0,138.0,10.272201776504517
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,141.425,11.02847456932068
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,4.0,4,0,142.45,11.482709884643556
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,141.975,8.931664943695068
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,138.625,7.560691356658935
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,138.875,7.586360692977905
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,136.075,7.627479791641235
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,140.825,7.459878444671631
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,2.0,2,0,139.75,7.402652978897095
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,136.675,7.177548408508301
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,138.075,7.4660139083862305
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,135.6,7.805960655212402
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,2.0,2,0,138.9,9.370474338531494
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,2.0,2,0,138.075,9.79381823539734
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.5,1,1,135.175,9.780569791793823
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,3.0,3,0,140.8,11.288551807403564
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,137.1,10.162198781967165
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,140.75,9.735317945480348
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,140.3,10.399040937423706
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,140.35,10.027559041976929
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,138.15,12.724261045455933
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.5,2,1,136.225,11.781130313873293
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,137.3,10.475223302841188
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,137.175,9.510941982269289
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,140.725,10.741087198257446
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,138.05,10.553444623947144
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.5,1,1,137.85,10.06307315826416
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,137.45,9.904089450836182
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,2.5,3,1,134.875,10.808006048202516
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,137.425,11.175885677337646
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,4.0,4,0,139.175,10.611167430877686
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,136.075,10.626750469207764
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.5,2,1,137.75,10.060985326766968
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,136.075,10.589602947235107
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,135.725,9.715268850326538
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,140.175,9.775160789489746
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,138.475,10.890838623046877
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,3.0,3,0,139.725,9.544169902801514
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,135.975,10.888614416122437
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,139.175,9.888102054595947
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,134.775,9.573347806930542
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,136.85,11.61372709274292
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.5,1,1,136.675,10.97749090194702
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,138.975,10.209624767303469
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,139.25,11.62803077697754
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,136.925,10.535844326019289
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,138.65,10.980677604675291
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,135.575,10.900875806808472
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,139.275,10.385916471481323
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,141.375,9.309572219848633
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,136.875,9.917293787002563
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,139.525,11.285028457641602
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,134.025,9.993908882141112
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,137.925,8.799676656723022
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,139.725,10.66787338256836
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,138.25,10.576101064682009
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,135.9,11.88801336288452
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.5,1,1,140.9,10.442800045013428
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,139.975,11.094383001327516
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,140.8,9.232091903686523
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,2.0,2,0,139.95,9.370291233062744
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.5,2,1,141.75,10.074292421340942
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,136.025,9.59239625930786
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,2.0,2,0,138.0,9.902494192123411
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,138.1,11.422338724136353
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.5,2,1,141.85,9.29244828224182
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,140.475,9.290287017822266
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,141.425,9.785412549972534
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,138.9,10.628298997879028
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,138.8,10.70636773109436
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,138.2,8.214184045791626
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,2.0,2,0,138.225,10.7124981880188
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,2.0,2,0,138.75,11.547329425811768
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,1.0,1,0,136.575,11.562675952911375
A,noisy,9,0.0005,0.0005,1,1.0,1,9,9,9,40,0.0,0,0,141.05,10.137490510940552
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,2.0,2,0,67.9,9.7647967338562
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,68.05,9.582241535186768
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,2.0,2,0,70.45,9.350750923156738
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,70.4,9.15593123435974
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,70.95,8.653710126876831
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,68.925,8.351246118545532
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,68.85,9.723705053329468
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,70.8,8.300817012786865
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,69.875,9.74320101737976
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,68.425,8.974931478500366
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,68.55,8.557052850723267
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,69.7,11.3125159740448
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,66.7,9.76629900932312
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,70.175,8.905348777770996
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,67.05,9.219861268997192
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,67.5,8.008897304534912
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,66.275,7.316073417663574
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,67.6,6.594865560531616
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,65.925,6.205119848251343
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,67.025,6.232113599777222
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,67.125,6.483094215393066
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,68.075,6.756839752197266
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,68.075,8.644598484039307
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,68.175,8.085959434509277
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,67.825,8.024584770202637
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,71.625,9.37382435798645
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,67.275,9.01450252532959
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,68.075,8.858999490737915
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,71.425,9.245591640472412
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,69.725,8.58340311050415
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,70.325,10.5546236038208
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,67.075,8.954689025878906
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,2.0,2,0,69.425,8.626285314559937
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,69.15,9.21903920173645
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,65.725,9.266201734542848
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,67.5,9.369903326034546
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,69.325,9.147557258605955
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,2.0,2,0,67.95,9.64431357383728
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,2.0,2,0,68.675,10.132864952087402
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.5,1,1,66.425,10.553589820861816
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,69.15,9.077897548675535
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,2.0,2,0,68.425,8.359230041503906
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,67.8,9.057760000228882
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,67.875,9.253867864608765
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,67.575,10.046656608581545
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,68.425,9.598005533218384
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,70.775,8.476516246795654
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,69.05,8.359942436218262
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.5,2,1,69.525,9.15522527694702
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,69.625,9.599544286727903
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.5,1,1,68.025,9.274401903152466
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,68.075,8.260237455368042
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,68.975,8.730133533477783
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,68.85,8.886813640594482
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,69.425,8.991811513900757
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,69.9,8.948148727416992
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,68.35,8.363009929656982
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,66.125,9.461241960525513
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,70.175,10.85719633102417
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,67.15,8.94407033920288
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.5,1,1,68.2,9.3606538772583
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,68.0,9.317541122436523
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,68.925,8.999006032943726
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,71.275,9.501443147659302
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,66.45,9.988492250442505
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,67.9,9.18669891357422
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.5,1,1,69.1,9.125094413757324
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,2.5,3,1,68.05,9.675764083862305
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,67.225,9.810144901275637
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,2.0,2,0,69.85,9.98170804977417
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,68.65,10.70176649093628
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,65.125,9.845597982406616
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,68.85,9.693718194961548
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,68.45,11.06210207939148
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,69.725,9.723566770553589
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,67.875,8.724101066589355
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,70.0,10.602837562561035
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,69.2,9.764312982559204
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,69.25,8.929823875427246
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,70.425,12.150382280349731
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,68.975,9.273473739624023
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,68.7,10.198932647705078
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,69.175,11.37356948852539
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,67.775,9.534528017044067
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,66.25,9.152697563171388
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.5,1,1,67.75,10.261891841888428
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,68.525,10.934661865234377
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,70.575,12.027963876724243
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,68.275,11.154866218566896
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,67.975,11.648867845535278
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,68.75,10.976346731185911
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,69.875,10.410098791122437
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,2.0,2,0,71.0,11.242157697677612
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.5,1,1,69.55,10.751355171203612
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,70.525,11.09930682182312
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,69.85,9.644418954849243
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,69.675,9.9806227684021
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,67.55,10.583132028579712
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,68.25,9.63460612297058
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,68.375,10.03109574317932
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,70.475,9.540496587753296
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,67.5,9.88658332824707
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,67.95,10.372653722763062
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,68.675,9.798884153366089
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,68.95,10.606321096420288
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.5,1,1,69.0,8.987847089767456
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,66.525,8.586778402328491
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,69.825,10.185437440872192
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,68.2,10.24673843383789
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.5,1,1,69.85,8.541159629821777
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,67.375,10.619874000549316
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,68.9,9.608426332473757
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,68.875,10.083386659622192
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,67.9,10.687412977218628
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,68.125,9.721916437149048
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.5,1,1,70.275,10.40207028388977
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,2.0,2,0,69.325,11.238858222961426
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.5,1,1,70.275,8.755825996398926
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,69.375,8.731020450592041
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,68.55,9.123988628387451
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,2.0,2,0,69.25,11.032021045684814
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,70.3,9.491162061691284
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,67.45,11.911755800247192
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,67.575,8.568215370178223
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,69.35,8.727198600769043
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,68.475,9.235917568206789
B,noisy,9,0.001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,70.925,10.286271572113035
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.125,7.156642913818359
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.1,4.402794122695923
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.675,4.145668029785156
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.525,4.314629793167114
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.55,6.075055837631226
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.775,4.537505626678467
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.9,4.944617986679077
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.025,5.217938184738159
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.85,5.5564563274383545
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.325,5.975795269012451
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.425,4.45561408996582
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.25,6.326277732849121
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.15,4.9141693115234375
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.7,5.1023359298706055
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.0,6.33621621131897
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.025,4.366712093353272
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.9,6.53743577003479
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.55,6.49530839920044
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.725,4.926758527755737
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.625,5.86649751663208
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,6.2,5.057681560516357
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.825,5.904730558395386
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.575,6.104944944381714
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.775,4.6882874965667725
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.075,5.068119287490845
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.375,5.340869665145874
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.825,5.399420261383057
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.55,5.5321879386901855
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.425,5.273270130157471
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.0,4.739858150482178
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.4,5.073575019836426
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.25,6.047011375427246
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.9,5.6129677295684814
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.425,4.577062845230103
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.075,4.642441987991333
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.25,4.138639450073242
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.95,3.855459213256836
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.4,4.342266798019409
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.175,6.173697233200073
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.575,6.062991142272949
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.025,4.980235576629639
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.75,5.542461633682251
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.775,4.797802925109863
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.075,4.620887517929077
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.2,5.544888257980347
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.8,5.60077714920044
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.95,4.5577545166015625
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.55,4.720552206039429
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.575,5.189831256866455
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.225,4.60707950592041
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.475,5.683608055114746
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.85,6.815765619277954
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.45,5.480037927627564
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.675,6.77191162109375
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.825,5.255263328552246
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.1,6.085465669631958
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.35,5.914937257766724
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.9,5.569785833358765
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.775,5.626041889190674
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.325,6.471141338348389
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.025,4.978229522705078
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.1,5.466125965118408
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.925,5.6969287395477295
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.3,5.261730194091797
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.45,6.485173463821411
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.45,5.433001279830933
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.125,7.334850311279297
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.55,7.067815542221069
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.125,7.426480054855347
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.175,5.926557779312134
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.425,5.763274908065796
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,7.325,5.668213367462158
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.4,5.102235794067383
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.275,5.8615710735321045
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.5,6.542318820953369
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.825,5.433735370635986
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.475,7.333982229232788
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.8,4.865358114242554
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.2,5.724370718002319
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.925,6.28571367263794
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.825,5.127052307128906
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.225,5.926634073257446
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.75,5.118061304092407
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.475,4.252426385879517
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.425,6.173601150512695
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.975,5.459015846252441
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.625,5.458800554275513
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.275,6.122581720352173
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.65,5.432386636734009
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.1,4.716174364089966
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.975,5.961947679519653
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.625,6.231162071228027
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.875,6.95836353302002
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.875,6.047298431396484
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,8.025,5.279955863952637
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.075,5.497052431106567
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.875,6.044328212738037
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.0,5.818079710006714
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.0,4.890124082565308
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.925,5.741878032684326
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.725,5.112128257751465
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.75,5.413220405578613
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.675,6.359041929244995
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.7,5.351157903671265
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.3,5.65615439414978
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.3,6.473501205444336
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.625,5.770772695541382
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.425,5.353761672973633
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.95,5.713128328323364
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.475,5.146205902099609
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.425,5.657716751098633
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.0,5.681986331939697
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.625,5.831013917922974
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.4,6.605489730834961
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.875,5.2394609451293945
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.475,5.70636773109436
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.85,5.519934415817261
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.5,5.687585353851318
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.15,5.410822868347168
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.5,4.41220498085022
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.0,5.27906608581543
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.65,5.323702335357666
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.475,5.628340482711792
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.9,5.668072700500488
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.775,5.850311994552612
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.4,6.745651960372925
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.95,6.459179878234863
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.375,5.282718420028687
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.025,5.903350353240967
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.7,4.8158934116363525
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.425,5.195716857910156
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.775,5.122242450714111
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.7,6.503125905990601
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.9,5.587656259536743
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.575,5.057088851928711
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.15,4.685425281524658
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.425,4.579962253570557
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.75,5.332107782363892
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.425,5.645890712738037
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.425,5.6617350578308105
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.625,5.790580034255981
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.225,5.8044114112854
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.125,5.729875564575195
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.2,4.629678010940552
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.75,5.167231321334839
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.975,5.977144479751587
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.725,5.780597448348999
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.975,6.229109764099121
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.425,4.8592517375946045
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.625,4.835296392440796
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.475,5.174292325973511
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.15,5.844811201095581
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.675,6.442371368408203
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.3,7.048238754272461
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.525,5.4159324169158936
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.575,6.733445167541504
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.225,5.739350318908691
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.05,6.564462423324585
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.4,4.86632776260376
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.05,6.872948884963989
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.1,5.002109527587891
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.075,5.521606922149658
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.725,4.744839191436768
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.575,5.4868223667144775
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.8,5.633261442184448
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.45,4.732255458831787
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.225,5.3139424324035645
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.225,4.707763910293579
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.9,5.218271970748901
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.575,4.819734573364258
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.7,6.5761542320251465
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.325,4.541671752929688
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.625,4.811963796615601
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.15,4.86990761756897
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.675,5.809014797210693
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.1,4.816787004470825
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.25,5.513882875442505
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.625,5.311939477920532
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.1,5.434244155883789
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.45,4.910249710083008
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.35,4.97035551071167
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.875,6.5605528354644775
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.675,6.444808483123779
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.1,6.313689470291138
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.625,5.9660162925720215
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,1.0,1,0,6.1,4.428390026092529
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.425,7.11261248588562
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.325,8.275561094284058
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.425,6.626311540603638
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.175,7.166000843048096
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.925,5.971671581268311
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.65,5.696796894073486
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.1,5.35619854927063
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.9,7.360527038574219
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.05,5.712601661682129
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.925,4.512145757675171
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.875,6.954334735870361
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.8,5.96957540512085
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.05,6.437129974365234
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.575,5.558053255081177
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.925,6.693186521530151
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.375,6.968255996704102
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.975,5.9845592975616455
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.95,4.785413980484009
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.525,4.650737524032593
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.225,5.883215427398682
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.85,5.035547494888306
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.95,4.710976600646973
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.225,6.387669801712036
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.075,5.66117000579834
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.5,6.273929834365845
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,6.6,6.263347148895264
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.275,7.268998384475708
B,noisy,9,0.0001,0.0,1,1.0,1,9,9,9,40,0.0,0,0,7.2,7.056051731109619
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.39,9.11065125465393
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.81,7.456984043121338
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.19,9.885063409805298
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.18,7.651264905929565
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.88,8.403908252716064
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.67,8.824077606201172
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.02,8.175071477890015
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.04,7.718812942504883
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.84,8.340296030044556
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.9,7.787845849990845
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,1.0,1,0,9.44,8.193864583969116
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.16,7.270789384841919
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.29,8.257920265197754
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.5,1,1,10.19,6.801471710205078
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.93,7.579113483428955
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,1.0,1,0,9.67,8.014828205108643
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.7,7.756292343139648
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.51,7.72488260269165
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.85,8.569801330566406
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.5,1,1,10.26,8.415157318115234
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,1.0,1,0,9.63,7.406417369842529
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.86,7.814891815185547
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.67,7.88458251953125
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.58,7.564383268356323
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.5,1,1,10.26,7.985306978225708
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,2.0,2,0,9.82,7.988569974899292
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.21,9.178192138671877
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,1.0,1,0,9.46,7.640517473220825
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.0,8.23911166191101
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,1.0,1,0,10.21,6.968082666397095
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,1.0,1,0,9.51,5.453983783721924
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.67,5.761525630950928
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.23,5.9761738777160645
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.65,5.601172685623169
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.34,5.511478662490845
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.68,5.418823719024658
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.82,5.909315347671509
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.53,6.5685508251190186
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.53,6.196314334869385
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,1.0,1,0,9.93,6.507950305938721
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,1.0,1,0,9.84,6.476478099822998
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,1.0,1,0,10.06,6.484032154083252
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.95,6.576105117797852
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.6,5.68843936920166
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.0,6.825918674468994
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.86,6.590287923812866
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.81,5.4417829513549805
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.77,5.579545259475708
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.97,6.139985084533691
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.89,5.772435426712036
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.8,6.38059139251709
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.15,6.020636796951294
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.81,5.690672159194946
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.7,6.338700532913208
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.19,5.903345584869385
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.5,1,1,9.85,5.876471042633057
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.51,6.3031227588653564
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.93,5.640580415725708
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.61,5.4550957679748535
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.52,5.817198038101196
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.5,1,1,10.32,6.913404226303101
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.43,5.686772346496582
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.87,5.918552875518799
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.87,5.909572601318359
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.23,5.868059158325195
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,1.0,1,0,10.33,6.069481611251831
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.65,6.2090795040130615
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.22,6.252106189727783
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.77,5.507933855056763
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.24,6.275791883468628
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.48,5.763629913330078
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.74,6.135461330413818
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.25,6.844491004943848
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.92,6.157962322235107
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.32,4.963388919830322
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.89,5.37467885017395
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.53,6.440171480178833
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.09,6.505096197128296
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.66,5.353335857391357
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,1.0,1,0,10.0,5.95241641998291
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.46,6.651480436325073
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.81,5.864607095718384
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.12,6.6259284019470215
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.63,5.943831920623779
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.5,1,1,9.74,6.289039373397827
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.69,5.552292823791504
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.62,5.706611633300781
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.78,5.0698230266571045
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.5,5.249854803085327
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,2.0,2,0,9.48,5.880083799362183
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.67,5.7656567096710205
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.69,5.675081491470337
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.87,5.677947044372559
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.08,6.593544960021973
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.5,1,1,10.47,5.69348931312561
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.5,1,1,9.87,6.5758161544799805
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.5,5.884946346282959
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.98,5.4752280712127686
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.78,5.496488332748413
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,1.0,1,0,9.48,5.467982053756714
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.35,5.601880073547363
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.19,5.329980134963989
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.61,5.7907326221466064
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.15,5.793036937713623
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.98,6.458922863006592
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.63,5.861172437667847
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.35,6.4199419021606445
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.87,6.284554958343506
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.03,6.291449546813965
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,1.0,1,0,10.08,6.18770694732666
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.12,6.055173873901367
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.71,5.747964859008789
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.44,6.779120683670044
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,1.0,1,0,9.42,6.188263177871704
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.04,5.5708091259002686
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.85,5.698405027389526
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.74,5.8123064041137695
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.65,5.341756105422974
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.0,6.287350654602051
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.94,6.264196157455444
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.72,6.110226631164551
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.01,5.370274782180786
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.75,5.653949499130249
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.5,6.030595064163208
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.18,6.504507541656494
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.31,6.07029914855957
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.1,5.854548454284668
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.76,5.236616611480713
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.2,5.334189414978027
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.5,1,1,10.34,5.385549545288086
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.1,5.568718433380127
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.07,5.874374151229858
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.67,6.824834823608398
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.84,6.871336221694946
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.5,1,1,9.8,5.562687158584595
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.91,5.332242965698242
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.15,7.780094385147095
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.61,5.851317882537842
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.36,6.667926788330078
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,1.0,1,0,10.02,6.67777156829834
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.82,6.235631942749023
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.84,5.713351011276245
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.07,6.0785722732543945
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.75,6.230079889297485
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.1,5.696749210357666
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.7,6.634705543518066
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.95,6.014528512954712
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.78,6.6986236572265625
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.83,5.929818391799927
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.17,6.423283338546753
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.5,1,1,9.71,6.734077453613281
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.86,6.153982877731323
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.81,6.25630521774292
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.85,5.985400199890137
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.95,5.841485261917114
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,1.0,1,0,9.91,6.6278581619262695
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.33,6.4543280601501465
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.01,6.731390476226807
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.22,6.07170295715332
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.08,5.926710367202759
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.67,5.886487722396851
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.94,5.61517596244812
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.2,6.447953462600708
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,1.0,1,0,9.07,5.725831747055054
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.38,5.306325674057007
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,1.0,1,0,9.68,5.8817126750946045
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.91,5.049667119979858
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.63,5.692652940750122
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.04,5.515197038650513
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.3,6.709860324859619
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.53,5.481995582580566
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.81,5.812303304672241
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.87,6.157749176025391
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.85,5.724389314651489
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.73,5.664794921875
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.78,5.392577886581421
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.21,5.161056995391846
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.1,6.212469339370728
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.33,5.809077024459839
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.86,5.861197471618652
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.8,5.170021057128906
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.81,5.865121364593506
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.99,6.392635345458984
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.79,5.902069330215454
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,1.0,1,0,10.0,5.912084341049194
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.63,6.1218907833099365
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.94,6.145371675491333
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,1.0,1,0,9.61,6.064811706542969
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,9.91,5.376709699630737
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.19,5.997334957122803
B,noisy,7,0.0003,0.0,1,1.0,1,7,7,7,100,0.0,0,0,10.19,7.064422369003296
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.6,4.050952196121216
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.58,4.104014873504639
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,1.0,1,0,6.13,4.452585697174072
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.86,3.911920309066773
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.9,4.245765686035156
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.73,3.8747591972351074
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.25,4.214017391204834
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.89,4.721172094345093
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.5,1,1,5.68,4.519517183303833
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.21,3.5674283504486084
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.28,3.658934831619263
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,1.0,1,0,5.8,3.907914400100708
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,1.0,1,0,5.79,3.866539478302002
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.69,3.837956428527832
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.59,4.433880090713501
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.49,4.069027900695801
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.64,3.982761859893799
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.63,3.8963851928710938
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.31,4.031482934951782
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.53,3.7939958572387695
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.5,1,1,6.1,4.34994649887085
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.64,4.392338752746582
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.97,3.7011289596557617
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,1.0,1,0,6.03,3.449779510498047
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.58,4.226095676422119
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.52,3.4935412406921387
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.55,3.497666597366333
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.54,3.671862602233887
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.8,3.73427963256836
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,1.0,1,0,5.87,4.320528984069824
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.58,4.604067325592041
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.65,4.052928447723389
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.65,4.16044545173645
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.38,3.904507637023926
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.74,4.4237282276153564
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.43,4.137872934341431
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.65,3.960616111755371
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.93,4.830709934234619
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.33,4.090068101882935
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.57,3.7458176612854
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.75,4.693725824356079
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.58,4.37747859954834
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.97,4.065911769866943
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.46,4.167558431625366
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.83,3.5333619117736816
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.82,4.120283842086792
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.41,3.415710926055908
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.07,3.943859815597534
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.62,4.011721611022949
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,1.0,1,0,5.77,3.9347035884857178
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.12,3.6535274982452393
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.8,3.6386239528656006
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.87,3.30654239654541
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.64,3.9973652362823486
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.93,3.688027620315552
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.59,3.889173030853272
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.57,3.839827537536621
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.64,4.473533153533936
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.0,3.78286337852478
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.6,3.8571038246154794
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.13,4.250622034072876
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.61,4.003702878952026
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,1.0,1,0,5.98,4.8499915599823
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.73,3.999948740005493
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.86,4.284235954284668
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.64,3.801000833511353
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.02,4.1490888595581055
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.46,3.835779428482056
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.59,3.542547702789306
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.12,4.196747541427612
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.71,3.696038484573364
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.96,3.87040376663208
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.91,3.8728981018066406
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.34,3.761813163757324
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.21,3.1801130771636963
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.49,3.251173734664917
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.04,3.811730146408081
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.63,3.636008739471436
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.73,4.842724084854126
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.5,1,1,5.63,4.1226115226745605
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.65,4.394869804382324
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.45,4.062691688537598
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.74,4.058700084686279
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.28,4.280517101287842
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.09,3.849823474884033
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.78,4.106151103973389
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.61,4.2287278175354
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.03,3.829686164855957
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.45,4.773851156234741
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.15,4.071195602416992
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.63,3.974387645721436
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.65,4.055842399597168
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.96,4.376055240631104
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.56,4.064393520355225
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.68,5.047167539596558
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.87,4.6855950355529785
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.8,4.533024787902832
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.58,4.100734710693359
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.73,3.590064764022827
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.09,4.81341552734375
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.24,4.254040956497192
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.94,4.517729997634888
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.59,3.803425550460816
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.86,4.118126153945923
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.4,4.138829469680786
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.71,3.739586114883423
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.88,4.10857367515564
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,1.0,1,0,6.09,4.452062129974365
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.86,3.932507276535034
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.85,3.793794870376587
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.86,3.6996195316314697
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.23,3.1337039470672607
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.81,3.602752685546875
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.1,4.247356176376343
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.69,3.568129539489746
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.87,4.215342998504639
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.93,4.265743017196655
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.84,4.06151556968689
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.57,4.0761473178863525
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.13,4.240097999572754
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.71,3.4906044006347656
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.79,4.149421453475952
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.62,3.784844398498535
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.64,4.711648464202881
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,1.0,1,0,5.58,3.861894130706787
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.49,4.2753190994262695
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.57,3.9167211055755615
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,1.0,1,0,5.56,4.115004301071167
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.38,4.009747266769409
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.72,3.913548231124878
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.55,3.623183965682984
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.71,3.949390411376953
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.45,3.639333486557007
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,1.0,1,0,5.93,3.826160907745361
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.83,4.367009878158569
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.79,3.838635683059693
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.85,3.9439573287963863
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.51,3.8389852046966553
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.45,4.650632858276367
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.91,4.4521214962005615
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.55,4.497884750366211
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.84,4.26052451133728
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.44,4.363172054290772
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.62,4.089816093444824
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.55,4.316816091537476
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.6,4.124569416046143
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.42,3.980838060379028
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.69,4.637289047241211
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,1.0,1,0,5.99,4.667416572570801
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.5,4.063206434249878
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.31,4.15724515914917
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.78,3.9119160175323486
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.89,4.800422430038452
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.74,4.202068567276001
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.65,4.141666889190674
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.59,4.861822128295898
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.66,3.662909030914306
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.66,4.125860214233398
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.87,4.151706695556641
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.83,3.9681146144866943
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.68,4.153878927230835
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.25,3.944521427154541
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.57,3.679603576660156
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.8,4.349246978759766
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.33,3.939832448959351
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.99,5.13828444480896
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.87,3.880715847015381
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.7,4.742999076843262
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.77,4.1404407024383545
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.37,3.964896678924561
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.5,3.7837188243865967
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.41,3.9634950160980233
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.45,3.8906660079956055
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.72,4.268189430236816
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,1.0,1,0,5.77,4.0420684814453125
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.77,3.880655288696289
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.29,3.452328681945801
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.67,4.073801279067993
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.6,3.751644372940064
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.74,4.594250679016113
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.12,4.154541730880737
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,1.0,1,0,5.63,4.107031583786011
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.95,3.68852972984314
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.5,1,1,5.6,4.8934595584869385
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.9,4.689831256866455
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.63,3.750587701797485
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.82,4.2326085567474365
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.52,4.237925052642822
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.5,1,1,5.5,4.027280569076538
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.94,4.227255582809448
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.47,4.110225200653076
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.99,4.249282598495483
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,1.0,1,0,5.52,3.911277294158936
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.99,4.1214141845703125
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.68,3.753721237182617
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.28,4.7632155418396
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.87,4.23835563659668
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.66,4.052558898925781
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.33,4.452843189239502
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.08,3.853234052658081
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.77,4.117304563522339
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.0,4.045219898223877
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.68,4.119056463241577
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.44,3.834016084671021
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.28,4.030817747116089
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.34,3.872207403182984
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.94,3.924607038497925
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.58,3.665219783782959
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.5,1,1,5.84,3.5473897457122803
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.88,3.977640628814697
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.49,4.0540430545806885
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.64,3.800793647766113
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.52,4.981405019760132
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,1.0,1,0,6.14,4.310872554779053
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.33,3.933559894561768
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.5,1,1,5.5,4.008690118789673
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.27,4.3189239501953125
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.66,4.509004354476929
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,1.0,1,0,5.71,4.301364898681641
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.98,4.334695339202881
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.61,3.888196229934693
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.64,3.858524799346924
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.08,4.732144117355347
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.92,4.717256546020508
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.19,4.493163824081421
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.47,3.824605941772461
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.89,4.069242715835571
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.87,3.885503053665161
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.14,4.309180736541748
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.74,3.569888114929199
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,1.0,1,0,5.76,3.625844240188599
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.36,3.4362235069274902
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.59,5.1756432056427
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.66,3.504880905151367
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.28,3.621429920196533
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.55,3.377872467041016
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.55,3.114054679870605
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.66,3.460246324539185
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.89,3.4493603706359863
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.66,3.8538873195648193
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.65,3.756484746932984
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.17,4.241320610046387
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.5,1,1,5.63,3.942348003387451
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.4,3.722368240356445
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.75,3.825634002685547
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,1.0,1,0,5.76,3.584408760070801
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.66,3.7138960361480713
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.78,3.737344741821289
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.28,4.0470147132873535
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.44,4.067791700363159
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.58,3.965759038925171
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.1,4.056697368621826
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.67,3.842620611190796
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.34,3.883968591690064
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.94,4.226410150527954
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.01,4.36466383934021
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.91,3.872128009796143
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.45,3.5753941535949707
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.44,3.866527318954468
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.92,3.924649953842163
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.08,4.447230100631714
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.32,4.416738510131836
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,1.0,1,0,6.06,4.512662172317505
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.8,3.862821578979492
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.74,3.8628759384155273
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.81,3.784337520599365
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.39,3.838270425796509
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.88,3.8055689334869385
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.75,4.041722059249878
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.76,4.430624961853027
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,1.0,1,0,5.64,4.015634536743164
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.93,4.056361436843872
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.71,3.636521816253662
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.91,4.873020648956299
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.73,3.904717922210693
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.21,4.013103008270264
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.8,3.9663174152374263
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.84,4.059043884277344
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.38,4.174716711044312
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.22,3.768902778625488
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.52,3.675713300704956
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.15,3.6379234790802
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.48,3.5622544288635254
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.09,3.8545894622802734
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.92,3.682104110717773
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.86,4.4313740730285645
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.6,4.095040082931519
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.77,4.231600046157837
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.57,4.211379051208496
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.8,4.962750196456909
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.78,4.357750654220581
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.75,3.664906740188599
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.85,4.34528660774231
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.45,3.613787651062012
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.72,3.9425227642059326
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,6.26,3.989501237869263
B,noisy,7,0.0002,0.0,1,0.6,1,7,5,7,100,0.0,0,0,5.62,3.560698986053467
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.1525,8.810561895370483
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,2.5,3,1,10.2375,8.058639287948608
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.15,8.598042488098145
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.2875,8.638678073883057
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.1775,8.659079790115356
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,2.0,2,0,10.2,8.923666954040527
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.3575,8.019122838973999
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.2375,8.188361883163452
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.2375,8.435158729553223
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.1275,8.815268993377686
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.55,9.438055753707886
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,9.94,8.887647151947021
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.37,8.527400016784668
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.5,1,1,10.175,9.514063596725464
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.3575,8.673944234848022
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,2.0,2,0,10.17,8.538533687591553
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,2.0,2,0,10.315,8.605820894241333
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.14,8.408496141433716
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.1875,8.944999933242798
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.5,1,1,10.2125,9.032185792922974
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.195,8.333991050720215
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,4.0,4,0,10.3925,9.123918533325195
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,2.0,2,0,10.4225,9.144491195678713
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.1525,8.305656909942627
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.5,1,1,10.09,8.551160335540771
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,2.0,2,0,9.89,8.350690364837646
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.0175,8.769007921218872
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.215,8.394745826721191
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,9.9875,8.367199659347534
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,3.0,3,0,10.2175,8.54306173324585
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.02,9.03524661064148
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,2.0,3,2,10.4125,9.033527612686155
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.145,9.41180419921875
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,2.0,2,0,10.1375,8.95834732055664
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.29,8.685052156448364
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.1875,8.593676805496216
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,9.9375,9.016830921173096
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.0625,8.684943199157715
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,2.5,3,1,9.9,8.475723028182983
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,2,2,10.0525,8.386107444763184
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,9.9875,8.639340877532959
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.5,2,1,9.85,9.107048511505129
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,9.985,8.79178762435913
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,9.9125,8.562570095062256
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.5,2,1,10.1425,8.65493655204773
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.3575,8.518160104751587
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,9.7725,9.25671124458313
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,2.0,2,0,9.98,9.090641260147097
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.5,2,1,10.215,8.818846464157104
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.345,9.119117975234984
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.115,8.534884691238403
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.5,1,1,9.925,9.015169382095335
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.125,9.44233465194702
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.5,2,1,9.975,8.876769065856934
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.0825,8.925658702850342
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,9.915,8.642669439315796
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.4175,8.889781951904297
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,9.92,8.58399772644043
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,9.9075,9.46238088607788
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.025,9.161145448684692
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,9.9825,8.778284788131714
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,3.0,3,0,10.0775,9.4663724899292
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,9.87,8.973612785339355
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.0025,9.578776597976685
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.5,1,1,10.2275,9.16775631904602
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.0925,9.426371812820436
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.455,9.547138690948486
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.2275,9.320810794830322
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.28,8.78984785079956
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,9.915,9.835412502288818
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.125,8.974542379379272
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.1875,8.404082298278809
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.1675,9.122918367385864
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.1425,8.970993041992188
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,9.775,8.722812414169312
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,2.0,2,0,9.9775,9.842911005020142
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.5,2,1,10.265,10.110304355621338
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,9.9725,10.649628400802612
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,2.0,2,0,10.255,9.261664628982544
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,9.8325,9.47727918624878
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,9.9775,8.777445316314697
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.3775,8.852837324142456
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.11,9.166315793991089
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.2375,8.450312852859497
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,9.88,8.241741418838501
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.2125,8.724802017211914
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,9.945,8.748582601547241
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.5,3,3,10.2625,9.41926407814026
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,9.99,8.921874046325684
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,2.0,2,0,10.2225,8.49810791015625
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.0425,9.190304279327393
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.5,2,1,10.2975,8.159438848495483
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.5,1,1,10.1975,8.380119800567627
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.5,1,1,10.3775,9.342011213302612
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.195,9.290237426757812
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.36,8.934342861175537
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,2.0,2,0,9.96,8.346029281616211
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,2.0,2,0,10.155,9.037559986114502
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.235,9.215104818344116
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.2675,9.369314908981323
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,9.855,8.450741291046143
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.2575,8.962208271026611
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.18,9.075817346572876
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.26,8.794360637664795
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,2.0,2,0,10.07,9.00909948348999
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.0725,9.163479566574097
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.255,8.920542240142822
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.075,8.832104444503784
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.2325,9.266957998275757
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,2.5,3,1,10.1025,9.136902332305908
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,9.83,8.685592651367188
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.24,8.815691471099854
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,9.9675,8.769858598709106
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,9.9125,8.36609411239624
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.125,8.730625867843628
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,9.99,8.748779058456421
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.03,8.985336780548096
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.015,8.630357027053833
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,2.0,2,0,10.165,8.54289174079895
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.145,8.677136182785034
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,9.865,8.848812341690063
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,9.855,8.656232595443726
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.12,8.547662734985352
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.5,2,1,10.215,8.84360408782959
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,2.0,2,0,10.02,8.870133638381958
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.205,9.032600402832031
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,3.0,3,0,10.0225,8.798205614089966
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.1225,8.966614246368408
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.44,9.024340867996216
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.5,1,1,9.995,9.08410358428955
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.045,9.152891397476196
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.1675,8.772578239440918
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,1.0,1,0,10.3075,9.133855819702148
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,10.3025,8.65859866142273
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.0,0,0,9.91,8.541281938552856
A,noisy,5,0.0002,0.0002,1,1.0,1,5,5,5,400,0.5,1,1,10.1375,9.189440965652466
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.1275,6.906850337982178
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.5,3,1,6.085,6.95145058631897
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.5,1,1,6.265,5.964718818664551
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.0775,5.588735342025757
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,4.5,5,1,6.1225,6.452085494995117
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.5,1,1,6.075,6.176278591156006
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.2125,6.237345933914185
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.0,2,0,6.2325,6.651011228561401
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.09,5.757109642028809
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.0,2,0,6.07,6.323576927185059
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.1075,6.142059803009033
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.5,1,1,6.1125,6.082257032394409
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,3.0,4,2,6.12,6.113534688949585
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.5,1,1,6.1275,5.907348394393921
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.5,2,1,6.095,5.810593605041504
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,3.0,3,0,6.25,6.215245723724365
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.2325,6.020229816436768
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.075,5.920217514038086
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,5.9,6.659477949142456
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.0,2,0,6.07,6.737919330596924
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.5,1,1,6.0275,6.94407320022583
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.1925,6.516651153564453
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.5,1,1,5.875,5.90582013130188
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.1775,5.830740690231323
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.5,1,1,6.3975,6.776191234588623
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,5.995,6.331429719924927
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.2725,6.111021518707275
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.5,1,1,6.125,6.606053113937378
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.5,2,1,6.1275,6.118688106536865
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.0,2,0,6.1375,6.190633773803711
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.5,3,3,6.1325,6.109707355499268
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.5,2,1,5.9875,6.602571487426758
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.1175,6.334962368011475
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.06,5.920397758483887
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.0925,6.591291427612305
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.0475,5.970893621444702
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.4275,6.84503173828125
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.2575,5.900246620178223
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.5,1,1,6.1375,6.353301048278809
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.075,6.254792213439941
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,3.0,4,2,6.265,6.374606609344482
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.0,2,0,6.17,5.876679182052612
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,5.9325,6.011293411254883
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.0,3,2,6.0575,5.783153533935547
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.02,5.809453010559082
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.5,3,1,5.995,5.71659255027771
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,5.99,5.736003398895264
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.14,5.872251987457275
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.0,2,0,5.98,6.099788188934326
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.0,3,2,6.1625,6.502439498901367
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.2025,6.190275430679321
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,5.85,5.70781397819519
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.5,3,1,5.97,5.7320942878723145
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.0,3,2,5.9675,5.771997690200806
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.5,2,1,6.04,6.025571346282959
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.13,6.238054037094116
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.065,5.804641962051392
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.24,6.109568595886231
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.5,1,1,6.035,5.961960315704346
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.5,3,1,6.1275,5.946590423583984
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.5,2,1,6.23,5.548609018325806
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,2,2,6.04,6.152234554290772
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.0,2,0,6.2325,6.0444536209106445
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,3.0,3,0,6.0975,6.352620601654053
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.215,6.328220367431641
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.5,1,1,6.06,6.413912057876587
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.0525,6.791092395782471
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,5.945,6.282947540283203
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.5,1,1,5.9525,6.064316034317017
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.0,2,0,6.2425,6.039067983627319
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.0,2,0,6.185,6.33922004699707
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.135,6.144114971160889
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.1925,6.087847471237183
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.2575,6.179428339004517
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.5,1,1,6.2325,5.916611433029175
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.2,6.101687431335449
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.0,2,0,6.045,5.721432447433472
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,5.8425,5.766906023025513
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.075,5.91891622543335
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.0,2,0,6.0125,6.505124807357788
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,5.9475,6.367839574813843
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.2825,5.98249888420105
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.0,2,0,6.0,7.077910423278809
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.065,6.035151243209839
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.135,5.947251796722412
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,5.9675,5.9848268032073975
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.5,2,1,6.16,6.081689596176148
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.19,5.72188401222229
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.5,2,1,6.21,5.916966915130615
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.165,5.510089159011841
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.5,1,1,6.2075,5.924025058746338
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.2025,5.887170791625977
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.15,6.0170111656188965
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.5,2,1,6.235,5.666290283203125
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.3525,5.759585618972778
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.5,1,1,6.035,5.960520267486572
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.245,5.816972970962524
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,5.8,5.92073130607605
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.5,3,1,6.0225,5.723283052444458
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.2525,5.701848030090332
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,5.94,6.533167839050293
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.5,1,1,6.0075,6.378876447677612
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.0775,6.036736965179443
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,5.9725,6.266496419906616
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.135,6.405622243881226
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,5.9525,5.89142107963562
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.0,3,2,6.1975,5.79769778251648
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.0,3,2,6.0125,5.735235929489136
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.5,1,1,6.0425,5.737500190734863
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.0,2,0,6.09,6.003973484039307
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.06,6.350350379943848
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.25,5.869912624359131
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.1775,5.757472515106201
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.1,5.962482213973999
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.0,2,0,6.175,6.054167032241821
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,5.91,5.761985063552856
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.5,1,1,5.985,6.088924884796143
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.0,2,0,6.2325,5.916788339614868
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.5,2,1,6.1575,6.130387306213379
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.0,2,0,5.855,5.85944676399231
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,5.8875,6.283598184585571
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.0825,6.0907368659973145
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.095,5.924160003662109
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.0475,5.969820022583008
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.0,2,0,6.19,5.471882343292236
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.5,3,1,6.1225,5.828229188919067
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.0775,5.981236219406128
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.215,5.970080137252808
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,2,2,6.0575,5.473260879516602
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.1425,5.8735668659210205
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.0,2,0,6.37,5.595645427703857
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,5.9025,6.03078031539917
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.0175,5.62639045715332
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.5,2,1,6.2125,5.692097902297974
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.12,6.893669128417969
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.315,5.983612775802612
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.0,5.8125550746917725
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.5,2,1,6.0475,5.773960590362549
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,3.0,3,0,6.1375,5.947272062301636
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,5.9425,5.647751092910767
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,3.0,3,0,5.83,5.920759677886963
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,3.0,3,0,5.8425,6.380335569381714
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.015,5.946065664291382
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.5,1,1,6.11,5.706174612045288
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.0675,6.009938478469849
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,5.9725,5.592281341552734
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,5.8875,5.760318994522095
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.1975,5.74180269241333
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,2.0,2,0,6.14,6.240015506744385
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.3075,5.923595905303955
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.195,5.669875383377075
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.5,2,1,6.1625,5.769483327865601
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.2325,5.787490129470825
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.1275,6.03389310836792
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.5,1,1,6.21,5.974412202835083
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.1125,5.7925567626953125
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.1425,5.7900097370147705
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,1.0,1,0,6.2675,5.866248846054077
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.01,6.29118013381958
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,3.5,4,1,6.0275,5.9789879322052
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.13,5.632913112640381
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.5,1,1,6.1775,5.964355230331421
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.1475,5.838458299636841
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.1425,5.776325464248657
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,3.5,4,1,6.22,5.704033374786377
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.5,1,1,6.145,5.986963033676148
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.0,0,0,6.19,5.725283861160278
B,noisy,5,0.0005,0.0,1,1.0,1,5,5,5,400,0.5,1,1,5.865,5.925970077514648
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.97,1.7827749252319336
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.81,1.7322959899902344
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.89,1.728191614151001
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,2.0,2,0,1.805,1.742659568786621
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.815,1.6200802326202393
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.84,1.6160292625427246
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.83,1.6866483688354492
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.905,1.6428663730621338
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.5,1,1,1.8,1.686126947402954
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.83,1.6376714706420898
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.855,1.6971452236175537
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.965,1.635514497756958
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.78,1.666057825088501
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,2.0,2,0,1.65,1.6157894134521484
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.5,1,1,1.87,1.6708884239196775
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.83,1.6748533248901367
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.74,1.6571919918060305
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.86,1.6819078922271729
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.765,1.7374658584594729
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,2.14,1.940948247909546
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.68,1.6207420825958252
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.605,1.5858280658721924
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.825,1.5669186115264893
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.5,1,1,1.96,1.625133991241455
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.63,1.5920281410217283
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.99,1.683358907699585
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.75,1.7173235416412354
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.5,1,1,1.875,1.633601427078247
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.925,1.6189415454864502
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.85,1.768507957458496
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.795,1.7542157173156738
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.885,1.6378231048583984
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.66,1.709026575088501
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,2.0,2,0,1.865,1.889462947845459
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.815,1.6199240684509275
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.87,1.5938866138458252
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.875,1.8700921535491943
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,2.0,2,0,1.78,1.8360185623168943
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.905,1.9164631366729736
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,3.0,3,0,1.785,1.7241530418395996
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.725,1.6041769981384275
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.79,1.7358160018920898
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.73,1.8419671058654783
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.955,1.888786792755127
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.905,1.876556158065796
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.725,1.8759970664978027
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.72,1.869415283203125
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.9,1.7202374935150146
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.87,1.6481163501739502
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.815,1.6855487823486328
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.91,1.8007957935333252
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.735,1.7317438125610352
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.775,1.804759740829468
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.785,1.6212143898010254
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.81,1.823493480682373
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,2.0,2,0,1.775,1.687978744506836
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.905,1.6902942657470703
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.58,1.7062811851501465
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.73,1.6548762321472168
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.83,1.7449610233306885
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.86,1.6553149223327637
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.88,1.5831928253173828
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.75,1.6792523860931396
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.73,1.7732198238372805
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.88,1.579530954360962
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.9,1.839071273803711
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.76,1.667508840560913
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.835,1.6680445671081543
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.83,1.67189621925354
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.945,1.653824806213379
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.855,1.5403876304626465
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.7,1.8181309700012207
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.5,1,1,1.75,1.720555305480957
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.87,1.7155640125274658
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.935,1.8085198402404783
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.5,2,1,1.935,1.7143521308898926
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.845,1.9590835571289065
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,2.0,2,0,1.73,1.889223337173462
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.745,1.6086111068725586
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.765,1.6537559032440186
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,2.01,1.7315800189971924
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.79,1.889081716537476
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.91,1.876795768737793
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.855,1.691441774368286
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.785,1.7852532863616943
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.835,1.704146385192871
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.5,2,1,1.95,1.7960410118103027
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.735,1.7124519348144531
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,2.0,2,0,1.87,1.678164005279541
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.76,1.7467610836029053
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.5,1,1,1.97,1.7211177349090576
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.865,1.669677972793579
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.745,1.6917357444763184
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.865,1.7453889846801758
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.83,2.104065179824829
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.88,1.7209622859954834
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.5,1,1,1.695,1.6316752433776855
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.765,1.6809909343719482
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.8,1.6708762645721436
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,2.02,1.712391376495361
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.885,1.7073643207550049
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.915,1.799119234085083
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.845,1.7217066287994385
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.75,1.766540288925171
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.905,1.739159107208252
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.855,1.8442411422729488
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.775,1.8653898239135744
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.5,2,1,1.775,1.672107458114624
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.78,1.6680829524993896
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.685,1.6305766105651855
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.78,1.727891206741333
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.85,1.7183172702789309
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.5,1,1,1.895,1.603623390197754
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.735,1.7343192100524902
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.92,1.826537132263184
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.655,1.808696746826172
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.79,1.7547380924224854
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,2.03,1.7395062446594238
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.83,1.883931398391724
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.75,1.7524423599243164
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,2.095,1.6782758235931396
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.86,1.653930425643921
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.845,1.9407880306243896
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.8,1.8356525897979736
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,2.02,1.741175651550293
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.995,1.9962637424468996
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.735,1.6991891860961914
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.825,1.6220827102661133
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.79,1.738661527633667
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.85,1.6037940979003906
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.725,1.728829860687256
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.775,1.7310688495635986
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.86,1.750523567199707
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.75,1.7050745487213137
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.92,1.7119064331054688
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.685,1.7026116847991943
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.835,1.558523654937744
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.915,1.6784460544586182
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.96,1.6638221740722656
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.66,1.5132408142089844
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.74,1.9251046180725095
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.69,1.9044060707092283
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.765,1.7910494804382324
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.85,1.7282438278198242
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.785,1.7017273902893066
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.905,1.90064001083374
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.79,1.8066730499267576
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.785,1.8595449924468996
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,2.0,2,0,1.875,1.796431303024292
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.74,1.5852820873260498
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.89,1.633274793624878
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.855,1.7647902965545654
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.5,1,1,1.845,1.6742675304412842
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.795,1.701138257980347
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.875,1.733119010925293
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,2.0,2,0,1.71,1.676185131072998
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.75,1.679553747177124
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.5,1,1,1.955,1.8003225326538086
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.855,1.7200956344604492
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.98,1.923656702041626
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.745,1.7493648529052734
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.685,1.713975429534912
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.945,1.883486032485962
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.755,1.7500030994415283
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.9,1.7394523620605469
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,2.0,2,0,1.895,1.80759859085083
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.84,1.7064664363861084
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.925,1.6366329193115234
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.895,1.6411712169647217
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.735,1.788538694381714
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.655,1.6881153583526611
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.96,1.6956565380096436
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.775,1.7436635494232178
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.82,1.7293076515197754
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.88,1.8846065998077397
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.74,1.7216298580169678
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.845,1.7244949340820312
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,2.0,2,0,1.96,1.9335918426513672
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.84,1.8020930290222168
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.955,1.923774242401123
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.715,1.788877010345459
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.675,1.8033604621887207
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.5,1,1,1.675,1.7056951522827148
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.95,1.7307870388031006
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.865,1.7193808555603027
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,2.0,2,0,1.82,1.826469898223877
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.75,1.944457769393921
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.785,2.053427457809448
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.905,1.908657550811768
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,3.5,4,1,1.695,1.7972733974456787
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.94,1.9870688915252688
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.74,1.825805902481079
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.645,1.7221887111663818
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.5,1,1,1.805,1.728173017501831
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.715,1.6673104763031006
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.84,1.7122392654418943
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.79,1.7152421474456787
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.84,2.075627565383911
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.855,1.8089392185211184
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.845,1.8410823345184328
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.77,1.8825252056121824
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.5,2,1,1.82,1.7651798725128174
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.725,1.815197229385376
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.845,1.7045185565948486
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.925,1.7369871139526367
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.835,1.5941100120544434
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.915,1.670788049697876
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.875,1.658048391342163
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.65,1.8475227355957031
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.69,1.6787397861480713
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.76,1.9857375621795652
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.98,1.898148536682129
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.915,1.747537612915039
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.715,1.7958340644836426
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.86,1.8052666187286377
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.805,1.749420404434204
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.59,1.7094099521636963
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.77,1.8503727912902832
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.855,1.8328816890716555
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,2.0,2,0,2.075,1.844087600708008
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,2.0,2,0,1.81,1.8980398178100584
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,2.0,2,0,1.95,1.7520837783813477
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.72,1.776118278503418
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.66,1.5491881370544434
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.82,1.9147231578826904
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,2.01,1.8141498565673828
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.82,1.8193726539611816
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.86,1.6724050045013428
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,2.0,2,0,1.86,1.7796766757965088
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.72,1.743950128555298
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,2.05,1.7605485916137695
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.705,1.6118261814117432
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.92,1.7682936191558838
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.81,1.7954950332641602
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.865,1.9405522346496584
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.75,1.854842185974121
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.5,2,1,1.81,1.779141902923584
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.805,1.8359806537628172
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.915,1.787045955657959
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.5,1,1,1.885,1.736066818237305
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.675,1.7842442989349363
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.685,2.049770832061768
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.695,1.8496227264404297
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.875,1.7593374252319336
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.75,1.719498872756958
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.74,1.6978673934936523
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.885,2.0426392555236816
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.89,1.840649127960205
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,2.0,2,0,2.01,1.8386116027832031
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.795,1.7836284637451172
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,2.0,2,0,1.895,1.6514978408813477
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.98,1.811400413513184
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,2.045,1.9809367656707764
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.5,1,1,1.925,2.0861570835113525
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.845,1.7999675273895264
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.765,1.765040159225464
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.695,1.6113314628601074
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.93,1.7532963752746582
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,2.03,2.0272552967071533
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.74,1.7972745895385742
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.985,1.927600622177124
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.815,1.7686293125152588
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.855,1.9801595211029053
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.81,1.7517449855804443
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.815,1.6236324310302734
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.85,2.03967833518982
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.83,1.7721796035766602
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.805,1.8724136352539065
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.935,1.8157100677490237
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.815,2.012051820755005
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.875,1.933266878128052
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.755,1.9845545291900637
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.775,1.909522533416748
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,2.0,2,0,1.685,1.952300786972046
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.83,1.8068938255310056
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.94,1.7607803344726562
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.78,1.7646770477294922
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.95,1.7145020961761477
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.865,1.8893072605133057
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.85,1.80272912979126
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.885,1.7394931316375732
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.5,1,1,1.88,1.8251633644104004
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.96,2.2782890796661377
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.86,1.7922441959381104
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.905,1.992207765579224
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,2.5,3,1,1.69,1.7710697650909424
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.83,1.894279718399048
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.5,2,1,1.61,1.8684043884277344
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.78,1.7650156021118164
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,2.005,1.8601820468902588
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.805,1.892054557800293
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.78,1.7701756954193115
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.705,1.7673850059509275
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.5,1,1,1.795,1.815983772277832
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.72,1.891369104385376
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.92,1.876132249832153
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.5,2,1,1.935,1.781235933303833
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.92,1.7481367588043213
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.7,1.8787450790405276
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.99,1.9170036315917969
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.78,1.817044019699097
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.79,1.802165508270264
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.785,1.8771185874938965
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,2.09,1.95011305809021
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.865,2.00603985786438
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.68,1.7783281803131104
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.76,1.8586626052856443
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.89,1.6837496757507324
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.85,1.8879270553588867
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.8,1.9265706539154053
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.84,1.7884197235107422
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.82,1.799346923828125
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.795,1.7701749801635742
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.66,1.749563694000244
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.82,1.8003008365631104
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.945,1.7397761344909668
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.745,1.5763835906982422
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.965,1.7651889324188232
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.83,1.7489490509033203
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.73,1.7458000183105469
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.99,1.7397449016571045
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.885,1.9399917125701904
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.82,1.845705509185791
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.725,1.816540241241455
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.81,1.7837059497833252
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.925,1.9121863842010496
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.74,1.8566138744354248
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.5,2,1,1.95,1.98193883895874
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.945,1.9215681552886963
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.95,1.7675917148590088
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.775,1.6624093055725098
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.8,1.7451205253601074
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.885,1.7350711822509766
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.82,1.747182846069336
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,2.055,1.823720932006836
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.79,1.5865230560302734
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.5,2,1,2.015,1.7059733867645264
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.83,1.8032031059265137
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.935,1.7085902690887451
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.695,1.8860788345336916
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.77,1.8109402656555176
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.7,1.824593544006348
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.9,1.7199068069458008
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.87,1.7279813289642334
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.875,1.8007290363311768
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.77,1.92062783241272
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,2.0,2,0,1.69,1.80401873588562
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.68,1.8262665271759035
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.925,1.8747873306274412
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.67,1.8294970989227293
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.79,1.7248945236206057
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.76,1.8352086544036863
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.74,1.8670949935913088
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.75,1.8370897769927976
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.83,1.6519172191619873
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.925,1.7865386009216309
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.87,1.7716126441955566
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.97,1.781700611114502
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.825,1.7860572338104248
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.905,1.873645544052124
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.865,1.962335586547852
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,2.045,1.9403283596038816
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.81,1.800694227218628
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.77,1.7560093402862549
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,2.01,1.6299231052398682
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.72,1.7697794437408447
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.5,1,1,1.78,1.7102057933807373
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.735,1.7582998275756836
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.81,1.738255739212036
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.79,1.8751437664031985
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,2.015,1.87782621383667
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.71,1.8786046504974363
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.8,1.9699902534484863
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.83,1.8771679401397705
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,2.05,1.9845573902130127
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.725,1.8781230449676516
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.705,1.81585955619812
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.75,1.7752702236175537
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.77,1.817781925201416
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.7,1.8325161933898928
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.5,1,1,1.9,2.187139749526977
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.95,1.790719509124756
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.87,1.8891961574554443
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.88,1.9307878017425537
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.875,1.8112785816192627
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.865,1.8499844074249268
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.72,1.957862138748169
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.93,1.907069444656372
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,2.0,2,0,1.65,1.7852764129638672
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.86,1.7140769958496094
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.825,2.0664544105529785
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.0,0,0,1.76,1.7355666160583496
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.5,1,1,1.85,1.7652175426483154
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,2.025,1.855879545211792
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.77,1.8278019428253172
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,0.5,1,1,1.915,1.8683409690856931
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.685,1.791924238204956
B,noisy,5,0.0002,0.0,1,0.4,1,5,2,5,200,1.0,1,0,1.7,1.916095733642578
A,noisy,3,0.0005,0.0005,1,1.0,1,3,3,3,400,23.5,24,1,6.01,2.482093572616577
A,noisy,3,0.0005,0.0005,1,1.0,1,3,3,3,400,26.0,26,0,5.7075,2.5093910694122314
A,noisy,3,0.0005,0.0005,1,1.0,1,3,3,3,400,15.0,16,2,6.0575,2.2991831302642822
A,noisy,3,0.0005,0.0005,1,1.0,1,3,3,3,400,14.0,15,2,5.845,2.3119378089904785
A,noisy,3,0.0005,0.0005,1,1.0,1,3,3,3,400,27.0,27,0,6.1675,2.3845784664154053
A,noisy,3,0.0005,0.0005,1,1.0,1,3,3,3,400,11.5,13,3,5.9725,2.329877614974976
A,noisy,3,0.0005,0.0005,1,1.0,1,3,3,3,400,21.0,21,0,5.98,2.2472033500671387
A,noisy,3,0.0005,0.0005,1,1.0,1,3,3,3,400,18.5,20,3,6.085,2.253793954849243
A,noisy,3,0.0005,0.0005,1,1.0,1,3,3,3,400,16.0,16,0,5.825,2.13794493675232
A,noisy,3,0.0005,0.0005,1,1.0,1,3,3,3,400,17.5,18,1,5.915,2.321281671524048
A,noisy,3,0.0005,0.0005,1,1.0,1,3,3,3,400,18.0,18,0,6.0025,2.351301908493042
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,8.0,8,0,2.42,2.0954432487487797
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,3.0,3,0,2.3425,1.9321422576904297
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,4.0,4,0,2.4925,1.8721790313720703
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,1.0,1,0,2.2775,2.129507541656494
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,4.0,4,0,2.2775,1.9509756565093996
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,2.0,2,0,2.505,1.9456465244293213
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,5.0,5,0,2.4225,2.0955028533935547
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,6.0,6,0,2.44,1.9460973739624023
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,2.0,2,0,2.325,2.055338859558105
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,5.5,6,1,2.3975,1.848763465881348
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,5.0,5,0,2.5175,1.9746594429016116
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,3.0,3,0,2.3575,2.0678277015686035
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,3.0,3,0,2.455,2.026867151260376
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,5.5,6,1,2.4875,1.9029831886291504
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,5.0,5,0,2.285,1.8248462677001955
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,2.5,3,1,2.39,2.121661424636841
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,3.0,3,0,2.3925,1.999286413192749
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,3.0,3,0,2.2775,1.950056076049805
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,0.0,0,0,2.41,1.929532527923584
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,2.0,2,0,2.425,1.7352876663208008
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,5.0,5,0,2.4975,1.7483315467834473
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,3.0,3,0,2.3925,1.7516863346099854
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,5.0,5,0,2.4425,1.7889952659606934
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,4.0,4,0,2.485,1.6069302558898926
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,2.0,2,0,2.4425,1.873645305633545
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,6.5,7,1,2.315,1.9845621585845947
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,1.5,2,1,2.4125,1.8997812271118164
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,2.0,2,0,2.44,1.8512592315673828
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,2.0,2,0,2.4625,2.012760877609253
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,5.0,5,0,2.395,2.01929783821106
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,3.5,4,1,2.375,1.8587980270385744
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,4.0,4,0,2.3725,2.198472023010254
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,7.5,8,1,2.4125,1.8591504096984863
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,2.0,2,0,2.4425,1.8035955429077148
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,2.5,3,1,2.245,1.9658927917480469
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,3.0,3,0,2.3875,1.8323771953582764
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,3.0,3,0,2.4625,1.987612009048462
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,4.0,4,0,2.5375,2.016408443450928
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,2.0,2,0,2.565,2.2149746417999268
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,5.0,5,0,2.4075,2.01485538482666
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,3.0,3,0,2.3525,2.269484519958496
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,3.0,3,0,2.3775,1.886753797531128
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,3.0,3,0,2.52,1.8905797004699707
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,3.0,3,0,2.3575,1.969527006149292
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,3.5,4,1,2.425,2.2132482528686523
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,4.0,4,0,2.375,2.078865528106689
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,1.0,1,0,2.38,1.705554485321045
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,3.0,3,0,2.5,2.0061206817626958
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,5.0,5,0,2.5075,1.8198137283325195
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,3.5,4,1,2.5225,1.7760214805603027
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,2.0,2,0,2.3975,1.9795639514923096
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,4.0,4,0,2.3675,1.929274082183838
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,3.0,3,0,2.2875,1.734595775604248
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,2.0,2,0,2.5125,1.7843196392059326
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,2.0,2,0,2.315,1.909078598022461
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,4.0,4,0,2.445,1.8224189281463623
A,noisy,3,0.0002,0.0002,1,1.0,1,3,3,3,400,3.5,4,1,2.385,1.8844225406646729
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,3.0,4,2,1.345,1.5467655658721924
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.0,1,0,1.415,1.5222702026367188
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,0.0,0,0,1.4,1.6041102409362793
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,3.0,3,0,1.31,1.5151233673095703
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.5,3,1,1.405,1.4863526821136477
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,0.0,0,0,1.2975,1.400101900100708
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,5.0,5,0,1.3425,1.4487082958221436
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.0,1,0,1.4425,1.4583616256713867
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,5.5,6,1,1.3825,1.3424930572509766
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.0,1,0,1.39,1.388969659805298
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.0,2,0,1.425,1.4008076190948486
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.5,2,1,1.3075,1.7115657329559326
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.0,1,0,1.3525,1.247034788131714
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,0.0,0,0,1.3675,1.3138647079467771
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.0,2,0,1.4075,1.5219459533691406
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,3.0,4,2,1.325,1.4576215744018557
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.5,2,1,1.38,1.4554357528686523
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,0.0,0,0,1.35,1.4388375282287598
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,3.0,3,0,1.325,1.4535980224609375
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.5,3,1,1.3475,1.5964200496673584
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,0.0,0,0,1.42,1.3795368671417236
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,3.5,4,1,1.4825,1.265458345413208
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.0,2,0,1.3625,1.4261982440948486
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.0,2,0,1.38,1.363208532333374
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,3.0,3,0,1.3675,1.5320038795471191
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,4.0,4,0,1.4325,1.3542029857635498
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,0.0,0,0,1.39,1.3998408317565918
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,0.0,0,0,1.3675,1.4575698375701904
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.5,2,1,1.42,1.326244592666626
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.0,1,0,1.3625,1.39825177192688
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,3.0,3,0,1.4275,1.4684646129608154
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.0,2,0,1.325,1.4298183917999268
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.5,3,1,1.4375,1.6735296249389648
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.5,2,1,1.465,1.5440330505371094
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.0,2,0,1.35,1.3585577011108398
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,0.5,1,1,1.41,1.494166374206543
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.5,3,1,1.3075,1.3646364212036133
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.0,1,0,1.3,1.3462753295898438
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.0,1,0,1.3825,1.5127034187316897
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.0,1,0,1.395,1.5285875797271729
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.5,3,1,1.315,1.5299527645111084
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,0.5,1,1,1.4075,1.3218154907226562
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,3.0,3,0,1.4725,1.3766047954559326
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.5,2,1,1.3625,1.4282560348510742
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.0,1,0,1.3775,1.388397216796875
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,3.0,3,0,1.37,1.618668794631958
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.0,1,0,1.35,1.5145878791809082
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,0.5,1,1,1.4925,1.4511756896972656
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,0.0,0,0,1.3675,1.3786225318908691
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,3.0,3,0,1.49,1.4513258934020996
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.0,2,0,1.2675,1.6696419715881348
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.5,3,1,1.37,1.62138032913208
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.0,2,0,1.3225,1.375239610671997
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.5,3,1,1.4375,1.3780279159545898
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,3.5,4,1,1.37,1.4539783000946045
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,0.0,0,0,1.43,1.4263286590576172
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,4.0,4,0,1.4,1.5659971237182615
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,3.0,3,0,1.335,1.4475173950195312
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.0,2,0,1.4125,1.475764274597168
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,0.5,1,1,1.35,1.4092397689819336
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.0,1,0,1.43,1.4776864051818848
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.0,3,2,1.3225,1.4384617805480957
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.5,3,1,1.4275,1.6094059944152832
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,5.5,7,3,1.215,1.489659547805786
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,3.0,3,0,1.4425,1.574437141418457
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,4.0,4,0,1.505,1.541013479232788
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,0.5,1,1,1.34,1.3580689430236816
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,0.0,0,0,1.37,1.5341839790344238
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,3.0,3,0,1.3725,1.6377294063568115
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.0,1,0,1.43,1.723344802856445
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,3.0,3,0,1.2875,1.3555495738983154
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.0,1,0,1.3575,1.3535754680633545
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,0.0,0,0,1.36,1.5928013324737549
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.0,1,0,1.3175,1.7011361122131348
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,0.0,0,0,1.455,1.3905308246612549
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,5.5,6,1,1.4725,1.4529550075531006
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,3.0,3,0,1.335,1.4469845294952393
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.0,1,0,1.4,1.5084104537963867
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.5,2,1,1.3925,1.4911739826202393
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,0.0,0,0,1.4275,1.6416246891021729
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.5,3,1,1.3975,1.3743817806243896
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.0,2,0,1.385,1.4268407821655271
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.0,1,0,1.4075,1.5144906044006348
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,3.0,3,0,1.4025,1.3825151920318604
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.0,2,0,1.4125,1.4575681686401367
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,3.0,3,0,1.3525,1.6556777954101562
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,0.0,0,0,1.4375,1.4534080028533936
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.0,1,0,1.275,1.5701467990875244
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,0.0,0,0,1.46,1.5633416175842283
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.0,1,0,1.3275,1.9342710971832275
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.0,2,0,1.34,1.7507574558258057
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.5,2,1,1.3225,1.6748371124267578
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,3.0,3,0,1.37,1.6798903942108154
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,0.0,0,0,1.3075,1.614213466644287
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.0,1,0,1.2875,1.5427439212799072
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,1.0,1,0,1.3225,1.6317119598388672
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.0,2,0,1.51,1.3471693992614746
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.5,3,1,1.3725,1.7190911769866943
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,2.5,3,1,1.4775,1.5983266830444336
B,noisy,3,0.0005,0.0,1,1.0,1,3,3,3,400,3.0,3,0,1.4225,1.7733888626098633
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,81.425,8.750733613967896
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.4,7.109462738037109
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,83.6,6.660339832305908
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,84.625,6.842074632644653
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.575,6.246245384216309
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,85.35,6.929266452789307
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,85.375,6.387717247009277
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,83.675,7.636475801467895
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,84.075,6.3237035274505615
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,80.1,6.635320901870728
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,86.425,7.003644227981567
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,84.275,7.316241025924683
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.975,6.832076787948608
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,83.275,6.570947170257568
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.35,7.266475915908813
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.775,6.738963842391968
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,84.775,7.151643991470337
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.5,1,1,81.25,9.166522741317747
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.975,9.423449993133543
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,81.725,9.911439895629885
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,80.9,9.456434726715088
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,84.675,9.505516529083252
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,1.0,1,0,84.55,9.442389726638794
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,81.9,9.7245774269104
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,79.575,7.95309042930603
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,81.925,9.868031978607178
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,83.6,9.79577136039734
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,80.875,9.485073328018188
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.6,8.792522430419922
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,1.0,1,0,81.575,9.417058944702148
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,83.975,7.560611009597778
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.375,9.014575719833374
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,85.075,9.28804349899292
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,81.45,8.780208587646484
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.775,10.490597486495972
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,84.375,9.277843952178957
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,1.0,1,0,82.95,9.562893152236938
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,84.625,10.082558155059814
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,81.975,8.629133701324463
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,80.925,9.418372631072998
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,1.0,1,0,83.725,8.48502492904663
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,1.0,1,0,83.775,9.85518765449524
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,84.0,10.153518676757812
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,85.175,8.945289373397827
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,84.025,9.8270845413208
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,1.0,1,0,83.7,10.649949073791504
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.875,9.458463430404665
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.1,10.011573076248167
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.0,9.526532411575316
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,81.875,8.123690843582153
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,83.825,9.51457905769348
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,84.05,8.910507678985596
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.5,1,1,86.575,10.417984008789062
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,81.725,10.779696702957152
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.05,9.978066205978394
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,83.7,9.39428973197937
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,83.2,7.835312604904175
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,84.725,7.181934356689453
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,1.0,1,0,83.05,6.484866380691528
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.425,7.8960230350494385
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,81.975,7.63006854057312
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,1.0,1,0,83.525,7.745020627975464
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,85.0,8.171313524246216
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.6,7.850062608718872
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.075,8.98142123222351
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,81.8,8.668837547302246
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,83.675,8.286402702331543
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,84.55,9.531584978103638
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.575,10.046061992645264
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,83.4,9.67873740196228
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,85.15,9.477643013000488
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,1.0,1,0,81.15,8.579021215438843
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,81.675,9.7182297706604
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.375,11.37536644935608
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,80.675,9.795425176620483
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,84.725,9.47727084159851
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.175,9.285199165344238
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,85.05,9.331979751586914
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,81.3,9.648402214050291
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,83.8,9.186441898345947
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,85.2,8.553200960159302
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,81.25,9.161587238311768
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.5,1,1,82.525,8.714388370513916
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,81.975,9.2651469707489
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,81.925,9.7407865524292
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,83.825,9.54359245300293
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.075,9.613765239715576
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.5,1,1,82.925,8.322355270385742
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,81.55,9.129069089889526
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,84.1,9.647635459899902
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.2,10.216010808944702
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,80.9,8.809222221374512
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.9,8.751404523849487
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,83.825,9.137728691101074
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,1.0,1,0,83.875,9.041470289230348
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.3,11.285082817077637
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,83.95,9.535611629486084
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,83.55,8.417906045913696
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,84.525,8.98255705833435
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,84.4,10.519135475158691
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,86.1,9.406364440917969
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,80.3,10.64588475227356
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,1.0,1,0,83.975,10.563421249389648
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,81.875,10.336528778076172
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.175,7.479912996292114
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,1.0,1,0,84.925,9.557329893112184
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,78.55,8.630173444747925
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,84.525,9.92422366142273
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,83.5,9.63108205795288
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,1.0,1,0,82.825,9.074191570281982
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,83.35,7.6286303997039795
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,81.475,9.34949803352356
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,81.375,9.007373809814451
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,1.5,2,1,83.0,9.326200723648071
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,1.0,1,0,83.675,9.321102857589722
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.75,9.092488527297974
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,83.45,8.861074447631836
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,84.35,9.522817611694336
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,81.75,10.280551433563232
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,83.3,10.26724624633789
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,83.0,10.058687686920166
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,1.0,1,0,82.575,10.415277004241943
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,84.275,9.251591444015505
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,81.225,8.653359174728394
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,1.0,1,0,85.225,8.419278144836426
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.35,9.304810047149658
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,83.375,7.840925693511963
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,81.925,8.78594160079956
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,84.9,9.07054567337036
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,84.0,8.972551107406616
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,84.45,9.09087347984314
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,82.375,9.721931219100952
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,1.0,1,0,83.0,8.706290006637573
A,noisy,9,0.0003,0.0003,1,1.0,1,9,9,9,40,0.0,0,0,84.95,10.014368534088137
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,1.0,1,0,36.475,8.605933904647827
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.15,9.619783401489258
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.5,9.198928594589232
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,36.3,8.785534381866455
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.925,8.918983936309814
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.025,8.402316331863403
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.8,7.911479949951172
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,1.0,1,0,35.475,7.268657684326172
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.125,7.997560739517212
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.975,10.107847929000854
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.075,7.393601417541504
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.675,9.676809310913086
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.4,7.293952941894531
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.675,6.756434679031372
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.125,8.028189897537231
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.325,8.171167850494385
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.075,7.742270708084106
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,32.25,7.05587911605835
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.925,7.054208993911743
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,36.05,5.5454041957855225
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.175,4.891056776046753
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.95,5.07611346244812
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.25,5.185574054718018
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.825,5.9139955043792725
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.75,5.945127248764038
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,36.4,7.165506839752197
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.4,7.293331861495972
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,1.0,1,0,33.675,8.32999038696289
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.85,7.500492095947266
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.925,6.714063405990601
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.05,7.529499530792236
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.45,7.266113996505737
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.2,8.442263841629028
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,1.0,1,0,35.125,8.185844659805298
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.85,8.603475570678711
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.3,7.29845929145813
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.775,6.993683338165283
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.875,7.708897590637207
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,1.0,1,0,33.75,6.962883710861206
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,32.9,6.984055757522583
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,32.95,8.179547309875488
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.125,7.879003524780273
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.85,6.977957725524902
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.075,7.839682817459106
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.35,7.018366575241089
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.975,10.577949047088625
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.55,8.164652824401855
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.4,7.337754726409912
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.8,8.741628170013428
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.5,1,1,32.375,9.597723960876465
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.05,9.125927448272703
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.475,8.032694578170776
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.45,8.902350664138794
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.15,7.360114097595215
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,1.0,1,0,35.575,8.690850973129272
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.9,7.602284908294678
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.15,8.317415237426758
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,1.0,1,0,34.425,8.364381551742554
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,1.0,1,0,33.625,7.898950099945068
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.275,6.295680999755859
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.9,6.748903274536133
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.5,1,1,33.275,8.429452896118164
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.925,7.606868743896484
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.3,8.236084222793579
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,1.0,1,0,33.85,7.8669819831848145
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,1.0,1,0,33.175,7.6640965938568115
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,1.0,1,0,33.8,8.20541262626648
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.35,8.34812307357788
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.425,7.294054746627808
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.25,7.986083745956421
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.775,7.963799476623535
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.7,7.565980672836304
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,1.0,1,0,34.375,7.323886871337891
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,1.0,1,0,35.625,7.35218071937561
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.0,7.652988433837891
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.225,7.271503210067749
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,1.0,1,0,35.0,8.6471586227417
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.575,7.480078458786011
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.875,7.666261434555054
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.775,7.495677471160889
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.175,7.556722640991211
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.35,7.754650831222534
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.925,9.206133127212524
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.375,8.043391704559326
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,1.0,1,0,32.975,9.13418173789978
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.725,7.328674554824829
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.6,8.816636085510254
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,1.0,1,0,35.825,8.261284828186035
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,1.0,1,0,33.425,8.784452438354492
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.7,8.366939067840576
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.15,7.621978044509888
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.7,8.086700201034546
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.35,8.79177713394165
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.475,9.26359486579895
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.425,8.335693836212158
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.525,8.16372537612915
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.425,8.455679178237915
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.175,7.841524362564087
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,32.775,8.669651508331299
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.75,9.78428030014038
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.775,9.52495288848877
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.875,8.931369304656982
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.3,8.69010305404663
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.325,10.246121168136597
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.3,9.796631336212158
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.375,9.441467523574827
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.05,10.211265802383425
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.175,9.71176266670227
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,32.0,9.388940334320068
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.175,7.549725532531738
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.0,9.126646995544434
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.5,9.60278582572937
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.725,8.08917784690857
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.65,8.353424549102783
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,36.9,9.264233827590942
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.5,1,1,35.475,9.625669717788696
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.1,9.00007152557373
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.525,9.26863932609558
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.15,8.474900245666504
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,1.0,1,0,32.225,9.015244960784912
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.35,8.99146318435669
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,1.0,1,0,34.675,9.03010320663452
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.45,8.669667959213257
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.5,8.706063985824585
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.475,8.496134757995605
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.875,8.196657180786133
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.775,8.853220701217651
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,32.95,8.34437084197998
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.95,7.925416946411133
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.075,8.31741189956665
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.675,8.00336217880249
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.025,7.680815935134888
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.475,8.341231346130371
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,32.325,7.941717624664307
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.7,9.833274841308594
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.6,8.2375967502594
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.475,9.01858115196228
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,32.35,9.787922859191896
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,36.375,9.57077980041504
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.3,8.956605434417725
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.475,8.664667129516602
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,35.575,9.459850072860718
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,32.8,8.404262781143188
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.3,8.141189575195312
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.325,8.935206651687622
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,34.575,8.831804752349854
B,noisy,9,0.0005,0.0,1,1.0,1,9,9,9,40,0.0,0,0,33.175,8.434276342391968
A,noisy,7,0.001,0.001,1,1.0,1,7,7,7,100,19.0,20,2,133.66,11.697922229766846
A,noisy,7,0.001,0.001,1,1.0,1,7,7,7,100,14.5,15,1,133.1,11.526092052459717
A,noisy,7,0.001,0.001,1,1.0,1,7,7,7,100,21.5,23,3,133.56,12.33747673034668
A,noisy,7,0.001,0.001,1,1.0,1,7,7,7,100,13.0,13,0,134.04,12.347247838974
A,noisy,7,0.001,0.001,1,1.0,1,7,7,7,100,16.0,17,2,133.25,12.798755168914797
A,noisy,7,0.001,0.001,1,1.0,1,7,7,7,100,22.0,22,0,136.03,12.702713012695312
A,noisy,7,0.001,0.001,1,1.0,1,7,7,7,100,19.5,20,1,133.94,11.974803447723389
A,noisy,7,0.001,0.001,1,1.0,1,7,7,7,100,17.5,19,3,134.14,11.915492057800291
A,noisy,7,0.001,0.001,1,1.0,1,7,7,7,100,15.0,16,2,135.37,12.29956841468811
A,noisy,7,0.001,0.001,1,1.0,1,7,7,7,100,24.5,26,3,133.1,11.853055238723757
A,noisy,7,0.001,0.001,1,1.0,1,7,7,7,100,18.5,21,5,132.76,10.881446123123167
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.83,10.465707778930664
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.25,10.707059144973757
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,41.21,10.029462575912476
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.66,9.97146773338318
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.59,10.395994901657104
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.59,7.834864854812622
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.28,7.5979392528533936
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.64,10.835108995437622
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,41.14,9.56835389137268
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,40.35,10.323793411254885
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,38.77,10.522409200668337
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.27,11.741159677505491
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.5,1,1,40.86,10.849240064620972
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.05,10.19494915008545
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.45,12.330356121063232
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,40.01,11.74074101448059
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.5,1,1,40.38,11.715000867843628
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,40.29,12.296358108520508
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.22,9.967644453048706
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.5,1,1,39.56,11.840412616729736
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.64,10.073259592056274
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,40.09,12.930680990219116
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.8,12.633586883544922
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.73,10.940917015075684
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.39,11.138602018356323
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,40.11,11.22741436958313
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.96,11.932554006576538
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.18,11.232268810272217
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.47,11.180785655975342
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.05,10.023012161254885
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.22,9.429113864898682
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.5,10.41769814491272
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.15,9.40796160697937
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,40.4,9.61223077774048
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.76,11.348798036575316
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.6,11.271785020828249
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,41.14,10.215842247009276
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.86,10.364899158477783
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.5,11.11885380744934
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.6,11.001479864120483
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,40.17,13.211003303527832
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.99,10.859370708465576
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.4,12.377697467803957
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.5,1,1,40.89,10.584401369094849
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.5,1,1,40.64,10.233383655548096
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.88,8.463494300842285
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.27,10.870165586471558
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.63,11.801740646362305
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,40.75,10.977583169937134
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.85,10.864366292953491
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.12,10.693103313446043
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,40.49,9.348375082015991
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.08,11.341017484664915
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.58,9.901493549346924
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,39.6,10.671828508377075
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,40.75,11.21852707862854
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.57,10.226081371307371
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,40.93,10.39483904838562
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.09,11.129181623458862
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.34,9.13452672958374
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,2.0,2,0,40.54,10.107542991638184
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.22,10.827261686325071
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,39.37,10.4589946269989
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.7,10.26965618133545
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,40.19,8.749999761581421
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.53,9.587090492248535
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.73,10.646294593811035
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,38.4,10.258572101593018
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,40.84,12.07949185371399
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.61,12.113561630249023
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.68,9.78853464126587
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.56,11.180806398391724
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,41.4,10.169841289520264
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.5,2,1,40.87,10.477436780929564
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.95,12.014941692352297
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,41.11,10.473633766174316
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.78,10.732536554336548
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.4,10.183876752853394
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,39.51,9.703948974609377
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.66,10.818869590759276
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,40.16,10.095725774765016
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,40.17,10.395620822906494
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.12,10.391605615615845
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,40.42,12.085413455963137
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,40.41,12.061105966567991
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,39.98,11.453791618347168
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,40.29,11.53967308998108
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,2.0,2,0,39.36,10.542136192321776
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.03,11.47881031036377
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.86,11.69622254371643
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.61,11.432595252990724
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.83,11.4889075756073
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.5,1,1,39.84,11.583501815795898
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.2,11.795837879180908
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.72,11.30641269683838
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.22,12.239724397659302
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,40.66,10.825461626052856
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.86,11.491214752197266
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.8,11.73777985572815
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.72,10.805287837982178
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,40.43,10.995958805084229
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.77,11.45784306526184
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.94,11.87161660194397
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,2.0,2,0,38.78,11.412344694137571
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,39.82,9.742021799087524
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,41.23,10.43709111213684
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.8,11.14969253540039
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,39.52,11.163294315338137
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,40.21,11.093974351882936
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,38.33,9.930619716644289
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,1.0,1,0,39.15,9.977355480194092
A,noisy,7,0.0003,0.0003,1,1.0,1,7,7,7,100,0.0,0,0,39.87,10.438140153884888
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.78,8.019805431365967
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.11,7.821109056472778
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.58,7.7111976146698
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.48,7.878162145614624
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.52,7.527775049209595
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.72,8.802619934082031
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.23,8.182479858398438
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.95,6.769256114959717
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,14.11,7.614452123641968
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.01,7.562066078186035
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.9,7.84171462059021
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.02,6.836699724197388
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.48,8.147686958312988
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.41,5.344587802886963
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.5,1,1,13.59,6.030595541000366
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.44,5.534404754638672
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.35,5.531580924987793
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,1.0,1,0,12.52,5.794210195541382
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.24,5.874331474304199
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.9,5.757563352584839
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.2,6.513385057449341
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.47,5.782561540603638
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.32,6.044243097305298
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.03,6.561335325241089
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.27,6.203461885452271
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.21,5.7595062255859375
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.09,5.680722713470459
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.11,6.424546480178833
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.17,5.873769521713257
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.74,6.6470208168029785
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.91,5.689895868301392
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.65,5.996980667114258
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.23,6.337661504745483
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.73,6.388482332229614
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.5,1,1,13.29,6.320595264434815
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.47,6.0720295906066895
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.56,6.222209215164185
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.26,6.06430721282959
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.87,6.672025680541992
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.48,5.494017601013184
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.56,5.8556318283081055
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.68,6.557904481887817
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.24,5.904906272888184
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.05,6.168560981750488
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,14.0,5.6327409744262695
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.6,5.704903364181519
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,1.0,1,0,13.63,5.365764617919922
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.62,5.801329851150513
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.23,6.070237636566162
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.47,6.389352798461914
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.73,6.049878835678101
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.67,6.26401424407959
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.81,6.124829530715942
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.45,5.863024950027466
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.02,6.680400848388672
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,14.01,6.61184549331665
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.56,6.221779823303223
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.69,6.058972358703613
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.82,5.6028032302856445
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.83,6.033971548080444
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,1.0,1,0,14.01,6.054296970367432
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.0,6.315540790557861
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.3,5.965866327285767
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.17,6.170682907104492
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.03,6.702493190765381
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.76,6.437878370285034
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.73,6.49951434135437
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.62,5.667078018188477
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.11,5.717470169067383
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.67,5.628750801086426
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.31,5.848590612411499
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,1.0,1,0,12.75,5.487131357192993
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.75,6.08115291595459
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.87,6.010077238082886
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.92,5.927453279495239
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.16,5.933257818222046
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.45,6.221750259399414
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.51,5.95769739151001
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.28,5.711267948150635
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,14.13,6.382476568222046
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.81,5.810222387313843
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,1.0,1,0,13.83,5.542362213134766
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,1.0,1,0,13.03,6.011820077896118
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.8,5.668421030044556
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.47,5.872942209243774
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.69,5.874195575714111
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.14,5.795335292816162
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.53,5.831577301025391
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.24,5.985569000244141
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.97,5.97325587272644
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.8,6.1542277336120605
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.33,6.159632205963135
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.71,5.969834089279175
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.95,6.094660758972168
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.32,5.709916591644287
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.87,6.710026502609253
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.04,5.632489442825317
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.31,6.001757383346558
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.42,5.996737957000732
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.55,5.582942724227905
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.24,5.350510120391846
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.39,5.573840856552124
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.69,5.664648532867432
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.14,5.945209741592407
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.93,5.887256383895874
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,1.0,1,0,12.97,6.214998006820679
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.99,5.805365800857544
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.52,6.048290729522705
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,14.18,5.565742254257202
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.46,5.618874788284302
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.77,5.179245233535767
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.03,5.909485578536987
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.81,5.352320194244385
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.64,5.441642284393311
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.08,6.260420560836792
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.33,6.153620719909668
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.46,6.876090288162232
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.67,6.342494249343872
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.6,6.161319732666016
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,14.13,5.6407318115234375
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.03,5.226780414581299
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.9,6.108706712722778
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.5,6.792619228363037
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.9,6.430715322494507
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.73,6.465480327606201
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.65,5.885098934173584
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.51,6.265300512313843
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.96,6.010228633880615
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.93,6.205065488815308
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.4,5.955576181411743
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.08,6.680304765701294
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.87,6.600311040878296
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.86,6.015005350112915
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.61,6.89771294593811
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.23,6.882283926010132
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.04,6.854494094848633
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.95,7.0897650718688965
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.54,6.104481935501099
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.9,7.097699165344238
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.76,6.0562803745269775
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.6,6.202903270721436
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.48,6.52722954750061
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.26,5.741279602050781
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.54,5.604473114013672
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.59,5.703912973403931
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.37,6.481602668762207
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.34,5.579196214675903
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.39,5.667778968811035
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.58,5.728221654891968
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.17,5.602008581161499
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,14.15,5.628504514694214
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.34,6.089495897293091
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,14.05,6.254096269607544
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.91,6.03422999382019
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,2.0,2,0,13.98,5.519231796264648
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.16,6.271193981170654
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.97,5.598324775695801
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.18,4.965506076812744
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.84,5.966168403625488
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.38,5.537442445755005
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.01,5.326520681381226
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.11,5.595099449157715
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.23,5.685387134552002
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.7,6.480457782745361
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.1,5.736440658569336
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.71,5.437524318695068
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.66,5.382107973098755
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.25,6.001742362976074
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.62,5.7045605182647705
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.39,5.861089468002319
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.98,6.040374517440796
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.0,6.133363246917725
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.23,5.616325378417969
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.75,6.3350090980529785
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.37,6.194139242172241
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.25,5.95992112159729
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.92,6.287771940231323
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.26,6.580907344818115
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.8,6.616568326950073
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.27,6.261997699737549
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,12.88,6.175265550613403
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,1.0,1,0,13.24,5.88288164138794
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,1.0,1,0,13.64,6.102787256240845
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.3,6.325354337692261
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.6,5.449536085128784
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.52,6.412497043609619
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.5,1,1,13.92,6.1262311935424805
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.13,7.470496654510498
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.17,5.5667808055877686
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.51,5.625491380691528
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.86,5.764652252197266
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.56,5.388487339019775
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.4,5.390807151794434
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.31,5.155649662017822
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.33,6.227263450622559
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.81,6.60251259803772
A,noisy,7,0.0001,0.0001,1,1.0,1,7,7,7,100,0.0,0,0,13.44,6.143570184707642
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,1.0,1,0,6.88,5.2958760261535645
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.55,5.523231029510498
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.49,4.354429244995117
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.19,4.80750036239624
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.56,4.51754093170166
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.34,4.904860973358154
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.7,4.442378044128418
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.35,4.992513418197632
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.48,4.842288970947266
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.38,4.67973518371582
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.51,4.339662313461304
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.34,4.740267276763916
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,7.21,4.75348687171936
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.05,4.198665857315064
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.5,1,1,6.55,4.194202184677124
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.73,4.036686182022095
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.32,4.070033311843872
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.88,4.540827512741089
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.48,4.0626442432403564
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.11,4.978838205337524
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.45,4.85195779800415
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.5,1,1,6.49,4.58208703994751
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.45,5.221708297729492
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.82,5.137767791748047
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.25,4.365992546081543
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.6,5.219192981719971
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.98,5.123441457748413
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.02,4.038707494735718
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.74,4.320098400115967
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.78,4.955016613006592
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.87,4.863381862640381
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.49,4.310559988021851
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.77,4.972816228866577
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,3.0,3,0,6.0,4.285968065261841
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,5.82,4.725478887557983
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.82,5.392774820327759
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,1.5,2,1,6.76,6.172566652297974
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.82,5.001559019088745
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.78,4.295412540435791
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.59,5.436915397644043
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,1.0,1,0,6.17,5.620777130126953
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.41,5.325257301330566
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,1.0,1,0,6.88,4.80084228515625
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,1.5,2,1,6.39,4.990677833557129
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.46,4.68008279800415
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.39,4.822717189788818
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.67,5.675416231155396
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.58,5.26410174369812
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.44,4.701712608337402
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.46,5.558385610580444
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.9,5.257648468017578
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.5,5.348258256912232
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.49,4.679565668106079
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,1.0,1,0,6.65,4.84469485282898
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.89,5.320602178573608
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.71,4.712092161178589
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.64,5.321173429489136
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.54,4.669967651367188
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.41,3.916353464126587
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.27,3.8920140266418457
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.68,4.401771545410156
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.55,4.86691689491272
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.41,4.195748567581177
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.63,4.685991048812866
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.57,4.489531755447388
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.12,4.776970624923706
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.84,4.64098858833313
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.51,5.718713998794556
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.61,5.214545726776123
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,1.0,1,0,6.96,4.6347081661224365
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,1.0,1,0,7.08,4.714947462081909
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.67,5.197314977645874
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.87,5.566823720932007
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.5,1,1,6.6,5.563184499740601
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.68,5.033477306365967
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.93,5.070167303085327
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.62,4.413194179534912
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.55,4.694061756134033
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.47,4.4400506019592285
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.61,3.9269165992736816
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.74,3.9487109184265137
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.82,4.496412038803101
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.86,4.777521371841431
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.95,5.357125282287598
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,1.0,1,0,6.56,4.650361776351929
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,5.63,4.540980339050293
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.56,4.939285755157471
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.31,5.663369655609131
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.55,5.887047052383423
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.21,4.982597351074219
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.43,5.284321069717407
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.98,5.254309892654419
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,1.0,1,0,6.68,6.029428243637085
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.83,5.8179931640625
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.44,5.5672876834869385
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.37,4.907427787780762
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.66,4.798925399780273
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.17,5.217834711074829
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.46,6.089278697967529
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.76,4.937441110610962
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.06,6.305618047714233
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.71,4.60370659828186
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.48,4.694446563720703
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.64,5.589624643325806
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.41,5.112321853637695
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.55,5.280917882919312
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.53,5.352078914642334
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.45,4.88751745223999
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.58,4.738361835479736
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,1.0,1,0,6.47,4.734743118286133
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.17,4.451839208602905
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.7,5.254314661026001
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.65,4.604596376419067
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.55,4.709453344345093
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.78,4.097201347351074
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.53,4.928210973739624
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.37,4.280629873275757
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.7,4.161035537719727
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.61,4.947750091552734
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.59,4.797385692596436
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.15,4.837942361831665
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.71,4.5364320278167725
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,1.0,1,0,6.5,4.79977011680603
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.52,4.423705101013184
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.7,5.081632852554321
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.03,4.66505241394043
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.34,5.526360511779785
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.81,4.894970417022705
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.32,4.910647392272949
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.55,5.836379289627075
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.47,4.586615324020386
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.25,4.29117488861084
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.26,4.52329421043396
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.7,5.314255237579346
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.62,5.385404109954834
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.47,5.149218559265137
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,5.96,4.843888282775879
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.74,5.012770414352417
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.56,4.975276231765747
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.47,4.865235567092896
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.48,4.973426342010498
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.64,4.902549505233765
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.6,4.873352527618408
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.68,4.678579807281494
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.45,3.982922554016113
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.21,5.120179653167725
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.24,4.684494733810425
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.55,5.093959331512451
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.76,4.781522512435913
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.82,5.661032676696777
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.42,5.114664077758789
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.49,4.9930150508880615
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,1.0,1,0,6.98,5.91609525680542
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,1.0,1,0,7.07,5.06031060218811
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.88,5.12150764465332
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.29,5.231437921524048
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,7.19,5.015109539031982
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.65,4.597581148147583
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,1.0,1,0,6.82,5.507303237915039
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.82,4.675570011138916
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.64,4.613354921340942
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.5,1,1,6.94,5.030068635940552
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.45,4.531131744384766
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.72,4.860684871673584
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.61,5.212758779525757
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.48,4.81787371635437
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.58,4.020394563674927
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.37,4.109238386154175
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.38,4.33737325668335
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.52,4.554442405700684
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.45,4.206014156341553
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,1.0,1,0,6.61,4.749091625213623
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,7.0,4.656558513641357
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.5,1,1,6.03,4.466539621353149
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,7.2,4.592488527297974
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.56,4.661816358566284
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.58,5.097214221954346
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.09,4.978476524353027
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.73,4.743276834487915
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.14,4.522294282913208
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.79,4.608295917510986
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.87,5.449357986450195
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.44,4.949596166610718
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.46,4.936827182769775
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,1.0,1,0,6.75,4.684956789016724
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.52,5.216549634933472
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.6,4.5034801959991455
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,1.0,1,0,6.19,4.180481195449829
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.75,5.435832738876343
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.21,5.182022333145142
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.35,4.273065805435181
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.68,4.328978538513184
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.6,5.545136451721191
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.67,5.013011693954468
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.82,4.436002254486084
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,7.01,4.871084690093994
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.71,4.948800086975098
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.69,4.673197746276856
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.39,5.394176006317139
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.64,4.381653785705566
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.79,4.7475879192352295
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.58,4.865178108215332
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.34,4.968834161758423
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.08,4.557812929153442
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.06,3.998650550842285
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,1.0,1,0,7.26,4.413048267364502
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.47,4.635535001754761
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.18,4.627974271774292
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.56,4.528753757476807
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.84,4.847873210906982
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.37,4.818319082260132
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.4,4.693805932998657
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.11,4.739961624145508
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.12,4.682577610015869
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.45,4.2419047355651855
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.5,4.488469839096069
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.25,4.5386457443237305
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.06,4.425804615020752
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.41,4.604531288146973
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.81,5.217877864837647
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.27,4.8848795890808105
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.32,4.098585844039917
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.66,4.092657804489136
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.53,4.146202802658081
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.17,4.758350372314453
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.26,4.484019994735718
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.36,4.53992486000061
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.85,4.902740001678467
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.59,4.85235333442688
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.59,4.504649639129639
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.86,4.588803291320801
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,7.0,4.465253829956055
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.65,4.652549743652344
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.59,4.161064863204956
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.54,4.231128692626953
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,7.1,4.800279140472412
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.21,4.7790610790252686
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.85,5.125373840332031
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.26,4.670185804367065
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.47,5.506591081619263
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.62,4.769571781158447
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,7.02,5.14259672164917
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.62,4.645665645599365
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.39,4.538832426071167
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.49,5.322068929672241
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.13,4.264727592468262
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,7.02,4.619213104248047
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.86,4.594623565673828
B,noisy,7,0.0002,0.0,1,1.0,1,7,7,7,100,0.0,0,0,6.03,4.29372239112854
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,1.0,1,0,7.59,5.346451282501221
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,1.0,1,0,8.15,5.263970851898193
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.79,5.830384969711304
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.74,6.30362343788147
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.27,5.973092794418335
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.93,6.088365077972412
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.26,6.373101711273193
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.5,6.9571802616119385
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.09,7.114733695983887
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.7,6.445987224578857
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.19,5.689603090286255
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.34,5.364120960235596
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.16,6.423279762268066
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.5,6.110480070114136
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.95,5.440973281860352
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.53,5.565495729446411
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.05,5.908639907836914
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.66,5.910678625106812
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.03,5.72678279876709
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.94,6.002577543258667
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.36,6.405889272689819
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.3,6.428324222564697
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.23,6.832567930221558
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.1,6.075603008270264
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.32,6.579487085342407
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.45,7.01603889465332
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.67,6.074810028076172
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.04,6.024040937423706
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.11,6.221391916275024
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.57,5.807234764099121
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.08,6.694721937179565
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.16,6.401772737503052
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,1.0,1,0,8.13,6.373499155044556
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.3,6.540321350097656
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.43,5.979653358459473
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.88,6.082846164703369
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.59,6.849505424499512
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.19,6.193308115005493
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.15,6.251737117767334
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.89,6.14950704574585
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.17,6.391421556472778
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.25,6.46254301071167
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.19,7.154107570648193
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.73,6.919523000717163
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.5,1,1,8.05,6.129931688308716
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.2,6.602293252944946
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,1.0,1,0,8.23,6.206632375717163
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.34,6.058584213256836
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.9,6.03385853767395
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.47,6.114912033081055
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,1.0,1,0,8.19,6.799762725830078
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.93,5.7460033893585205
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.26,7.278238296508789
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.05,6.042372226715088
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.57,5.827916860580444
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.94,6.07667350769043
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.3,6.243496894836426
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.28,6.241595268249512
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.49,6.430975914001465
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.64,6.066753149032593
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.1,6.43898344039917
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.17,6.525757074356079
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.71,6.120580196380615
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.18,6.807605981826782
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.47,7.202263832092285
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.32,6.697317123413086
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.13,6.136619091033936
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.19,6.538934230804443
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.03,7.129180192947388
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.07,6.482225656509399
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.54,7.177609920501709
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.34,7.085444688796997
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.6,6.720544815063477
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.22,6.650391578674316
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,1.0,1,0,9.06,6.518695116043091
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.83,6.404495000839233
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.63,6.4557952880859375
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.95,6.050006866455078
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.12,6.607158660888672
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.9,6.374765396118164
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.06,5.352734088897705
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.97,6.226885318756104
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.48,6.529603481292725
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.2,7.188474893569946
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.01,6.296543121337891
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.15,7.16706395149231
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,9.21,6.954108476638794
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.13,6.5576653480529785
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.73,7.7183518409729
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.01,6.102010250091553
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.37,6.247554302215576
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.52,6.172722339630127
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.54,6.198176622390747
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.5,1,1,7.53,6.819095611572266
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.08,6.172396421432495
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.86,6.2641332149505615
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.0,5.878354072570801
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.25,6.392134189605713
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.22,6.398238182067871
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,1.0,1,0,8.68,6.393815040588379
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.04,7.316055059432983
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.82,5.955778121948242
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.0,6.284682989120483
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.47,6.161967754364014
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.9,6.067591428756714
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.0,5.929782867431641
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.73,6.656930685043335
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.05,7.292686700820923
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.45,6.043523788452148
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.09,6.476063013076782
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.08,5.952291011810303
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.65,5.544452667236328
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.89,6.509200811386108
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.05,6.47724986076355
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,1.5,2,1,8.33,6.458743572235107
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.67,6.318586349487305
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.28,5.992152690887451
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.09,6.617365837097168
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.98,6.7107994556427
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.9,5.823721408843994
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.11,6.406865358352661
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,1.0,1,0,8.31,6.465176343917847
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.86,6.0158398151397705
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.47,7.138607501983643
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.2,6.24849796295166
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.48,6.318769693374634
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.92,6.370656967163086
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.95,6.018761873245239
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.6,5.87716555595398
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.45,6.665934085845947
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.26,6.529689073562622
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,1.0,1,0,8.0,5.983492136001587
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.1,6.4371724128723145
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.0,6.063133239746094
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.27,5.93033504486084
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.23,5.785714626312256
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.37,5.4351208209991455
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.01,5.348818063735962
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,1.0,1,0,7.77,5.818233251571655
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.95,5.987083911895752
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.48,6.71956992149353
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.21,6.891219139099121
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.85,5.755108594894409
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.22,6.347440242767334
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.34,6.464420795440674
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.15,5.3828284740448
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.88,6.546583414077759
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.61,5.965881109237671
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.95,6.903890132904053
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.19,7.000237941741943
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.2,6.599820613861084
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.05,6.341882228851318
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.26,5.970342636108398
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.16,6.646565675735474
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.42,7.261944770812988
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.43,6.311263084411621
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.24,6.097070693969727
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.32,6.144635200500488
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.28,6.348396301269531
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.3,6.256793022155762
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.05,6.442912340164185
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.34,6.764378547668457
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.16,6.560319423675537
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.83,6.3263914585113525
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.37,6.411441326141357
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,1.0,1,0,8.0,5.8498289585113525
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.91,5.60506010055542
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.49,6.391695499420166
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.14,5.862583160400391
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.25,6.175983667373657
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.18,6.077888250350952
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.73,6.017750024795532
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.06,6.427231311798096
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.21,7.332246780395508
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.21,6.6556923389434814
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.4,6.723351001739502
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.4,6.412275314331055
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.3,6.553545951843262
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.4,5.813896894454956
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.3,6.152512788772583
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.6,6.410238742828369
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.85,5.37883734703064
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,7.79,6.072468519210815
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.55,5.828873634338379
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.05,6.1208367347717285
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.14,5.83584189414978
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.26,7.071811676025391
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.08,6.496866226196289
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.03,6.85248875617981
B,noisy,7,0.0002,0.0,1,1.5,1,7,11,7,100,0.0,0,0,8.0,5.960045337677002
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.5,2,1,3.65,5.780657529830933
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,2,2,3.6825,5.135536670684815
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.645,6.411756277084351
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.7325,5.848384857177734
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.6875,5.911138534545898
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.595,5.869145154953003
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.6575,6.364859580993652
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.865,6.093112945556641
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.655,5.705942630767822
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.64,5.97717022895813
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.62,5.82049036026001
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.605,6.033353805541992
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,2.0,2,0,3.55,5.826754570007324
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,3.0,3,0,3.6025,5.71535325050354
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.6825,5.962375164031982
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.6975,6.22228741645813
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.7,5.631424903869629
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.6175,6.1745665073394775
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.77,6.059720754623413
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.5675,5.66200065612793
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.665,5.682828664779663
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.575,5.208920478820801
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.6425,5.362780332565308
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.6475,5.725524663925171
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.6825,5.2746546268463135
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.66,5.016967296600342
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.81,5.482818365097046
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.7575,4.83715558052063
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.655,5.819572448730469
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.5,2,1,3.5275,5.886747360229492
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.6675,5.646793365478516
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.805,5.934822082519531
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.5125,5.310394048690796
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.4525,5.549776554107666
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.525,5.3081464767456055
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.7175,5.690687656402588
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.73,5.990983724594116
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.55,5.620234966278076
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.7425,5.315485715866089
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.7075,5.412907838821411
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.5975,5.905272483825684
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.635,5.55012321472168
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.5925,6.0253190994262695
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.5,2,1,3.675,5.31250786781311
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.7075,5.706910133361816
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.795,5.7017576694488525
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.83,7.00378680229187
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.9525,6.036868572235107
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.7325,5.772971153259277
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.5625,6.35226583480835
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.8125,5.659369707107544
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.655,5.5316033363342285
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.5725,5.4699623584747314
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.705,6.3237268924713135
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.43,5.553195476531982
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.3475,5.296320199966431
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.6925,5.748213052749634
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.66,5.530450344085693
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.5825,5.845951318740845
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.775,5.399417877197266
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.6625,5.318869829177856
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.53,5.726033687591553
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.79,5.694006681442261
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.62,6.178055763244629
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.5375,6.2385008335113525
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.6325,5.645539999008179
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.8325,6.003076791763306
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.64,5.573613882064819
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.5975,5.707615852355957
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.8,5.59062647819519
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.43,5.488434553146362
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.6675,5.401565313339233
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.7275,5.141388654708862
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,2,2,3.675,5.802030086517334
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.63,5.689736604690552
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.7075,5.602064609527588
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.8175,5.593135356903076
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.6125,5.009735345840454
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.69,5.264566421508789
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.5775,5.227246046066284
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.705,5.391396999359131
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.805,5.653022050857544
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.5775,6.320640802383423
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.655,6.090334177017212
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.6575,6.055628061294556
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.705,5.317577838897705
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,2,2,3.7725,5.363727331161499
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.8975,5.884501457214356
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.865,5.675297021865845
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.6275,5.741042137145996
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,2.0,2,0,3.6525,5.237394571304321
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.63,5.339494705200195
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.65,5.440654993057251
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,2.5,3,1,3.715,5.519701719284058
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.5775,5.446258306503296
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.685,5.833288669586182
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.69,5.061424016952515
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.7475,6.007776260375977
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.42,5.437429428100586
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.725,5.840350151062012
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.6075,5.521514177322388
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.6875,5.260980129241943
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.685,5.438657522201538
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.5525,4.936608791351318
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.4125,5.442245244979858
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.78,5.802913665771484
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.4475,5.621371269226074
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.8,5.489096164703369
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.5,2,1,3.6575,5.281525611877441
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.4925,5.594285011291504
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.485,5.4675281047821045
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.675,5.666427850723267
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.4675,4.800409078598023
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.69,5.382568359375
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.535,5.386906623840332
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.7325,5.374678134918213
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.47,5.257243633270264
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.5675,5.133021354675293
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.575,5.419678926467896
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.5,2,1,3.5975,5.555783748626709
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.625,5.256005048751831
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.7675,6.573734045028687
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.5,2,1,3.535,5.208298683166504
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.5,2,1,3.6325,4.998321533203125
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.6,5.581560850143433
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.6925,5.458463430404663
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.6275,5.033938407897949
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.58,5.365404605865479
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,2.0,2,0,3.6325,6.176218032836914
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.465,5.492629528045654
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,2.0,2,0,3.7425,5.753860712051392
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.8375,5.913395881652832
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.6425,5.239005565643311
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.5,2,1,3.54,5.351964712142944
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.5875,5.627567768096924
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.6725,5.632261514663696
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.5725,5.416114807128906
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.5825,5.51008939743042
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.605,5.472091913223267
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.66,5.438287973403931
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.6525,5.222243070602417
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.685,5.575423717498779
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.6375,5.526612997055054
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.5,2,1,3.7725,5.575224876403809
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.655,5.331977844238281
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.71,5.375870704650879
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.6075,5.490032434463501
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.6875,5.328869104385376
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.785,5.899644136428833
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.6625,5.623089075088501
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.6325,5.623410940170288
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.82,5.665733814239502
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.7075,5.844453811645508
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,2.0,3,2,3.765,5.824130773544312
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.47,5.408538103103638
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.9575,5.490596771240234
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.59,5.5921630859375
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.515,5.494211673736572
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.745,5.237413644790649
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.555,5.530845403671265
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.67,5.2643818855285645
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,2.0,2,0,3.6025,5.360342025756836
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.53,5.481675624847412
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.535,4.865819215774536
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.5,2,1,3.7,5.579876661300659
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.6725,5.5346763134002686
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.66,5.573084831237793
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.71,5.503854036331177
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.865,5.793550491333008
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.5275,5.843443870544434
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.4825,6.001001119613648
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.575,5.619817495346069
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.63,5.484813928604126
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.4325,5.536996603012085
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.7175,5.651876449584961
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,2.0,2,0,3.7425,5.504706382751465
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.58,5.222512483596802
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.7525,5.382129907608032
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.73,5.268959999084473
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.77,5.525163888931274
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.635,5.128329753875732
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.7125,5.239155292510986
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.68,5.306078910827637
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.6725,5.442080020904541
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.6425,5.426222801208496
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.6675,5.408120155334473
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.7375,5.681203603744507
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.7975,5.5641303062438965
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.785,5.526933908462524
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,2,2,3.5025,5.562464714050293
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.715,5.190471887588501
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.4375,5.042749643325806
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.635,5.376305103302002
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.9025,5.568264961242676
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.67,5.616878509521484
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.645,5.4639668464660645
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.62,5.506397247314453
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.5725,5.167818784713745
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.6475,5.051937103271484
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.745,5.380890369415283
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.575,5.809749364852905
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.56,5.354176044464111
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.77,5.826541423797607
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,2.0,2,0,3.7325,5.478183746337891
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.7,5.501628875732422
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.5,1,1,3.6775,5.541037559509277
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.6475,5.488186597824097
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,2,2,3.705,5.694234371185303
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.605,5.827056169509888
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.7625,5.615965843200684
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,1.0,1,0,3.665,5.382648468017578
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.6425,5.317167520523071
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.785,5.583563327789307
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.595,5.592612504959106
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,0.0,0,0,3.53,5.634904384613037
B,noisy,5,0.0003,0.0,1,1.0,1,5,5,5,400,3.0,3,0,3.6875,5.7526328563690186
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.09,2.2125463485717773
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.115,2.1469507217407227
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.02,1.982853889465332
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.965,2.1519439220428467
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.98,1.9772093296051023
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.935,1.8887526988983157
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.12,1.911935806274414
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.185,1.8603582382202148
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.8,1.989856481552124
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.015,1.9118502140045168
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.045,2.066713094711304
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.995,1.9001479148864744
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.26,2.103426933288574
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.12,1.995020866394043
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.09,2.054680347442627
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.015,1.835606813430786
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.895,1.8552489280700684
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.005,2.050601959228516
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.94,2.0511460304260254
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.005,1.953689813613892
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.095,1.9812815189361568
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,2.0,3,2,2.065,1.917545318603516
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.955,2.0810811519622803
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.94,2.090028524398804
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.01,2.1877615451812744
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.895,1.9545273780822756
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.21,2.2558884620666504
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.92,2.1507608890533447
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.97,1.881089687347412
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.025,1.930037498474121
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.02,1.9475336074829104
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.005,1.9821643829345703
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.205,2.197226524353028
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.0,2.286496162414551
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.07,1.972947359085083
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.95,1.8156955242156985
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.17,2.016753435134888
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.15,1.986151933670044
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.01,1.897783279418945
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.09,2.1266098022460938
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.045,1.908974885940552
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.005,2.079732179641724
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,1.995,1.861968755722046
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.085,2.171211004257202
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.19,1.892395496368408
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.11,1.9375333786010744
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.235,2.002646207809448
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.94,2.154049158096313
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.055,2.009255170822144
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.05,1.874256610870361
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.1,1.8413732051849363
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.0,1.7758054733276367
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.21,2.130682468414306
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.02,2.3567748069763184
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.045,2.139371156692505
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.06,1.9307243824005127
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.19,1.9274060726165767
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.5,2,1,2.12,1.967029094696045
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.08,2.424983024597168
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.13,2.1638259887695312
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.1,2.029487371444702
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.905,1.792219161987305
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.815,1.818028688430786
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.075,2.08575701713562
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.07,2.061445951461792
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.165,2.276360988616944
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.965,2.0615737438201904
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.02,2.2020668983459477
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.1,1.977060317993164
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.8,2.009681224822998
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.175,2.135962963104248
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.87,1.8885376453399656
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.05,1.9830679893493648
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.06,2.1203579902648926
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.5,2,1,1.935,1.9402031898498533
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.015,2.3745296001434326
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.09,2.1024088859558105
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.265,2.0210442543029785
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.995,2.0473015308380127
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.15,2.148136138916016
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.13,2.0553324222564697
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.98,2.0079824924468994
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.89,1.9839112758636477
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.85,2.104173421859741
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.9,2.427377462387085
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.975,2.144010066986084
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,1.905,1.9324629306793213
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.955,1.972597599029541
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.005,1.9382107257843015
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.97,1.9840404987335205
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.08,1.7942559719085691
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.025,1.739170789718628
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.03,2.1623687744140625
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.94,1.945419788360596
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.06,2.0353572368621826
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.1,1.9809961318969729
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.195,2.174480438232422
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.145,2.1764204502105717
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.82,1.911247968673706
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,1.98,1.9768931865692136
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.145,2.01277756690979
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.0,1.8822007179260252
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.945,2.165701389312744
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,2.0,2,0,2.215,2.0967912673950195
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.13,2.186612844467163
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.025,2.1152210235595703
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.88,1.909918546676636
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.05,1.9587249755859373
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.015,1.872364282608032
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.92,2.059065103530884
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,1.92,2.2511255741119385
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.01,2.011478900909424
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,1.955,2.2961437702178955
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.005,1.7853703498840332
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.025,1.8745875358581543
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.925,1.957728385925293
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.085,2.004124879837036
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.99,1.8521981239318848
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.0,1.87188982963562
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.115,2.0611257553100586
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.1,2.1793038845062256
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.055,2.185732841491699
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.025,2.08367133140564
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.07,2.013740539550781
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.105,2.313950777053833
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.155,1.9729382991790767
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.885,2.1197428703308105
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.945,1.9544949531555176
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.125,1.9826891422271729
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.02,1.82543420791626
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.105,2.0386483669281006
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.1,1.9412176609039309
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.195,1.8782448768615725
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.785,2.2106752395629883
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.14,2.155583381652832
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.87,2.1163923740386963
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.865,2.141587734222412
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.095,2.175357818603516
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.165,2.0595951080322266
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.16,2.1451566219329834
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.965,2.09765362739563
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.95,2.100822687149048
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.03,2.086256742477417
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.91,2.139810562133789
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.03,2.04353141784668
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.0,2.00766658782959
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.03,1.9430944919586184
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.065,1.9243805408477783
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.985,1.8951683044433596
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.12,1.9247026443481443
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.975,1.802555799484253
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.1,1.949429035186768
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,1.9,1.8178379535675049
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.885,1.75829815864563
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.025,1.9817802906036377
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.05,2.004957437515259
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,3.0,3,0,2.13,2.2117514610290527
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.12,2.0939300060272217
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.13,1.8849167823791504
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,1.945,1.7997517585754397
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,1.935,1.8875248432159424
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.99,1.9083583354949951
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.88,2.000496387481689
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.08,2.111093521118164
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.115,2.3236987590789795
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.09,2.164690494537353
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.155,2.1888155937194824
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.82,2.053337574005127
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.88,2.0031545162200928
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.005,2.106264352798462
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.96,2.2098662853240967
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.995,2.139711141586304
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.215,2.1761693954467773
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.06,1.9923405647277832
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.055,2.002852439880371
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.06,2.240696907043457
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.955,2.2899587154388428
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.065,2.1593985557556152
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,1.94,1.95028018951416
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.98,1.9468727111816408
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.115,2.1743855476379395
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.055,1.949596643447876
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,1.87,1.9870491027832031
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.01,2.019519090652466
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.09,1.917421579360962
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.875,1.8357999324798584
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.985,2.2510344982147217
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.0,2.0620310306549072
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.945,2.187394618988037
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,1.915,2.123215436935425
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.165,2.201079130172729
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.95,1.99322509765625
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.085,2.027834177017212
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.93,2.032992362976074
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.0,2.193570137023926
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.0,1.906904220581055
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.925,2.057536363601685
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.945,1.8872127532958984
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.07,2.0203919410705566
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.905,2.067657232284546
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.945,1.9307758808135984
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.065,2.1436667442321777
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.135,2.061564445495605
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.07,1.828561782836914
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.95,2.2932288646698
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.055,1.9657740592956543
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.05,2.250361442565918
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.095,2.105649948120117
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.11,2.0251033306121826
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.145,2.0910816192626958
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.06,2.143271446228028
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.115,2.031357288360596
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.135,1.9416987895965576
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.05,1.923677682876587
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.04,1.926624774932861
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.07,1.892125368118286
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.195,2.2215375900268555
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.01,1.892906188964844
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.13,1.931568384170532
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.065,1.9673957824707031
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.15,1.9867310523986816
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.115,1.8182339668273928
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.955,1.9772617816925049
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.06,1.95231294631958
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.015,1.9721794128417969
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.02,1.9887244701385496
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.04,2.021855115890503
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.075,2.158160924911499
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.045,1.9938950538635252
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.055,2.014352321624756
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.06,2.0603725910186768
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.235,2.2490551471710205
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.065,2.0766608715057373
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.11,1.8965158462524412
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.175,2.026727199554444
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.94,1.887311935424805
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.045,2.233439445495605
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.96,1.912806749343872
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.045,2.154520034790039
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.095,2.2446606159210205
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.03,1.974968433380127
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.96,2.065081834793091
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.19,2.069374084472656
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.175,2.001145362854004
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.01,2.224970579147339
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.03,2.0853166580200195
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.155,1.9749131202697756
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.2,2.152923345565796
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.975,1.943337202072144
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.855,2.217827081680298
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.16,2.1457040309906006
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.87,1.8056070804595947
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.875,2.0726423263549805
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.06,2.075676202774048
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.995,2.0958340167999268
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.975,1.866593837738037
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.22,2.1384711265563965
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.72,1.961347341537476
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.045,1.9834363460540767
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.01,1.937861442565918
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.885,2.013277769088745
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.08,1.9227452278137207
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.985,1.8596765995025637
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.07,2.138068199157715
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.03,1.9969172477722168
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.02,1.7836909294128418
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.0,1.9397783279418943
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.965,1.940781116485596
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.785,1.7730588912963867
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.04,2.178934335708618
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.27,2.0011236667633057
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.965,1.848610639572144
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.18,1.870514154434204
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.88,2.002294540405273
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.05,2.110768795013428
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.985,1.906691551208496
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.075,1.8963661193847656
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.055,1.7688965797424316
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.91,2.111309766769409
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.095,2.076942682266236
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.0,1.8918399810791016
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.775,1.925702810287476
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.065,1.8810656070709229
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.055,1.836615800857544
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.115,2.106215238571167
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.955,1.9778430461883545
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.11,1.8296561241149905
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.155,1.9654030799865725
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.99,2.039185047149658
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.07,2.107211112976074
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.07,2.023207902908325
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.055,1.887826919555664
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.195,2.008760452270508
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.01,2.1082262992858887
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.14,1.992138385772705
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.0,1.859612226486206
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.05,2.234666347503662
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.145,2.146169185638428
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.075,1.948368549346924
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.93,1.8984293937683103
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.085,1.867517709732056
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.125,1.9364032745361328
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.165,1.933117389678955
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.175,2.04236388206482
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.93,1.9679818153381348
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.99,2.129602432250977
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.045,2.139314889907837
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.855,2.050733089447021
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.04,2.055627822875977
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.99,2.365809440612793
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.01,2.135371208190918
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.935,2.0796263217926025
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.22,1.9749631881713867
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.065,1.9550471305847168
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.215,2.0104730129241943
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.095,1.9826006889343264
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.08,2.0002167224884038
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.21,1.8759751319885252
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,1.855,2.181849956512451
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.105,2.588704824447632
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.165,2.2852232456207275
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.99,2.3860080242156982
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.1,2.2771787643432617
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.025,2.0983593463897705
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.21,2.1077210903167725
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.045,2.2722392082214355
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.045,1.9911184310913088
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.19,2.129018068313598
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.115,2.0360589027404785
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.135,1.9863677024841309
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.135,1.991043329238892
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.095,1.8933513164520264
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.04,1.8794240951538088
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.135,2.011104106903076
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.125,2.1085245609283447
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.945,1.911585807800293
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.04,2.085688352584839
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.165,1.9669163227081297
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.995,1.96786904335022
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.06,2.1160900592803955
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.92,2.133302927017212
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.045,2.0593574047088623
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.08,2.0010669231414795
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.055,2.0145363807678223
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.935,1.890596389770508
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.07,1.9797956943511963
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.0,1.97602653503418
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.2,2.09870719909668
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.08,2.100417375564575
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.88,2.056950330734253
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.05,2.1943483352661133
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.145,2.250169277191162
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.265,2.2026941776275635
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.055,2.1419897079467773
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.065,1.951685905456543
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.825,1.9853835105896
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.125,1.947117567062378
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,1.99,2.160457134246826
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.135,2.140047311782837
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.285,1.953420162200928
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.115,2.089557409286499
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.09,2.113977670669556
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.025,2.127648115158081
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.02,1.922872543334961
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.05,1.9587101936340328
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,1.93,2.083660364151001
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.09,2.009565591812134
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.01,1.9614298343658447
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.9,2.041111469268799
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.065,2.054161548614502
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.16,2.141059637069702
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,1.915,1.9304304122924805
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.9,2.0457584857940674
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.02,1.9535741806030276
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.155,2.212857723236084
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.88,2.141700029373169
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.835,2.0514135360717773
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.15,2.008328437805176
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.88,2.1118834018707275
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.86,1.845388889312744
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.805,2.01949405670166
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.005,2.3190879821777344
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.18,2.2746593952178955
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.17,2.178722858428955
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.035,2.0887391567230225
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.88,1.8122897148132324
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.065,1.953843355178833
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.24,1.9715399742126465
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.205,2.294478178024292
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.035,2.074720859527588
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.06,2.196676254272461
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.045,2.5799927711486816
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.305,2.0337815284729004
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.07,2.1041531562805176
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.035,2.2436273097991943
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.105,2.128891706466675
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.985,2.54778265953064
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.99,1.9358267784118648
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.1,2.047222375869751
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,2.0,2,0,2.055,2.918875217437744
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.985,2.6210474967956543
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.025,2.461737871170044
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,1.975,2.4594945907592773
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.82,1.932241678237915
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.045,1.9041764736175537
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.15,2.011024713516236
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.0,1.9837393760681152
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.01,2.062662124633789
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.055,2.267908811569214
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.885,2.256067037582397
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.02,1.9662823677062988
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.07,2.034695863723755
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.08,2.2304930686950684
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,1.995,2.0304672718048096
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.99,2.232540845870972
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.08,2.197164535522461
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,1.99,2.049994945526123
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.095,2.483370304107666
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.05,2.320390462875366
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.87,1.931430101394653
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.225,2.36253023147583
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.855,2.2309563159942627
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.095,2.166639804840088
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.975,2.0967414379119877
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.18,2.4681336879730225
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.155,2.748387098312378
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.065,2.259727716445923
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.06,2.199802875518799
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.12,2.3517110347747803
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.035,2.323934316635132
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,1.955,2.286082029342652
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.105,2.168300151824951
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.15,1.9278903007507324
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.985,2.079540967941284
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.19,1.9828870296478271
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.095,1.8838646411895752
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.02,2.016679286956787
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.5,2,1,2.03,2.067307710647583
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.145,2.141561985015869
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.01,1.974992036819458
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.105,1.95001220703125
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.235,2.1495981216430664
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.09,2.1106226444244385
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.01,2.082108736038208
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.11,1.9017021656036377
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.09,2.0696730613708496
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.96,1.8979167938232424
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.085,1.9218661785125728
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.26,1.8688335418701167
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,2,2,1.965,1.9980812072753904
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.81,1.956792116165161
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.105,1.943376064300537
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.09,2.0052530765533447
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.0,1.929725646972656
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.05,1.9469611644744875
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.025,1.9102911949157717
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.085,2.143563985824585
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.96,1.944403886795044
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.965,1.8626744747161863
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.99,2.002424478530884
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.095,1.8983032703399656
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.155,2.094982385635376
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.025,1.9876728057861328
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.115,1.9319698810577397
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.955,1.8916244506835935
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.15,2.005923271179199
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.03,1.927553653717041
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.085,2.1310713291168213
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.1,1.9637327194213867
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.23,1.90685510635376
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.1,2.003669261932373
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.0,1.943158864974976
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.175,2.040923833847046
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.25,1.951622486114502
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.11,1.9359016418457031
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.055,1.8759472370147705
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.005,2.0049192905426025
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.045,1.9464633464813232
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.995,2.126060724258423
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.02,1.899139404296875
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.04,2.009693384170532
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.75,2.2721242904663086
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.215,2.04745888710022
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.04,2.0141079425811768
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,1.995,2.181513547897339
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.86,2.1052987575531006
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.045,2.104604959487915
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.175,2.107205867767334
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,2.0,2,0,1.955,2.3100686073303223
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.84,2.2776310443878174
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.92,1.9947001934051516
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.155,2.0258374214172363
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.27,1.962278127670288
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.11,2.143329381942749
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.805,2.218043088912964
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.18,2.0719103813171387
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.185,2.0848307609558105
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.175,2.481290817260742
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.155,2.273763656616211
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.915,2.43420147895813
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.87,2.2128849029541016
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.0,1.9032700061798096
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.895,2.138807773590088
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.245,2.210440158843994
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.035,1.9475696086883545
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.1,2.2939910888671875
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.005,2.062598466873169
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.03,1.9131395816802976
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.005,2.097993850708008
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.065,2.052077054977417
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.005,2.1969263553619385
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.04,2.0151708126068115
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.115,2.096310615539551
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.93,2.134342670440674
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.05,2.211613655090332
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,1.985,2.06611967086792
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.035,2.1183650493621826
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.005,1.8610200881958008
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,1.985,1.7978341579437256
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.0,1.9927494525909424
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,1.88,1.9330785274505613
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.03,1.947479248046875
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.885,1.810163974761963
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.325,1.954563856124878
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.185,2.024430513381958
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.0,1.9349098205566408
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,1.855,2.0533607006073
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.03,1.9611425399780276
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.945,2.0882375240325928
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.06,1.9991722106933596
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.025,1.9390480518341064
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.155,1.977749347686768
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.09,1.972075700759888
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.895,1.954967975616455
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.0,1.8724632263183596
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.015,2.0384106636047363
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.92,2.1605372428894043
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,1.915,2.024598360061645
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.11,2.27742600440979
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.985,1.889570713043213
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.175,1.9431324005126955
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.12,2.0694971084594727
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.07,1.92284631729126
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.145,1.9796881675720213
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.075,2.050783634185791
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.1,2.0599851608276367
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.975,2.2044289112091064
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,1.98,1.991325855255127
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.98,1.9596529006958008
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.98,2.093338966369629
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.02,2.0886733531951904
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.11,2.2148420810699463
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.895,2.132678985595703
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.08,2.063218593597412
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.02,2.333281755447388
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.975,1.9879698753356931
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.11,2.3319599628448486
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.085,2.416365146636963
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.1,2.1475603580474854
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.96,2.3613474369049072
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.18,2.104609966278076
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.225,2.258384704589844
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,1.0,1,0,2.205,2.049188137054444
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.205,2.446802854537964
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.93,2.100111246109009
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.035,2.4231135845184326
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.885,2.283071756362915
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.03,2.160062789916992
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.055,2.378467798233032
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,1.96,2.0872840881347656
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.09,2.146248579025269
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.5,1,1,2.08,2.009256601333618
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.08,2.3546993732452397
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.01,2.087752103805542
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,1.895,2.0126538276672363
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.1,2.3040452003479004
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.165,2.2766737937927246
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.145,2.239396095275879
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.025,2.0929083824157715
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.105,2.127523422241211
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.08,2.166421413421631
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.11,2.2454540729522705
B,noisy,5,0.0002,0.0,1,0.6,1,5,3,5,200,0.0,0,0,2.07,2.878282308578491
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.1275,1.771287202835083
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.21,1.9855306148529053
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.225,2.0033633708953857
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.22,1.968364953994751
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.0875,1.7620325088500977
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.1975,1.7961502075195312
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.2025,1.9261670112609863
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.155,1.9172093868255613
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.2825,1.8130278587341309
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.165,1.7761051654815674
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.1175,1.742764234542847
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.1975,1.7096643447875977
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.085,1.6748790740966797
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.5,2,1,1.2,1.7333598136901855
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.2775,1.8076283931732176
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.5,2,1,1.1825,1.8784537315368648
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,4.0,4,0,1.2175,1.8326761722564695
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.2375,1.865081548690796
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.2975,1.870394229888916
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.225,1.849369525909424
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.155,1.9584743976593015
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.1325,1.8622384071350095
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,3.0,3,0,1.3175,2.1906607151031494
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.125,2.194450616836548
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.2125,1.82597017288208
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.2175,1.7627804279327393
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,3.0,3,0,1.2475,1.5681345462799072
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.1375,1.680347919464111
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.175,2.061208486557007
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.3,1.8477225303649905
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.13,1.8825902938842771
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.5,3,1,1.13,1.6700677871704102
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.2575,1.8170511722564695
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.145,1.788416624069214
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.1625,1.958052158355713
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.5,1,1,1.145,1.6780025959014893
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.18,1.7705023288726809
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.135,1.7586290836334229
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.1125,1.7037770748138428
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.1175,1.666578769683838
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,3.0,3,0,1.2025,1.694439172744751
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,3.0,3,0,1.2025,1.7043824195861816
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.2375,1.787477970123291
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.5,1,1,1.2,1.6833064556121826
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.1825,1.9185097217559812
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.155,1.93140172958374
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.1775,1.8347103595733645
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.145,1.650153398513794
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.1925,1.9392714500427248
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.1925,1.9869868755340576
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,3.0,3,0,1.19,2.2549920082092285
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.5,1,1,1.1875,2.100592374801636
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.2575,2.074069738388061
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.2275,1.8279592990875244
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.28,1.857743263244629
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.23,1.862825870513916
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.1975,1.886113405227661
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.2825,1.784101963043213
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.1875,1.7200696468353271
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.1775,2.023155212402344
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.5,3,1,1.1875,2.442659854888916
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.2125,2.139202117919922
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.115,2.109872817993164
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,3.0,3,0,1.1325,1.831230640411377
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.1875,1.8571891784667969
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.1375,2.2877204418182373
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.1525,1.909578561782837
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.2225,2.155601739883423
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.0925,2.0703744888305664
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.23,2.028506278991699
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.2725,2.0299222469329834
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.155,2.2130744457244877
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.0575,1.9011213779449463
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.2025,1.8082215785980225
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.2,2.04604721069336
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.14,2.224161386489868
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,4.0,4,0,1.2825,1.7805261611938477
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.1325,1.792158842086792
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.2175,1.5781242847442627
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.285,2.002539873123169
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.15,1.6475472450256348
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.13,1.6543560028076172
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.2825,1.9981334209442136
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.1825,2.20377254486084
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.235,2.080041885375977
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.235,1.8744192123413088
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,3.0,3,0,1.2225,1.904348373413086
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,3.0,3,0,1.2175,2.2180678844451904
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,3.0,3,0,1.225,2.000584602355957
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.1925,2.139465570449829
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.2875,1.7175719738006592
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.1775,1.714643955230713
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.2725,1.7407069206237793
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.2325,1.6904349327087402
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.2225,1.685737133026123
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.1525,2.022531270980835
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.09,1.9042623043060305
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.2075,2.180241107940674
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.18,2.1344552040100098
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.315,2.0780107975006104
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.15,2.1425905227661133
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.1475,2.249521493911743
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.2525,1.7786049842834473
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.1425,2.004463195800781
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.15,1.8968756198883057
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.19,1.7646877765655518
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.205,1.7265496253967283
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.2175,1.903458833694458
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.1,2.1066811084747314
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.13,1.6704044342041016
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.18,2.181634426116944
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.045,1.8760871887207031
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.215,1.765331745147705
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.18,2.354619264602661
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.15,2.5311105251312256
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.2025,2.475862979888916
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.245,1.980663061141968
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.14,2.5801379680633545
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.185,2.1420905590057373
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.5,1,1,1.11,2.114710807800293
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.1875,2.3649449348449707
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.2175,2.104962110519409
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.08,1.903188705444336
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.1625,1.8917508125305176
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.3,2.113638162612915
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.28,1.8757917881011963
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.2225,1.9639313220977783
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,3.5,4,1,1.2275,1.9816248416900637
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.19,2.1222410202026367
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.1775,1.804600715637207
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,3.0,3,0,1.165,2.006317377090454
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,3.0,3,0,1.25,1.6719553470611572
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.1875,1.6686787605285645
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.1475,1.93593168258667
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,4.0,4,0,1.1775,1.7704331874847412
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,3.0,3,0,1.24,1.8546388149261477
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.21,1.658724308013916
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.1625,1.8966920375823972
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.2,2.081200122833252
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.2225,2.130432605743408
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.275,1.889491319656372
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.1375,1.6771092414855957
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.2825,1.9326121807098389
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.2425,2.207204341888428
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.1975,2.295879364013672
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.225,1.838193655014038
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.09,1.8923604488372805
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.5,2,1,1.1525,1.9536464214324951
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.1925,2.249013900756836
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,2.0,2,0,1.2275,1.8664829730987549
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.1475,2.0906803607940674
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.1325,1.9399185180664065
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.1775,2.0584421157836914
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.27,2.5138816833496094
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.1675,1.9860570430755613
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.225,2.1093521118164062
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.275,2.15010929107666
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.0975,1.9459869861602783
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.155,2.914088487625122
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.0825,1.906338453292847
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.1875,2.078974962234497
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.5,1,1,1.17,2.161362648010254
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.255,1.9296250343322756
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.24,2.016819953918457
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,1.0,1,0,1.2525,2.290295362472534
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.19,1.991398572921753
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,0.0,0,0,1.2925,1.875281810760498
A,noisy,3,0.0001,0.0001,1,1.0,1,3,3,3,400,4.0,4,0,1.1475,2.3753623962402344
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.8575,1.7497777938842771
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.82,1.5547621250152588
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.75,1.6507527828216553
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.845,1.3775174617767334
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.875,1.4499735832214355
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.785,1.507376194000244
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.875,1.7222506999969482
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.8675,1.4280509948730469
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.7575,1.4212579727172852
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.8225,2.1424806118011475
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.865,1.785306453704834
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.845,1.9669208526611328
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.735,1.5463712215423584
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.8425,1.7648587226867676
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.78,2.176064729690552
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.83,1.451371192932129
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.79,1.5957703590393066
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.8775,1.6311657428741455
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.76,2.057877779006958
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,4.0,4,0,0.8475,1.825174331665039
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,4.5,5,1,0.84,1.7837316989898682
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.8,1.6535773277282717
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.83,1.546039342880249
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.82,1.418976068496704
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.865,1.5271196365356443
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.7425,1.6290266513824463
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.9,1.3734822273254397
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.8675,1.4470925331115725
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.8125,1.825687885284424
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.8925,1.7160565853118896
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,3.0,3,0,0.7975,1.3685526847839355
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.8425,1.6975383758544922
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.8325,1.7761507034301758
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.8125,1.372342824935913
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.8425,1.23691987991333
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.845,1.3942439556121826
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.795,1.6089084148406982
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.76,1.3622279167175293
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.79,1.4099664688110352
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.8375,1.7885847091674805
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.725,1.667149543762207
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,3.0,3,0,0.8075,1.5878381729125977
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.83,1.3671636581420898
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.8175,1.560832977294922
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.875,1.6462624073028564
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.8325,1.5180346965789795
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.8125,1.5091238021850586
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.86,1.749605417251587
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.805,1.742213249206543
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.835,1.5033173561096191
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.815,1.4895100593566897
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.8475,1.4659857749938965
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.7325,1.3727514743804932
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.805,1.624377965927124
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.7925,1.8005564212799072
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,3.0,3,0,0.8325,1.811370611190796
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.8425,1.7837989330291748
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.8,1.9921181201934812
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,4.0,4,0,0.8175,2.1403377056121826
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.7875,1.9537370204925537
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.8125,2.0001561641693115
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.75,2.188192844390869
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.7975,2.245133638381958
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.8525,1.88519287109375
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.82,1.9224402904510496
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.82,1.9250071048736568
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.5,3,1,0.87,2.061603546142578
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.87,1.616490602493286
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.8825,2.185657024383545
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.865,1.7600088119506836
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,3.0,3,0,0.8025,1.6238036155700684
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.835,1.7394342422485352
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.8025,2.077235698699951
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.8525,1.7468767166137695
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.88,1.7275819778442385
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.8325,2.2532432079315186
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.89,2.337336778640747
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.8675,2.102107524871826
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.8,1.8612523078918457
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.8075,1.532733678817749
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.81,1.94576096534729
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.79,2.128979206085205
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.875,1.6800227165222168
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.845,1.4796569347381592
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.89,1.6339490413665771
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.8675,1.676196575164795
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.8325,2.2081704139709477
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.8125,1.849585771560669
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.8425,1.808478832244873
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.9075,1.7298214435577393
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.7825,1.7271230220794678
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.83,1.8453834056854248
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.8425,1.957628965377808
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.8525,1.8913912773132324
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.795,1.8615682125091555
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.8875,1.6074013710021973
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.815,2.1210198402404785
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.8225,1.7147765159606934
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.8025,1.929424524307251
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,4.0,4,0,0.795,1.9472410678863523
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.7575,2.204457521438598
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,3.0,3,0,0.805,2.0266342163085938
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.985,1.9549365043640137
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.83,1.708503007888794
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.8525,1.831773042678833
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.8075,1.8218178749084477
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.825,1.5191996097564695
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.795,1.6434619426727295
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.5,3,1,0.86,1.6669504642486572
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.8075,1.5722179412841797
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.8275,1.5785763263702393
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,3,2,0.79,1.9928138256073
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,3.5,4,1,0.8975,1.7077066898345947
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.8725,1.7689275741577148
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.795,1.7579734325408936
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.7925,1.7549645900726318
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.8525,1.8948006629943848
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.8725,1.8020379543304443
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.8425,1.6736440658569336
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.9075,1.4091346263885498
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.78,1.6292133331298828
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.84,1.5354628562927246
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.85,1.43752121925354
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,3.0,4,2,0.7975,1.587551832199097
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.85,1.5121002197265625
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.84,1.566826343536377
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.8425,1.459156513214111
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.785,2.098100423812866
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.805,1.72625994682312
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.905,1.6492042541503906
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.81,2.073411464691162
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.875,1.646538496017456
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.81,1.6928579807281494
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.825,1.8066508769989011
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.805,1.6405835151672363
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.8075,1.6803293228149414
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.5,3,1,0.9275,1.622025966644287
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.7775,1.612959623336792
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.81,1.6793603897094729
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.885,1.6454801559448242
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.79,1.7761166095733645
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.86,1.4937095642089844
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.7975,1.8776969909667969
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.8425,1.4771921634674072
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.8225,1.6256873607635498
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.855,1.52358078956604
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.79,1.7549140453338623
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.89,1.6291182041168213
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.9,1.588754415512085
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.735,1.7987616062164309
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.8275,1.5477936267852783
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.8625,1.4780077934265137
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.7425,1.4156725406646729
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.845,1.4127411842346191
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.82,1.3120336532592771
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.5,3,1,0.8125,1.583658218383789
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.9325,1.5451014041900637
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.7875,1.4849915504455566
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.83,1.4864599704742432
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.84,1.4860177040100098
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.83,1.6299107074737549
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.8325,1.5804991722106934
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.76,1.6621878147125244
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.795,1.6086225509643557
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.8225,1.5412640571594238
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,3,2,0.88,1.6514363288879397
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.755,1.4483304023742676
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.825,1.75213360786438
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.845,1.567134141921997
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.8075,1.4821219444274902
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.8525,1.9884839057922363
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,3.0,3,0,0.7925,1.4681627750396729
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.7375,1.583479881286621
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.78,1.48293137550354
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.83,1.5396485328674316
B,noisy,3,0.0003,0.0,1,1.0,1,3,3,3,400,2.0,3,2,0.775,1.4542038440704346
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.4,6.784695625305176
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.65,6.203613758087158
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.65,5.423207998275757
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.875,5.989346504211426
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.05,5.703497886657715
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,58.4,6.049750804901123
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,57.425,5.297912836074829
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.575,6.100566387176514
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.7,6.170886993408203
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.7,5.9450554847717285
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.5,1,1,55.425,5.627346992492676
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.5,1,1,53.975,5.055079460144043
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,53.4,5.342644453048706
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.75,5.95951509475708
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.85,6.179031848907471
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,1.0,1,0,56.825,5.762239456176758
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,57.925,6.0570831298828125
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.175,5.833377122879028
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,53.125,5.795946836471558
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.9,6.290160894393921
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,53.825,9.237814664840698
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,57.175,7.835031270980835
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.0,7.419255971908569
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.125,7.8662269115448
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.1,7.753300189971924
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.125,8.001724243164062
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.175,8.206462860107422
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.125,8.218395233154297
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,57.425,7.502958059310913
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,53.05,8.907468557357788
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.225,8.144297361373901
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.175,7.108698844909668
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.125,8.388763904571533
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,53.275,7.006189346313477
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.025,7.734405994415283
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.775,8.702728271484375
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,57.275,7.987082719802856
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.0,8.187196254730225
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,53.025,7.564234495162964
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.925,8.358160257339478
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,53.2,9.16434907913208
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.375,8.241859912872314
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.05,7.378371477127075
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,58.325,7.852000713348389
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.975,8.207287311553955
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.125,8.193092107772827
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.5,1,1,54.975,8.522996664047241
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,2.0,2,0,56.9,7.752976894378662
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.05,8.135403156280518
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.975,9.560947179794312
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.5,8.235610246658325
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.875,8.886737823486328
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.85,7.221210956573486
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.625,8.187450408935547
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.375,8.029045343399048
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.85,7.656161785125732
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.075,7.226562738418579
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,57.275,8.288490533828735
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,57.125,8.75919771194458
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.25,8.154468297958374
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.175,7.391092300415039
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.55,9.161783933639526
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,52.75,8.397526025772095
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.225,7.942468643188477
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.8,8.433172225952148
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.725,6.603368282318115
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.35,5.741273880004883
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.15,5.78619909286499
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.1,5.801567077636719
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.5,6.336773872375488
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.075,6.0192036628723145
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.625,5.201751708984375
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.6,6.359315633773804
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.6,6.898890733718872
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,53.8,5.881352663040161
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,57.775,5.906150579452515
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.675,7.04010009765625
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.5,8.692987442016602
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,1.0,1,0,55.975,8.583353996276855
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.775,7.761751174926758
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,1.0,1,0,57.275,8.432912111282349
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.225,7.947551012039185
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,53.575,8.47648811340332
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,57.125,8.144410133361816
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.725,9.214799404144289
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.5,8.496880769729614
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,1.0,1,0,56.8,8.918369770050049
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.2,8.807640790939331
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,57.6,7.392408609390259
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,57.075,7.581737756729126
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,57.1,7.519452810287476
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,57.8,7.976533889770508
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.55,6.895146369934082
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.175,7.846563100814819
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.775,7.501713514328003
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.4,7.501531362533569
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.3,7.414455652236938
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.55,8.139706134796143
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.125,8.6752188205719
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,1.0,1,0,55.825,8.452357530593872
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,1.0,1,0,55.0,7.461258411407471
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.0,8.390635967254639
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.975,7.153684616088867
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.5,1,1,56.55,6.922571420669556
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,57.575,7.961160182952881
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.675,7.146366834640503
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.5,1,1,53.6,7.463203191757202
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.675,8.983718872070312
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.45,7.194218873977661
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.85,7.737874746322632
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.6,8.289970874786377
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.25,8.997862815856934
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,57.675,8.34108567237854
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.05,8.145753622055054
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.825,7.481879472732544
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,52.375,9.109816312789915
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.2,8.612457752227783
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.575,7.133684158325195
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.45,8.909091711044312
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.55,7.642552852630615
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.6,7.9955010414123535
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.025,7.699115514755249
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.775,8.673926591873169
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,53.975,7.039011240005493
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.8,8.684867143630981
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,57.375,7.14024543762207
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.3,8.175812721252441
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,58.175,8.968038082122803
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.725,6.76959753036499
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.675,7.990615844726562
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.25,7.873564720153809
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.575,8.091105937957764
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.175,8.134999513626099
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.95,7.048886060714722
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.4,7.464295864105225
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.025,7.5142502784729
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.65,8.379591941833496
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.725,8.296449184417725
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.275,7.833187580108643
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,1.0,1,0,54.725,8.322458505630493
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.45,8.281378269195557
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.525,8.256416082382202
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.65,8.302585124969482
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,53.575,7.540266275405884
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,58.05,6.780904769897461
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.075,6.647087812423706
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.5,1,1,56.05,7.966600894927978
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.775,8.048937559127808
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.9,7.620039939880371
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.75,7.597122430801392
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.55,6.860731363296509
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,54.5,6.840157747268677
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,1.0,1,0,55.275,9.047977685928345
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,52.375,8.238953351974487
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.725,7.792671203613281
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.2,7.734339952468872
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,56.1,9.313819408416748
A,noisy,9,0.0002,0.0002,1,1.0,1,9,9,9,40,0.0,0,0,55.675,9.171786546707152
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,19.875,9.48124933242798
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.1,7.798427581787109
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.55,8.152674913406372
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.225,6.985897779464722
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.1,8.053137063980103
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.35,7.108400821685791
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.075,6.826894998550415
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.925,8.063960075378418
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,19.925,7.541480302810669
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.15,8.29783010482788
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.8,8.218754053115845
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,19.925,9.11485505104065
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.7,8.353509426116943
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.125,7.314488172531128
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.7,7.494183540344238
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.25,8.025873184204102
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.05,7.965865135192871
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.825,6.657386779785156
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.225,7.533159255981445
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.9,5.969375133514404
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.825,5.406788110733032
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.55,5.285727024078369
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,19.4,5.572063446044922
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.55,5.2166829109191895
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.725,5.486768960952759
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.45,5.877529859542847
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.325,7.241601467132568
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.25,5.909179210662842
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.1,6.981346607208252
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,19.85,7.365559101104736
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.225,8.214716672897339
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.05,6.587064981460571
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.675,8.403571128845215
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.825,7.717467784881592
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,18.8,8.175625085830688
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.7,7.453065633773804
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.675,7.123329639434815
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.4,7.198237180709839
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.95,7.732842922210693
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.3,6.625046730041504
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.625,7.41512393951416
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,19.775,7.579395532608032
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,19.925,7.813114881515503
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.325,7.386054039001465
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.725,7.906815528869629
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.375,6.795425415039063
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.75,8.806263446807861
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,1.0,1,0,20.125,8.282866477966309
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.125,9.209519863128662
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,17.95,7.771714925765991
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.025,7.4657204151153564
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.25,6.051723480224609
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.75,7.46414589881897
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,22.2,7.768282651901245
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.3,7.826967000961304
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.725,7.676111698150635
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.95,8.45986294746399
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.625,7.026766777038574
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.7,6.261128902435303
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.825,6.832692861557007
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.075,8.920135021209717
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,19.825,6.959938049316406
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.475,8.080451011657715
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.7,8.754313945770264
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,19.7,6.8129637241363525
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.05,7.274052143096924
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.725,7.331580400466919
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.075,8.36707353591919
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,19.45,6.230748891830444
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.85,6.775148630142212
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,19.9,7.885616302490234
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.05,7.297547101974487
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.8,8.113350629806519
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.15,6.989559650421143
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.45,6.779370307922363
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.225,8.001067638397217
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,19.675,7.517467021942139
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.975,7.773306846618652
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,19.975,8.022631406784058
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.425,6.882723808288574
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.375,9.014046669006348
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.525,6.725411176681519
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.9,6.641851186752319
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.05,9.469285011291504
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.4,8.36673092842102
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.15,8.332074165344238
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.225,9.23415756225586
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.925,7.600459814071655
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.95,8.012235641479492
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.825,8.051999568939209
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.5,7.354918718338013
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,19.725,8.32826042175293
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.85,8.146387577056885
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.65,8.776895999908447
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,1.0,1,0,18.2,6.985724687576294
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.5,1,1,21.525,8.151409387588501
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.225,8.624566316604614
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.7,8.673088073730469
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,19.8,9.111552238464355
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,19.65,7.415643215179443
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.475,7.972165584564209
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.5,8.84901213645935
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.675,7.950240850448608
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,1.0,1,0,20.0,8.223286867141724
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.825,9.015968799591064
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,22.2,9.130762100219728
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.45,9.084972620010376
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.8,9.9726083278656
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.425,9.284031867980955
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.5,1,1,21.325,10.63156270980835
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,1.0,1,0,19.425,8.18480396270752
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,19.625,10.513977527618408
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.4,9.585420846939089
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.275,9.501734495162964
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.925,8.368014335632324
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.775,7.351475954055786
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.2,8.99284839630127
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,19.9,6.702290296554565
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.0,8.459269523620605
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.575,7.666802883148193
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.425,8.139909029006958
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.825,8.625174283981323
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.175,7.556295394897461
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.4,8.79150938987732
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.625,8.863220691680908
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,19.625,8.32166862487793
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.5,7.5906150341033936
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,19.925,8.445705890655518
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.325,9.059147119522097
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.4,8.803262948989868
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.45,8.297112464904785
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.05,9.317980289459229
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.125,8.672291278839111
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.2,8.854032278060913
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.15,8.090999126434326
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.5,8.07503366470337
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.6,9.693418741226196
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.6,9.491584300994871
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.25,7.6023218631744385
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.45,7.466233730316162
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.825,7.8021135330200195
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.95,8.20110273361206
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,18.775,8.628486633300781
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,19.75,6.741922616958618
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.675,9.030410528182983
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,21.025,8.976226329803467
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,19.2,8.637855052947998
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.65,8.072999715805054
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.525,7.609996318817139
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.6,8.47743844985962
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,22.3,8.20440149307251
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,1.0,1,0,20.75,7.1407470703125
B,noisy,9,0.0003,0.0,1,1.0,1,9,9,9,40,0.0,0,0,20.6,9.73117184638977
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,66.02,9.176751375198364
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,2,0,66.77,9.088453769683838
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,2,0,66.31,10.007262706756592
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,2,0,66.34,9.84296989440918
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,66.82,10.534590244293211
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.5,2,1,65.64,9.709912061691284
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.5,4,3,68.75,8.80945873260498
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,66.22,10.186926364898682
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,66.7,9.891288995742798
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,3,2,68.01,10.637113809585571
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,2,0,67.1,11.05707573890686
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,2,0,66.5,10.3657705783844
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,5.0,5,0,67.41,8.91543960571289
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,2,0,67.45,10.69631028175354
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,0.0,0,0,66.5,10.143611907958984
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,3.0,3,0,68.04,9.13910984992981
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,3.0,3,0,66.76,8.600804090499878
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,0.5,1,1,65.81,10.823657274246216
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,65.62,9.149872541427612
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.5,2,1,65.49,7.317857980728149
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,66.69,6.783197402954102
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,0.0,0,0,67.0,11.398822546005247
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,4.0,4,0,66.81,9.872768878936768
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,67.17,11.136206150054932
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,67.75,10.459556341171265
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,0.5,1,1,67.01,9.878171682357788
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,67.22,10.860396146774292
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,66.03,10.87056303024292
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,68.56,11.30354905128479
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,66.65,10.4311625957489
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,3,2,68.04,10.071712255477903
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,66.49,10.089649438858032
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,4.0,4,0,67.0,11.109061002731323
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,0.0,0,0,67.78,9.654243230819702
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,2,0,66.37,10.679603576660156
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,2,0,66.37,11.32247519493103
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,0.0,0,0,67.68,13.33351182937622
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.5,3,1,66.25,11.427970170974731
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,2,0,67.43,11.487285852432253
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,3,2,68.12,11.233432531356812
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,66.3,10.192855596542358
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,2,0,66.96,12.12667465209961
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,0.0,0,0,67.77,11.87354016304016
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.5,3,1,65.56,9.50860857963562
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,0.0,0,0,65.39,10.659181833267212
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,66.18,10.41622757911682
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,0.5,1,1,67.36,10.628520727157593
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,3.0,3,0,68.02,9.412405252456663
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,0.0,0,0,68.76,9.930284976959229
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,2,0,65.9,10.532717227935793
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,4.5,5,1,67.41,10.789988994598389
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.5,2,1,65.83,10.55230951309204
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,0.0,0,0,65.19,9.989392042160034
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,68.27,10.399013996124268
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,0.5,1,1,67.6,9.266409158706663
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,66.65,9.83594799041748
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,0.5,1,1,67.9,10.350867986679075
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,67.34,9.816571474075316
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,0.0,0,0,66.05,9.728516578674316
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,0.0,0,0,67.02,11.236188650131226
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,0.0,0,0,67.54,10.68975019454956
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,66.53,9.59260630607605
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,66.76,9.93643355369568
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,0.0,0,0,66.04,12.365142583847046
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,67.81,11.702809572219849
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,65.88,10.145294666290283
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,68.91,9.574186086654665
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.5,2,1,67.45,10.081202745437622
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,0.0,0,0,67.07,10.126242637634276
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.5,2,1,66.07,10.206426858901978
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.5,2,1,67.62,10.146084308624268
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,67.07,9.645614385604858
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,65.92,9.520716190338137
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.5,3,1,67.49,9.797008514404297
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,2,0,66.57,9.82568645477295
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,3.0,3,0,67.27,9.81319260597229
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,66.75,10.543746709823608
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,2,0,66.16,10.66394567489624
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,0.0,0,0,67.46,10.929521083831789
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,0.5,1,1,65.99,10.038684844970703
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,65.19,12.37090277671814
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,3.0,3,0,67.3,10.883002042770386
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,2,0,67.55,8.625503778457642
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,66.82,10.919556856155396
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,2,0,66.73,9.490604400634766
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,2,0,66.84,10.050037145614624
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,2,0,65.87,11.31462335586548
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,0.5,1,1,67.84,11.166172742843628
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.5,2,1,67.36,10.356855630874634
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,0.0,0,0,66.98,8.137375116348267
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,3.0,3,0,67.01,9.718886375427246
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,66.5,9.82495379447937
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,66.99,11.590448141098022
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,6.5,7,1,66.96,10.932517766952516
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,0.0,0,0,66.28,8.582431554794312
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.5,2,1,68.28,10.724315404891968
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,65.73,10.041502714157104
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,3.0,4,2,66.78,10.418507814407349
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,3.5,4,1,66.45,11.451312065124512
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,0.0,0,0,66.92,11.811038494110107
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,2,0,65.98,11.766529083251951
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,67.9,11.956890106201172
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,4.0,4,0,67.75,12.328392505645752
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,3.0,3,0,65.46,11.65199899673462
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.5,3,1,66.47,9.889111280441284
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,4.0,4,0,66.98,10.506885051727297
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,2,0,64.34,9.742167949676514
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,2,0,66.16,10.849042654037476
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.5,2,1,66.99,9.910008430480955
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,2,0,67.54,9.912227869033812
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,4.0,4,0,66.57,10.332950592041016
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.5,3,1,67.65,10.328707456588743
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,3.0,3,0,66.96,11.565164804458618
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.0,2,0,67.08,11.130585193634031
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,1.0,1,0,66.42,11.287750482559204
A,noisy,7,0.0005,0.0005,1,1.0,1,7,7,7,100,2.5,3,1,67.24,10.716961145401
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,25.76,10.43955659866333
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,25.73,11.06565237045288
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.39,11.198944091796877
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.0,12.426985263824465
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.0,11.263592720031738
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.61,10.710978031158447
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.44,9.620614528656006
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.48,9.700451850891112
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.14,10.523133993148804
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.54,10.11541986465454
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.77,9.527671813964844
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.5,1,1,27.46,10.042079210281372
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.35,10.398484230041504
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.74,11.476597547531128
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.43,10.85864758491516
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.67,10.925517082214355
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.85,9.57628870010376
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,2.0,2,0,27.33,8.588697910308838
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,25.58,9.123607635498049
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,28.36,10.00986933708191
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.67,8.388796329498291
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,25.98,10.061390161514282
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.86,8.6343412399292
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,1.5,2,1,26.83,8.183245182037354
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.06,6.823791265487671
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.22,6.6912829875946045
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.4,6.763095378875732
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.64,6.926837205886841
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.57,6.944047927856445
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,25.66,7.338824033737183
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.09,7.426846027374268
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.63,7.363167762756348
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,25.33,7.43993616104126
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.99,6.973950147628784
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,1.0,1,0,26.43,6.960748672485352
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.22,7.321415424346924
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.11,7.316904067993164
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.54,6.566311359405518
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.15,6.619231939315796
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.79,6.84663987159729
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.91,6.966753244400024
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.67,6.891197204589844
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.77,7.127804279327393
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.58,7.42443060874939
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.87,6.736436128616333
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.81,7.517667770385742
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,1.0,1,0,26.13,6.972391366958618
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.47,7.4278624057769775
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.77,7.04363226890564
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.11,7.315015554428101
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.86,6.933418273925781
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.21,6.756895542144775
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.8,6.832450866699219
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.61,7.545721769332886
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.71,7.3355491161346436
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.29,7.339932918548584
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.84,6.935572385787964
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.31,7.297123193740845
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,25.43,7.175327062606812
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,1.0,1,0,26.92,7.671678781509399
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.85,7.131779909133911
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,1.0,1,0,27.51,6.770448684692383
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.53,7.182067155838013
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,1.0,1,0,25.97,7.109106779098511
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.5,1,1,25.84,6.917799472808838
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.62,7.010680198669434
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.73,7.032315492630005
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.66,7.0349860191345215
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,25.89,7.356769800186157
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.13,7.606488227844238
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.42,6.45575475692749
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.54,6.5367653369903564
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.2,6.5608885288238525
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.9,6.36633038520813
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.21,7.88670539855957
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.34,6.64799690246582
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,25.87,6.525326728820801
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,25.92,7.155303716659546
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.53,6.982741594314575
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,25.61,6.8242104053497314
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.5,1,1,26.54,6.739948749542236
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.92,6.7184388637542725
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.04,6.452618360519409
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.59,6.412123203277588
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.65,6.610519170761108
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.59,6.745498895645142
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.96,6.939458847045898
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,1.0,1,0,26.75,6.632831573486328
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.01,7.122316837310791
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.03,6.487910985946655
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,25.96,6.609911203384399
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.18,6.683169841766357
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.52,6.9481940269470215
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,1.0,1,0,27.78,6.845808029174805
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.53,7.701223373413086
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,2.0,2,0,26.33,6.471928596496582
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.79,6.86290454864502
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.38,6.443385362625122
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.89,6.060185670852661
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,25.96,6.730272054672241
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,25.95,6.594203472137451
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,1.5,2,1,26.88,6.86462664604187
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.25,7.210832118988037
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,2.0,2,0,26.05,6.96105432510376
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.15,6.905548095703125
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.38,6.953841924667358
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.0,6.635270118713379
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.8,6.493287563323975
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.55,6.770747900009155
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.85,6.641883611679077
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.67,6.410383701324463
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.66,7.521826982498169
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.21,7.962751388549805
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,25.64,6.928978681564331
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.18,6.76276969909668
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.5,7.601184129714966
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.59,6.920997381210327
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.26,7.858564853668213
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,1.0,1,0,25.72,8.538299083709717
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.11,7.527170181274414
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.98,7.4544196128845215
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.27,6.847417593002319
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.83,7.469187021255493
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.3,7.3468337059021
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.68,8.061240196228027
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.87,7.519064426422119
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.08,8.349230766296387
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,25.89,7.268144607543945
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.32,7.253250122070312
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.48,7.931401014328003
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.47,7.7753472328186035
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.0,7.612865209579468
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.67,7.327018976211548
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.23,7.938120126724243
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.36,7.99625039100647
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.51,6.546770334243774
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.51,6.85228419303894
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.74,6.629012823104858
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.6,6.712841033935547
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.16,7.41052508354187
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,1.0,1,0,26.77,6.665121793746948
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,25.66,6.993083953857422
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,1.0,1,0,27.28,7.073859930038452
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.29,6.632144927978516
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.38,7.252686977386475
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,25.7,6.837037324905396
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.12,6.65181827545166
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.71,7.058277606964111
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.7,6.404193162918091
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.93,7.151057243347168
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.87,7.420533895492554
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,25.89,6.468121767044067
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.44,6.971642255783081
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,27.7,7.382860898971558
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.45,7.037208318710327
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.39,6.7960288524627686
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.38,7.090810775756836
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.01,7.121742486953735
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.7,6.736630916595459
A,noisy,7,0.0002,0.0002,1,1.0,1,7,7,7,100,0.0,0,0,26.45,7.561180591583252
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.43,7.38581109046936
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.24,8.280675172805786
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,33.66,7.026708126068115
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,33.04,7.522231578826904
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,33.95,6.948884725570679
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.74,6.269341468811035
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.79,6.416624546051025
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,2.5,3,1,32.5,6.976982831954956
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.34,6.695104122161865
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.5,1,1,31.63,6.685439348220825
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.35,7.238781690597534
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,2.0,2,0,33.5,6.76495099067688
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,33.01,7.464092016220093
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,2.5,3,1,33.81,6.721736907958984
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.5,2,1,32.59,6.49271297454834
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.69,6.557609558105469
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,2.5,3,1,32.75,5.973238945007324
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,33.44,7.671404361724853
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.76,6.780140161514282
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,2.0,2,0,33.11,6.622192144393921
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.71,6.85599160194397
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.23,6.992333173751831
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.5,2,1,32.4,6.634300708770752
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,33.09,6.90354061126709
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,33.03,6.468801736831665
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,2.5,3,1,32.73,6.670320272445679
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.09,5.716728925704956
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.41,6.982190608978272
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,31.79,5.901325941085815
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,31.89,6.534414291381836
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.18,6.385211944580078
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.68,5.979147434234619
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,3.0,3,0,31.96,6.663528203964233
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,2.0,2,0,33.11,6.4910924434661865
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,33.01,7.129469394683838
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,33.05,6.505250930786133
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.5,1,1,33.03,6.29916262626648
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.23,8.018877744674683
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.21,6.852140426635742
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.79,5.8861894607543945
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,33.09,6.624560117721558
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.09,6.388235569000244
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,33.12,6.280208349227905
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,2.5,3,1,33.02,6.712245225906372
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.58,6.435532808303833
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,33.42,6.7403340339660645
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.07,7.58342719078064
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.86,6.9838173389434814
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,33.23,7.2031025886535645
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,33.06,6.029473304748535
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,33.69,6.665152072906494
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,2,2,33.12,6.305765390396118
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,2.0,2,0,33.55,6.610226631164551
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.82,6.699126243591309
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,2.0,2,0,32.45,8.168623685836792
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.2,6.930740118026733
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.58,8.095292091369629
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.92,6.772013425827026
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.3,6.8141608238220215
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,33.27,7.574190855026245
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.5,2,1,33.17,6.660095453262329
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,33.3,7.422686576843262
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.41,6.581328630447388
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.5,2,1,32.71,6.475148916244507
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.92,7.368159770965576
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,33.3,6.656209468841553
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.64,6.731845378875732
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.78,6.238529682159424
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.25,7.211885213851929
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.77,6.619595289230347
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.64,5.850410461425781
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.75,6.599099397659302
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.8,6.707505941390991
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,33.55,7.471380949020386
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,33.56,7.224593639373779
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.34,6.222007989883423
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.67,7.201748847961426
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.84,6.822355508804321
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,33.82,5.948553085327148
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,31.9,6.513826847076416
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.71,6.408254861831665
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.34,6.655960559844971
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.43,6.702605724334717
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.85,6.702341079711914
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.97,7.686432838439941
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.42,6.655931234359741
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.46,7.395989418029785
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.83,7.190776824951172
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.21,7.544457197189331
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.75,7.062233686447144
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,33.06,7.315218687057495
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,2.0,2,0,32.97,7.328423023223877
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,31.73,7.306545257568359
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.5,2,1,33.43,6.533370018005371
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,3.0,3,0,32.95,8.202752113342285
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.96,7.295620679855347
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.06,6.826658487319946
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,33.12,6.976444959640503
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.37,7.431089401245117
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.5,1,1,33.1,6.557708978652954
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.93,6.890367031097412
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,2.0,2,0,34.03,6.182690620422363
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,2.0,2,0,31.26,6.091557502746582
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.62,6.631332159042358
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.58,6.678081750869751
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.92,6.258663892745972
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.98,7.33623480796814
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.5,2,1,31.98,6.745068311691284
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.3,6.752023458480835
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.82,6.973225593566895
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.5,1,1,32.94,7.8401854038238525
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.5,1,1,33.0,7.271831750869751
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.5,1,1,32.59,7.407979249954224
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,31.83,6.726745843887329
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.51,7.604164600372314
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,2.0,2,0,31.83,6.807963371276856
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.5,1,1,32.44,6.5316808223724365
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.94,7.051154375076294
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.82,6.770403861999512
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,2.0,2,0,31.96,6.450566530227661
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.31,6.500337362289429
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.63,5.895259618759155
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.99,5.866076946258545
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,2.0,3,2,33.24,6.657208681106567
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,31.91,6.23391580581665
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.5,1,1,32.46,6.881836891174316
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.14,7.901081562042236
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,2,2,32.69,6.905230522155762
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,2.0,2,0,32.77,6.448359966278076
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.5,2,1,32.7,7.071626901626587
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,31.71,6.628637790679932
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,33.77,7.045173168182373
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,33.43,7.017271757125855
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,33.78,7.339203834533691
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.66,6.886723756790161
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.25,6.810678958892822
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,2.0,3,2,31.41,5.892231225967407
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,33.13,6.50821590423584
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.65,6.182113170623779
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.52,6.739970207214356
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.5,1,1,32.9,5.948301076889038
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.97,5.863092422485352
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,31.55,6.345331192016602
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.94,6.574828624725342
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,31.57,6.528550148010254
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.99,6.802408456802368
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.65,5.849810123443604
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,33.28,6.147860765457153
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,33.09,6.280733823776245
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,2.0,2,0,32.04,6.91303300857544
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.68,7.061212778091431
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.24,6.571095705032349
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.95,6.191016435623169
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,33.56,6.625977993011475
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.5,1,1,32.96,6.341731309890747
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.95,6.911518335342407
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,33.1,6.963561058044434
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.22,6.653502941131592
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.5,1,1,32.75,6.492016315460205
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,33.21,6.991213798522949
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.54,6.854120016098023
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.26,6.991236448287964
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.25,6.219177484512329
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,33.32,6.267479181289673
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.5,2,1,32.71,7.012633800506592
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.5,1,1,32.98,6.793717384338379
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.08,6.942477226257324
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.29,6.38027811050415
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.5,2,1,33.59,6.416747331619263
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.69,6.2910473346710205
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.37,6.326096773147583
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.79,6.470088243484497
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.94,6.124231338500977
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,32.28,7.146756410598755
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,33.22,7.392568111419678
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,32.25,7.446666240692139
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,2.0,2,0,33.46,6.920035123825073
B,noisy,7,0.001,0.0,1,1.0,1,7,7,7,100,2.0,2,0,33.55,6.192652225494385
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,3.31,3.250067949295044
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.23,3.078364372253418
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.4,3.2165651321411133
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.17,3.3991024494171143
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.13,3.635019540786743
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.36,3.242053985595703
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.24,3.839750051498413
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.61,3.5391502380371094
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.5,1,1,3.26,3.233909845352173
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.38,4.120518684387207
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.09,3.477018117904663
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.22,3.618476152420044
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.05,3.2496209144592285
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.32,3.4752488136291504
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,2.91,3.167224884033203
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,3.11,3.413492441177368
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.14,3.0787370204925537
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.39,3.337017059326172
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.57,3.4675331115722656
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.63,3.389132022857666
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.12,3.233439683914185
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.5,3.2724626064300537
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.38,3.3987183570861816
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.21,3.4358668327331543
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.12,3.44895076751709
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.06,3.3528406620025635
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.23,3.555062055587769
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.34,3.371860980987549
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.51,3.584059715270996
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.48,3.586916446685791
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.45,3.7569098472595215
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.0,3.238132953643799
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.31,3.788883686065674
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.11,3.499022483825684
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.35,3.2982945442199707
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.09,3.260500192642212
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.14,3.168449640274048
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,3.03,3.569647789001465
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.13,2.9646434783935547
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.4,3.237035751342773
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.07,3.5403623580932617
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.4,3.129141330718994
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,3.4,3.314568281173706
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.26,3.373617172241211
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.21,3.2853269577026367
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.13,3.327145576477051
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.14,3.2518527507781982
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.43,3.513286590576172
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.41,3.738471269607544
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,2.97,3.133359670639038
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.3,3.3726229667663574
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.26,3.2541749477386475
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.04,3.6217968463897705
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.19,3.431062936782837
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.31,3.64766263961792
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.03,3.274287462234497
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.51,3.1154561042785645
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.17,3.4404959678649902
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.01,3.2869441509246826
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.07,3.4182913303375244
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.46,3.3053250312805176
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.0,3.0338566303253174
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.21,3.2369844913482666
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.24,3.365556478500366
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.38,3.555921792984009
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,2.92,3.0006353855133057
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.28,2.98207688331604
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,2.92,3.076656579971313
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.06,3.100792646408081
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.15,3.42089319229126
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.45,3.452077865600586
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.33,3.135019540786743
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.1,3.1802940368652344
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,3.23,3.3615312576293945
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.15,3.3322057723999023
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.01,2.938727378845215
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.13,3.3744237422943115
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.22,3.3791446685791016
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.25,3.013010740280152
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.58,3.470080137252808
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.24,3.3187191486358643
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.56,3.621938467025757
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,2.69,3.659596920013428
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.07,3.1212899684906006
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.27,3.445013999938965
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.14,3.374555826187134
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.01,3.2225136756896973
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.5,3.484814405441284
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.59,3.8021957874298096
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.46,3.523191452026367
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,2.97,3.3321783542633057
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.49,3.804516315460205
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.17,3.373497486114502
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.24,3.605202436447144
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.28,3.1927435398101807
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,3.33,3.5385475158691406
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.2,3.53735899925232
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.11,3.2131097316741943
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.4,3.78188419342041
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.34,3.4487433433532715
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.39,3.6794159412384033
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.34,3.5658657550811768
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.15,3.5250911712646484
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.25,3.4628360271453857
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.2,3.3394691944122314
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.2,3.221921682357788
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.25,3.6356494426727295
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.32,4.307009935379028
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.26,3.379078149795532
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.3,3.619905471801758
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.24,4.118743181228638
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.34,3.4303152561187744
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.16,3.56852388381958
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.24,3.314897060394287
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.19,3.655117750167847
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.01,3.073918581008911
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.5,3.325761079788208
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.24,3.2931957244873047
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.24,3.639242887496948
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.08,3.2861785888671875
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.11,3.6247718334198
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.31,3.0838265419006348
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.12,3.4539103507995605
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.24,3.673889398574829
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.48,3.4698565006256104
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.24,3.432047843933105
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,2.96,3.2579495906829834
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.33,3.3910014629364014
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.23,3.5063271522521973
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,3.32,3.3954906463623047
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.57,3.342975378036499
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.33,3.727596044540405
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,3.04,3.6602728366851807
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,2.98,3.3993732929229736
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.38,4.102503299713135
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.01,3.3798952102661133
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.67,3.2413697242736816
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.15,3.4986660480499268
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.07,3.578733444213867
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.4,3.533975601196289
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.25,3.210124969482422
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.6,3.759815216064453
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.42,3.541412591934204
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.49,3.644617080688477
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.43,3.574787378311157
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.32,3.178816318511963
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,2.0,2,0,3.44,3.5940651893615723
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.22,3.3265490531921387
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.07,3.691807746887207
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.37,3.5457534790039062
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.09,3.652317523956299
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.34,3.353127956390381
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.61,3.746595621109009
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.45,3.574122667312622
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.5,1,1,3.75,4.184418678283691
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.23,4.0801286697387695
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.27,3.64772629737854
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.21,3.905043125152588
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.03,3.794667959213257
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.54,3.605088949203491
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,3.33,3.615446329116821
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.43,4.213303565979004
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.29,3.677293539047241
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.0,3.856184720993042
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.5,1,1,3.52,3.75520396232605
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.13,3.7734506130218506
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.18,3.301506519317627
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.4,3.551085948944092
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,2.98,3.305976867675781
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.4,3.912132024765015
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.67,3.820683240890503
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.03,3.5478522777557373
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.12,3.5645880699157715
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.8,4.091881275177002
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.52,4.6438071727752686
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.14,3.495447158813477
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,3.09,3.750556945800781
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.54,3.386317253112793
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.28,3.582491874694824
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.0,3.566569566726685
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.49,3.652148962020874
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.0,3.390690565109253
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.3,3.757066011428833
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.49,3.5214614868164062
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.57,3.4184157848358154
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.21,3.416741371154785
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.44,3.584757089614868
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.22,3.2994384765625
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.46,3.5675477981567383
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.24,3.0368926525115967
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.64,3.462071657180786
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.45,3.5320143699646
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.52,3.542257070541382
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,2.0,2,0,3.7,4.297332286834717
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.15,3.2320916652679443
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.44,3.28436541557312
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.26,3.5848333835601807
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.56,4.385406732559204
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.07,3.804945230484009
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.3,4.039052486419678
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.19,3.886031150817871
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.41,3.230212926864624
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.44,3.311800479888916
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.54,3.5297982692718506
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.03,3.39969539642334
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.41,3.389782190322876
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.27,3.8515119552612314
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.11,3.608668804168701
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.25,3.684748411178589
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.57,3.2228517532348637
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.34,3.2667837142944336
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.55,3.479771137237549
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.29,3.623723030090332
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.24,3.500866651535034
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.37,3.316549301147461
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.05,3.168204069137573
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.12,3.476125955581665
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.34,3.4877266883850098
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.23,3.611929655075073
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.24,3.354386329650879
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.13,3.128426074981689
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.31,3.7537004947662354
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.22,3.67077112197876
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.29,3.5341241359710693
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.35,3.250861883163452
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.3,3.4844117164611816
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.16,3.608397483825684
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.41,3.5976274013519287
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.65,3.428872585296631
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.48,4.095625877380371
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.28,3.783227443695069
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.09,3.3700802326202397
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.09,3.4769270420074463
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.19,3.7436230182647705
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.01,3.6844756603240967
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,1.5,2,1,3.4,3.340687990188598
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.21,3.438755512237549
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.32,3.514521360397339
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,3.44,3.693090915679932
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.33,3.715197324752808
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.5,3.5329043865203857
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.45,3.6263303756713863
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.38,3.5134036540985107
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.06,3.2633676528930664
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.34,3.426991701126098
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.34,3.0998315811157227
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.33,3.240023374557495
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.14,3.5899486541748047
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.0,3.237761974334717
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.13,3.592010259628296
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.16,3.5349416732788086
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.52,3.6547446250915527
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.24,3.109338521957397
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.49,3.8683812618255615
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.43,3.418443441390991
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.1,3.082984447479248
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.03,3.213110446929932
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.45,3.327899217605591
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.15,3.1995720863342285
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.06,3.485292196273804
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,2.97,3.654599905014038
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.01,3.631735324859619
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.3,3.301409721374512
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,3.3,2.88660216331482
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.28,3.6621406078338623
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.24,3.1767024993896484
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.32,4.02336049079895
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.07,3.0471763610839844
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.17,3.2588016986846924
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.27,3.617490530014038
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.33,3.4060800075531006
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.31,3.788287401199341
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.26,3.621767282485962
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.35,3.097833633422852
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.3,3.352123498916626
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.07,3.780893564224243
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.54,3.329545259475708
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.17,3.271023750305176
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.15,3.706909418106079
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.46,3.439654588699341
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.5,1,1,3.57,3.3054022789001465
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.16,3.3974766731262207
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.39,3.550992488861084
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.42,3.5877795219421387
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.13,3.048869132995605
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,3.37,3.116742849349976
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.08,2.952197551727295
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.49,3.733036994934082
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.37,3.387914657592773
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.56,2.973515510559082
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.5,3.574866533279419
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.42,3.7243685722351074
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.33,3.634636163711548
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.16,3.4825189113616943
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,3.24,3.347414255142212
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.22,3.5760321617126465
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.44,3.839874505996704
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.22,3.721721410751343
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.14,3.232893705368042
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.18,3.3310370445251465
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.21,2.960233688354492
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.3,3.185650587081909
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.29,3.1671817302703857
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.5,1,1,3.3,3.645282745361328
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.29,3.09208345413208
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,3.42,3.083200454711914
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.15,3.6252617835998535
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.17,3.498132467269897
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,2.94,3.5048916339874268
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.23,3.361640930175781
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.43,3.712458848953247
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.41,3.110696315765381
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.36,3.4963865280151367
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.29,3.353055238723755
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.2,3.3940558433532715
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.44,3.146564722061157
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.22,3.4534952640533447
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.42,3.517497062683105
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.1,3.7545690536499023
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.24,3.6421146392822266
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.35,3.2420222759246826
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,3.31,3.717842817306519
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.17,3.192249298095703
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.46,3.711129903793335
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.41,3.675281047821045
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.54,3.482994079589844
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.3,3.631525039672852
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.09,3.00333309173584
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.34,3.6640474796295166
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.27,3.413644552230835
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.23,3.641720771789551
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.14,2.9715688228607178
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.23,3.5648584365844727
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.27,3.6202704906463614
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.27,3.922672510147095
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,1.0,1,0,3.15,3.5054681301116943
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.53,3.814910650253296
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.56,3.86139178276062
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.31,3.470934629440308
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.04,3.189537286758423
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.21,3.484670639038086
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.22,3.3477888107299805
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.52,3.5009732246398926
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.28,3.2350432872772217
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.08,3.064527988433838
B,noisy,7,0.0001,0.0,1,1.0,1,7,7,7,100,0.0,0,0,3.21,3.3147549629211426
A,noisy,5,0.001,0.001,1,1.0,1,5,5,5,400,74.0,78,8,51.385,11.62089705467224
A,noisy,5,0.001,0.001,1,1.0,1,5,5,5,400,75.0,77,4,50.3875,11.128232717514038
A,noisy,5,0.001,0.001,1,1.0,1,5,5,5,400,66.5,69,5,51.01,11.242223262786863
A,noisy,5,0.0005,0.0005,1,1.0,1,5,5,5,400,9.0,9,0,25.27,10.52752161026001
A,noisy,5,0.0005,0.0005,1,1.0,1,5,5,5,400,6.5,7,1,25.1975,9.610360860824583
A,noisy,5,0.0005,0.0005,1,1.0,1,5,5,5,400,7.5,8,1,25.1825,9.28487992286682
A,noisy,5,0.0005,0.0005,1,1.0,1,5,5,5,400,7.0,8,2,25.2675,9.363747119903564
A,noisy,5,0.0005,0.0005,1,1.0,1,5,5,5,400,10.0,11,2,24.9525,9.628960132598875
A,noisy,5,0.0005,0.0005,1,1.0,1,5,5,5,400,6.5,7,1,25.3275,9.145084142684937
A,noisy,5,0.0005,0.0005,1,1.0,1,5,5,5,400,7.5,9,3,24.8625,9.18352723121643
A,noisy,5,0.0005,0.0005,1,1.0,1,5,5,5,400,4.0,4,0,25.2175,9.55577039718628
A,noisy,5,0.0005,0.0005,1,1.0,1,5,5,5,400,6.0,6,0,25.4975,9.751919984817505
A,noisy,5,0.0005,0.0005,1,1.0,1,5,5,5,400,12.0,12,0,24.9175,9.7080659866333
A,noisy,5,0.0005,0.0005,1,1.0,1,5,5,5,400,13.0,13,0,25.075,10.9227876663208
A,noisy,5,0.0005,0.0005,1,1.0,1,5,5,5,400,6.5,7,1,25.3675,9.86125946044922
A,noisy,5,0.0005,0.0005,1,1.0,1,5,5,5,400,7.0,7,0,25.08,9.840392112731934
A,noisy,5,0.0005,0.0005,1,1.0,1,5,5,5,400,15.0,16,2,25.1375,9.762290716171265
A,noisy,5,0.0005,0.0005,1,1.0,1,5,5,5,400,10.0,10,0,25.7325,10.09875464439392
A,noisy,5,0.0005,0.0005,1,1.0,1,5,5,5,400,15.0,15,0,25.715,9.618759155273438
A,noisy,5,0.0005,0.0005,1,1.0,1,5,5,5,400,9.0,9,0,24.9675,9.7373149394989
A,noisy,5,0.0005,0.0005,1,1.0,1,5,5,5,400,12.5,13,1,25.3575,9.74836015701294
A,noisy,5,0.0005,0.0005,1,1.0,1,5,5,5,400,7.0,8,2,25.135,10.133421421051024
A,noisy,5,0.0005,0.0005,1,1.0,1,5,5,5,400,5.5,7,3,25.175,10.309686422348022
A,noisy,5,0.0005,0.0005,1,1.0,1,5,5,5,400,8.5,9,1,25.39,9.906190872192385
A,noisy,5,0.0005,0.0005,1,1.0,1,5,5,5,400,11.0,12,2,25.0675,10.305498600006104
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.015,6.614129304885864
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.095,6.675419807434082
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.105,6.673909664154053
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.19,6.756821870803833
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.1175,7.137545824050903
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0875,6.675102233886719
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.89,6.640563488006592
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.995,7.3897833824157715
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.9525,7.322110176086426
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.16,7.024848937988281
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.1225,7.095861911773682
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.095,7.084919214248657
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0,7.041388273239136
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,5.0125,7.285868167877197
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,5.085,6.72658896446228
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,5.3275,6.633424997329712
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.5,1,1,5.0375,7.080398082733154
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.05,6.779629230499268
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,5.035,7.217925548553467
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.8575,6.8370771408081055
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.085,7.062928915023804
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.1525,7.589341402053833
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.005,7.669275522232056
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,5.04,7.206213474273682
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.9775,7.182521820068359
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0075,6.9842588901519775
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.9775,6.801888704299927
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,4.935,6.858688831329346
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.125,7.050072908401489
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.23,7.122669219970703
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.1875,6.696266889572144
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.025,7.147745609283447
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.9275,6.859250068664551
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.1575,6.8170325756073
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.9725,7.644691705703735
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.12,7.4611546993255615
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0175,7.372540712356567
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.045,7.2668137550354
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.08,8.018810033798218
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,4.9925,7.010056495666504
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.2475,6.99291205406189
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.94,7.0398430824279785
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.165,6.80777907371521
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.5,2,1,4.95,7.002519607543945
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.1225,6.682860851287842
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.9375,6.997957706451416
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.245,6.640776872634888
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.23,6.593416690826416
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,5.03,6.744505882263184
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.1275,6.877647876739502
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.5,1,1,5.0325,6.8247435092926025
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0925,7.208565950393677
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.1475,7.703181505203247
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.085,7.070024967193604
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.1175,7.22980260848999
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.915,7.411440134048462
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.2025,6.98650050163269
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.1375,6.691496849060059
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,5.0375,6.825291633605957
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0225,6.5264668464660645
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.97,7.101012945175171
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.125,6.974041223526001
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.1575,6.887877702713013
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.8975,6.870928525924683
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,5.1,7.28949236869812
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0775,7.151081323623657
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.2175,7.020384073257446
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,5.04,7.187573671340942
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.05,7.041786670684815
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.5,1,1,5.0075,7.097175121307373
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.79,6.685735702514648
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0575,6.574985980987549
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.1225,6.869107723236084
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.2325,6.7164177894592285
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0975,6.550810098648071
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0525,7.238467216491699
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.075,6.7549779415130615
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.17,6.384476900100708
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.1825,6.758896827697754
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,5.0675,6.927233457565308
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.1275,6.805288314819336
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.1825,7.536860227584839
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,4.955,6.868949174880981
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,4.935,6.88788652420044
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0925,6.480244159698486
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,4.9625,6.806890964508057
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0675,6.691422939300537
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0325,7.631458282470703
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.035,7.064353466033936
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.21,7.238014936447143
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.9975,6.989197731018066
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.995,7.10460638999939
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0275,6.628549098968506
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,4.865,6.694185495376587
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,2.0,2,0,5.0825,6.479375600814819
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.155,6.846842050552368
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0975,6.478972434997559
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,5.06,6.323458909988403
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.1575,7.09385085105896
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.5,1,1,5.23,6.774620771408081
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,5.39,6.960708856582642
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0075,6.658939361572266
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.1175,7.145852327346802
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,4.8775,7.008295774459839
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.9925,6.782637119293213
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.2,6.843560695648193
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.885,6.696106672286987
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.1325,7.227280378341675
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.1275,6.891779899597168
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0425,6.5511767864227295
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.5,1,1,5.2225,6.672795295715332
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.9075,6.932902574539185
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0225,6.739537954330444
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.015,6.874893426895142
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.98,6.790096998214722
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.145,6.905126571655273
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.9825,6.91466760635376
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.05,7.035882234573364
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.9525,6.315246820449829
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.985,7.335684776306152
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.9275,7.100369930267334
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.5,1,1,5.0825,7.180492162704468
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0175,6.874300479888916
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.14,7.002541542053223
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.965,6.867395401000977
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0775,6.898955821990967
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,5.0575,6.4411780834198
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,5.17,6.632007598876953
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.06,6.800955057144165
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.015,6.766859292984009
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.1675,6.912740468978882
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0875,6.797588348388672
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.07,6.841293096542358
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.935,6.579543828964233
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.5,1,1,5.1125,6.976527214050293
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.8825,6.801057815551758
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.15,6.382018089294434
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.895,6.662514925003052
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.8375,6.8107733726501465
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.9525,6.782899618148804
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,4.9725,6.965782403945923
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.065,6.899404287338257
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0675,6.774894714355469
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,5.3175,6.952466249465942
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.03,6.701252698898315
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.985,6.848594665527344
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.9575,6.74482536315918
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.9925,7.258217573165893
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,4.9675,6.778172254562378
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,4.945,6.6945106983184814
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.5,1,1,5.0925,7.108973264694214
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,5.0525,7.239301681518555
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.035,6.736963272094727
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0225,6.763740301132202
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.02,6.972498416900635
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,5.105,6.831883668899536
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.38,7.112323999404907
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.1475,7.358143806457519
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.305,7.292001962661743
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0475,6.856197834014893
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.9275,7.2237958908081055
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.03,7.320467948913574
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0525,6.854782342910767
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0875,6.998175144195557
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,4.9025,7.242937564849853
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,5.1025,6.909366130828857
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.05,6.811874151229858
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.5,1,1,5.0325,7.066624164581299
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.1125,7.0860595703125
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0925,7.228182077407837
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,1.0,1,0,5.055,7.171227931976318
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.0,0,0,5.0575,7.136032104492188
A,noisy,5,0.0001,0.0001,1,1.0,1,5,5,5,400,0.5,1,1,4.965,6.831667423248291
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.5,1,1,2.3675,5.130550622940064
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.2875,4.811561107635498
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.3725,5.134838581085205
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4725,4.658134937286377
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.48,4.87546443939209
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4325,5.123918056488037
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.41,4.974242210388184
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.5,1,1,2.605,4.843742370605469
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.485,5.253044843673706
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.445,4.864812850952148
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4375,4.559839010238648
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.3875,4.718741893768311
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.53,4.885814189910889
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.405,5.039945363998413
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4175,4.949916362762451
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.45,4.619798421859741
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.5475,5.038604021072388
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.395,4.90807056427002
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4625,4.884061574935913
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.5,1,1,2.53,5.069427251815796
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.5,4.9337158203125
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.5525,4.937485218048096
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.375,4.767958402633667
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.375,4.76340389251709
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,2.0,2,0,2.6375,4.602822303771973
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4,4.36070728302002
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.635,4.92822003364563
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.5,2,1,2.4475,4.878316879272461
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.405,4.438639163970947
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.3175,4.673216819763184
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.415,4.836448669433594
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4275,4.846643686294556
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.33,4.693387269973755
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.435,4.815609693527222
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.48,4.878313779830933
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.47,4.891401767730713
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.52,5.171428442001343
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.3725,4.688100576400757
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4975,5.22063946723938
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.5,1,1,2.4275,4.517559051513672
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.5075,4.698077440261841
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4775,4.59445071220398
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.4125,5.004918098449707
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.395,4.745060205459595
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.5,4.866150379180908
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.3975,5.083684206008911
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.46,4.995607137680054
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.5,1,1,2.5125,4.822494029998779
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4475,5.081895112991333
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.47,4.654664516448975
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.41,4.804470539093018
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,2.5,3,1,2.4975,4.918211936950684
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.3375,4.716794729232788
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4125,4.996432065963745
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.375,4.774821519851685
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.5075,5.296442985534668
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.45,4.704170942306519
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.52,4.899802684783936
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4025,4.352472543716431
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4825,4.69088888168335
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.39,4.411137580871582
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.5175,4.571624517440796
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4875,4.751382827758789
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.4725,4.798429489135742
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.4425,4.673127889633179
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4325,5.233693838119507
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.495,4.815239191055298
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.375,4.709919691085815
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.555,4.920079946517944
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.5,1,1,2.45,4.628366947174072
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.395,4.938645839691162
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.5025,4.889227151870728
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.36,4.835146427154541
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.5,1,1,2.41,4.89915657043457
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.38,5.100352764129639
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.41,4.985740184783936
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4125,4.58786678314209
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.5725,4.7387330532073975
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4975,4.887844085693359
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.585,4.789731979370117
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.465,4.853550434112549
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.44,4.79086971282959
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.5,4.798254013061523
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.4625,4.607733249664307
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.3825,4.79820990562439
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.435,4.726027011871338
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.405,4.905489206314087
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.495,4.466534376144409
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.5875,4.74974250793457
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.515,4.684262275695801
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.5675,4.676918029785156
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.4125,4.674067735671997
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4325,4.660931348800659
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.435,4.701269865036011
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.5,1,1,2.485,4.955507516860962
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.355,4.685837030410767
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.4175,4.516318082809448
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4025,4.6872169971466064
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4,4.554266691207886
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.485,4.592820882797241
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.4925,4.555178642272949
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.445,4.948129415512085
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.5,1,1,2.46,4.87945556640625
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4,4.627244710922241
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.5675,4.8343377113342285
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.31,4.842023849487305
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4975,5.196249723434448
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.4075,5.320227384567261
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.5025,4.774650096893311
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.44,4.800544261932373
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4075,4.56930947303772
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.475,5.00933313369751
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.46,5.687638282775879
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.5025,5.153409957885742
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.3825,4.748550415039063
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4175,4.957072496414185
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.57,4.871843576431274
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.305,4.633235454559326
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.3525,4.59865927696228
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.405,4.949576377868652
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.555,4.829384803771973
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.315,4.785153150558472
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.345,4.832309484481812
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.405,4.934347629547119
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.42,5.054717063903809
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.41,4.988655090332031
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.395,4.957913875579834
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.5,1,1,2.4775,4.851982116699219
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.3275,5.032896041870117
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.495,5.33786940574646
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4825,4.748576402664185
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.415,4.825999736785889
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.385,4.972410678863525
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.54,4.868798494338989
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4275,5.182125091552734
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.5325,4.7995078563690186
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.3425,4.999094724655151
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.465,5.133569717407227
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.5,1,1,2.485,5.217501878738403
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4925,4.724927663803101
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.485,5.418483018875122
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.27,5.075065851211548
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.345,5.068129539489746
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.34,4.962470769882202
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4875,5.171708106994629
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.3675,5.54331636428833
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.38,5.426058292388916
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.3375,4.956941604614258
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.41,4.815816879272461
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.51,6.003641128540039
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.5525,5.095165967941284
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.355,4.964768648147583
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.4275,5.077885150909424
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.5,1,1,2.305,5.3956139087677
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,2.0,2,0,2.4275,5.264023065567017
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.5,1,1,2.3675,5.459064960479736
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.3525,5.589483737945557
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.505,5.696821451187134
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4625,5.107696294784546
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.305,5.19825005531311
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.3725,5.258205413818359
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.5725,4.95388388633728
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.3275,4.961017608642578
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.49,5.31246542930603
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.435,4.724240779876709
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4875,4.71558403968811
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.48,4.875609874725342
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.385,4.633169889450073
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.5,1,1,2.5325,4.992001533508301
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.305,4.81494402885437
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.455,4.753201723098755
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.465,4.497510671615601
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.57,4.747297525405884
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.3775,4.907535552978516
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.5375,4.892165899276733
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.56,4.870349645614624
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.5,2,1,2.4725,4.79192328453064
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,2.0,2,0,2.4,4.805144309997559
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.5125,4.664160251617432
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.5625,4.866693496704102
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.4725,4.878201007843018
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.525,5.033361196517944
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.5,1,1,2.295,5.099592685699463
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.58,4.783672571182251
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.47,4.871323585510254
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.5,1,1,2.3425,5.060368537902832
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.5,1,1,2.53,4.996658563613892
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.445,5.334581613540649
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.245,5.16062068939209
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.4325,5.145641326904297
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4625,5.048618316650391
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.3875,5.001213550567627
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4625,5.006433963775635
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4775,4.991720199584961
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.48,4.807718515396118
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.485,4.777366161346436
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.3725,4.630265951156616
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.345,4.898460626602173
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.365,4.792003154754639
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.3075,4.892282485961914
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.455,4.909494400024414
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.415,4.81557559967041
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.5525,4.87252140045166
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.2825,5.010983943939209
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.545,4.760911464691162
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.3725,4.575595855712891
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.475,4.931330919265747
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4325,4.855152130126953
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.2275,4.837336778640747
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.5,1,1,2.54,5.0619940757751465
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.285,5.189566135406494
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.5325,4.706441879272461
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.42,4.939548015594482
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.335,4.975865602493286
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.5975,5.433567523956299
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.5625,5.757594585418701
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.5,1,1,2.5225,5.432241678237915
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4275,5.6733787059783936
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.46,5.487804889678955
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.23,5.249394655227661
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.3025,4.741102457046509
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.5,1,1,2.3475,5.044542551040649
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.405,4.860088586807251
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.3375,5.262238025665283
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.3675,4.696599245071411
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.5125,4.730560541152954
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.605,4.905036926269531
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.515,5.017261028289795
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.5175,4.863000154495239
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.3325,4.957741975784302
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.5225,5.684766530990601
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.5,1,1,2.37,5.136918783187866
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.5,1,1,2.4125,5.43546724319458
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,2.0,2,0,2.4325,5.056754112243652
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.455,4.612389087677002
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.5,1,1,2.3725,4.979195833206177
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.54,4.974668025970459
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4725,4.616515398025513
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.405,4.899449348449707
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.5275,5.267595052719116
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4,5.446819305419922
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,1.0,1,0,2.4775,5.051559209823608
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.3025,5.448947429656982
B,noisy,5,0.0002,0.0,1,1.0,1,5,5,5,400,0.0,0,0,2.4425,5.947142124176025
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.07,4.408493518829346
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.875,3.7159619331359854
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.5,1,1,2.895,3.7396199703216553
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.12,3.697064161300659
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.5,1,1,3.18,3.5318970680236816
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.96,3.552659749984741
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.335,3.927128791809082
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.125,3.807843446731568
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.165,3.373586654663086
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.005,2.856996774673462
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.98,3.044028759002685
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.86,3.3448269367218018
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,2.0,2,0,3.04,3.592731475830078
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.205,3.3000943660736084
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,3.105,3.491872549057007
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.96,3.8255455493927
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.01,3.1179356575012207
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,2.865,3.243807315826416
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,2.91,3.1135153770446777
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.975,3.5789060592651367
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.835,4.084338903427124
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.97,3.679324388504028
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.935,3.603491067886353
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.945,3.453529357910156
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.17,3.4565775394439697
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.165,3.462851047515869
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.485,3.849160671234131
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.145,3.669036865234375
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.055,4.345992565155029
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.805,4.639189720153809
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.96,4.045966386795044
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.94,3.847370862960816
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.1,4.1647255420684814
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.97,4.238468408584595
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.095,3.629013776779175
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,3.075,3.4922680854797363
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.93,3.6740896701812735
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.905,3.821515083312988
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.095,3.9893388748168954
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.145,4.210252046585083
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.5,1,1,3.035,3.4755985736846924
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,2.995,3.7517247200012207
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.075,3.36017107963562
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.125,4.041609764099121
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.245,4.256658315658569
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.035,4.194780349731445
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.08,4.030065298080444
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.055,4.469332695007324
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.03,4.041192054748535
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,3.085,3.698508739471436
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.115,3.613274335861206
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.265,3.6936440467834473
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,3.105,3.843440294265747
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.09,3.712947368621826
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.055,3.404212951660156
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.345,3.760066747665405
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.16,3.844067096710205
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.095,3.9504690170288086
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.03,3.609544515609741
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,3.09,4.091510772705078
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.07,3.739746809005737
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.81,4.092320919036865
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.05,3.863970279693604
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.785,4.2805609703063965
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.11,3.725801706314087
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.98,3.7696373462677
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.08,3.956195831298828
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.23,3.6319580078125
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.045,3.78694748878479
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.965,3.4297895431518555
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.88,3.710427761077881
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.995,3.3012561798095703
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.125,3.711189031600952
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.85,3.1064910888671875
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.135,4.157200336456299
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,3.04,3.5785646438598637
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.775,3.293079137802124
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.97,3.326412916183472
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.86,3.8045499324798575
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.1,3.080914258956909
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.985,3.3095808029174805
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.1,3.8326735496521
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.5,2,1,3.12,3.820526361465454
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.24,4.594762563705444
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.05,4.834911108016968
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.04,4.248250246047974
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.165,4.966266632080078
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,3.045,3.8129148483276367
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.92,3.901147603988648
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.885,4.170893669128418
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.915,4.780481815338135
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.09,4.663346529006958
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.165,3.715759754180908
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.87,4.355803489685059
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.04,3.449455976486206
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.005,3.619208335876465
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.985,4.210122346878052
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.195,3.7045371532440186
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.02,3.714496374130249
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.09,4.078496932983398
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.23,3.918622970581055
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.08,4.0385284423828125
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.035,4.211318492889404
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.5,1,1,3.185,4.100127696990967
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.035,4.127153158187866
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.135,3.714689016342163
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.865,3.478097915649414
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.03,3.558790445327759
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.99,3.516923427581787
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.125,3.409782648086548
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.155,3.4624781608581543
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.135,3.973315954208374
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.975,3.4121017456054688
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.035,3.2710325717926025
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.5,1,1,2.95,3.628474712371826
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.82,3.97862720489502
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.02,3.2895309925079346
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.24,4.267591953277588
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.93,4.303647756576538
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.055,3.691927671432495
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.0,3.4749293327331543
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.0,3.7271881103515625
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.03,3.758810520172119
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.005,3.518280029296875
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.055,3.939253091812134
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.93,3.5834333896636963
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.5,1,1,3.14,3.536141395568848
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.195,3.8052053451538086
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.98,3.413020133972168
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.09,3.579756259918213
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.01,3.836058616638184
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.075,4.204632759094238
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.935,3.913430213928223
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,2.915,3.6231415271759033
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.265,3.706652879714966
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.155,3.566602230072021
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.05,4.092190980911255
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.11,4.164095401763916
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.065,4.064646720886231
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.89,3.202953815460205
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.965,3.3772132396698
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.88,3.1951472759246826
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.945,3.5511631965637207
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.115,3.610232591629028
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.025,3.3802173137664795
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.985,3.569588422775269
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.1,3.7539467811584473
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,2.965,3.8950235843658447
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.015,3.969969272613525
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.015,3.570058822631836
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.42,3.865592002868652
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.855,4.273488998413086
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.195,3.759001731872559
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.97,3.880028009414673
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.81,3.8966526985168457
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,2.965,3.633629322052002
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.225,3.578972339630127
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.73,3.669189214706421
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.13,3.142763137817383
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.955,3.6885483264923096
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.84,3.273634195327759
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.235,4.067975044250488
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.03,3.6334190368652335
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.945,2.8693649768829346
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.01,3.466668128967285
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.195,3.8253211975097656
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.13,3.390080451965332
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.5,1,1,3.195,3.651606321334839
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.88,3.187326192855835
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.73,3.4335179328918457
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.5,1,1,3.025,3.5701770782470703
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,3.03,3.6228625774383545
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.12,3.71440863609314
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.1,3.55755615234375
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.905,3.187365770339966
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,3.115,3.482276678085327
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.94,3.295238971710205
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.03,3.506397247314453
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.05,3.540342330932617
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.1,3.381535053253174
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.01,3.4435646533966064
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.11,3.3946022987365723
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.14,3.197756052017212
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.965,3.678767681121826
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.935,3.406716823577881
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,3.055,3.202802896499634
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,3.085,3.419100284576416
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.11,3.915633201599121
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.235,3.3383007049560547
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.895,3.388115644454956
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.93,3.245143175125122
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.135,3.4992711544036865
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.145,3.191011905670166
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.095,3.2330236434936523
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.085,3.470309972763061
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.97,3.405586242675781
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.98,3.827420711517334
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.145,4.238409757614136
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.045,4.418807744979858
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.835,3.3237476348876958
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.5,2,1,2.96,3.5904252529144287
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.5,1,1,3.02,3.561969995498657
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.185,4.21812629699707
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.835,3.959005832672119
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.5,1,1,3.215,3.958808422088623
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.005,3.4700193405151367
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.99,3.668904066085816
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.12,3.912894487380981
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.045,3.379584789276123
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.96,4.614360809326172
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.91,3.467234373092652
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.16,3.9684228897094727
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,3.09,3.6696701049804688
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.935,3.519369125366211
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.025,3.2427473068237305
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.975,3.9712531566619873
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.11,3.679919481277466
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,2.955,3.857618570327759
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,3.235,3.897334337234497
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,3.025,3.547999620437622
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.06,3.750258684158325
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.285,3.996257781982422
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.01,3.7413077354431152
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.0,3.4011459350585938
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.405,4.213382005691528
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.995,3.5149779319763184
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.92,3.651439666748047
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.83,3.9686439037323
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,2.985,3.471123218536377
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.0,3.122579336166382
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,3.225,3.473040819168091
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.06,4.019816637039185
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.77,3.881234645843506
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.115,3.6162188053131104
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.145,3.728792428970337
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,3.085,3.5601954460144043
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.265,3.3276851177215576
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.19,3.2888400554656982
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,2.955,3.553513526916504
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.04,3.3639566898345947
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.99,3.535025119781494
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.285,4.062747955322266
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.925,4.378753900527954
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.17,3.941828012466431
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.175,3.96352744102478
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.975,3.654679775238037
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.075,4.106671333312988
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.765,3.4615237712860107
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.5,1,1,3.23,4.269721746444702
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.07,3.8837499618530273
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.145,4.090151786804199
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.775,3.275980234146118
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.005,4.037889242172241
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.745,3.683883666992188
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.02,3.2822265625
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.865,4.3762805461883545
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,2.0,2,0,2.835,3.748559713363648
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.345,3.81313157081604
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.11,3.944989442825317
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.245,4.137105703353882
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,2.815,3.673639059066773
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.995,3.608582019805908
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.15,4.460555553436279
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.075,3.934299945831299
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.005,3.6095898151397705
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.945,3.627450704574585
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.18,3.310314178466797
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.065,3.5952019691467285
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.995,3.473170042037964
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.08,3.592104911804199
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.055,3.5188395977020264
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.15,4.37690281867981
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.5,1,1,3.1,3.439051628112793
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.97,3.845834016799927
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.925,3.806735038757324
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.22,3.9265410900115967
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.325,3.391005039215088
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.945,3.3857460021972656
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.0,3.628732204437256
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.885,3.914416551589966
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.13,3.6650502681732178
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.94,3.3259730339050293
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.5,1,1,3.175,3.401468276977539
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.06,3.4823334217071533
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.22,3.4638946056365967
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.165,4.037064552307129
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.275,3.710968017578125
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.005,3.929043531417847
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.075,4.397084951400757
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.96,4.354971408843994
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.095,4.602219581604004
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.955,3.505485773086548
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.985,3.711190223693848
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.115,3.5969042778015137
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.04,3.509538173675537
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.005,4.444818019866943
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.955,3.5897161960601807
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.165,3.663935899734497
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.15,4.229487419128418
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.19,3.2414262294769287
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.885,3.5967352390289307
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.915,3.6251158714294434
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.94,3.1617932319641118
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.175,3.453576803207397
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.29,3.913547992706299
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.5,2,1,3.14,4.359847068786621
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.97,4.114864826202393
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.045,3.748575210571289
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.13,3.890282154083252
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.96,3.485166549682617
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.5,1,1,3.14,3.8581275939941406
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.015,3.533026695251465
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.935,3.463827610015869
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.085,4.117354393005371
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.5,2,1,3.315,3.772822856903076
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.845,4.297364950180054
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,2.0,2,0,3.165,3.757678985595703
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.125,3.527583122253418
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,1.0,1,0,3.16,4.040336608886719
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,2.945,3.7451868057250977
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.175,3.177377939224243
B,noisy,5,0.0002,0.0,1,1.5,1,5,8,5,200,0.0,0,0,3.17,3.7205631732940674
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.545,1.5714192390441897
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5475,1.7163946628570557
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.595,2.1720073223114014
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5275,1.512031078338623
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5625,1.6218640804290771
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.5525,1.4948036670684814
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.5325,1.475416660308838
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.485,1.9771151542663572
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.535,1.6133885383605957
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.555,1.460733413696289
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.585,1.614851951599121
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.57,1.3505263328552246
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.58,1.595994472503662
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.58,1.6072819232940674
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.6625,1.3974261283874512
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5325,1.5850753784179688
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.575,1.7948389053344729
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.5925,1.896101474761963
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5925,1.4297826290130615
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5925,1.3890061378479004
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.0,3,2,0.59,1.426534652709961
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5775,1.3388714790344238
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5725,1.3779642581939695
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5475,1.2488689422607422
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.495,1.4125652313232422
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5375,1.4803941249847412
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.635,1.8127129077911377
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5525,1.719987392425537
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.545,1.687239170074463
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.57,1.850613355636597
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.56,1.734394073486328
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.565,1.5853452682495115
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.6025,1.6758530139923096
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.535,1.4829518795013428
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.4875,1.5239686965942385
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.64,1.5405724048614502
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.58,1.436950922012329
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.635,1.5162601470947266
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.475,1.6423935890197754
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.5575,1.5113177299499512
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5625,1.5941510200500488
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.575,1.580848217010498
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.58,1.4513611793518066
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.53,1.52673077583313
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5575,1.8206791877746584
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.56,1.8573338985443115
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5175,1.5784320831298828
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.6025,1.3307673931121826
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.56,1.333449125289917
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5875,1.5381460189819336
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.57,1.6462290287017822
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5825,1.534440040588379
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.515,1.7172455787658691
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.53,1.4295997619628906
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.53,1.3818411827087402
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.59,1.7387323379516602
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5325,1.4546334743499756
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.54,1.5648269653320312
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.58,1.5149540901184082
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.5125,1.5676591396331787
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,3.0,3,0,0.58,1.561882734298706
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5725,1.578301191329956
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.59,1.5902631282806396
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.575,1.5362396240234375
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5325,1.574054479598999
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.57,1.7862577438354492
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.48,1.642890453338623
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.4775,1.4738662242889404
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.545,1.3624005317687988
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5575,1.661424160003662
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5775,1.6038143634796145
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.535,1.327141284942627
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.525,1.514622688293457
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.605,1.645638942718506
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,3.0,3,0,0.5325,1.3599247932434082
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5525,1.5086393356323242
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5425,1.3409254550933838
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.58,1.5001966953277588
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5575,1.2685155868530271
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.52,1.579643487930298
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.55,1.3341670036315918
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.4475,1.807292461395264
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5,1.6129281520843506
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5075,1.5718634128570557
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5,1.6459989547729492
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.515,2.293808698654175
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5925,1.9348957538604736
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5575,1.6579349040985107
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.57,1.4448492527008057
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5575,1.7220523357391355
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5975,1.6427388191223145
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.6,1.666588306427002
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5025,1.7173659801483154
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.62,1.416095495223999
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5625,1.4206113815307615
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5,1.3209004402160645
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5475,1.4973437786102295
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.565,1.8070049285888672
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.52,1.418553113937378
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5325,1.5854730606079102
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.5975,1.614302635192871
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.58,1.853442668914795
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.49,1.5747337341308594
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5425,1.5895264148712158
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5825,1.3177495002746582
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5375,1.5268309116363523
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5225,1.4107353687286377
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5225,1.3268442153930664
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5375,1.5426082611083984
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.565,1.4113647937774658
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5425,1.2895915508270264
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.56,1.4210717678070068
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5475,1.488924264907837
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5825,1.391298532485962
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.5175,1.5473427772521973
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.5725,1.5679786205291748
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.525,1.6124210357666016
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.0,3,2,0.58,1.9104692935943604
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.555,1.6780619621276855
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.535,1.7131154537200928
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.55,1.776592254638672
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.54,1.6512296199798584
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5775,1.6509652137756348
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.545,1.384326696395874
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.6,1.34716796875
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.545,1.6820926666259766
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.6225,1.5708074569702148
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.5675,1.5453195571899414
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.55,1.6904892921447754
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.555,1.4415197372436523
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.485,1.324759006500244
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.54,1.380729913711548
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.58,1.568042278289795
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.57,1.4247841835021973
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.535,1.336181402206421
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.5125,1.492412805557251
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.515,1.4500961303710938
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5525,1.3152732849121094
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.51,1.49241042137146
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5525,1.5515413284301758
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.555,1.441941738128662
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5375,1.4195160865783691
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.6,1.3976354598999023
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.6,1.5381038188934326
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5425,1.5157928466796875
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5525,1.3751611709594729
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.6075,1.3896868228912354
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.55,1.7797114849090576
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5625,1.5774257183074951
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.57,1.538651466369629
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.58,1.4471700191497805
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.6275,1.383155107498169
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.55,1.4281535148620603
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.5175,1.357142448425293
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,2,2,0.525,1.4580786228179932
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5775,1.4401586055755615
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5725,1.4400453567504885
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.64,1.707963466644287
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5325,1.374058485031128
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.565,1.523643970489502
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.54,1.7781157493591309
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5625,1.4715332984924316
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.595,1.5340027809143066
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.54,1.4484810829162598
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.545,1.4198739528656006
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5725,1.5473248958587646
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,3.5,4,1,0.5175,1.516798973083496
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5675,1.3101482391357422
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.4925,1.6392841339111328
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5275,1.6709136962890625
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.52,1.4678635597229004
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.55,1.323972463607788
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5375,1.5084078311920166
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.565,1.4681997299194336
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5375,1.6106393337249756
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5225,1.4439454078674316
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.53,1.5633814334869385
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5925,1.5053958892822266
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.595,1.4908673763275146
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.57,1.461991548538208
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.525,1.5198373794555664
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.485,1.504265069961548
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.6025,1.575333595275879
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.63,1.7374813556671145
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.53,1.6221764087677002
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5275,1.4465975761413574
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.56,1.369307279586792
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.55,1.516960859298706
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5575,1.5945963859558103
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.575,1.6404521465301514
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.58,1.555018663406372
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.55,1.5407187938690186
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.515,1.496391534805298
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.58,1.6729533672332764
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.535,1.6947736740112305
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.535,1.60443377494812
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.62,1.5088512897491455
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5325,1.49102520942688
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.56,1.147758960723877
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5575,1.167099952697754
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5025,1.260939359664917
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.555,1.1938090324401855
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5875,1.491995096206665
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.535,1.553349494934082
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5725,1.5341603755950928
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.555,1.4523961544036863
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.555,1.411172866821289
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5475,1.3110172748565674
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5125,1.482028245925903
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.4875,1.5876681804656982
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5125,1.6541085243225098
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.6025,1.2568247318267822
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,3.5,4,1,0.5575,1.2861895561218262
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.4975,1.6026561260223389
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.605,1.5613245964050293
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.6025,1.669640064239502
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.5375,1.5937440395355225
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.59,1.4430873394012451
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5425,1.418527126312256
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5175,1.6624832153320312
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.4875,1.4097754955291748
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5575,1.45341157913208
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5175,1.3948068618774414
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,3.0,3,0,0.6125,1.592643976211548
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.565,1.5913922786712646
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.485,1.612919807434082
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.5225,1.358041524887085
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.4975,1.377912998199463
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.575,1.4143528938293457
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.585,1.5294067859649658
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5275,1.4231929779052734
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5175,1.3016948699951172
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5425,1.2644267082214355
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.5325,1.5638792514801023
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5475,1.5177032947540283
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.55,1.5364301204681396
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5825,1.5730531215667725
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.6275,1.87495756149292
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.575,1.5933690071105957
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.52,1.512019395828247
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.5725,1.368854284286499
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.495,1.3453423976898191
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.47,1.5017664432525637
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.0,3,2,0.565,1.5800681114196775
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.59,1.742274284362793
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.48,1.590115308761597
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.56,1.3847641944885254
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.4875,1.420229196548462
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.535,1.5932729244232178
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5075,1.8665416240692136
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.52,1.3329124450683594
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5725,1.3737404346466064
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.555,1.4238097667694092
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.6025,1.2418420314788818
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.58,1.6161952018737793
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.6525,1.5802969932556152
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.59,1.7975780963897705
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5975,1.2750751972198486
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.6225,1.393303632736206
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.545,1.2860703468322754
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.54,1.543111801147461
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5875,1.665510654449463
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5075,1.4371891021728516
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.595,2.0226891040802
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.58,1.6263105869293213
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5675,1.4978020191192627
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5725,1.337568998336792
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.5775,1.3998444080352783
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5875,1.469475269317627
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.5,3,1,0.58,1.858385801315308
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5825,1.4298968315124512
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.565,1.3873226642608645
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.495,1.442171335220337
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.52,1.670494556427002
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5125,1.4486198425292969
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.5,2,1,0.6025,1.657888650894165
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5375,1.5832281112670898
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,3.0,3,0,0.5025,1.6502962112426758
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.5875,1.4862794876098633
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.55,1.745866060256958
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.52,1.7142791748046875
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.48,1.529078722000122
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.56,1.305321216583252
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.595,1.487943410873413
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,3.0,3,0,0.545,1.222811222076416
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.5625,1.4828107357025146
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.54,1.5062096118927002
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.555,1.3771061897277832
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5525,1.4408345222473145
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.585,1.5907886028289795
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.535,1.4849653244018557
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.545,1.5517306327819824
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5875,1.7672851085662842
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.54,1.4954869747161863
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5875,1.165390968322754
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.5575,1.4821360111236572
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.6575,1.414332389831543
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.535,1.3894567489624023
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.53,1.3403563499450684
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5275,1.3237085342407229
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.575,1.4860296249389648
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.555,1.4350802898406982
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.475,1.365659236907959
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.5,1,1,0.5575,1.4671416282653809
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5775,1.41302227973938
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.5525,1.4460184574127195
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.4425,1.320176362991333
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5125,1.4631812572479248
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.5,3,1,0.5375,1.2893004417419434
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.5325,1.4920258522033691
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.0,2,0,0.545,1.6247942447662354
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,2.5,3,1,0.515,1.5534026622772217
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.515,1.4922361373901367
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.4975,1.406346321105957
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5275,1.328998565673828
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.575,1.2180330753326416
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.555,1.320176601409912
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.545,1.4312541484832764
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.52,1.3097317218780518
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.6025,1.4057292938232422
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.53,1.3655104637145996
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.5275,1.379796028137207
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,0.0,0,0,0.5525,1.2833948135375977
B,noisy,3,0.0002,0.0,1,1.0,1,3,3,3,400,1.0,1,0,0.525,1.286978244781494
"""
EMBED_CAL = """scenario,d,p,p_idle,shots,rounds,PL_x,PL_z,pL_per_round
A,3,0.002,0.002,400000,9,0.0208925,0.0207275,0.0047122991286524
A,3,0.003,0.003,400000,9,0.0451475,0.0435125,0.0102626058879012
A,3,0.004,0.004,400000,9,0.0733575,0.07181,0.0172769226213095
A,3,0.006,0.006,400000,9,0.14178,0.137995,0.0358101837851487
A,5,0.002,0.002,400000,15,0.0080575,0.008295,0.0010985764574328
A,5,0.003,0.003,400000,15,0.02602,0.0258775,0.0035465408317352
A,5,0.004,0.004,400000,15,0.056525,0.0555075,0.0078900917560479
A,5,0.006,0.006,400000,15,0.1460725,0.14552,0.0227203208000839
A,7,0.002,0.002,200000,21,0.00276,0.002735,0.0002623538339543
A,7,0.003,0.003,200000,21,0.01282,0.012585,0.001224645060988
A,7,0.004,0.004,200000,21,0.03725,0.03636,0.0036343514239208
A,7,0.006,0.006,200000,21,0.137855,0.138235,0.0152674118755562
A,9,0.002,0.002,100000,27,0.00073,0.00086,5.8934320877557234e-05
A,9,0.003,0.003,100000,27,0.00595,0.00564,0.0004316747616033
A,9,0.004,0.004,100000,27,0.02414,0.02308,0.0017899460535197
A,9,0.006,0.006,100000,27,0.12923,0.12512,0.0108119991514165
A,11,0.002,0.002,100000,33,0.00024,0.00018,1.2729918114073868e-05
A,11,0.003,0.003,100000,33,0.00252,0.00269,0.0001582793977821
A,11,0.004,0.004,100000,33,0.01337,0.01385,0.0008359366295414
A,11,0.006,0.006,100000,33,0.11586,0.11593,0.0079588306565758
B,3,0.002,0.0,400000,9,0.00485,0.0046375,0.0010586407429796
B,3,0.003,0.0,400000,9,0.0110275,0.010415,0.002405535892685
B,3,0.004,0.0,400000,9,0.01877,0.0180525,0.0041599689205648
B,3,0.006,0.0,400000,9,0.039665,0.039115,0.009075958010966
B,5,0.002,0.0,400000,15,0.0013175,0.0013975,0.0001812299292068
B,5,0.003,0.0,400000,15,0.0050075,0.00504,0.0006729945911855
B,5,0.004,0.0,400000,15,0.0112825,0.0115225,0.0015367569392947
B,5,0.006,0.0,400000,15,0.0361125,0.0360225,0.0049788333357908
B,7,0.002,0.0,200000,21,0.00037,0.000355,3.4535739193986004e-05
B,7,0.003,0.0,200000,21,0.001965,0.001725,0.0001760250998786
B,7,0.004,0.0,200000,21,0.005645,0.005455,0.000531386492007
B,7,0.006,0.0,200000,21,0.026935,0.027045,0.0026389677360992
B,9,0.002,0.0,100000,27,8e-05,9e-05,6.296813501005527e-06
B,9,0.003,0.0,100000,27,0.00058,0.00047,3.890877912121615e-05
B,9,0.004,0.0,100000,27,0.00241,0.0021,0.0001674025578251
B,9,0.006,0.0,100000,27,0.01821,0.01869,0.001391554962761
B,11,0.002,0.0,100000,33,0.0,1e-05,3.030332415554504e-07
B,11,0.003,0.0,100000,33,0.00016,0.0002,1.0911019016013324e-05
B,11,0.004,0.0,100000,33,0.00095,0.00106,6.096870652305686e-05
B,11,0.006,0.0,100000,33,0.01258,0.01222,0.0007607034543122
"""
def _load(path, embedded):
    if os.path.exists(path):
        print('read', path); return pd.read_csv(path)
    print(path, 'not found -> embedded copy (production run)'); return pd.read_csv(io.StringIO(embedded))
mc = _load(f'{RESULTS}/d4_noisy_mc.csv', EMBED_MC)
cal = _load(f'{RESULTS}/surface_code_plog.csv', EMBED_CAL)
print(len(mc), 'Monte Carlo chunks,', len(cal), 'calibration points')

## 第I部 表面符号の論理誤り率の較正（Litinski 方式の入力）
1 ラウンドあたりの論理誤り（X と Z の和）$p_L$ を $p_L = A\,(p/p_{\rm th})^{(d+1)/2}$ に当てはめる（GoSC 式 (10) と同じ形。GoSC は $A=0.1$, $p_{\rm th}=0.01$）。閾値より十分下の $p\le0.004$ で、失敗が 20 回以上ある点を使う。

In [ ]:
PLOG = {}
rows = []
for s in SCENARIOS:
    c_ = cal[(cal.scenario == s) & (cal.pL_per_round > 0)]
    w_ok = c_[((c_.PL_x + c_.PL_z) * c_.shots >= 20) & (c_.p <= 0.004)]   # enough failures, below threshold
    A, pth = fit_plog(list(zip(w_ok.d, w_ok.p, w_ok.pL_per_round)))
    PLOG[s] = make_plog(A, pth) if PLOG_SOURCE == 'calibrated' else plog_gosc
    rows.append(dict(scenario=s, A=A, p_th=pth, points=len(w_ok),
                     ratio_to_GoSC_at_p1e3_d11=float(make_plog(A, pth)(1e-3, 11) / plog_gosc(1e-3, 11))))
pd.DataFrame(rows).round(5)

## 第II部 Kobayashi 方式
### II-1. モンテカルロの結果
ショットごとの不忠実度（0, 1/2, 1）の平均 $\hat\varepsilon_K$ と標準誤差。`window = noisy` が本来のシミュレーション（全ての位置に雑音）、`window = quiet` は**診断用**で、切り替えの時間窓（D4 相最後のラウンド、$U$ 直前の $Z$ ラウンド、$U$、switch-out、FSC に戻った最初のラウンド）だけ雑音を 0 にしたもの。本研究の復号器が 1 個の故障を訂正できない場所はこの窓に集中している（`validation.ipynb` N5）ので、quiet は「窓の誤りを理想的に扱えた場合」の楽観的な限界を与える（物理的なモデルではない）。$\kappa_{\rm in}\neq1$ の行は switch-in のラウンド数の感度。

In [ ]:
g = mc.groupby(['window', 'scenario', 'd', 'p', 'kappa_in']).agg(
    shots=('shots', 'sum'), s=('sum_infidelity', 'sum'), n_fail=('n_fail', 'sum'), n_half=('n_half', 'sum'),
    errors_per_shot=('mean_errors', 'mean')).reset_index()
g['eps'] = g.s / g.shots
n_one = g.n_fail - g.n_half
g['stderr'] = np.sqrt(np.maximum((n_one + 0.25 * g.n_half) / g.shots - g.eps ** 2, 0) / g.shots)
g['eps_over_p'] = g.eps / g.p
main = g[(g.kappa_in == 1.0) & (g.window == 'noisy')].copy()
quiet = g[(g.kappa_in == 1.0) & (g.window == 'quiet')].copy()
g.round({'eps': 6, 'stderr': 6, 'errors_per_shot': 1, 'eps_over_p': 2})

### II-2. $d$ と $p$ への依存の当てはめ（外挿の根拠）
閾値以下の標準的な形 $\varepsilon_K \approx A\,\Lambda(p)^{-d}$（$\Lambda$ は $p$ のべき）を仮定し、
$$\log\varepsilon_K = a + c\,d + (b + e\,d)\log p$$
を重み付き最小二乗で当てはめる（noisy は失敗 10 回以上、quiet は 5 回以上の点）。$d$ 方向の抑制率は $-(c+e\log p)$。**$d\le 9$ のデータからの外挿であり、確定値ではない**。
noisy では $\varepsilon_K$ が $d$ でほとんど下がらず、$\varepsilon_K\approx(3\text{–}8)\,p$ にとどまる：切り替えの窓で復号器が 1 個の故障を訂正できない場所が一定数あり、その寄与（$\propto p$）が支配する（`validation.ipynb` N5）。quiet では $d$ とともに急速に下がる。

In [ ]:
FIT = {}
rows = []
for w_, data, nmin in (('noisy', main, 10), ('quiet', quiet, 5)):
    for s in SCENARIOS:
        m_ = data[(data.scenario == s) & (data.n_fail >= nmin) & (data.eps > 0)]
        if len(m_) < 4:
            continue
        X_ = np.stack([np.ones(len(m_)), m_.d, np.log(m_.p), m_.d * np.log(m_.p)], axis=1)
        y_ = np.log(m_.eps.values)
        wt = (m_.eps / np.maximum(m_.stderr, 1e-12)).values           # 1/sigma of log(eps)
        coef, *_ = np.linalg.lstsq(X_ * wt[:, None], y_ * wt, rcond=None)
        FIT[(w_, s)] = coef
        a, c, b, e = coef
        rows.append(dict(window=w_, scenario=s, a=a, c=c, b=b, e=e, points=len(m_),
                         suppression_per_2d_at_p1e4=float(np.exp(-2 * (c + e * np.log(1e-4)))),
                         suppression_per_2d_at_p1e3=float(np.exp(-2 * (c + e * np.log(1e-3)))),
                         rms_log_residual=float(np.sqrt(np.mean((X_ @ coef - y_) ** 2)))))
def eps_K_model(s, d, p, w='noisy'):
    a, c, b, e = FIT[(w, s)]
    return float(min(np.exp(a + c * d + (b + e * d) * np.log(p)), 1.0))
pd.DataFrame(rows).round(4)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True, sharey=True)
mk = {3: 'o', 5: 's', 7: '^', 9: 'D'}
cols = {3: '#8fb8ea', 5: '#5a95dc', 7: '#2a78d6', 9: '#174a8a'}
pp = np.logspace(-4.2, -2.9, 40)
for i, (w_, data) in enumerate((('noisy', main), ('quiet', quiet))):
    for j, s in enumerate(SCENARIOS):
        ax = axes[i, j]
        for d in sorted(data.d.unique()):
            m_ = data[(data.scenario == s) & (data.d == d)].sort_values('p')
            pos = m_[m_.n_fail > 0]; zero = m_[m_.n_fail == 0]
            ax.errorbar(pos.p, pos.eps, yerr=pos.stderr, fmt=mk.get(d, 'o'), color=cols.get(d, 'k'), capsize=2,
                        label=f'd = {d}')
            if len(zero):                      # no failure: 95% upper limit 3/shots
                ax.plot(zero.p, 3 / zero.shots, 'v', color=cols.get(d, 'k'), mfc='none')
            if (w_, s) in FIT:
                ax.plot(pp, [eps_K_model(s, d, x, w_) for x in pp], '-', color=cols.get(d, 'k'), lw=1, alpha=0.7)
        ax.plot(pp, pp, ':', color='#52514e', lw=1, label='error = p (raw injected state)')
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_title(f'scenario {s}, ' + ('full simulation' if w_ == 'noisy' else 'switching window noise-free (diagnostic)'),
                     fontsize=9)
        if i == 1: ax.set_xlabel('physical error rate p')
        if j == 0: ax.set_ylabel('output infidelity  ε_K(d, p)')
axes[0, 1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{FIGURES}/d4_logical_error.png', dpi=150); plt.show()

### II-3. 資源（同じ回路から数える）
量子ビット数（ピーク）と時間ステップ数を、シミュレーションに使ったのと同じ回路（$\kappa_0=\kappa_{\rm in}=\kappa_{\rm out}=1$）から $d=3,\dots,11$ で数え、$d$ の多項式に当てはめて大きな $d$ に延長する。体積は「各ブロックで使う量子ビット数 × そのブロックの時間ステップ数」の和（雑音なしの最終ラウンドは含めない）。
**含めていない費用**：FSC から通常の回転表面符号パッチへの変換（未構成、[KZH] にも記述なし）、古典処理の待ち時間。

In [ ]:
def kobayashi_counts(d):
    circ = D4SwitchingCircuit(d, d, d, d)
    comp = CompiledProtocol(circ, NoiseModel(1.0))
    vol, steps = 0, 0
    for blk in comp.blocks:
        if blk['name'][0] == 'FIN':
            continue
        code_ = blk['code']
        active = set(circ.data_rb if (code_ == 'FSC') else circ.data_all) | (set(circ.anc[code_]) if code_ in ('FSC',) else set())
        if code_ == 'D4':
            active |= set(circ.anc['D4'])
        n_steps = len(blk['moms'])
        vol += len(active) * n_steps
        steps += n_steps
    return dict(d=d, qubits=circ.nq, time_steps=steps, volume=vol)
KC = pd.DataFrame([kobayashi_counts(d) for d in (3, 5, 7, 9, 11)])
POLY = {k: np.polyfit(KC.d, KC[k], 3 if k == 'volume' else 2) for k in ('qubits', 'time_steps', 'volume')}
def kobayashi_cost(d):
    return {k: float(np.polyval(POLY[k], d)) for k in POLY}
KC['volume_poly'] = [kobayashi_cost(d)['volume'] for d in KC.d]
print({k: np.round(v, 2) for k, v in POLY.items()})
KC

## 第III部 Litinski 方式（誤り予算モデル）
* **GoSC 15-to-1**（Fig. 17c：11 tiles × 11τ、1τ = $d$ cycles）：単一の距離 $d$。
* **GoSC 225-to-1**（Fig. 19：176 tiles × 15τ）：1 段目 $d_1$、2 段目 $d_2$。
* **Litinski 2019 15-to-1** $(d_X,d_Z,d_m)$ と **(15-to-1)×(15-to-1)**：Litinski（2019）の配置と費用式（公開コードと同じ）。
* 入力の魔法状態：確率 $p$ の Pauli 誤り（GoSC Sec. 4 の仮定）。出力誤りは事後選択後の不忠実度、時間は成功確率で割る。
* 量子ビット数はデータ＋測定量子ビット（1 tile = $2d^2$）、時間は code cycle × `STEPS_PER_CYCLE`。

In [ ]:
import itertools as _it
def litinski_candidates(p, s):
    pl = PLOG[s]
    out = []
    for d in range(3, 52, 2):
        out.append(gosc_15to1(p, d, pl))
    lvl1 = {d1: run_steps(gosc_15to1_steps(p, d1, pl), out_qubit=4) for d1 in range(3, 26, 2)}
    for d1 in range(3, 26, 2):
        for d2 in range(d1, 52, 2):
            out.append(gosc_225to1(p, d1, d2, pl, level1=lvl1[d1]))
    for dx in range(5, 40, 2):
        for dz in range(3, dx + 1, 2):
            out.append(litinski2019_one_level(p, dx, dz, dz, pl))
    for dx, dz, dx2, dz2, nl1 in _it.product((7, 9, 11, 13), (3, 5), range(13, 42, 2), (5, 7, 9, 11, 13), (4, 6)):
        if dz < dx and dz2 < dx2:
            out.append(litinski2019_two_level(p, dx, dz, dz, dx2, dz2, dz2, nl1, pl))
    df = pd.DataFrame(out)
    df['family'] = df.protocol.str.extract(r'^(GoSC 15-to-1|GoSC 225-to-1|Litinski2019 \(15-to-1\)x|Litinski2019 \(15-to-1\))')[0]
    df['time_steps'] = df.cycles * STEPS_PER_CYCLE
    df['volume_steps'] = df.qubits * df.time_steps
    return df
import time as _time
t0 = _time.time()
LIT = {(s, p): litinski_candidates(p, s) for s in SCENARIOS for p in P_EVAL}
print('evaluated', sum(len(v) for v in LIT.values()), 'factory configurations in %.0f s' % (_time.time() - t0))

## 第IV部 比較
### IV-1. 出力誤り vs 時空間体積
各方式で、ある体積で到達できる最小の出力誤り。Litinski 方式は候補の中のパレート前線。Kobayashi 方式は $d=3,\dots,9$ がシミュレーション（マーカー）、線は II-2 の当てはめ（$d>9$ は外挿）。実線＝本来のシミュレーション（本研究の復号器）、点線＝切り替えの窓を雑音なしにした診断（楽観的な限界）。実際の方式の性能は、最適な復号器を使えば両者の間にあると考えられる。

In [ ]:
def pareto(df):
    df = df.sort_values('volume_steps')
    best, rows = np.inf, []
    for _, r in df.iterrows():
        if r.eps < best:
            best = r.eps; rows.append(r)
    return pd.DataFrame(rows)
show_p = [1e-4, 3e-4, 1e-3]
fig, axes = plt.subplots(2, len(show_p), figsize=(4.2 * len(show_p), 7.6), sharex=True, sharey=True)
ds = np.arange(3, 61, 2)
vk = np.array([kobayashi_cost(d)['volume'] for d in ds])
for i, s in enumerate(SCENARIOS):
    for j, p in enumerate(show_p):
        ax = axes[i, j]
        fr = pareto(LIT[(s, p)])
        ax.step(fr.volume_steps, fr.eps, where='post', color=C_LIT, label='Litinski (best of GoSC / 2019 factories)')
        ax.plot(vk, [eps_K_model(s, d, p) for d in ds], '-', color=C_KOB, lw=1.2, label='Kobayashi (fit; d > 9 extrapolated)')
        if ('quiet', s) in FIT:
            ax.plot(vk, [eps_K_model(s, d, p, 'quiet') for d in ds], ':', color=C_KOB, lw=1.5,
                    label='Kobayashi, switching window noise-free (optimistic)')
        sim = main[(main.scenario == s) & (main.p == p)]
        ax.errorbar([kobayashi_cost(d)['volume'] for d in sim.d], sim.eps, yerr=sim.stderr, fmt='o', color=C_KOB,
                    label='Kobayashi (simulated)')
        ax.axhline(p, color='#52514e', ls=':', lw=1)
        ax.set_xscale('log'); ax.set_yscale('log'); ax.set_ylim(1e-14, 1)
        ax.set_title(f'scenario {s},  p = {p:g}')
        if j == 0: ax.set_ylabel('output error of one T state')
        if i == 1: ax.set_xlabel('space-time volume [physical qubits x time steps]')
axes[0, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{FIGURES}/cost_comparison.png', dpi=150); plt.show()

### IV-2. 要求誤りごとの最小費用
Litinski 方式は要求を満たす候補のうち体積最小のもの。Kobayashi 方式は要求を満たす最小の $d$（$d\le61$）とその体積：`kobayashi_*` は本来のシミュレーションの当てはめ、`kobayashi_*_opt` は切り替えの窓を雑音なしにした診断（楽観的）。`—` は $d\le61$ で届かないこと、`extrapolated` は $d>9$（外挿）を示す。

In [ ]:
def kob_min(s, p, t, w_):
    if (w_, s) not in FIT:
        return None
    return next((d for d in range(3, 62, 2) if eps_K_model(s, d, p, w_) <= t), None)
rows = []
for s in SCENARIOS:
    for p in P_EVAL:
        df = LIT[(s, p)]
        for t in TARGETS:
            ok = df[df.eps <= t]
            lit = ok.loc[ok.volume_steps.idxmin()] if len(ok) else None
            r = dict(scenario=s, p=p, target=t,
                     litinski_protocol=lit.protocol if lit is not None else '—',
                     litinski_eps=lit.eps if lit is not None else np.nan,
                     litinski_qubits=lit.qubits if lit is not None else np.nan,
                     litinski_time_steps=lit.time_steps if lit is not None else np.nan,
                     litinski_volume=lit.volume_steps if lit is not None else np.nan)
            for w_, suf in (('noisy', ''), ('quiet', '_opt')):
                dk = kob_min(s, p, t, w_)
                kc = kobayashi_cost(dk) if dk else None
                r.update({f'kobayashi_d{suf}': dk if dk else '—',
                          f'kobayashi_extrapolated{suf}': bool(dk is not None and dk > 9),
                          f'kobayashi_qubits{suf}': kc['qubits'] if kc else np.nan,
                          f'kobayashi_time_steps{suf}': kc['time_steps'] if kc else np.nan,
                          f'kobayashi_volume{suf}': kc['volume'] if kc else np.nan})
            rows.append(r)
CMP = pd.DataFrame(rows)
CMP['volume_ratio_K_over_L'] = CMP.kobayashi_volume / CMP.litinski_volume
CMP['volume_ratio_K_opt_over_L'] = CMP.kobayashi_volume_opt / CMP.litinski_volume
pd.set_option('display.max_colwidth', 50)
CMP[['scenario', 'p', 'target', 'litinski_protocol', 'litinski_volume', 'kobayashi_d', 'kobayashi_volume',
     'volume_ratio_K_over_L', 'kobayashi_d_opt', 'kobayashi_extrapolated_opt', 'kobayashi_volume_opt',
     'volume_ratio_K_opt_over_L']]

### IV-3. Kobayashi 方式が同じ体積に収まるために必要な改善
要求誤りごとに、Litinski 方式の最小体積以下に収まる Kobayashi 方式の最大の $d$（$d_{\max}$）を求め、そこでの $\varepsilon_K(d_{\max},p)$（本来のシミュレーションの当てはめ）が要求誤りの何倍か（`required_improvement`）を示す。復号器・スケジュールの改良でこの倍率だけ $\varepsilon_K$ が下がれば、体積で並ぶ（FSC → パッチ変換の費用は含めていない）。

In [ ]:
rows = []
for _, r in CMP.iterrows():
    if not np.isfinite(r.litinski_volume):
        continue
    ds_ = [d for d in range(3, 62, 2) if kobayashi_cost(d)['volume'] <= r.litinski_volume]
    dmax = max(ds_) if ds_ else None
    ek = eps_K_model(r.scenario, dmax, r.p) if dmax else np.nan
    rows.append(dict(scenario=r.scenario, p=r.p, target=r.target, litinski_volume=r.litinski_volume,
                     kobayashi_d_max=dmax, eps_K_at_d_max=ek, required_improvement=ek / r.target if dmax else np.nan))
IMP = pd.DataFrame(rows)
CMP = CMP.merge(IMP[['scenario', 'p', 'target', 'kobayashi_d_max', 'required_improvement']], how='left',
                on=['scenario', 'p', 'target'])
CMP.to_csv(f'{RESULTS}/cost_comparison.csv', index=False)
IMP.round({'eps_K_at_d_max': 8, 'required_improvement': 1})

## 結論
@@CONCLUSION@@